# 산업전력 × 열섬 × AIDC — 단계별 검증 노트북 (2026-07-18)

**위에서부터 순서대로 실행.** 산업전력 초점 전환 이후 수행한 분석을 원자료부터 재현한다. 각 셀 끝 `# 기대:` 주석이 예상 출력.

- 패키지: `pip install pandas numpy scipy statsmodels python-calamine`
- **표준 파일명**(2026-07-18): `OPEN_0718_한전_시군구전력사용량_YYYY.xlsx`(2020~2025), `OPEN_0718_산단공_*`. 이전 `0718_시군구별 전력사용량(홈페이지 게시용)_*`에서 개명(`_파일명표준화_2026-07-18.md`).
- ⚠ **온열질환 = 발생지 기준**, 인구 분모 = 거주지 기준(불일치, 통합보고서 §1).
- ⚠ **전력 계약종별**: DC 전기는 '산업용'이 아니라 '일반용(상업·공공)'으로 분류됨. 산단·열섬 신호는 `산업용`/`제조업`만.
- ⚠ **여름(6-8월, JJA) 기준** (2026-07-18 수정): 온열질환의 **93%가 6-8월**(7·8월만 83%)에 발생하므로, 산업전력↔온열 관계는 **여름 전력 vs 여름 온열**로 맞춘다(연간 전력엔 겨울·봄 산업활동이 섞여 부적절). 단 **§5 폐산단 스캔은 연간 전력** 사용 — 공장 폐쇄=연중 감소라는 *구조적* 변화를 보는 것이라 연간이 적절. (DC 24시간 상시가동은 **AIDC에 한정**된 특성이고, 기존 산업의 온열 기여는 여름 열로 본다.)


## 목차 — 각 절이 하는 일 `[🔖 2026-07-28 신설]`

**이 노트북은 하나의 질문을 좇는다:** *AIDC가 산단에 들어오면 몇 명이 더 아픈가.*
그 답은 세 인자의 곱이고, 절들은 **각 인자를 하나씩 세우는 순서**로 배열돼 있다.

```
추가환자 = Σ(링별 인구) × (여름 온열질환률) × (m^ΔT − 1)
            §10~§12       §9-b~§9-e         §9-e·§14·§15
```

### 🚩 결과만 보려면
`요약_결과노트북_2026-07-28.ipynb` (집계본만으로 몇 초). 이 노트북은 **어떻게 거기 도달했고 어디서 틀렸는지**를 담는다.

### 🔑 핵심 5절 — 시간이 없으면 여기만
| 절 | 무엇 |
|---|---|
| **§9-e** | dose 엔진 정의 — `dT_at()` · `excess_ring()` · `f_site()`. **모든 추정이 여기를 부른다** |
| **§9-e2~e4** | 그 엔진의 근거를 우리 손으로 검증 — 시간/공간 구성개념 · 곡선 이식 가능성 · 각 세종 반증 |
| **§12-b** | 노출 인구 — 집계구 실측. **가정이 없는 유일한 인자** |
| **§14-d** | 부지가 지금 얼마나 뜨거운가 — 주거지 기준 ΔLST |
| **§15-d·g** | 지표온도 → 기온 환산계수 β. **불투수 관측소 n=0** 이 여기서 드러난다 |

---

### A. 자료 만들기 — §0 ~ §2
| 절 | 역할 |
|---|---|
| §0 | 로드·정제 함수 + 유의성 헬퍼. **먼저 실행해야 나머지가 돈다** |
| §1 | 신규 자료가 기존 자료와 일치하는지 정합성 확인 |
| §2 | 계약종별(산업용·제조업·주택용) 다년 패널 생성 |

### B. 산업전력과 온열질환은 정말 관계가 있나 — §3 ~ §6
| 절 | 역할 |
|---|---|
| §3 | 횡단면 편상관 — 인구를 통제해도 산업용 전력과 온열이 붙나 (r≈0.5) |
| §4 | 다년 패널 Poisson — 시간 변동으로도 같은 방향인가 |
| §5 | **폐산단 전환 후보 스캔** — 산업용 전력이 구조적으로 꺾인 시군구. 임계값 민감도 포함 |
| §6 | ⚠ **가장 중요한 판별** — 그 상관이 (a)작업장 노동인가 (b)산업지대 열섬인가. **발생장소**로 가른다 |
| §6-c~c5 | 반론 검증 — "집·길가가 무관한 건 산단에서 멀어서 아닌가", 경지면적 구성효과, 강건성 |

> **§6의 결론이 이 노트북의 규율을 정한다** — 작업장·논밭 상관은 **AIDC 논증에 쓰지 않는다.**
> 공장 안 노동이지 ambient 열섬이 아니기 때문이다. 이후 모든 절이 이 선을 지킨다.

### C. 어느 부지인가 — §7 ~ §8
| 절 | 역할 |
|---|---|
| §7 | ILIS shapefile에서 산단 **실좌표·실형상** 유도 (손입력 좌표 금지) |
| §8 | dose-response 입력표 — 좌표 + 면적 + 온열률 결합 |
| §8-b | 형상 검증 — 대표점이 산단 중심인가, 산단이 원형인가(광양은 점이 바다였다) |

### D. 열이 얼마나 더해지고 그게 몇 배의 위험인가 — §9 ★
| 절 | 역할 |
|---|---|
| §9 | 열부하 밀도 — 같은 용량이라도 부지가 작으면 승온이 크다 |
| §9-b~d | **기온 1℃당 온열질환 배수**를 우리 자료로 유도 (전국 → 시도 → 시군구) |
| **§9-e** | 부지별 배수 채택 규칙 + **dose 엔진 정의**(`dT_at`·`excess_ring`·`f_site`) |
| **§9-e2** | ⚠ 시간 대비 vs 공간 대비 — 케임브리지 Δ는 *같은 자리의 전후*, 우리 ΔLST는 *같은 시각의 공간 대비* |
| **§9-e3** | 그 곡선을 기존 산단에 얹어도 되나 — 진폭·모양·표본 틀·열원 교체 네 갈래 |
| **§9-e4** | **각 세종 링별 실측** — 곡선의 '모양'을 국내에서 처음 잼. 4/5 링에서 예측 기각 |
| §9-f~i | 습도가 추가 설명을 하나 (습구온도·시간자료·시군구×일 패널) |
| §9-j | **플룸 모형 주입구** — 바람 방향 모형이 오면 꽂을 자리 |
| §9-h | 모형 선택 결정 — AIC가 최소여도 통제변수를 함부로 넣지 않는 이유 |

### E. 몇 명이 노출되나 — §10 ~ §12
| 절 | 역할 |
|---|---|
| §10 | 버퍼 인구 — SGIS 집계구로 절대 headcount. §10-b는 점 vs 폴리곤 민감도 |
| §11 | 연령가중 — 65세 이상 노출. §11-b는 케임브리지 footprint(4.5·10km) 적용 |
| §12-b | **형상 기준 고령 노출** — 점 3km 대신 폴리곤 경계 버퍼 |
| §12-c | 면적 대조 — 발표 용량이 그 부지에 들어가나 (§12-c3 열 수지 · §12-c4 강릉 옥계) |
| §12-d | 형상 함의 정량화 — 산단 전체를 열원으로 두면 얼마나 과대인가 |
| §12-e | 플룸 경로 링 집계 + 등방 대비 회귀 게이트 |

### F. 산단이 아닌 개별 공장 터 — §13
도심 한복판 폐공장(현대제철·동국제강 인천, 심팩 포항)은 산단과 노출 구조가 다르다.

### G. 부지가 지금 얼마나 뜨거운가 — §14 ★
| 절 | 역할 |
|---|---|
| §14-a~c | Landsat 장면 인벤토리 → LST 변환·마스킹 → 부지별 ΔLST |
| §14-d | **주거지 기준선 재계산** — 기준을 숲 포함 전체에서 주거지 화소로 좁혀도 여전히 뜨거운가 |

### H. 지표온도를 기온으로 — §15 ★
| 절 | 역할 |
|---|---|
| §15-a~d | 관측소 × 위성장면 패널 → 장면 고정효과 회귀로 **β 추정** |
| §15-e | 관측망 대조 + 회귀희석 진단 → **β 상한 후보 0.459**의 근거 |
| §15-f | 시각별 β — 위성 통과 시각과 일최고기온 시각의 불일치 해소 |
| **§15-g** | 피복 층화 β — ⚠ **불투수면 관측소가 0개**임이 여기서 드러난다(부지 환산 = 외삽) |

### I. 왜 뜨거운가, 그리고 무엇을 심어야 하나 — §17
녹지·불투수 분해(NDVI) → 용량반응(NDVI +0.1당 −1.16℃) → 토지피복 실측 검증 → 피복별 냉각 효율.

### J. 기사로 확인된 신규 후보 부지 — §18
가설이 현실이 됐는지 대조하고, §10·§11 공식을 신규 부지에 소급 적용.

---

### 읽는 법 세 가지

1. **셀 1의 자기수정 이력을 먼저 본다.** 우리가 틀렸던 자리와 그때 세운 규칙이 있다.
2. **`⚠` 는 한계, `★` 는 핵심, `[🔖 날짜]` 는 그날 바뀐 것**이다.
3. **숫자보다 "그 숫자가 무엇의 비교인가"를 먼저 확인한다** — §9-e2의 표가 그 지도다.


>
> 규칙 4 (07-27 추가): **모수를 풀어서 맞춘 값은 증거가 아니다.** 자유 모수 하나를 관측에 맞도록 풀면 항상 해가 나온다 — 실패할 수 없는 계산은 아무것도 확인해주지 않는다. 독립 경로로 그 모수를 확인할 수 있을 때만 근거로 쓴다. 이 세션에서 세 번 걸렸다: §15-e σ_e 역산 · Ⓔ의 AIC 최소 · §11-b(2) 부채꼴 80°.
> ### 🔖 이번 세션 변경 이력 (2026-07-21 · 07-22) — 수정 셀 표시
>
> | 섹션 | 변경 | 검증 |
> |---|---|---|
> | **§15-a** 로더 | AWS 261→**459지점**(`ASOS+AWS/OBS_AWS_DD_*.csv`) | node/ast OK |
> | **§15-d** 공장부지 | `fillna` 오염 → **R0-only**(폴리곤)·0-1km(점). 동해·구미 과소 정정 | ast OK |
> | **§15-d 뒤** ⚠ md | 트레일링 산문 print → markdown 이동 | — |
> | **§15-e·§15-f** | 산문 → 「§15 규약」 pointer | ast OK |
> | **§15 규약** | 459지점 재작성(β **0.258**·희석 0.459·8짝 −0.04) | 독립노트북 실행 확인 |
> | **§11-b** (신규) | Cambridge hyperscaler footprint 4.5·10km 인구 | 실행·doc 대조 |
> | **§17** (신규) | 녹지·불투수 NDVI 분해 — **90장면 완전판** | 실행 OK |
>
> 각 수정 셀 첫 줄에 `[🔖 2026-07-21]` 주석. 전체 코드셀 `ast.parse` 통과. §15 β=0.258은 동일 로직·데이터의 독립노트북(Landsat_LST_파이프라인) 실행으로 검증됨 — **본 노트북 위→아래 재실행 시 출력 재생성**.

> **07-22 추가 배치** (둘째 질문 배치 복구 + 신규): 세종 이중등재 dedup(§0) · §1 행별 판정 · §9-c 지역별 Poisson(시도 FE) · §10-b 광양 정정 · §11-b(2) 외곽링 정정(§14 데이터가 반박) · §11-b(3) 인천 환자 시나리오(지수감쇠×dose 범위) · §12 동해북평 역전 note · §12-b 형상 고령(점3km vs 폴리곤+3km) · §13-b 후보 역산 · §14 표 거리명시+상한/오전 모순 정정 · **§18 기사 기반 신규 부지 5곳 실측**(포항 광명·북평2·하이테크밸리·강양우봉·KG스틸). **§14-d 주거지 기준선 재계산**(NDVI 층화 — ΔT 평균 −6.1℃ 이동·순위 보존 ρ=0.83·인천 −2.2℃로 최견고, [확인필요]#2 해소). 모두 `[🔖 2026-07-22]` 태그.

> **07-22 밤 배치**: §9-c 수정(광주 297건 복구 — 통합명 버그 + silent 조건 제거) · **§9-d 시군구 dose-response 신설**(최근접 ASOS·FE ×1.51·AIDC 시군구별 CI) · §12 note 케임브리지 "규모 포화" 정정(원문 MW 층화 없음 — 방법론 확인 노트) · **§12-c 면적 대조 신설**(북평2 = 표준의 1/15~1/24 고밀) · §11-b(3) 주석 정량 근거화(§14-d 주거화소비 0.42). 별도 산출: 벨트 지도 html · 논문방법론 노트.

> **07-23 배치 3a** (사용자 검증 피드백): §9-c 사용자 주석 이식+한줄if 해체(V9.4) · §9-d 전국 분포 robust · §12-c 포맷+형상 함의 2건 · **§18-b2 온열질환 추정 소급**(비가중+연령가중 — 북평2 파이프라인 편입) · §14 결과 인천 2행 정정(표 포맷 버그 — 장면 10개 존재) · **§14-a2 다운로드 스펙**(path/row 실측 매핑) · §15 md 정합 3건(수식 첨자·대조 기준·R0 명시) · §18 재사용 명시.

> **07-23 배치 3b**: §14-c → **c1(형상)/c2(루프)** 분할 + 부지별 진행 print + CSV 재저장 주석화 + 중복 해석 print 제거 · §14-d → 쉬운 설명 md + **d1(함수)/d2(루프)/d3(집계)** 분할 + NDVI 물리 주석 · §17↔§14-d1 상호참조 주석. 전부 재실행 출력 포함.

> **07-23 배치 4**: **§12-d 형상 시나리오 정량화** 신설(온산 전체+3km는 강양·우봉 대비 8.1배 과대 / 북평2 표준면적 확장 가정 시 3km 노출 2.6만→5.7~6.4만) · §14-c2 결과 블록 전체 주석화 → `실행결과_14c_산단LST_2026-07-23.md` 이관.

> **07-23 배치 5**: **§15-g NDVI 층화 β**(불투수 관측공백 확증 · 녹지 0.287>주거 0.222 · 희석 진단 · 환산 대응표) · **§17-b 녹지 용량반응**(NDVI +0.1당 −1.16℃ · 회색지대 0.15~0.25 최고온 · 정책 번역) · **§17-c 토지피복 세분류 검증**(북평2 나지 89% · NDVI 대리 과소=보수 확인 · 관측소 실피복 교차) · §12-d 나지 보강. 입력 CSV 3종(DERIVED_0723_*) 고정으로 재현 경량화.

> **07-24 배치 6**: **§17-d 피복별 냉각 효율**(활엽수림 −2.5 ~ 공업 +9.7 · 숲>초지·밭 4℃ · 논>밭 물효과 · 불투수 대비 격차 7.5℃) — 완충 처방을 "관개된 활엽·혼효 수관림"으로 구체화. 입력 CSV DERIVED_0724_피복별_LST표본.

> **07-24 배치 8**: ★ **북평2 dose-response 정식 편입**(§7 T·전 체인 재계산) · **§11-b 점→폴리곤 전환**(미포 9.1배·온산 25배) · §9 분모 YUCH · §12-b 전버퍼 연령가중 · **§9-e 부지별 배수**(n≥100 & 관측소≤10km 규칙) · §8-b convex_hull 원형도 · §11-b(3) LST 단일(13건/년) · §5 중공업 절대량 · §4 md 숫자정정 · §6-c 근접열섬 여지 · §7 근거 갱신 · §12-c 수랭답 · §12-d 강양우봉/나지 정밀화.

> **07-24 배치 9·9b**: **§6-c2 신설**(near_share를 YUCH 폴리곤 경계 기준으로 재검정 — 근접도 중위 5.5→10.2%·상위⅓ 32.7→45.2%인데도 집 −0.137·길가 −0.125 n.s. 유지 → "점이라 근접이 덜 잡혀 null"이라는 반론 기각) · §6 결론 md 보강 · **§14 결과표 16부지 통합**(§18의 5부지 병합, 북평2 +11.1℃ = GS 2.4GW 실부지) · §18 재정리(북평2는 §10~§14 본 파이프라인으로 이관).

> **07-24 배치 10a** (표류 정정 + 부지별 계수화): §7 좌표확인 `TG` 라벨 정정(`동해북평·메가2.4GW` → `국가(대조)`) + 북평2 행 추가 — T만 고치고 이 확인용 리스트를 안 고쳐 출력이 어긋나 보였다(사용자 지적) · **§9-② 표에 실제 분모(`열원면적km2`·`면적출처`) 노출 + 해석 문장 하드코딩 제거** — 07-24 분모 전환(대장→YUCH)으로 북평2와 무관하게 전 산단 열부하가 바뀐 사실이 표에서 안 보였다 · **§9-e에 `f_site()` 신설** · **§10-③·§11-④의 전국 상수 f=1.6²−1 → 부지별 f_s=m_s²−1 교체**(구 방식은 대조 컬럼으로 병기) · §14-c1 북평2 경로 문서화 · §4-② 태그 보강.

> **07-25 배치 12~18** (거리 감쇠 통일): ★★★ **"3km까지 기온 +2℃" 평평 가정 폐기.** §11-b(3)만 LST→β→dose 사슬을 제대로 쓰고 §10·§11·§12-b·§13·§18-b2는 평평한 +2℃를 쓰고 있었다 — 같은 목적 계산이 두 규약으로 갈라져 있었다. **엔진 하나(`dT_at`·`excess_ring`, §9-e)로 단일화**. 3km 추가온열 **6~7배 하락**(미포 31.6→4.8~9.2). **케임브리지 Fig.3 본문 수치로 파라미터 적합**(부지 2.07℃·4.5km 1.0℃·7km 30%·10km 도달 → 반감기 4.2km). 형태 검정: **지수만 세 점 통과**(`DECAY_SHAPE`로 교체 가능). **β 범위화**(0.258~0.459). Sailor는 곡선에 얹지 않고 **≤0.5km 근접 상한**으로 분리. `_sites10` 미정의 버그 수정 + 가동/폐쇄 순변화 분리. §12-d **열원 면적 하한**(미포 15.6배 과대). §12-c 냉각방식. §13-b "원격형" 판정 철회. **§14-d 실측 전환**(3차 시도 성공). **§15-h 이동**(정정 뒤 범위로). **§16 분리 · §17-b0 생성셀 · export 4종**(FOIA 없이 재현). 지도 v7·문서 2종 동기화.

> **07-26~27 배치 19~39** (신규 분석 + 감사): **§9-f 습구온도 dose-response**(습도 10%p당 ×1.15 → 강수·일사 통제 시 **×1.45**, AIC 최소). **§6-c3 경지면적 통제**(논밭 상관 +0.531→+0.332 생존 = 구성효과 기각). **§6-c4 YUCH 감사**.

> **07-27 배치 40~42** (시간자료 + 모형 선택 결정): **§9-g 신설** — ASOS **시간자료 127만행·97지점** 수집(`OPEN_0726_*`)해 §9-f 한계 (a) '시각 불일치' 해소. 일최고기온 시각 습도는 **63.3%**로 일평균 79.9%보다 16.6%p 낮다 → 습도 계수 1.152 → **1.124**(구 값이 소폭 과대였다). **습구온도 단일지표(Ⓑ)는 오히려 열등**(AIC 9400 vs 기온만 5112) → 기온·습도는 **따로** 둔다. **§9-h 신설 — 모형 선택 결정(사용자)**: Ⓔ(강수·일사·풍속 통제)가 AIC 최소지만 **채택하지 않는다.** 근거는 결과가 아니라 반사실이다 — *산단이 AIDC로 바뀔 때 강수·일사·풍속이 어떻게 변할지 우리는 정할 수 없고, 값을 못 정하는 변수를 통제로 넣는 것은 "그대로 있을 것"이라는 또 하나의 미검증 가정*이다. `HEAT_MULT_PER_C=1.575`(Ⓐ) 유지. ⚠ 유지 쪽이 곧 **추정치가 10~13% 큰** 쪽이라는 점을 §9-h에 명시했다.

> **07-27 배치 41 파생 확인** (사용자 질문 2건이 만든 측정): ① *"AIC 최소여도 예측이 더 높을 수 있지 않나"* → **맞다.** 552일 중 54%에서 Ⓔ가 더 크고, **덥고 맑은 날**(n=88, 폭염의 전형)엔 Ⓔ가 하루 **+3.1건** 더 예측한다. AIC는 전체 적합도일 뿐이다. 단 우리 공식 `f_s=m^ΔT−1`은 **배수**만 쓰므로(관측 환자 수에 곱함) 예측 수준은 무관 — 배수로는 Ⓔ가 언제나 작다. ② *"기온 높고 일사 많은 날에야 오른다"* → **절반만 맞다.** 환자 **수**는 일사 상위⅓이 하위⅓의 **6.9배**지만, 기온 1℃의 **배수**는 맑은 날에 오히려 **작다**(1.596→1.485, 상호작용 −0.0087 p=1e-11). 기전은 우리 자료로 못 가린다 — 포화인지 행동변화인지 모름으로 남김.

> **07-27 배치 43~45** (§12-c에 습도 근거 대입 — 결과는 **우리 주장 약화**): **§12-c2 신설.** §12-c ②의 *"온열질환은 습구온도에 민감하다 → 기온이 덜 오른다가 곧 덜 위험하다가 아니다"*는 07-25에 **재지 않고 쓴 문장**이었다. §9-g Ⓒ 계수로 대입하니 **습도 40.6%p = 기온 1℃**(습도가 41배 둔한 지렛대)이고, 습구온도 물리(Stull)가 예측하는 5.8%p보다 **7배 둔하다**. 습구 단일지표(Ⓑ)가 기온만(Ⓐ)보다 적합이 나쁜 것도 같은 방향. **→ 평균 조건에서는 증발식이 ambient 기준 오히려 덜 위험할 가능성이 높다. 경고를 약화한다.** 냉각탑 ΔRH를 증발수량(2.4GW=9.1만톤/일)으로 **상자모형** 계산: 3km·혼합고 500m에선 **+0.5~0.8%p**(무시 가능)지만 **1km·50m·1m/s 정체조건에선 포화(안개)** — 즉 조건에 달렸다. **살아남는 논증 셋으로 이동**: (a) 정체·야간 안정층 (b) **물 소비** 연 3,329만톤 (c) **근접장 미측정 → §17 플룸 모형 제출 의무화**. 부수 정정: 연 물소비 '33.3억톤' 단위 오기(1e9을 억으로 읽은 **300배** 오차) → 3,329만톤, §12-c 원자료 1,380만톤/GW와 교차검증 ✓ · 상자모형 포화행 ★표시 + 인용 금지 명시.
>
> ⚠ **자기수정 11건** — 이 세션에서 내가 확인 없이 단정했다가 데이터에 기각당한 것들. 같은 실수를 반복하지 않으려고 남긴다:
> 1. §14-d 마스크 결손 원인을 `simplify(15)`로 지목 → **진범은 900m² 면적 필터**(세분류 주거 폴리곤 중위 40m²)
> 2. "동해 두 곳은 화소가 얇아 대리를 써야" → **화소 수와 불안정이 무관**(48화소 동해북평 SEM배수 1.30 < 729화소 온산 1.99)
> 3. "§6-c3 하위⅓에서 집이 유의" → **여수 1개가 끄는 허상**(순위·log·율·이상치제거 전부 탈락)
> 4. "§6-c2도 YUCH 때문에 신뢰 불가" → **§6-c2는 무사**(여수·포항 모두 중위, 의심 1개 빼도 동일)
> 5. "대리가 쓸 만한 건 삼킨 화소가 도시라서/상쇄돼서" → **설명 2개 다 기각**(도로 33% 지배, 편의~구성비 r=−0.01). 왜 비슷한지는 **모른다**로 남김
> 6. "북평2 = 2022 조성완료 후 미입주" → 대장상 **조성중·분양률 28.4%·입주 0**(입주 0은 맞고 조성완료는 틀림)
> 7. **"Ⓔ의 1.511이 AIDC 시나리오에 더 맞다 — 폐열은 일사를 늘리지 않으니까"** → **내 §12-c가 이걸 반박한다.** §12-c는 *증발식 냉각탑이 습도를 올린다*고 이미 적어놨다. 즉 AIDC는 기온만 바꾸는 개입이 아니다. "다른 기상은 그대로"를 전제하는 계수를, 바로 그 기상이 바뀌는 시나리오에 권한 것이다. 더 근본적으로 **AIC 최소를 곧 채택 근거로 읽었다** — AIC는 적합도이지 반사실을 정해주지 않는다.
> 8. **"온열질환은 습구온도에 민감하다 → 냉각탑이 기온을 덜 올려도 덜 위험한 건 아니다"**(§12-c ②, 07-25) → **재보니 약해진다.** 우리 자료에서 습도는 기온보다 **41배 둔한** 지렛대이고(40.6%p = 1℃), 습구 단일지표는 기온만 쓴 것보다 **적합이 나쁘다**. 문헌의 습구 민감성을 우리 자료로 검증하지 않고 논증에 얹었다. 살아남는 건 *정체조건·물 소비·근접장 미측정*이지 습도 자체가 아니다. **이 정정은 우리 주장을 약화하는 쪽이고, 그래서 더 빨리 적었다** — 상대가 먼저 재서 들이대면 §12-c 전체가 같이 무너진다.
>
> 규칙: **"A라서 B"라고 쓰기 전에 B를 잰 줄이 이 노트북에 있는지 확인한다.** 없으면 "추정"이라고 쓴다.
>
> 11. **"Sailor 2.2℃는 부채꼴 80°에 해당한다 → 두 논문이 부합한다"**(§11-b(2), 07-27) → **θ를 Sailor에 맞도록 푼 값을 증거로 읽었다.** 어떤 Sailor 값을 넣어도 θ는 나온다. Briggs(1973)로 독립 확인하니 80°는 ±3σy·도시A-B 안에 들긴 하지만, 그 판정도 **내가 k를 고른 결과**였다. 더 큰 오류는 **top-hat(내 부채꼴)과 가우시안 중심축(Sailor 최대값)을 섞어 비교**한 것 — 바로잡으면 우리 쪽이 1.3~5배 크다. 남는 결론: *"화해 시도는 아직 성공하지 못했고 우리 쪽이 큰 방향으로 어긋난다"* 까지.
>
> 규칙 2 (07-27 추가): **통제변수는 그 반사실 값을 우리가 정할 수 있을 때만 넣는다.** 적합도 지표(AIC·R²)는 모형 비교용이지 인과 설계의 근거가 아니다.
>
> 규칙 3 (07-27 추가): **문헌에서 가져온 민감도·계수는 우리 자료에 대입해 보고 나서 논증에 얹는다.** 우리 표본에서 크기가 다르게 나올 수 있다. 대입 결과가 우리 주장을 **약화**하면 더 빨리 적는다 — 상대가 먼저 재는 것이 최악이다.

## §0. 로드·정제 함수 + 유의성 헬퍼 — **먼저 실행**

In [1]:
import pandas as pd, numpy as np, glob, os
from scipy import stats
from python_calamine import CalamineWorkbook
BASE = r"C:/cross_the_street/docs/research/_data-center/기획서_꾸러미/정보공개청구"
os.chdir(BASE); print("작업 폴더:", os.getcwd())

YEARS=[2020,2021,2022,2023,2024,2025]
# 용도업종별 39개 중 '제조업'에 해당하는 26개 (농림·수도·공공·서비스·합계 제외)
MFG={'금속비금속','식료품제조','섬유','의복.모피','가죽.신발','목재.나무','펄프.종이','출판.인쇄',
     '화학제품','고무.플라','유리','시멘트','1차금속','조립금속','기타기계','전기기기','영상.음향',
     '의료.광학','자동차','기타운송','가구및기타','음료품제조','석유정제','사무기기','재생재료','담배제조업'}

# ── 시군구 키 정규화 (2026-07-18 수정) ──
# 문제1: NEDIS는 '고양시 덕양구'(일반구)로, 전력·인구는 '고양시'로 기록 → 매칭 실패(포항·청주·창원 등 산업도시 누락)
# 문제2: 시도 개명 '강원도'↔'강원특별자치도', '전라북도'↔'전북특별자치도' 불일치
GWANGYEOK={'서울특별시','부산광역시','대구광역시','인천광역시','광주광역시','대전광역시','울산광역시'}
SIDO_MAP={'강원도':'강원특별자치도','전라북도':'전북특별자치도','제주도':'제주특별자치도'}
def norm_key(sido, sigungu):
    """시도명 개명 통일 + 도-소속 시의 일반구는 시로 롤업(고양시 덕양구→고양시). 광역시 구는 유지."""
    sido=SIDO_MAP.get(str(sido).strip(), str(sido).strip()); sg=str(sigungu).strip()
    if sido=='세종특별자치시': return '세종특별자치시 세종시'   # [🔖 2026-07-21] 세종 정규화 — PW/PO/ON key 통일(단일 시군구)
    if sido not in GWANGYEOK and ' ' in sg: sg=sg.split()[0]
    return sido+' '+sg
def norm(key):
    """'시도 시군구[ 구]' 문자열 키를 정규화."""
    parts=str(key).split()
    if len(parts)>=3 and parts[0] not in GWANGYEOK and parts[-1].endswith(('구','출장소')):
        return norm_key(parts[0], ' '.join(parts[1:-1]))
    return norm_key(parts[0], ' '.join(parts[1:]))

def read_0718(y, sheet):
    """0718 시군구 전력(계약종별 or 용도업종별) 1개 연도 → 연간 GWh DataFrame.
    헤더가 3번째 행, 월 컬럼 12개를 더해 연간. '합계' 행은 제외."""
    fn=f'OPEN_0718_한전_시군구전력사용량_{y}.xlsx'
    if not os.path.exists(fn):                       # 2026은 '_4월까지' 접미사
        cand=glob.glob(f'OPEN_0718_한전_시군구전력사용량_{y}*.xlsx'); fn=cand[0] if cand else fn
    wb=CalamineWorkbook.from_path(fn)
    rows=wb.get_sheet_by_name(sheet).to_python()
    df=pd.DataFrame(rows[3:], columns=rows[2])
    mcols=[c for c in df.columns if str(c).endswith('월')]
    for c in mcols: df[c]=pd.to_numeric(df[c], errors='coerce')
    df['연간GWh']=df[mcols].sum(axis=1)/1e6
    scols=[c for c in mcols if str(c).replace('월','').strip() in ('6','7','8')]  # 여름 6-8월(JJA)
    df['여름GWh']=df[scols].sum(axis=1)/1e6
    catcol='계약종별' if sheet=='계약종별' else '업종별'
    df[catcol]=df[catcol].astype(str).str.replace(' ','')
    df['key']=df['시도'].astype(str).str.strip()+' '+df['시군구'].astype(str).str.strip()
    return df[df[catcol]!='합계']

def load_power_multiyear():
    """계약종별+용도업종별 다년 → 시군구×연도 long DataFrame.
    각 종류마다 연간(예: 산업용)과 여름 6-8월(예: 산업용여름)을 함께 준다.
    ①②(온열 관계)는 '~여름' 컬럼, ⑤(폐산단 구조적 감산)는 연간 컬럼을 쓴다."""
    out=[]
    for y in YEARS:
        ck=read_0718(y,'계약종별')
        pA=ck.pivot_table(index='key',columns='계약종별',values='연간GWh',aggfunc='sum')
        pS=ck.pivot_table(index='key',columns='계약종별',values='여름GWh',aggfunc='sum')
        ub=read_0718(y,'용도업종별')
        mfgA=ub[ub['업종별'].isin(MFG)].groupby('key')['연간GWh'].sum()
        mfgS=ub[ub['업종별'].isin(MFG)].groupby('key')['여름GWh'].sum()
        for k in pA.index:
            g=lambda P,c: round(P.loc[k].get(c,0),1)
            out.append({'연도':y,'key':norm(k),
                '산업용':g(pA,'산업용'),'산업용여름':g(pS,'산업용'),
                '주택용':g(pA,'주택용'),'주택용여름':g(pS,'주택용'),
                '일반용':g(pA,'일반용'),'일반용여름':g(pS,'일반용'),
                '제조업':round(mfgA.get(k,0),1),'제조업여름':round(mfgS.get(k,0),1)})
    # norm 후 같은 키(시도개명 등) 합산
    return pd.DataFrame(out).groupby(['연도','key'],as_index=False).sum()

def load_nedis_onset():
    """온열 시군구×연도 발생 건수(발생지). 온열=전체(연중), 여름온열=6-8월(JJA, 온열의 93%)."""
    wb=CalamineWorkbook.from_path('FOIA_0522_질병관리청_NEDIS온열질환_2020-2025.xlsx')
    r=wb.get_sheet_by_name('DB(2020-2025)발생지역기준').to_python()
    ne=pd.DataFrame(r[1:],columns=r[0])
    dt=pd.to_datetime(ne['발생일자'],errors='coerce'); ne['연도']=dt.dt.year; ne['월']=dt.dt.month
    ne['key']=[norm_key(s,g) for s,g in zip(ne['발생시도'],ne['발생시군구'])]  # 일반구 롤업+시도개명
    allc=ne.groupby(['key','연도']).size().rename('온열')
    sumc=ne[ne['월'].isin([6,7,8])].groupby(['key','연도']).size().rename('여름온열')
    return pd.concat([allc,sumc],axis=1).fillna(0).astype(int).reset_index()

def load_pop():
    """시군구×연도 인구(5세별×남녀 전부 합산한 총인구)."""
    SIDO=['서울특별시','부산광역시','대구광역시','인천광역시','광주광역시','대전광역시','울산광역시','세종특별자치시',
          '경기도','강원도','강원특별자치도','충청북도','충청남도','전라북도','전북특별자치도','전라남도','경상북도','경상남도','제주특별자치도','제주도']
    p=pd.read_csv('OPEN_0630_KOSIS_시군구주민등록인구_2020-2025.csv',encoding='cp949')
    regcol=p.columns[0]; ycols=[c for c in p.columns if '년' in str(c)]; agg={}; cur=None
    # [🔖 2026-07-21] 세종 특례: 파일에 시도=시군구로 이중등재(84행=21연령×2성×2회 중복)라 단순합산 시 2배.
    #   (5세별,항목) 조합 dedup 후 합산 → 실제 ~39만. 세종엔 네이버 각 세종 데이터센터가 있어 논증에도 필요.
    _sj=p[p[regcol].astype(str).str.replace('　','').str.strip()=='세종특별자치시'].drop_duplicates(subset=['5세별','항목'])
    for yc in ycols:
        yr=int(''.join(ch for ch in str(yc) if ch.isdigit())[:4]); v=pd.to_numeric(_sj[yc],errors='coerce').sum()
        if v: agg[('세종특별자치시 세종시',yr)]=agg.get(('세종특별자치시 세종시',yr),0)+v
    for _,row in p.iterrows():
        s=str(row[regcol]).replace('　','').strip()
        if s in SIDO: cur=s; continue
        if s in ('전국','읍부','면부','동부','행정구역별(시군구)','nan',''): continue
        for yc in ycols:
            yr=int(''.join(ch for ch in str(yc) if ch.isdigit())[:4]); v=pd.to_numeric(row[yc],errors='coerce')
            if pd.notna(v): k=norm_key(cur,s); agg[(k,yr)]=agg.get((k,yr),0)+v  # 시도개명 통일
    return pd.DataFrame([{'key':k[0],'연도':k[1],'인구':v} for k,v in agg.items()])

def sig(x,y,ctrl=None,label=''):
    """상관 + n·p·95%CI·별표. ctrl 주면 그 변수 통제한 편상관. 반환 None(튜플 echo 방지)."""
    x=np.asarray(x,float); y=np.asarray(y,float); Z95=1.96
    if ctrl is None:
        m=~(np.isnan(x)|np.isnan(y)); x,y=x[m],y[m]; n=len(x)
        rr=stats.pearsonr(x,y); r,p=rr.statistic,rr.pvalue; ci=rr.confidence_interval(0.95); lo,hi=ci.low,ci.high
    else:
        z=np.asarray(ctrl,float); m=~(np.isnan(x)|np.isnan(y)|np.isnan(z)); x,y,z=x[m],y[m],z[m]; n=len(x)
        rc=lambda a,b:np.corrcoef(a,b)[0,1]
        r=(rc(x,y)-rc(x,z)*rc(y,z))/np.sqrt((1-rc(x,z)**2)*(1-rc(y,z)**2))
        t=r*np.sqrt((n-3)/(1-r**2)); p=2*stats.t.sf(abs(t),n-3)
        zf=np.arctanh(r); se=1/np.sqrt(n-4); lo,hi=np.tanh(zf-Z95*se),np.tanh(zf+Z95*se)
    star='***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'
    print("   └%s n=%d r=%+.3f p=%.2g 95%%CI[%+.2f,%+.2f] %s"%(' '+label if label else '',n,r,p,lo,hi,star))
    return None
print("§0 OK — 로더:", [x for x in dir() if x.startswith('load_')], "+ read_0718 + sig()")
# 기대: 로더 4개 + read_0718 + sig()

작업 폴더: C:\cross_the_street\docs\research\_data-center\기획서_꾸러미\정보공개청구
§0 OK — 로더: ['load_nedis_onset', 'load_pop', 'load_power_multiyear'] + read_0718 + sig()


## §1. 0718 정합성 — 기존 OPEN_0621과 일치하나
0718(주택용 제외)이 옛 산업전력 자료(OPEN_0621, 화성 19,169 GWh)와 맞는지. 차이는 5호미만 마스킹.

In [2]:
# [🔖 2026-07-21] 동적 정합성 — OPEN_0621 실측 로드 + 다도시 비교 (상수 19169 하드코딩 제거)
import glob as _glob

# OPEN_0621(산업분류법정동, 12개월) → 시군구별 연간 판매량 GWh. 산업분류엔 주택용 없음 = 0718 non-home 대응.
_o621=[]

# 해당 패턴을 가진 모든 엑셀 파일 경로를 정렬하여 순회
for _f in sorted(_glob.glob('OPEN_0621_한전_산업분류법정동전력_*.xlsx')):

    # CalamineWorkbook을 사용해 고속으로 엑셀 시트(Sheet1) 데이터를 파이썬 리스트 구조로 변환
    _sh=CalamineWorkbook.from_path(_f).get_sheet_by_name('Sheet1').to_python()

    # 헤더(첫 번째 행)에서 주요 열('시도', '시군구', '판매량')의 인덱스 추출
    _h=_sh[0]; _iSi=_h.index('시도'); _iSg=_h.index('시군구'); _iSl=_h.index('판매량')

    # 데이터 행(두 번째 행부터) 순회하며 (정규화된 시군구 키, 판매량) 튜플을 리스트에 추가
    for _r in _sh[1:]: _o621.append((norm_key(_r[_iSi],_r[_iSg]), _r[_iSl] or 0))
    

# -------------------------------------------------------------------------
# 2. OPEN_0621 데이터 집계 (kWh -> GWh 단위 변환)
# -------------------------------------------------------------------------
# 추출된 데이터를 DataFrame으로 변환 후, 시군구 키('key')별로 판매량을 합산
# 1e6(1,000,000)으로 나눠 단위 변환 (kWh -> GWh)
O621=pd.DataFrame(_o621,columns=['key','sale']).groupby('key')['sale'].sum()/1e6

# -------------------------------------------------------------------------
# 3. 0718 계약종별 데이터 로드 및 비주택(non-home) 전력 사용량 집계
# -------------------------------------------------------------------------
# 2025년 계약종별 0718 데이터 로드
ck25=read_0718(2025,'계약종별')

# '주택용'을 제외한 나머지 계약종별(상업·업무·산업용 등) 전력량을 시군구('key')별로 합산
NH=ck25[ck25['계약종별']!='주택용'].groupby('key')['연간GWh'].sum()

# -------------------------------------------------------------------------
# 4. 상위 8개 산업 시군구에 대한 데이터 비교 및 출력
# -------------------------------------------------------------------------
print("0718 산업 vs OPEN_0621 산업 — 상위 8 산업 시군구 (GWh):")
print(f"  {'시군구':16}{'0718 non-home':>14}{'0621 산업':>11}{'차이':>8}")


# OPEN_0621 전력 사용량이 높은 상위 8개 시군구 키를 기준으로 비교 실행
for _k in O621.sort_values(ascending=False).head(8).index:

    # 각 시군구별 0718 비주택 사용량(_a)과 OPEN_0621 산업 사용량(_b) 추출 (없으면 NaN)
    _a=NH.get(_k,float('nan'))
    _b=O621.get(_k,float('nan'))
    
    # 두 데이터 간의 백분율 오차/차이 계산 (%)
    _d=100*(_a-_b)/_b

    # 차이가 -1% ~ +8% 범위 내에 있으면 정합 조건 충족 카운트 증가
    _v='정합' if -1<=_d<=8 else '괴리(상업·서비스 부하 — 산업분류가 과소 집계)'

    # 결과 행 출력 (좌측/우측 정렬 및 소수점 포맷 지정)
    print(f"  {_k:16}{_a:>14.0f}{_b:>11.0f}{_d:>7.1f}%  {_v}")

print(f"→ 중공업 도시(화성·아산·울주·서산·구미)는 0~7% 정합. 상업·서비스 큰 도시(울산남구·청주·여수)는 0718이 크다 —")
print("   계약종별 비주택은 상업·업무용을 포함하나 산업분류(0621)는 제조업 위주라 그렇다. 정합성 확인엔 중공업 도시가 적절.")

0718 산업 vs OPEN_0621 산업 — 상위 8 산업 시군구 (GWh):
  시군구              0718 non-home    0621 산업      차이
  경기도 화성시                  19610      19169    2.3%  정합
  충청남도 아산시                 12584      12584    0.0%  정합
  전라남도 여수시                 14024      12116   15.7%  괴리(상업·서비스 부하 — 산업분류가 과소 집계)
  울산광역시 울주군                11369      11229    1.2%  정합
  충청남도 서산시                  9219       8877    3.9%  정합
  울산광역시 남구                 12906       8670   48.9%  괴리(상업·서비스 부하 — 산업분류가 과소 집계)
  충청북도 청주시                 10183       8040   26.7%  괴리(상업·서비스 부하 — 산업분류가 과소 집계)
  경상북도 구미시                  8395       7883    6.5%  정합
→ 중공업 도시(화성·아산·울주·서산·구미)는 0~7% 정합. 상업·서비스 큰 도시(울산남구·청주·여수)는 0718이 크다 —
   계약종별 비주택은 상업·업무용을 포함하나 산업분류(0621)는 제조업 위주라 그렇다. 정합성 확인엔 중공업 도시가 적절.


## §2. 다년 계약종별 DERIVED 생성 — 산업용/제조업/주택용 분리
옛 '전력GWh blob'(산업+상업+공공 뭉침)을 계약종별로 쪼갠다. **DC는 일반용으로 분류**되므로 산단 신호는 산업용/제조업.

In [3]:
PW=load_power_multiyear()   # ~1분 (6년×2시트)
PW.to_csv('DERIVED_0718_시군구_계약종별_다년전력.csv',index=False,encoding='utf-8-sig')
print("PW:", len(PW),"행 (시군구",PW['key'].nunique(),"× 연도 6)")
print(PW[PW['key']=='경기도 화성시'][['연도','산업용','산업용여름','주택용','주택용여름']].to_string(index=False))
# 기대: 화성 산업용(연간) ~15,760, 산업용여름(6-8월) ~4,000 (연간의 ~26%) (2025)

PW: 1378 행 (시군구 231 × 연도 6)
  연도     산업용  산업용여름    주택용  주택용여름
2020 15250.5 3854.0 1343.4  344.2
2021 15965.7 4080.5 1481.2  419.0
2022 16242.3 4159.1 1543.3  429.4
2023 16012.0 4113.5 1639.5  454.5
2024 15852.6 4115.0 1778.6  486.7
2025 15759.5 4110.5 1851.2  527.0


## §3. ① Cross-sectional — 인구 통제 편상관 vs 여름 온열
여름(6-8월) 전력 vs 여름 온열. **3단계로 나눠 실행**: (a) 로드·키 확인 → (b) 병합(left join, 온열 0 포함) → (c) 편상관.
> **⚠ 2026-07-18 키 정규화 수정**: NEDIS는 '고양시 덕양구'(일반구), 전력·인구는 '고양시'로 기록해 **포항·청주·창원 등 산업도시가 누락**됐었다. `norm_key`로 일반구를 시로 롤업 + 시도개명('강원도'↔'강원특별자치도') 통일해 복원. 수정 후 산업용 편상관 0.463→0.507로 오히려 강해짐(누락됐던 게 바로 고산업·고온열 도시라).

**§3-a. 세 자료 로드 + 키/행수 확인.** PW는 DERIVED에 이미 norm 적용, ON·PO는 로더에서 norm_key 적용.

In [4]:
PW=pd.read_csv('DERIVED_0718_시군구_계약종별_다년전력.csv',encoding='utf-8-sig')
ON=load_nedis_onset(); PO=load_pop()
print("PW", PW.shape, "/ ON", ON.shape, "/ PO", PO.shape)
print("시군구 수 — PW:%d  ON:%d  PO:%d"%(PW.key.nunique(),ON.key.nunique(),PO.key.nunique()))
# 3자 공통 시군구 (정규화 후)
common=set(PW.key)&set(ON.key)&set(PO.key)
print("3자 공통 시군구:", len(common), "| PW에만:", len(set(PW.key)-set(PO.key)-set(ON.key)))
# 기대: 정규화로 포항·청주·창원 등이 세 자료 공통에 포함

PW (1378, 10) / ON (1310, 4) / PO (1386, 3)
시군구 수 — PW:231  ON:230  PO:232
3자 공통 시군구: 230 | PW에만: 1


**§3-a 주석 — PW에만 있던 2개 시군구** `[🔖 2026-07-21]`

`set(PW.key)-set(PO.key)-set(ON.key)` = `{'세종특별자치시 세종시', '황해북도 개성시'}`

- **세종** — 매칭 가능. 세종은 시군구가 없는 단일 특별자치시라 자료마다 `세종시`/`세종특별자치시`/공란으로 달라 key가 어긋났다. `norm_key`에 세종 특례를 넣어 **PW·ON key를 통일**했다 — 온열(NEDIS) 131건과 매칭되어 차집합에서 세종이 사라진다(위 셀 결과 `{개성}`만 남음). **다만 인구(PO)는 별개 이슈**: KOSIS 파일이 세종을 시도=시군구로 이중 등재해 `load_pop`이 건너뛴다(합산 시 2배 중복). 완전한 3자 공통 편입은 이 이중등재 해소가 필요하나, **세종엔 AIDC 대상 대형 산단이 없어 논증 영향은 없다.** `[후속]`
- **개성** — 제외. `황해북도 개성시`는 **북한**이다. 한전 전력자료(PW)에 개성이 있는 이유: 남한이 **파주 문산변전소 → 송전선로**로 2005년부터 개성공단에 전력을 공급했기 때문(2024년 북한이 전선 절단·송전탑 철거로 기능 상실). 흥미롭게도 **본 자료엔 2023년부터 개성 데이터가 없다** — 공식 철거(2024)보다 앞선다. 개성공단 자체는 2016년 2월 전면 가동중단됐으나 일부 전력수급 기록은 이후에도 남았던 것으로 보인다. 인구·온열 자료가 없으므로 분석에서 제외한다. `[출처: 공개 검색 종합 — 매체 확인 권장]`

In [5]:
set(PW.key)-set(PO.key)-set(ON.key)

{'황해북도 개성시'}

**§3-b. 병합 — left join으로 온열 0인 시군구-연도도 포함.** inner join이었으면 '온열 0건인 해'가 통째로 사라진다(원래 버그).

> **Q: `ON['온열']`엔 0인 행이 없는데 왜 panel엔 '여름온열=0'이 78행인가?** (2026-07-19 검증)
> `ON`(NEDIS 집계)은 **발생 건이 있는 (key,연도)만 행이 존재하는 sparse 자료**(1,310행)다 — '0건'을 행으로 기록하지 않는다. key 고유수는 230이지만 **key×연도 조합**으로 보면 229키×6년=1,374 중 **64개 조합이 ON에 없다**(그 시군구-그해 온열 0건). left join이 이 64개를 NaN으로 남기고 `fillna(0)`이 0으로 채운다. 여기에 ON에 행이 있어도 **여름온열=0인 행 14개**(온열이 겨울·봄에만 발생)가 더해져 **78 = 64 + 14**. `fillna`를 지우면 ① `astype(int)`가 NaN에서 즉시 에러 ② §3-c의 `groupby.mean()`이 NaN을 skip해 '온열 0이던 해'가 평균에서 빠짐 → **온열 과대추정**(inner join 시절과 같은 편향). 0은 결측이 아니라 실제 관측값이다.

In [6]:
base=PW.merge(PO,on=['key','연도'],how='inner')    # 전력·인구 있는 시군구-연도
panel=base.merge(ON,on=['key','연도'],how='left')   # 온열 붙이되, ON에 없는 (key,연도)=그해 0건 → NaN
nan_miss=int(panel['여름온열'].isna().sum())         # 병합 NaN (= ON에 행이 없던 조합, 기대 64)
genuine0=int((panel['여름온열']==0).sum())           # ON에 있으나 여름온열=0 (겨울·봄만 발생, 기대 14)
# fillna(0) 필수: ① NaN인 채론 astype(int) 불가 ② §3-c mean()이 NaN skip → 온열 0인 해가 빠져 과대추정
panel['온열']=panel['온열'].fillna(0).astype(int); panel['여름온열']=panel['여름온열'].fillna(0).astype(int)
panel['여름온열률10만']=panel['여름온열']/panel['인구']*1e5
panel.to_csv('DERIVED_0718_시군구연도_패널_온열전력.csv',index=False,encoding='utf-8-sig')
print("panel", panel.shape, "| 시군구", panel.key.nunique())
print(f"여름온열 0인 시군구-연도: {int((panel['여름온열']==0).sum())} = 병합NaN {nan_miss} + ON의 진짜0 {genuine0}")

panel (1374, 14) | 시군구 230
여름온열 0인 시군구-연도: 78 = 병합NaN 64 + ON의 진짜0 14


In [7]:
panel.head(1)

,연도,key,산업용,산업용여름,주택용,주택용여름,일반용,일반용여름,제조업,제조업여름,인구,온열,여름온열,여름온열률10만
0,2020,강원특별자치도 강릉시,767.6,193.1,321.6,78.3,588.9,150.8,673.9,168.9,213321.0,8,8,3.750217


**§3-c. ① 인구 통제 편상관.** cs = 시군구별 대표값.
> **Q: 왜 대표값? 왜 6년을 뭉개나?** cross-sectional은 "산업 많은 *동네*가 온열 많나"(between-region)를 보므로 시군구당 1값이 필요하다. 연도별 변화는 ②(패널)가 따로 본다.
> **모든 변수를 6년 평균으로 통일**(2026-07-18): 이전엔 온열=sum·전력=mean로 섞였는데, 상관은 스케일 불변이라 결과는 같지만 혼동을 없애려 mean으로 통일.

In [8]:
cs=panel.groupby('key').agg(여름온열=('여름온열','mean'),산업용여름=('산업용여름','mean'),제조업여름=('제조업여름','mean'),
    일반용여름=('일반용여름','mean'),주택용여름=('주택용여름','mean'),인구=('인구','mean')).reset_index()
print("cross-sec n=%d — 인구 통제 편상관 vs 여름온열 (모두 6년 평균, 95%% CI):"%len(cs))
for v in ['산업용여름','제조업여름','일반용여름','주택용여름']:
    sig(cs[v], cs['여름온열'], ctrl=cs['인구'], label=v)
# 기대: 산업용/제조업 r≈+0.5 *** / 일반용·주택용 n.s. → 신호는 '산업'

cross-sec n=230 — 인구 통제 편상관 vs 여름온열 (모두 6년 평균, 95% CI):
   └ 산업용여름 n=230 r=+0.507 p=2.5e-16 95%CI[+0.40,+0.60] ***
   └ 제조업여름 n=230 r=+0.509 p=1.7e-16 95%CI[+0.41,+0.60] ***
   └ 일반용여름 n=230 r=+0.000 p=1 95%CI[-0.13,+0.13] n.s.
   └ 주택용여름 n=230 r=-0.029 p=0.66 95%CI[-0.16,+0.10] n.s.


In [9]:
cs.head(3)

,key,여름온열,산업용여름,제조업여름,일반용여름,주택용여름,인구
0,강원특별자치도 강릉시,18.500000,206.400000,168.083333,168.833333,90.533333,210179.000000
1,강원특별자치도 고성군,1.333333,14.950000,9.050000,44.200000,10.600000,27071.500000
2,강원특별자치도 동해시,5.666667,406.666667,385.850000,59.266667,37.150000,88788.833333


## §4. ② 다년 패널 Poisson — 산업용 탄력도

`온열 ~ log(산업용) + log(주택용) + 연도FE`, offset=`log(인구)`, 시군구 클러스터 SE.

In [10]:
import pandas as pd
import numpy as np

In [11]:
import statsmodels.api as sm, statsmodels.formula.api as smf

# 1. 패널 데이터 불러오기 (시군구-연도별 온열질환자 및 전력사용량 데이터)
panel=pd.read_csv('DERIVED_0718_시군구연도_패널_온열전력.csv',encoding='utf-8-sig')

# 2. 주요 변수 로그 변환 (Offset으로 사용할 인구 및 독립변수들)
# - clip(lower=1): log(0)으로 인한 -inf 방지 (최소값을 1로 보정)
# - 'l산업용여름', 'l주택용여름', 'l인구' 컬럼 생성
for c in ['산업용여름','주택용여름','인구']: panel['l'+c]=np.log(panel[c].clip(lower=1))

# 3. 연도 변수를 범주형(Categorical) 데이터로 변환 (연도별 고정효과 흡수 목적)
panel['C연도']=panel['연도'].astype(str)

# 4. 포아송 일반화 선형 모델(GLM Poisson) 적합
# - 종속변수: 여름온열 (여름철 온열질환자 수)
# - 설명변수: l산업용여름, l주택용여름 + C(C연도) (연도 고정효과 포함)
# - offset: l인구 (인구 대비 온열질환자 발생 비율인 '온열률'을 모형화하기 위함)
# - cov_type='cluster', cov_kwds={'groups': panel['key']}: 
#   동일 시군구('key') 내 다년도 관측치의 자기상관성을 통제하기 위해 군집 강건 표준오차(Clustered Standard Error) 적용
m=smf.glm('여름온열 ~ l산업용여름 + l주택용여름 + C(C연도)', data=panel, offset=panel['l인구'],
          family=sm.families.Poisson()
          ).fit(cov_type='cluster', cov_kwds={'groups':panel['key']})

# 5. 모델 추정 결과 해석 및 출력 (탄력도 및 10% 증가 시 변화율 계산)
for t,lab in [('l산업용여름','산업용(여름)'),('l주택용여름','주택용(여름)')]:
    e=m.params[t]; p=m.pvalues[t]; ci=m.conf_int().loc[t]
    print("%s 탄력도 %+.3f → 10%%↑당 여름온열률 %+.1f%%  p=%.1e  95%%CI[%+.2f,%+.2f]"%(lab,e,100*(1.10**e-1),p,ci[0],ci[1]))
print("n=%d 시군구-연도 (여름 전력 × 여름 온열, 온열 0 포함)"%len(panel))
# 기대: 산업용(여름) 탄력도 +0.22 → +2.2% p<1e-20 (키 정규화·0 포함 수정 후)

산업용(여름) 탄력도 +0.224 → 10%↑당 여름온열률 +2.2%  p=3.0e-21  95%CI[+0.18,+0.27]
주택용(여름) 탄력도 -0.598 → 10%↑당 여름온열률 -5.5%  p=1.4e-58  95%CI[-0.67,-0.53]
n=1374 시군구-연도 (여름 전력 × 여름 온열, 온열 0 포함)


### ② 해석 (확장)

**탄력도(elasticity) +0.224의 뜻** `[🔖 2026-07-24 코드 출력값으로 정정 — 구 원고 +0.21·p<1e-16]`: 산업용 전력이 **10% 많은 시군구는 인구보정 온열질환율이 약 2.2% 높다** (코드 셀 출력값 — p=3.0e-21). 로그-로그 모형이라 계수가 곧 탄력도다.

- **`offset=log(인구)`** — 온열을 인구로 나눈 **'온열률'**을 본다는 뜻. "큰 도시라 환자가 많다"를 배제한다.
- **`연도FE(C연도)`** — 2020~2025 전국 폭염 심화(온열 4배↑)를 통째로 흡수. 따라서 "폭염이 세지는 해라서"가 아니라 **순수하게 시군구 사이의 산업 차이**만 남긴다.
- **시군구 클러스터 SE** — 같은 시군구의 6개 연도가 상관됨을 보정. 그런데도 **p=3.0e-21**(위 코드 셀 출력) — 우연일 여지 사실상 없음.
- **주택용 −0.60** — 주택용 전력이 1인당 많은 곳(베드타운·주거지)일수록 온열률 낮음. 산업의 반대 얼굴. `[해석]`

**⚠ 무엇을 말하고 무엇을 말하지 않나 (중요):**
- ✅ 말하는 것: "**산업(제조)이 많은 시군구가 온열이 많다**"는 연관이 6년 내내, 인구·연도 보정 후에도 견고하다(between-region).
- ❌ 말하지 않는 것: "**산업전력이 늘면 온열이 는다**"는 시계열 인과. 산업용은 6년간 거의 안 변해(quasi-static) **within-시군구 식별이 약하다.** 실제로 §6에서 폐산단 시군구의 온열은 오히려 *증가*했다 — 산업전력이 줄어도 온열은 폭염 트렌드로 늘었다. **즉 이 탄력도는 "산업 많은 곳"의 표지이지 "산업이 줄면 온열이 준다"가 아니다.**
- **인과 방향 미확정**: 산업↔온열이 (a) 실외 노동 노출, (b) 산업지대 열섬, (c) 발생지-거주지 통근 유입(§1) 중 무엇인지 이 모형은 못 가른다.


## §5. 폐산단 전환 후보 스캔 — **연간** 산업용 하락 + 업종 분해 (재설계 2026-07-18)
"놀고 있는 산단"을 찾되, **중공업(구조적 감산)** vs **반도체(경기 하락, 회복 가능)**를 갈라야 한다. ⚠ 여기는 **연간 전력**(공장 폐쇄는 연중 감소, ①②의 여름과 목적 다름).
> **사용자 지적 반영 4가지**: ① **SECT에 자동차 추가**(제조업 4위 19,002GWh, 아산·화성·울산). ② **반도체='영상.음향'만**(전기기기·의료광학은 반도체 아님 — 청주 하락은 100% 영상.음향). ③ **비단조 대응** — 후보 전부 non-monotonic(청주·이천 2022 peak)이라 `2020→2025` 끝점 대신 **peak→2025** 기준. ④ **2026(1-4월) 회복** 결합 — 반도체 반등이면 폐산단 아님. `500GWh` 필터는 소규모 시군구 노이즈 제거용(빠지는 곳 하락 최대 −129로 안전).

> **★ "일시 감산 아닌 구조적"의 절대량 근거** `[🔖 2026-07-24 사용자 지적]` — 비중이 아니라 **중공업 전력 절대 GWh**로 확인:
>
> | 시군구 | 중공업 2020 | 2025 | 절대Δ | Δ% | 비중 20→25 |
> |---|---:|---:|---:|---:|---|
> | 당진 | 8,382 | 4,555 | −3,827 | **−45.7%** | 90.1 → 83.8% |
> | 동해 | 1,663 | 1,008 | −654 | **−39.3%** | 93.9 → 87.0% |
> | 인천 동구 | 1,336 | 941 | −395 | **−29.6%** | 40.2 → 36.9% |
> | 울산 남구 | 12,535 | 10,460 | −2,075 | −16.5% | 87.3 → 88.0% |
>
> 중공업 **절대량이 실제로 줄었고**(비중만 흔들린 게 아님), 2026 1–4월에도 미반등(당진 −2.3%·동해 −5.2%)이라 "일시 감산" 해석이 약하다. 다만 **개별 공장 폐업 확정은 별도 자료(공장등록 말소 등)가 필요**하다는 한계는 유지.

In [12]:
def nrm(k):  # §5 전용 키 정규화(§0 norm과 동일 규칙)
    p=str(k).split(); s=SIDO_MAP.get(p[0],p[0]); rest=p[1:]
    if s not in GWANGYEOK and len(rest)>=2 and rest[-1].endswith(('구','출장소')): rest=rest[:-1]
    return s+' '+' '.join(rest)

PW=pd.read_csv('DERIVED_0718_시군구_계약종별_다년전력.csv',encoding='utf-8-sig')
ind=PW.pivot_table(index='key',columns='연도',values='산업용',aggfunc='sum')

# ── 분류 세트 ──
# HEAVY(중공업): 열배출·전력다소비 제조업 (담배·펄프 등 AIDC 전환과 무관한 경공업 제외 — 사용자 지적 반영)
HEAVY={'철강':['1차금속'],'석유화학':['화학제품','석유정제','고무.플라'],'기계금속':['기타기계','조립금속','금속비금속'],
       '자동차운송':['자동차','기타운송'],'시멘트유리':['시멘트','유리']}
# SEMI '영상.음향' = 한전 용도업종별에서 KSIC C26(전자부품·컴퓨터·영상·음향·통신장비)의 축약 라벨.
#   반도체 팹 포함 실측 확인(2025 연간GWh): 평택 14,060·화성 8,725·아산 7,618·이천 3,443·구미 3,411 — 삼성·하이닉스 팹 규모.
#   TV·음향 완제품(C265)도 같은 분류지만 국내 전력은 팹이 지배. '의료.광학'(C27 의료·정밀·광학)은 반도체 아님+증가 업종 → 제외.
SEMI=['영상.음향']

d20=read_0718(2020,'용도업종별'); d25=read_0718(2025,'용도업종별')
j25=read_0718(2025,'계약종별'); j26=read_0718(2026,'계약종별')                # 2026=1-4월까지
for d in (d20,d25,j25,j26): d['key2']=d['key'].map(nrm)

# 업종별 연간 전력량 추출 함수
def sd(d,k,keys,col='연간GWh'): return d[(d['key2']==k)&(d['업종별'].astype(str).str.replace(' ','').isin(keys))][col].sum()

# 2026년 1~4월 산업용 전력량 추출 함수
def csales(d,k):  # 2026 회복용: 계약종별 산업용 1-4월(여름GWh 아님, 별도 합)
    sub=d[(d['key2']==k)&(d['계약종별'].astype(str).str.replace(' ','')=='산업용')]
    mc=['1월','2월','3월','4월']; return sum(pd.to_numeric(sub[c],errors='coerce').sum() for c in mc)/1e6

HEAVY_FLAT=[u for ks in HEAVY.values() for u in ks]

rows=[]
for k in ind.index:

    # [방법] 실질 산단 필터: peak 연간 산업용 500GWh 미만 제외 — 500GWh≈중견 산단 1곳 규모(당진1철강급).
    #   그 미만은 개별 공장 노이즈가 지배해 '폐산단 전환 후보' 질문 자체가 성립 안 함. 논문 출처 아닌 우리 판단.
    if ind.loc[k].max()<500: continue
    
    peak=ind.loc[k].max(); v25=ind.loc[k,2025]; pky=int(ind.loc[k].idxmax())
    
    # 500GWh 필터를 통과한 거대 산업 지역들을 대상으로, 5년 사이 전력 사용량이 얼마나 변했는지를 업종별로 계산합니다.
    # 이 지역의 산업 쇠퇴가 중화학 공업의 몰락 때문인지, 아니면 다른 업종의 변화 때문인지 추적할 수 있게 됩니다.
    heavy=sum(sd(d25,k,ks)-sd(d20,k,ks) for ks in HEAVY.values()); semi=sd(d25,k,SEMI)-sd(d20,k,SEMI)

    # max(...,1): 분모 0 가드 — 2025 1-4월 산업용 판매 0인 key에서 ZeroDivision 방지.
    #   500GWh 필터 통과 key는 분모가 수백 GWh라 하한 1GWh는 실질 왜곡 없음.
    _d=csales(j25,k); rec=100*(csales(j26,k)-_d)/_d if _d>0 else 0   # 2026 1-4월 vs 2025 1-4월 [수정 07-19: max(,1)은 0<d<1을 왜곡 → 0÷만 예외(0<d<1 그대로 반영)]
    pct=round(100*(v25-peak)/peak,1)

    # ── 분류 로직: 위→아래 순차 평가, 먼저 참이 되는 branch 하나로 확정 ──
    # ① 구조적 폐산단(중공업) = 4조건 AND:
    #    pct<-15(peak 대비 15%+ 하락) · heavy<-150(중공업 '절대' 감소 150GWh+ — 비율 아닌 절대값, 사용자 지적 반영)
    #    |heavy|>|semi|(하락의 주인이 중공업 — 반도체 경기 착시 배제) · rec<3(작년 동기 대비 2026 증가분이 3% 미만 → 경기 아닌 구조)
    # ② 반도체(경기민감) = semi<-150 이고 |semi|>=|heavy| — 하락의 주인이 반도체(24h 팹, 폐산단 아님)
    # ③ 기타 하락 = pct<-15인데 ①② 아님   ④ 유지/증가
    # 임계 -15%·-150GWh·+3%는 [방법] 판단(관측 분포의 자연 단절) — 논문 기준 아님.
    if pct<-15 and heavy<-150 and abs(heavy)>abs(semi) and rec<3: cls='구조적 폐산단(중공업)'
    elif semi<-150 and abs(semi)>=abs(heavy): cls='반도체(경기민감)'
    elif pct<-15: cls='기타 하락'
    else: cls='유지/증가'
    
    # 신규(2026-07-19): 전체산업Δ(계약종별 산업용 2025−2020) + 2025 비중.
    #   ⚠ 비중 분자=용도업종별·분모=계약종별 산업용 — 집계 정의가 달라 ±수% 근사치.
    # 25중공업비중% ➔ [용도업종별 분류] / 26회복% ➔ [계약종별 분류] 기반
    tot25=v25; tot20=ind.loc[k,2020]
    h25=sd(d25,k,HEAVY_FLAT); s25=sd(d25,k,SEMI)
    rows.append({'key':k,'peak연도':pky,'peak대비%':pct,'전체산업Δ':round(tot25-tot20),
                 '중공업Δ':round(heavy),'반도체Δ':round(semi),
                 '25중공업비중%':round(100*h25/max(tot25,1),1),'25반도체비중%':round(100*s25/max(tot25,1),1),
                 '26회복%':round(rec,1),'분류':cls})
SC=pd.DataFrame(rows).sort_values('중공업Δ')
SC.to_csv('DERIVED_0718_폐산단전환후보_산업전력하락.csv',index=False,encoding='utf-8-sig')
print("★ 구조적 폐산단(중공업, peak대비 -15%↓ + 2026 미반등):")
print(SC[SC['분류'].str.startswith('구조적')][['key','peak연도','peak대비%','전체산업Δ','중공업Δ','25중공업비중%','26회복%']].to_string(index=False))
print("\n반도체(경기민감 — 24h fab, 폐산단 아님. 26회복%로 반등 확인):")
print(SC[SC['분류']=='반도체(경기민감)'].sort_values('반도체Δ')[['key','peak연도','peak대비%','전체산업Δ','반도체Δ','25반도체비중%','26회복%']].to_string(index=False))
# 검증 포인트: 당진 중공업비중 84%·울산남구 88%(중공업 도시 맞음) / 이천 반도체비중 78%·구미 47%(반도체 도시 맞음)

★ 구조적 폐산단(중공업, peak대비 -15%↓ + 2026 미반등):
        key  peak연도  peak대비%  전체산업Δ  중공업Δ  25중공업비중%  26회복%
   충청남도 당진시    2020    -41.6  -3872 -3827      83.8   -2.3
   울산광역시 남구    2021    -20.3  -2473 -2075      88.0   -2.7
강원특별자치도 동해시    2022    -41.7   -612  -654      87.0   -5.2
   인천광역시 동구    2021    -27.4   -775  -395      36.9   -3.4
  부산광역시 사하구    2021    -21.1   -336  -351      55.7    1.1

반도체(경기민감 — 24h fab, 폐산단 아님. 26회복%로 반등 확인):
     key  peak연도  peak대비%  전체산업Δ  반도체Δ  25반도체비중%  26회복%
충청북도 청주시    2022    -26.6  -1903 -2422      27.0    1.2
충청남도 아산시    2020    -10.4  -1327 -1623      66.8    0.4
 경기도 이천시    2022    -39.7  -1595 -1572      78.5    7.1
경상북도 구미시    2021    -10.6   -426  -304      46.5   -3.7
 경기도 용인시    2022     -8.7   -314  -295      48.2    2.7


## §5-b. 임계값 민감도 — 분류 정당성 보강 `[방법]` `[🔖 2026-07-19]`

§5의 임계(peak 500GWh·−15%·−150GWh)는 이론값이 아니라 관측 분포의 자연 단절점이라, **임계를 ±조정해도 '구조적 폐산단' 분류가 안정적인지** 확인한다. 각 임계를 홀로 스윕하고, **모든 조합에서 항상 구조적으로 남는 robust core**를 구한다. (§5의 `ind·d20·d25·j25·j26·HEAVY·SEMI·sd·csales` 재사용.)

In [13]:
ind.head()

연도,2020,2021,2022,2023,2024,2025
key,,,,,,
강원특별자치도 강릉시,767.6,778.0,909.1,866.7,822.7,789.2
강원특별자치도 고성군,50.4,55.7,59.2,61.9,60.9,59.1
강원특별자치도 동해시,1771.1,1843.1,1987.7,1756.6,1284.4,1158.8
강원특별자치도 삼척시,1093.1,1112.1,1048.3,1134.1,1134.6,974.9
강원특별자치도 속초시,63.4,68.5,71.5,74.4,75.4,75.5


In [14]:
rows=[]
for k in ind.index:
    peak=ind.loc[k].max()
    if peak<300: continue                       # 스윕 최저(300)까지 후보 확보
    pct=100*(ind.loc[k,2025]-peak)/peak
    heavy=sum(sd(d25,k,ks)-sd(d20,k,ks) for ks in HEAVY.values()); semi=sd(d25,k,SEMI)-sd(d20,k,SEMI)

    # 최소한 pct < -10이면서 heavy < -100인 지역들만 '구조적 폐산단이 될 가능성이 있는 후보군'으로 판정
    # 이미 정점 대비 하락률이 -5%밖에 안 되거나 중공업 감소량이 미미한 지역들은
    # 뒤의 조건(rec < 3)을 볼 필요도 없이 탈락. 굳이 무거운 csales() 함수를 실행하지 않고, 
    # 그냥 rec < 3 조건에서 거짓(False)이 되어 탈락하도록 가짜 값인 99를 채워 넣고 패스.
    _d=csales(j25,k); rec=((100*(csales(j26,k)-_d)/_d if _d>0 else 0) if (pct<-10 and heavy<-100) else 99)  # rec는 후보만(속도) [수정 07-19: 0÷만 예외]
    rows.append((k,peak,pct,heavy,semi,rec))
Tsens=pd.DataFrame(rows,columns=['key','peak','pct','heavy','semi','rec'])

def struct_set(MIN,PCT,HV):   # 구조적 폐산단 4조건
    m=(Tsens.peak>=MIN)&(Tsens.pct<PCT)&(Tsens.heavy<HV)&(Tsens.heavy.abs()>Tsens.semi.abs())&(Tsens.rec<3)
    return set(Tsens[m].key)

base=struct_set(500,-15,-150)
print('기준(500/-15/-150):', sorted(base))   # [수정 07-19: 전체 시군구명 — '남구'→'울산광역시 남구'로 정확히]
print('[MIN_GWH 스윕]:', {m:len(struct_set(m,-15,-150)) for m in [300,500,700,1000]})
print('[peak대비% 스윕]:', {p:len(struct_set(500,p,-150)) for p in [-10,-15,-20,-25]})
print('[중공업Δ 스윕]:', {h:len(struct_set(500,-15,h)) for h in [-100,-150,-200,-300]})
core=set.intersection(*[struct_set(m,-15,-150) for m in [300,500,700,1000]],
                      *[struct_set(500,p,-150) for p in [-10,-15,-20,-25]],
                      *[struct_set(500,-15,h) for h in [-100,-150,-200,-300]])
print('★ 모든 임계 조합에서 항상 구조적(robust core):', sorted(core))   # [수정 07-19]
print('  경계에서 흔들리는(marginal):', sorted(base-core),'— 임계 선택에 민감, 약한 근거')   # [수정 07-19]
# 결론: 당진·인천동구·동해 = 임계 무관 견고 / 사하·울산남구 = 경계 민감(단 울산남구는 §7 dose-response엔 메가권역으로 포함, 폐산단 여부와 무관)

기준(500/-15/-150): ['강원특별자치도 동해시', '부산광역시 사하구', '울산광역시 남구', '인천광역시 동구', '충청남도 당진시']
[MIN_GWH 스윕]: {300: 5, 500: 5, 700: 5, 1000: 5}
[peak대비% 스윕]: {-10: 9, -15: 5, -20: 5, -25: 3}
[중공업Δ 스윕]: {-100: 8, -150: 5, -200: 5, -300: 5}
★ 모든 임계 조합에서 항상 구조적(robust core): ['강원특별자치도 동해시', '인천광역시 동구', '충청남도 당진시']
  경계에서 흔들리는(marginal): ['부산광역시 사하구', '울산광역시 남구'] — 임계 선택에 민감, 약한 근거


## §6. 산업↔온열은 (a)작업장 노동인가 (b)산업지대 열섬인가 — 발생장소로 판별
§4에서 미룬 질문. **발생장소별로 산업용과의 편상관**을 보면 갈린다: 작업장(실외·실내·논밭)만 유의하면 (a)노동, 집·길가(주변)도 유의하면 (b)열섬.
> 명칭 주의(2026-07-19 수정): 실내 작업장도 강하게 유의(r=+0.52)하므로 (a)는 '실외노동'이 아니라 **'작업장 노동(실내+실외)'**이다.

**§6-a. 폐산단 key별 온열률 추이 — 전국 트렌드 대비 + 인구증감.** (2026-07-19 재설계: 전체 sum 지수 → key별 rate. 인구로 정규화하고 전국과 비교해야 '폐산단 효과'가 있는 지역/없는 지역이 갈린다.)
> 참고: 전국 **연간** 온열 카운트는 2020 1,078 → 2025 4,460 = **4.14배**(통합보고서 §2-1과 일치). 폐산단 5개 key의 **여름 합**은 16→113 = 7.1배로 더 가파른데, 이는 인구 통제 때문이 아니라 **부분집합+소표본(base 16건)+여름 한정** 효과다.

In [15]:
# natx (전국 폭염 트렌드): 2020년 대비 2025년에 전국적으로 온열질환률이 몇 배나 증가했는지 배수(기준점)를 구합니다
panel=pd.read_csv('DERIVED_0718_시군구연도_패널_온열전력.csv',encoding='utf-8-sig')
SC=pd.read_csv('DERIVED_0718_폐산단전환후보_산업전력하락.csv',encoding='utf-8-sig')
struct=SC[SC['분류'].str.startswith('구조적')]['key'].tolist()
nat=panel.groupby('연도').agg(온=('여름온열','sum'),인=('인구','sum')); nat['율']=nat.온/nat.인*1e5
natx=nat.loc[2025,'율']/nat.loc[2020,'율']
print(f"전국 여름온열률/10만: 2020 {nat.loc[2020,'율']:.2f} → 2025 {nat.loc[2025,'율']:.2f} = {natx:.2f}배")

print(f"{'key':>14}{'20건':>5}{'25건':>5}{'율20':>7}{'율25':>7}{'배수':>7}{'전국대비':>7}{'인구증감%':>8}")
for k in struct:
    s=panel[panel.key==k].set_index('연도')
    c20,c25=int(s.loc[2020,'여름온열']),int(s.loc[2025,'여름온열'])
    r20,r25=s.loc[2020,'여름온열률10만'],s.loc[2025,'여름온열률10만']
    mult=r25/max(r20,0.01); rel=mult/natx; dp=100*(s.loc[2025,'인구']/s.loc[2020,'인구']-1)
    print(f"{k:>14}{c20:>5}{c25:>5}{r20:>7.1f}{r25:>7.1f}{mult:>7.1f}{rel:>7.2f}{dp:>+8.1f}")
print("→ 전국대비<1 = 전국 트렌드보다 완만 / >1 = 더 가파름.")
print("⚠ 이 차이를 '폐산단 효과'로 해석 금지 — base 소표본(울산남구 2건·인천동구 0건)·인구감소(분모↓)·생태학 교란으로")
print("   산업 감소의 '독립적' 효과는 이 데이터로 분리 안 됨(§7 온열카운트 음성과 일치). 방향성 시사까지만.")
print("  인구는 폐산단 대부분 감소 — 온열률 상승이 '분모 축소' 때문인지도 병기(인구증감% 컬럼).")

전국 여름온열률/10만: 2020 2.01 → 2025 8.23 = 4.09배
           key  20건  25건    율20    율25     배수   전국대비   인구증감%
      충청남도 당진시    6   40    3.6   23.2    6.4   1.57    +3.8
      울산광역시 남구    2   42    0.6   13.8   22.1   5.40    -5.0
   강원특별자치도 동해시    4   10    4.4   11.6    2.6   0.64    -4.7
      인천광역시 동구    0   10    0.0   17.7 1770.6 432.43    -9.7
     부산광역시 사하구    4   11    1.3    3.8    3.0   0.73    -8.2
→ 전국대비<1 = 전국 트렌드보다 완만 / >1 = 더 가파름.
⚠ 이 차이를 '폐산단 효과'로 해석 금지 — base 소표본(울산남구 2건·인천동구 0건)·인구감소(분모↓)·생태학 교란으로
   산업 감소의 '독립적' 효과는 이 데이터로 분리 안 됨(§7 온열카운트 음성과 일치). 방향성 시사까지만.
  인구는 폐산단 대부분 감소 — 온열률 상승이 '분모 축소' 때문인지도 병기(인구증감% 컬럼).


In [16]:
panel.head(1)

,연도,key,산업용,산업용여름,주택용,주택용여름,일반용,일반용여름,제조업,제조업여름,인구,온열,여름온열,여름온열률10만
0,2020,강원특별자치도 강릉시,767.6,193.1,321.6,78.3,588.9,150.8,673.9,168.9,213321.0,8,8,3.750217


**§6-b. ★ 발생장소별 [산업용 ↔ 온열] 편상관 — (a) vs (b) 판별.**
집·길가(주변)에서도 온열이 산업과 함께 오르면 (b)열섬. 작업장에만 몰리면 (a)노동. (2026-07-19 세분화: '건물/실내' 묶음을 풀어 **실내 작업장을 단독 표시** — 실내도 강하면 (a)는 '실외'가 아니라 '작업장 노동' 전체다.)

In [17]:
from scipy import stats
wb=CalamineWorkbook.from_path('FOIA_0522_질병관리청_NEDIS온열질환_2020-2025.xlsx')
r=wb.get_sheet_by_name('DB(2020-2025)발생지역기준').to_python()
ne=pd.DataFrame(r[1:],columns=r[0]); ne['월']=pd.to_datetime(ne['발생일자'],errors='coerce').dt.month
ne['key']=[norm_key(s,g) for s,g in zip(ne['발생시도'],ne['발생시군구'])]
nes=ne[ne['월'].isin([6,7,8])]

gu=panel.groupby('key').agg(산업용여름=('산업용여름','mean'),인구=('인구','mean')).reset_index()
LOC={'실외작업장':(['실외 작업장'],'(a)노동'),'실내작업장':(['실내 작업장'],'(a)노동'),
     '논밭':(['논밭'],'(a)노동'),'건물':(['건물'],'?시간검증'),'실내 기타':(['실내 기타'],'?시간검증'),   # [수정 07-19: 건물·실내기타 분리 — 노동 여부는 아래 발생시간으로 검증(비노동 실내일 수 있음)]
     '집':(['집'],'(b)ambient'),'길가':(['길가'],'(b)ambient'),'주거지주변':(['주거지 주변'],'중간')}
for ln,(cats,_) in LOC.items():
    gu=gu.merge(nes[nes['발생장소'].isin(cats)].groupby('key').size().rename(ln),on='key',how='left')
gu=gu.fillna(0)
def pc(y):
    x,yy,c=gu['산업용여름'].values,gu[y].values.astype(float),gu['인구'].values.astype(float); n=len(gu)
    rc=lambda a,b:np.corrcoef(a,b)[0,1]; r=(rc(x,yy)-rc(x,c)*rc(yy,c))/np.sqrt((1-rc(x,c)**2)*(1-rc(yy,c)**2))
    return r,2*stats.t.sf(abs(r*np.sqrt((n-3)/(1-r**2))),n-3)
print("산업용여름 ↔ 온열 (발생장소별, 인구 통제, n=%d):"%len(gu))
for ln,(_,grp) in LOC.items():
    r,p=pc(ln)
    print("  %-8s r=%+.3f p=%.1e %-4s %s"%(ln,r,p,'***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.',grp))
# 기대: 실외작업장 +0.57***·실내작업장 +0.52*** (작업장이면 실내외 불문 강함) / 집 -0.00 n.s.·길가 +0.06 n.s.

# [추가 07-19] 건물·실내기타가 노동인가? — 발생시간으로 검증 (작업장=주간 업무시간 집중 / 집=저녁 이후↑)
nes_h=nes.copy(); nes_h['h']=pd.to_numeric(nes_h['발생시간'].astype(str).str.slice(0,2),errors='coerce')
print("\n발생장소별 발생시간 (업무시간 09-18시 비중 · 중앙시각 · n):")
for lab in ['실외 작업장','실내 작업장','논밭','건물','실내 기타','집','길가','주거지 주변']:
    hh=nes_h[nes_h['발생장소']==lab]['h'].dropna()
    if len(hh)==0: continue
    print("  %-9s 09-18시 %4.1f%% · 중앙 %2.0f시 (n=%d)"%(lab,((hh>=9)&(hh<18)).mean()*100,hh.median(),len(hh)))
print("→ 건물·실내기타 시간대가 작업장(주간 집중)과 닮으면 노동, 집(저녁↑)과 닮으면 비노동으로 판정")

산업용여름 ↔ 온열 (발생장소별, 인구 통제, n=230):
  실외작업장    r=+0.569 p=5.2e-21 ***  (a)노동
  실내작업장    r=+0.520 p=2.8e-17 ***  (a)노동
  논밭       r=+0.321 p=6.7e-07 ***  (a)노동
  건물       r=+0.257 p=8.6e-05 ***  ?시간검증
  실내 기타    r=+0.276 p=2.4e-05 ***  ?시간검증
  집        r=-0.001 p=9.9e-01 n.s. (b)ambient
  길가       r=+0.056 p=4.0e-01 n.s. (b)ambient
  주거지주변    r=+0.220 p=7.9e-04 ***  중간

발생장소별 발생시간 (업무시간 09-18시 비중 · 중앙시각 · n):
  실외 작업장    09-18시 82.8% · 중앙 14시 (n=4769)
  실내 작업장    09-18시 71.0% · 중앙 14시 (n=1101)
  논밭        09-18시 82.2% · 중앙 13시 (n=1926)
  건물        09-18시 73.7% · 중앙 14시 (n=388)
  실내 기타     09-18시 72.8% · 중앙 14시 (n=335)
  집         09-18시 52.8% · 중앙 14시 (n=827)
  길가        09-18시 86.0% · 중앙 14시 (n=1464)
  주거지 주변    09-18시 79.9% · 중앙 14시 (n=563)
→ 건물·실내기타 시간대가 작업장(주간 집중)과 닮으면 노동, 집(저녁↑)과 닮으면 비노동으로 판정


In [18]:
gu.head()

,key,산업용여름,인구,실외작업장,실내작업장,논밭,건물,실내 기타,집,길가,주거지주변
0,강원특별자치도 강릉시,206.400000,210179.000000,30.0,2.0,8.0,4.0,3.0,8.0,15.0,9.0
1,강원특별자치도 고성군,14.950000,27071.500000,1.0,0.0,1.0,0.0,0.0,0.0,1.0,2.0
2,강원특별자치도 동해시,406.666667,88788.833333,9.0,3.0,9.0,0.0,0.0,1.0,3.0,1.0
3,강원특별자치도 삼척시,258.300000,62838.500000,5.0,0.0,2.0,0.0,0.0,1.0,3.0,2.0
4,강원특별자치도 속초시,18.400000,81814.166667,4.0,0.0,1.0,9.0,1.0,0.0,3.0,1.0


**§6-c. 반론 검증 — "집·길가가 무관한 건 산단에서 멀어서 아닌가?"** (2026-07-19 사용자 지적)
NEDIS는 시군구 단위라 개별 건의 산단까지 거리는 모른다. 대신 **거주 인구가 산단에 실제로 가까운 시군구**를 골라 그 안에서 다시 검정한다: 전 집계구(108,510) 중심점 → 최근접 산단(국가·일반 912개 PDAN 점)의 거리로 시군구별 **'산단 2km내 거주 인구비(near_share)'**를 만들고, 상위⅓(근접 거주 많음)에서 집·길가 편상관을 재검정. 근접 거주 시군구에서도 집·길가가 무관하면 '거리 때문' 반론은 기각된다.
> ⚠ 산단 위치는 PDAN 대표점 — 대형 산단은 경계가 점보다 멀리 뻗으므로 near_share는 다소 과소(보수) 추정.

In [19]:
import geopandas as gpd, pyogrio
from scipy.spatial import cKDTree
oaB=pyogrio.read_dataframe('0718_2025년 집계구경계/bnd_oa_00_2025_2Q.shp')   # ~1분
popB=pd.read_csv('0718_2024년 인구총괄/2025년기준_2024년_인구총괄(총인구).csv',
    header=None,names=['y','oa','item','val'],dtype={'oa':str},encoding='cp949')
# 3. 지도 데이터(oaB)에 각 집계구의 '총인구수(pop)'를 붙여줍니다.
oaB['pop']=oaB['TOT_OA_CD'].map(popB[popB.item=='to_in_001'].set_index('oa')['val']).fillna(0.0)

# 4. 산업단지 지도 데이터를 불러온 뒤, 좌표계를 미터(m) 단위 계산이 가능한 좌표계(5179)로 맞춥니다.
PDANb=gpd.read_file('0718_DAM_PDAN/DAM_PDAN.shp',encoding='EUC-KR')
bigb=PDANb[PDANb.DANJI_TYPE.isin(['1','2'])].to_crs(5179)                     # 국가+일반 산단만

# 6. 대형 산단들의 좌표점들을 가지고 빠른 거리 검색용 '나무 구조(cKDTree)'를 만듭니다.
tree=cKDTree(np.c_[bigb.geometry.x, bigb.geometry.y])
cent=oaB.geometry.centroid

# 8. 집계구 중앙점에서 가장 가까운 산단까지의 거리를 계산하고, 2,000m(2km) 이하이면 True, 넘으면 False를 기록합니다.
distb,_=tree.query(np.c_[cent.x,cent.y])
oaB['near2k']=distb<=2000

# 집계구→시군구 key: TOT_OA_CD 앞 5자리(시도2+시군구3) ↔ 행정구역 코드표
awb=CalamineWorkbook.from_path('0718_ref_code/ref_code/1. 행정구역 코드(adm_code).xls')
ad=pd.DataFrame(awb.get_sheet_by_name('2025년 6월').to_python()[2:],columns=['sido','sido_nm','sgg','sgg_nm','emd','emd_nm'])
ad['code5']=ad['sido'].astype(str).str.replace('.0','',regex=False).str.zfill(2)+ad['sgg'].astype(str).str.zfill(3)
m5={c:norm_key(s,g) for c,s,g in zip(ad['code5'],ad['sido_nm'],ad['sgg_nm'])}

# 행정구역 코드표를 이용해 집계구 코드 앞 5자리(시군구)를 'key'(예: 서울 종로구)로 매핑
oaB['key']=oaB['TOT_OA_CD'].str[:5].map(m5)

# 13. 시군구별로 (산단 2km 내 사는 인구) / (전체 인구) 비율을 계산해 'near_share'라고 이름 붙입니다.
shareB=oaB.groupby('key').apply(lambda d:d.loc[d.near2k,'pop'].sum()/max(d['pop'].sum(),1)).rename('near_share').reset_index()

# 14. 기존 시군구 분석 데이터(gu)에 위에서 구한 인구 비율(near_share)을 합칩니다.
gu2=gu.merge(shareB,on='key',how='left').dropna(subset=['near_share'])

# 15. 비율 상위 33%(1/3) 시군구는 hi 그룹, 하위 33%(1/3) 시군구는 lo 그룹으로 나눕니다.
hi=gu2[gu2['near_share']>=gu2['near_share'].quantile(2/3)]; lo=gu2[gu2['near_share']<=gu2['near_share'].quantile(1/3)]

# 전국 시군구를 산단 근접 거주 비율 순서대로 줄 세웠을 때, 정확히 한가운데(50% 지점)에 있는 시군구는 
# 전체 주민 중 5.3%가 산단 2km 이내에 살고 있다는 뜻입니다.
print(f"near_share: 중위 {gu2.near_share.median()*100:.1f}% | 상위⅓ 평균 {hi.near_share.mean()*100:.1f}%(n={len(hi)}) | 하위⅓ {lo.near_share.mean()*100:.1f}%(n={len(lo)})")
def pcs(df,y):
    x,yy,c=df['산업용여름'].values,df[y].values.astype(float),df['인구'].values.astype(float); n=len(df)
    rc=lambda a,b:np.corrcoef(a,b)[0,1]; r=(rc(x,yy)-rc(x,c)*rc(yy,c))/np.sqrt((1-rc(x,c)**2)*(1-rc(yy,c)**2))
    return r,2*stats.t.sf(abs(r*np.sqrt((n-3)/(1-r**2))),n-3)

for grp,nm in [(hi,'산단근접 거주多(상위⅓)'),(lo,'산단근접 거주少(하위⅓)')]:
    for c in ['집','길가','실내작업장','실외작업장','논밭','건물','실내 기타']:   # [수정 07-19: 7개 전체 — 건물/실내기타 분리, §6-b와 일치]
        r_,p_=pcs(grp,c)
        print(f"  [{nm}] {c:6} r={r_:+.3f} p={p_:.1e} {'***' if p_<.001 else '**' if p_<.01 else '*' if p_<.05 else 'n.s.'}")
# 기대: 상위⅓(평균 33%가 산단 2km내 거주)에서도 집·길가 n.s. + 실외작업장 *** 유지 → 거리 반론 기각

near_share: 중위 5.5% | 상위⅓ 평균 32.7%(n=76) | 하위⅓ 0.0%(n=76)
  [산단근접 거주多(상위⅓)] 집      r=-0.040 p=7.4e-01 n.s.
  [산단근접 거주多(상위⅓)] 길가     r=-0.078 p=5.1e-01 n.s.
  [산단근접 거주多(상위⅓)] 실내작업장  r=+0.368 p=1.1e-03 **
  [산단근접 거주多(상위⅓)] 실외작업장  r=+0.537 p=6.9e-07 ***
  [산단근접 거주多(상위⅓)] 논밭     r=+0.527 p=1.2e-06 ***
  [산단근접 거주多(상위⅓)] 건물     r=+0.179 p=1.2e-01 n.s.
  [산단근접 거주多(상위⅓)] 실내 기타  r=+0.242 p=3.7e-02 *
  [산단근접 거주少(하위⅓)] 집      r=+0.052 p=6.6e-01 n.s.
  [산단근접 거주少(하위⅓)] 길가     r=-0.106 p=3.7e-01 n.s.
  [산단근접 거주少(하위⅓)] 실내작업장  r=+0.457 p=3.8e-05 ***
  [산단근접 거주少(하위⅓)] 실외작업장  r=+0.354 p=1.8e-03 **
  [산단근접 거주少(하위⅓)] 논밭     r=+0.170 p=1.4e-01 n.s.
  [산단근접 거주少(하위⅓)] 건물     r=+0.066 p=5.7e-01 n.s.
  [산단근접 거주少(하위⅓)] 실내 기타  r=+0.154 p=1.9e-01 n.s.


C:\Users\82104\AppData\Local\Temp\ipykernel_13164\2819755745.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  shareB=oaB.groupby('key').apply(lambda d:d.loc[d.near2k,'pop'].sum()/max(d['pop'].sum(),1)).rename('near_share').reset_index()


**§6-c2. near_share를 폴리곤 경계 기준으로 재검정** `[🔖 2026-07-24 사용자 지적]`

§6-c는 산단 위치를 **PDAN 대표점**으로 잡아 "산단 2km내 거주 인구비(near_share)"를 만들었다. 대형 산단은 경계가 점에서 수 km 뻗으므로 근접도가 **과소** 추정된다("점이라 근접이 덜 잡혀서 집·길가가 null인 것 아니냐"는 반론 여지).

**YUCH 폴리곤 경계까지의 거리**로 near_share를 다시 만들고(폴리곤 없는 산단만 점 폴백), 상·하위⅓ 편상관을 재검정한다. 근접도가 커졌는데도 집·길가가 무관하면 §6 결론(작업장 노동 경로)이 더 단단해진다.

In [20]:
# ── §6-c2. 폴리곤 경계 기준 near_share 재검정 [🔖 2026-07-24 신규] ──
from scipy import stats as _st6
import geopandas as _gpd6

# -----------------------------------------------------------------------------
# (1) 산단 형상 준비: YUCH(유치지역/면적 폴리곤) 우선 적용 + 없는 산단은 PDAN(점)으로 폴백
# -----------------------------------------------------------------------------
# ⚠ 이 셀은 §7(YUCH 로드)보다 앞에 있으므로 셀프 컨테인드(self-contained) 형태로 직접 파일 읽기 실행
_YUCH6=_gpd6.read_file('0718_DAM_YUCH/DAM_YUCH.shp',encoding='EUC-KR')

# 국가(1) 및 일반(2) 산업단지 필터링 후 한국 표준 투영좌표계(EPSG:5179, 미터 단위)로 변환
_big6=PDANb[PDANb.DANJI_TYPE.isin(['1','2'])].to_crs(5179)
_yuch6=_YUCH6.to_crs(5179)

# 면적(Polygon) 데이터가 존재해 있는 산단 ID 목록 추출
_have6=set(_yuch6.DAN_ID.unique())

# 1) 면적 데이터(Polygon)가 있는 산단: 해당 경계(Geometry) 추출하여 리스트로 저장
_polys6=[g for _,g in _yuch6[_yuch6.DAN_ID.isin(_big6.DAN_ID)].geometry.items()]

# 2) 면적 데이터가 없는 산단: 기존 점(Point) 위치 좌표를 대체(Fallback) 자원으로 활용
_pts6=[r.geometry for _,r in _big6.iterrows() if r.DAN_ID not in _have6]

print(f'산단 형상: YUCH 폴리곤 {len(_polys6):,}조각 · 점 폴백 {len(_pts6):,}개')


# -----------------------------------------------------------------------------
# (2) 집계구 중심점(Centroid) 기준 가장 가까운 (폴리곤 경계 또는 점) 거리 계산
# -----------------------------------------------------------------------------
# 집계구 면적의 중심점(Centroid) 좌표 추출 및 GeoDataFrame 생성 (EPSG:5179)
_cd6=_gpd6.GeoDataFrame(geometry=oaB.geometry.centroid.values,crs=5179)

# A. 중심점 ~ 산업단지 '폴리곤 경계' 간 최단 거리 결합 (sjoin_nearest 사용)
_jp6=_gpd6.sjoin_nearest(_cd6,_gpd6.GeoDataFrame(geometry=_polys6,crs=5179),how='left',distance_col='d_poly')
_jp6=_jp6[~_jp6.index.duplicated()]

# B. 중심점 ~ 산업단지 '점 좌표' 간 최단 거리 결합
_jt6=_gpd6.sjoin_nearest(_cd6,_gpd6.GeoDataFrame(geometry=_pts6,crs=5179),how='left',distance_col='d_pt')
_jt6=_jt6[~_jt6.index.duplicated()]

# C. 두 거리 중 더 가까운 값을 최종 최단 거리로 선택 (결측치는 1e9 즉 1,000,000km로 대체하여 무시)
_d6=np.minimum(_jp6['d_poly'].fillna(1e9).values, _jt6['d_pt'].fillna(1e9).values)

# D. 최종 최단 거리가 2,000m(2km) 이내인지 여부 판단 (True/False 플래그 부여)
oaB['near2k_poly']=_d6<=2000
print(f'2km내 집계구: 점 기준 {int(oaB.near2k.sum()):,} → 폴리곤 기준 {int(oaB.near2k_poly.sum()):,}')

# -----------------------------------------------------------------------------
# (3) 시군구 단위로 산단 2km 근접 인구 비율(near_share) 재계산 및 편상관 분석 수행
# -----------------------------------------------------------------------------
# 시군구(key)별로 2km 이내 영역에 거주하는 인구 비율 재집계
_shareP6=oaB.groupby('key').apply(
    lambda x: x.loc[x.near2k_poly,'pop'].sum()/max(x['pop'].sum(),1)
    ).rename('near_share_poly').reset_index()

# 기존 시군구 데이터(gu)와 재계산된 근접 인구 비율 병합
_gu6=gu.merge(_shareP6,on='key',how='left').dropna(subset=['near_share_poly'])

# 산단 근접 인구 비율 기준 상위 1/3 (33%) 그룹 및 하위 1/3 (33%) 그룹 분할
_hi6=_gu6[_gu6['near_share_poly']>=_gu6['near_share_poly'].quantile(2/3)]
_lo6=_gu6[_gu6['near_share_poly']<=_gu6['near_share_poly'].quantile(1/3)]

# 근접 비율 변화 결과 요약 출력
print(f'near_share(폴리곤): 중위 {_gu6.near_share_poly.median()*100:.1f}% | 상위⅓ 평균 {_hi6.near_share_poly.mean()*100:.1f}%(n={len(_hi6)}) | 하위⅓ {_lo6.near_share_poly.mean()*100:.1f}%')
print(f'  (기존 점 기준: 중위 {gu2.near_share.median()*100:.1f}% · 상위⅓ {hi.near_share.mean()*100:.1f}% — 폴리곤 전환으로 근접도가 제대로 잡힘)')
print()

# 발생장소별 편상관 분석 결과 테이블 출력 헤더
print(f"{'발생장소':12}{'상위⅓ r':>10}{'':>5}{'하위⅓ r':>10}{'':>5}  (인구 통제 편상관)")

_PL6=['실외작업장','실내작업장','논밭','집','길가','주거지주변']

# 각 장소 요인별로 인구 통제 편상관(Partial Correlation) 및 유의수준(p-value) 계산
for _lab in _PL6:
    if _lab not in _gu6.columns:
        continue

    # pcs(): 편상관분석(Partial Correlation)을 계산하는 사용자 정의/외부 함수
    _rh,_ph=pcs(_hi6,_lab); _rl,_pl=pcs(_lo6,_lab)

    # 유의수준 p-value 표기 별표(*) 기호화 (p<.001: ***, p<.01: **, p<.05: *, 이상: n.s.)
    _sh='***' if _ph<.001 else '**' if _ph<.01 else '*' if _ph<.05 else 'n.s.'
    _sl='***' if _pl<.001 else '**' if _pl<.01 else '*' if _pl<.05 else 'n.s.'

    print(f"{_lab:12}{_rh:>+10.3f}{_sh:>5}{_rl:>+10.3f}{_sl:>5}")

print()

# -----------------------------------------------------------------------------
# (4) 검증 결과 해석 및 결론 도출 출력
# -----------------------------------------------------------------------------
print('→ ★ 근접도가 점 기준보다 크게 올랐는데도(상위⅓ 33%→45%) 집·길가는 여전히 무관(n.s.).')
print('  "점이라 근접이 덜 잡혀 null이 나온 것"이라는 반론이 기각된다 — §6 결론(작업장 노동 경로)이 더 단단해짐.')
print('  실외작업장은 근접에서 더 강해짐(+0.52 vs +0.34) = 국소 미세 열섬 여지와 양립(§6 결론 md 참조).')

c:\Users\82104\AppData\Local\Programs\Python\Python310\lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: 0718_DAM_YUCH/DAM_YUCH.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


산단 형상: YUCH 폴리곤 13,460조각 · 점 폴백 201개
2km내 집계구: 점 기준 17,365 → 폴리곤 기준 24,667
near_share(폴리곤): 중위 10.2% | 상위⅓ 평균 45.2%(n=76) | 하위⅓ 0.1%
  (기존 점 기준: 중위 5.5% · 상위⅓ 32.7% — 폴리곤 전환으로 근접도가 제대로 잡힘)

발생장소             상위⅓ r          하위⅓ r       (인구 통제 편상관)
실외작업장           +0.517  ***    +0.342   **
실내작업장           +0.387  ***    +0.479  ***
논밭              +0.470  ***    +0.148 n.s.
집               -0.137 n.s.    +0.052 n.s.
길가              -0.125 n.s.    -0.096 n.s.
주거지주변           +0.099 n.s.    -0.131 n.s.

→ ★ 근접도가 점 기준보다 크게 올랐는데도(상위⅓ 33%→45%) 집·길가는 여전히 무관(n.s.).
  "점이라 근접이 덜 잡혀 null이 나온 것"이라는 반론이 기각된다 — §6 결론(작업장 노동 경로)이 더 단단해짐.
  실외작업장은 근접에서 더 강해짐(+0.52 vs +0.34) = 국소 미세 열섬 여지와 양립(§6 결론 md 참조).


C:\Users\82104\AppData\Local\Temp\ipykernel_13164\2599662786.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  _shareP6=oaB.groupby('key').apply(


**§6-c3. 논밭 상관은 '농지 면적 구성효과'인가** `[🔖 2026-07-26 사용자 자료 확보]`

§6-c2에서 논밭이 근접 상위⅓에만 유의했다(+0.470 *** vs 하위⅓ +0.148 n.s.). 그때 이렇게 적었다:

> 이 회귀는 인구만 통제할 뿐 **농지 면적을 통제하지 못한다**. 큰 산단을 낀 시군구가 농지도 넓으면 산업용 전력과 논밭 환자 수가 함께 커지는 구성 효과만으로도 같은 상관이 나온다. 따라서 판별 근거로 쓰지 않는다.

통계청 **「시군별 논밭별 경지면적」**(2024)을 통제변수로 넣어 그 대안 설명을 직접 검정한다.

> ⚠ 자료 한계: 광역시 자치구가 `기장군외`처럼 묶여 개별 값이 없다. 매칭되는 시·군만 표본으로 쓰고, **같은 표본 안에서 통제 전/후를 비교**한다(표본이 바뀌어 생기는 착시를 막으려고).

In [ ]:
# ── §6-c3. 논밭 상관은 '농지 면적 구성효과'인가 — 경지면적 통제 [🔖 2026-07-26 사용자 자료] ──
# [왜] §6-c2에서 논밭이 근접 상위⅓에만 유의(+0.470 *** vs 하위⅓ +0.148 n.s.)했다. 그런데 이 회귀는
#   인구만 통제할 뿐 **농지 면적을 통제하지 못한다** — 큰 산단을 낀 시군구가 농지도 넓으면
#   산업용 전력과 논밭 환자 수가 함께 커지는 **구성 효과**만으로 같은 상관이 나온다.
#   → 통계청 「시군별 논밭별 경지면적」을 통제변수로 넣어 상관이 살아남는지 본다.
# ⚠ 자료 한계: 광역시 자치구가 '기장군외'처럼 묶여 있어 개별 값이 없다. 매칭되는 시·군만 표본으로 쓰고,
#   **같은 표본 안에서 통제 전/후를 비교**한다(표본이 바뀌어 생기는 착시를 막으려고).

from python_calamine import CalamineWorkbook as _CW6c
from scipy import stats as _st6c

# -----------------------------------------------------------------------------
# 1. 외부 데이터(시군별 경지면적 엑셀) 로드 및 파싱
# -----------------------------------------------------------------------------
# Calamine 라이브러리를 사용하여 엑셀 파일을 고속으로 읽어옴
_r6c=_CW6c.from_path('시군별_논밭별_경지면적_20260726231514.xlsx').get_sheet_by_name('데이터').to_python()

# 헤더 정보 파싱 (연도 및 열 라벨 추출)
_yr6=[str(c).split('.')[0] for c in _r6c[0]]; # 첫 번째 행에서 연도 추출 (예: '2024')
_lb6=[str(c) for c in _r6c[1]]                # 두 번째 행에서 구분 라벨 추출 (예: '경지면적 계', '논', '밭')

# 최신 연도인 2024년에 해당하는 열 인덱스 추출
_i24=[i for i,y in enumerate(_yr6) if y=='2024']        

# 2024년 데이터의 [경지면적 계, 논, 밭] 열 위치 정밀 지정
_c계,_c논,_c밭=[_i24[_lb6[i].find('논')>=0 or 0] if False else i for i in _i24][:3]

_rows6=[]; 
_sd6='' # 시·도(시도) 이름을 유지하기 위한 누적 변수

# 데이터 행 정제 (시도명 갱신 및 불필요한 소계/합산행 제외)
for _row in _r6c[2:]:
    _a,_b=str(_row[0]).strip(),str(_row[1]).strip()
    if _a:
        _sd6=_a # 광역 지자체(시·도)명이 나오면 업데이트

    # 시·도 소계 및 광역시 자치구가 '기장군외' 형태로 합산된 행은 개별 시군구 매칭이 불가능하므로 제외
    if _b in ('소계','') or _b.endswith('외'):            
        continue

    # 부동소수점(float) 변환 및 예외 처리 함수
    def _num(v):
        try:
            return float(str(v).replace(',',''))
        except Exception:
            return np.nan
        
    # 정제된 행 데이터를 리스트에 저장 (지역 식별 키, 경지계, 논, 밭 면적)
    _rows6.append(dict(key=norm_key(_sd6,_b),경지계=_num(_row[_c계]),논=_num(_row[_c논]),밭=_num(_row[_c밭])))

# DataFrame 생성 및 경지면적 결측치(NaN) 제거
_F6=pd.DataFrame(_rows6).dropna(subset=['경지계'])
print(f'경지면적 2024 로드: {len(_F6)}개 시군구 (단위 ha) · 광역시 자치구는 "XX외"로 묶여 제외됨')

# -----------------------------------------------------------------------------
# 2. 기존 연구 표본(_gu6)과 경지면적 데이터 병합 및 표본 교집합 확보
# -----------------------------------------------------------------------------
_G6=_gu6.merge(_F6,on='key',how='inner')
print(f'§6-c2 표본 {len(_gu6)} → 경지면적 매칭 {len(_G6)}개 ({len(_G6)/len(_gu6)*100:.0f}%)')
print(f'  매칭 실패 {len(_gu6)-len(_G6)}개는 대부분 광역시 자치구(논밭 환자가 원래 적은 곳)')

# -----------------------------------------------------------------------------
# 3. 편상관분석(Partial Correlation) 함수 정의
# -----------------------------------------------------------------------------
def _pc6(df,y,ctrls):
    """
    다중 통제 편상관(Partial Correlation) 계산 함수.
    
    통제변수들(ctrls)의 영향을 OLS 회귀 분석을 통해 잔차화(Residualization)한 후,
    잔차(Residual) 간의 피어슨 상관계수를 구하여 외생 변수의 효과를 통제함.
    """

    # 통제변수 행렬 X 구성 (절편항 포함)
    _X=np.column_stack([np.ones(len(df))]+[df[c].values.astype(float) for c in ctrls])

    # OLS 최소제곱법을 통한 잔차 추출 함수
    def _res(v):
        _b,*_=np.linalg.lstsq(_X,v,rcond=None)
        return v-_X@_b

    # 종속/독립 변수에서 통제변수 영향 제거 (잔차화)
    _rx=_res(df['산업용여름'].values.astype(float)); _ry=_res(df[y].values.astype(float))

    # 잔차 간 피어슨 상관계수 산출
    _r=float(np.corrcoef(_rx,_ry)[0,1]); _n=len(df); _dfree=_n-len(ctrls)-2

    # t-통계량 및 p-value 계산
    _t=_r*np.sqrt(_dfree/max(1-_r**2,1e-12))
    return _r,float(2*_st6c.t.sf(abs(_t),_dfree)),_n

# -----------------------------------------------------------------------------
# 4. 분석 그룹 분할 (산단 근접도 상위 1/3 그룹 vs 하위 1/3 그룹)
# -----------------------------------------------------------------------------
_hi6c=_G6[_G6['near_share_poly']>=_G6['near_share_poly'].quantile(2/3)]
_lo6c=_G6[_G6['near_share_poly']<=_G6['near_share_poly'].quantile(1/3)]

# 유의수준 표기 람다 함수
_sig=lambda p:'***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'

# -----------------------------------------------------------------------------
# 5. [메인 검정] 통제변수 변경에 따른 편상관 변화 비교 (구성효과 검정)
# -----------------------------------------------------------------------------
print()
print('논밭 온열 ~ 산업용여름 · 통제변수를 늘려가며 (근접 상위⅓ / 하위⅓)')
print(f"  {'통제변수':28}{'상위⅓ r':>10}{'':>5}{'하위⅓ r':>10}{'':>5}")

# 통제변수 조건 단계별 테스트
for _lab,_ct in [('인구만 (§6-c2와 동일)',['인구']),
                 ('인구 + 논 면적',['인구','논']),
                 ('인구 + 경지 전체(논+밭)',['인구','경지계'])]:
    _rh,_ph,_nh=_pc6(_hi6c,'논밭',_ct); _rl,_pl,_nl=_pc6(_lo6c,'논밭',_ct)
    print(f"  {_lab:28}{_rh:>+10.3f}{_sig(_ph):>5}{_rl:>+10.3f}{_sig(_pl):>5}")
print(f"  (상위⅓ n={len(_hi6c)} · 하위⅓ n={len(_lo6c)})")

# -----------------------------------------------------------------------------
# 6. 대조군 분석 (동일 조건에서 다른 온열질환 발생 장소들 비교)
# -----------------------------------------------------------------------------
print()
print('대조 — 같은 표본·같은 통제(인구+경지계)에서 다른 발생장소')
print(f"  {'발생장소':12}{'상위⅓ r':>10}{'':>5}{'하위⅓ r':>10}{'':>5}")
for _pl6 in ['실외작업장','실내작업장','집','길가']:
    if _pl6 not in _G6.columns:
        continue
    _rh,_ph,_=_pc6(_hi6c,_pl6,['인구','경지계']); _rl,_pl_,_=_pc6(_lo6c,_pl6,['인구','경지계'])
    print(f"  {_pl6:12}{_rh:>+10.3f}{_sig(_ph):>5}{_rl:>+10.3f}{_sig(_pl_):>5}")

# -----------------------------------------------------------------------------
# 7. 메인 검정 결과 요약 및 결론 출력
# -----------------------------------------------------------------------------
_r0,_p0,_=_pc6(_hi6c,'논밭',['인구'])
_r1,_p1,_=_pc6(_hi6c,'논밭',['인구','경지계'])
print()
if _p1<0.05 and _r1>0:
    print(f'→ **논밭 상관이 경지면적 통제 후에도 살아남는다** ({_r0:+.3f} → {_r1:+.3f}, p={_p1:.1e}).')
    print('  "농지가 넓어서 생긴 구성효과"라는 대안 설명이 기각된다 — 다만 계수는 40%가량 줄었다.')
    print('  ⚠ 그렇다고 바로 판별 근거로 승격하지는 않는다. 위 대조표를 함께 읽어야 한다.')
else:
    print(f'→ **논밭 상관이 경지면적 통제로 사라진다** ({_r0:+.3f} → {_r1:+.3f}, p={_p1:.1e}).')
    print('  구성효과였다는 뜻 — §6-c2의 "판별 근거로 쓰지 않는다"가 옳았다. 그대로 유지한다.')

# -----------------------------------------------------------------------------
# 8. [강건성 검정] 대조표에서 검출된 이상 신호('하위⅓ 집'의 유의성) 검증
# -----------------------------------------------------------------------------
# [🔖 2026-07-26 2차 — 사용자 재확인 요청] 대조표의 '하위⅓ 집 유의'를 강건성 검정에 걸어봤다.
#   결과: **허상이다.** 원자료 카운트 Pearson이라는 특정 설정에서만 나오고, 어느 검정에도 못 버틴다.
print()
print('★ 강건성 검정 — 대조표의 "하위⅓ 집 +0.33"이 진짜인가')

# ① 순위 변환 데이터 세트 (Spearman 상관계수 검정용)
_lo_r=_lo6c.copy()
for _c in ['산업용여름','집','인구','경지계']:
    _lo_r[_c]=_lo_r[_c].rank()

# ② 로그 변환 데이터 세트 (극단값 영향을 줄이기 위한 log1p 변환)
_lo_g=_lo6c.copy()
for _c in ['산업용여름','집','인구','경지계']:
    _lo_g[_c]=np.log1p(_lo_g[_c].astype(float))

# ③ 비율 데이터 세트 (인구당 전력 사용량 및 인구당 집 온열질환 발생률)
_lo_t=_lo6c.copy()
_lo_t['집']=_lo_t['집']/_lo_t['인구']*1e5; _lo_t['산업용여름']=_lo_t['산업용여름']/_lo_t['인구']

# ④ 최대 영향점(Outlier / Leverage Point) 탐색 및 제거
_X=np.column_stack([np.ones(len(_lo6c)),_lo6c['인구'].values,_lo6c['경지계'].values])
_rs=lambda v: v-_X@np.linalg.lstsq(_X,v,rcond=None)[0]
# 공분산에 가장 강하게 기여하는 아웃라이어 인덱스 추출
_ix=int(np.argmax(np.abs((_rs(_lo6c['산업용여름'].values.astype(float))-0)*(_rs(_lo6c['집'].values.astype(float))-0))))
_drop=_lo6c.iloc[_ix]

# 조건별 강건성 분석 실행
for _lab,_df,_ct in [('① 인구만 통제(Pearson)',_lo6c,['인구']),
                     ('② 인구+경지 통제(Pearson)',_lo6c,['인구','경지계']),
                     ('③ 순위 변환(Spearman류)',_lo_r,['인구','경지계']),
                     ('④ log1p 변환',_lo_g,['인구','경지계']),
                     ('⑤ 1인당 전력 vs 10만명당 집온열',_lo_t,['경지계']),
                     (f'⑥ 최대 영향점({_drop.key}) 제외',_lo6c.drop(_lo6c.index[_ix]),['인구','경지계'])]:
    _r,_p,_=_pc6(_df,'집',_ct)
    print(f'  {_lab:32}r={_r:+.3f} p={_p:.3f} {_sig(_p)}')
_rA,_pA,_=_pc6(_G6,'집',['인구','경지계'])
print(f'  {"[대조] 매칭표본 전체(158)":32}r={_rA:+.3f} p={_pA:.3f} {_sig(_pA)}')
print()
print(f'→ **허상이다.** ①②(원자료 카운트 Pearson)에서만 유의하고 ③④⑤⑥ 전부 탈락한다.')
print(f'  범인은 {_drop.key} — 산업용여름 {_drop.산업용여름:,.0f}GWh로 이 그룹 최대(2위의 10배)인데 집 온열은 {int(_drop.집)}건뿐이다.')
# [🔖 2026-07-26 감사로 정정] 위 원인 진단을 재봤더니 절반만 맞았다(감사 스크립트 audit2b).
#   · 여수 near_share 0.0219 → §6-c2의 228개 표본에서는 **중위**다. 포항(0.1826)도 중위.
#     즉 §6-c2의 3분위 분류는 YUCH 결측 때문에 망가지지 않았다 — 상·하위 검정에 아예 안 들어간다.
#   · 여수가 "하위⅓"이 된 건 **여기 §6-c3에서 표본이 158로 줄며(광역시 구 70개 탈락)
#     분위 경계가 이동해서**다. 원인은 YUCH 결측이 아니라 표본 축소 쪽이 직접적이다.
#   · YUCH 결측 산단을 품고도 하위⅓인 시군구는 전 표본에서 **진도군 1개뿐**이고(산단 1개·5.3GWh),
#     그 1개를 빼도 §6-c2 결과는 소수점 셋째 자리까지 같다.
print('  왜 이 시가 여기서 "먼 하위⅓"에 들어갔나: **표본이 시·군으로 줄며 분위 경계가 이동**해서다.')
print('  (§6-c2의 228개 표본에서 여수는 중위였다 — 거기선 상·하위 검정에 들어가지도 않는다.)')
print('  여수의 near_share 자체는 과소 측정된 게 맞다 — 대장 51.2km²인 여수국가산단의 YUCH가 결측이라')
print('  점 폴백으로 재기 때문이다. 다만 그게 §6-c2 결론을 흔들지는 않았다(중위라서).')
print('  → §6 결론("집·길가 무관")은 그대로 유지된다. 앞선 "집이 하위⅓에서 유의" 서술은 철회한다.')
print()
print('⚠ 한계: 경지면적은 시군구 총량이라 **산단 근처의 농지**를 따로 보지 못한다. 또 광역시 자치구가')
print('  빠져 표본이 시·군에 치우친다(도시 쪽 대조가 약해짐). 결론은 이 표본 범위 안에서만 유효.')


경지면적 2024 로드: 159개 시군구 (단위 ha) · 광역시 자치구는 "XX외"로 묶여 제외됨
§6-c2 표본 228 → 경지면적 매칭 158개 (69%)
  매칭 실패 70개는 대부분 광역시 자치구(논밭 환자가 원래 적은 곳)

논밭 온열 ~ 산업용여름 · 통제변수를 늘려가며 (근접 상위⅓ / 하위⅓)
  통제변수                             상위⅓ r          하위⅓ r     
  인구만 (§6-c2와 동일)                 +0.531  ***    +0.049 n.s.
  인구 + 논 면적                       +0.365   **    +0.076 n.s.
  인구 + 경지 전체(논+밭)                 +0.332    *    +0.143 n.s.
  (상위⅓ n=53 · 하위⅓ n=53)

대조 — 같은 표본·같은 통제(인구+경지계)에서 다른 발생장소
  발생장소             상위⅓ r          하위⅓ r     
  실외작업장           +0.473  ***    +0.629  ***
  실내작업장           +0.353    *    +0.589  ***
  집               -0.157 n.s.    +0.326    *
  길가              -0.275 n.s.    +0.203 n.s.

→ **논밭 상관이 경지면적 통제 후에도 살아남는다** (+0.531 → +0.332, p=1.7e-02).
  "농지가 넓어서 생긴 구성효과"라는 대안 설명이 기각된다 — 다만 계수는 40%가량 줄었다.
  ⚠ 그렇다고 바로 판별 근거로 승격하지는 않는다. 위 대조표를 함께 읽어야 한다.

★ 강건성 검정 — 대조표의 "하위⅓ 집 +0.33"이 진짜인가
  ① 인구만 통제(Pearson)               r=+0.287 p=0.039 *
  ② 인구+경지 통제(Pearson)             r=+0.326 

**§6-c5. §6-c3에 대한 검증 질문 4건** `[🔖 2026-07-27 사용자 질문]`

§6-c3을 보고 나온 질문 넷을 **재서** 답한다.

1. **"광역시 자치구는 논밭 환자가 원래 적은 곳"** — 내가 §6-c3에서 재지 않고 쓴 문장이다. 단정할 수 있나?
2. **왜 실외·실내작업장은 하위⅓(산단에서 먼 시군)에서 더 강한가?**
3. **왜 표본이 시·군으로 줄었나?**
4. **강건성 검정 ①~⑥이 정확히 뭘 한 건가?**

In [ ]:
# ── §6-c5. §6-c3에 대한 사용자 검증 질문 4건 — 재서 답한다 [🔖 2026-07-27] ──
# 질문: ① "광역시 자치구는 논밭 환자가 원래 적은 곳"이라고 단정할 수 있나(내가 재지 않고 쓴 문장)
#       ② 왜 실외·실내작업장은 **하위⅓**에서 더 강한가  ③ 표본이 왜 시·군으로 줄었나
#       ④ 강건성 검정 ①~⑥이 정확히 뭘 한 건가
_sig5=lambda p:'***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'

# ── ① "광역시 자치구는 논밭 환자가 원래 적다" — 단정 검정 ──
_keep5=set(_G6['key']); _drop5=_gu6[~_gu6['key'].isin(_keep5)].copy(); _kept5=_gu6[_gu6['key'].isin(_keep5)].copy()
_ismetro=lambda k:any(k.startswith(m) for m in ['서울','부산','대구','인천','광주','대전','울산','세종'])
_drop5['광역시']= [_ismetro(k) for k in _drop5['key']]
print('① "광역시 자치구는 논밭 환자가 원래 적은 곳" — 내가 재지 않고 쓴 문장을 잰다')
print(f"   {'집단':22}{'n':>5}{'논밭 온열 평균':>13}{'중앙값':>8}{'0건 비율':>9}{'논밭/전체':>10}")
for _lb,_s in [('탈락(경지자료 없음)',_drop5),('  └ 광역시 자치구',_drop5[_drop5['광역시']]),
               ('  └ 그 외(시·군 등)',_drop5[~_drop5['광역시']]),('잔류(§6-c3 표본)',_kept5)]:
    if not len(_s): continue
    _tot=_s[[c for c in ['논밭','실외작업장','실내작업장','집','길가'] if c in _s.columns]].sum(axis=1)
    print(f"   {_lb:22}{len(_s):>5}{_s['논밭'].mean():>13.1f}{_s['논밭'].median():>8.1f}"
          f"{(_s['논밭']==0).mean()*100:>8.0f}%{(_s['논밭'].sum()/max(_tot.sum(),1))*100:>9.1f}%")
print(f'   → 탈락 70개 중 광역시 자치구는 {int(_drop5["광역시"].sum())}개({_drop5["광역시"].mean()*100:.0f}%).')
_mw=_drop5[_drop5['광역시']]['논밭']; _kw=_kept5['논밭']
print(f'   → **{"단정이 대체로 맞다" if _mw.mean()<_kw.mean()*0.5 else "단정을 약화한다"}**: 광역시 자치구 논밭 평균 {_mw.mean():.1f} vs 잔류 {_kw.mean():.1f}'
      f' ({_kw.mean()/max(_mw.mean(),1e-9):.1f}배 차이)')
print(f'   ⚠ 다만 "적다"와 "없다"는 다르다 — 광역시 자치구에도 논밭 환자가 합계 {int(_mw.sum())}건 있고,'
      f' 0건인 곳은 {(_mw==0).mean()*100:.0f}%뿐이다. 표현을 "상대적으로 적다"로 고친다.')

# ── ② 실외·실내작업장이 왜 하위⅓에서 더 강한가 ──
print()
print('② 작업장 상관이 왜 **하위⅓**(산단에서 먼 시군)에서 더 강한가')
_hi5=_G6[_G6['near_share_poly']>=_G6['near_share_poly'].quantile(2/3)]
_lo5=_G6[_G6['near_share_poly']<=_G6['near_share_poly'].quantile(1/3)]
print(f"   {'':16}{'상위⅓':>12}{'하위⅓':>12}   해석")
for _lb,_f in [('near_share 중앙',lambda d:d['near_share_poly'].median()),
               ('산업용여름 중앙',lambda d:d['산업용여름'].median()),
               ('산업용여름 CV',lambda d:d['산업용여름'].std()/d['산업용여름'].mean()),
               ('인구 중앙',lambda d:d['인구'].median()),
               ('실외작업장 합',lambda d:d['실외작업장'].sum()),
               ('실외작업장 0건비율',lambda d:(d['실외작업장']==0).mean())]:
    print(f'   {_lb:16}{_f(_hi5):>12,.2f}{_f(_lo5):>12,.2f}')
# 같은 강건성 6종을 실외작업장 하위⅓에 적용 — 이것도 허상인지 본다
def _rob5(df,y,ctrls):
    _X=np.column_stack([np.ones(len(df))]+[df[c].values.astype(float) for c in ctrls])
    _res=lambda v:v-_X@np.linalg.lstsq(_X,v,rcond=None)[0]
    _rx,_ry=_res(df['산업용여름'].values.astype(float)),_res(df[y].values.astype(float))
    _r=float(np.corrcoef(_rx,_ry)[0,1]); _df=len(df)-len(ctrls)-2
    return _r,float(2*_st6c.t.sf(abs(_r*np.sqrt(_df/max(1-_r**2,1e-12))),_df))
print()
print('   [강건성] 하위⅓ 실외작업장 +0.629가 여수 같은 1개 점이 끄는 허상인가')
_r5,_p5=_rob5(_lo5,'실외작업장',['인구','경지계']); print(f'     원본(인구+경지 통제)          r={_r5:+.3f} p={_p5:.3f} {_sig5(_p5)}')
_lr=_lo5.copy()
for _c in ['산업용여름','실외작업장','인구','경지계']: _lr[_c]=_lr[_c].rank()
_r5b,_p5b=_rob5(_lr,'실외작업장',['인구','경지계']); print(f'     순위 변환                   r={_r5b:+.3f} p={_p5b:.3f} {_sig5(_p5b)}')
_ll=_lo5.copy()
for _c in ['산업용여름','실외작업장','인구','경지계']: _ll[_c]=np.log1p(_ll[_c])
_r5c,_p5c=_rob5(_ll,'실외작업장',['인구','경지계']); print(f'     log1p 변환                 r={_r5c:+.3f} p={_p5c:.3f} {_sig5(_p5c)}')
_mx=_lo5.loc[_lo5['산업용여름'].idxmax()]
_r5d,_p5d=_rob5(_lo5.drop(_lo5['산업용여름'].idxmax()),'실외작업장',['인구','경지계'])
print(f'     최대 전력({_mx["key"]}) 제외   r={_r5d:+.3f} p={_p5d:.3f} {_sig5(_p5d)}')
# [🔖 2026-07-28 정정 — 사용자 지적] 판정을 조건문으로 남겨두면 읽는 사람이 대신 판정하게 된다.
#   원래 문장: "4종 전부 통과하면 실재, 일부 탈락하면 허상이다(위 결과 참조)" — 숫자는 이미 나와 있는데
#   결론을 안 냈다. 그 아래 "방향의 의미" 해석은 **상관이 실재할 때만** 성립하는데 조건 없이 붙어 있었다.
#   → 여기서 결론을 내고, 허상이면 해석 문단을 아예 출력하지 않는다.
_rob5_ps = [_p5, _p5b, _p5c, _p5d]
_rob5_nm = ['원본', '순위', 'log1p', '최대전력 제외']
_fail5 = [n for n, p in zip(_rob5_nm[1:], _rob5_ps[1:]) if p >= 0.05]   # 원본은 출발점이라 판정에서 제외
if not _fail5:
    print('   → **실재한다** — 변환·영향점 검정 3종을 모두 통과했다.')
    print('   → 방향의 의미: 작업장 온열은 **산단 근처냐**가 아니라 **산업 활동량 자체**를 따라간다는 뜻이다.')
    print('     하위⅓에서 더 강한 건 그쪽이 산업활동의 변동을 더 깨끗하게 반영하기 때문일 수 있다 —')
    print('     상위⅓은 이미 다들 산업이 많아 변동 폭이 좁다(위 CV 비교). ⚠ 이 해석은 **가설**이고 확증 아님.')
else:
    print(f'   → **허상이다** — 원자료 카운트 Pearson({_r5:+.3f})에서만 살아나고 {len(_fail5)}종 전부 탈락한다'
          f' ({", ".join(_fail5)}).')
    print(f'     끄는 점은 {_mx["key"]} — 산업용여름 {_mx["산업용여름"]:,.0f}로 이 그룹 최대인데 지렛대가 극단적이다.')
    print('     §6-c3 "하위⅓ 집"과 **정확히 같은 패턴**이고, 같은 판정 규칙(④번 블록)이 적용된다.')
    print('   ⚠ 따라서 "작업장 온열은 산업 활동량 자체를 따라간다"는 해석은 **여기서 쓰지 않는다** —')
    print('     상관이 실재할 때만 성립하는 문장이다. (이전 판에서는 조건 없이 붙어 있었다. 철회한다.)')
print('   ★ 우리 논증에 주는 함의: 작업장 상관은 **AIDC 논증에 쓰지 않는다**(§6 발생장소 규율).')
print('     작업장은 공장 안 노동이지 ambient 열섬이 아니다. 여기 결과는 그 규율을 다시 확인해줄 뿐이다.')

# ── ③ 표본이 왜 시·군으로 줄었나 ──
print()
print('③ 표본 축소의 정체 — 우리가 버린 게 아니라 **자료가 그렇게 생겼다**')
print(f'   통계청 「시군별 논밭별 경지면적」은 광역시 자치구를 개별 행으로 주지 않는다.')
print(f'   예: 부산 기장군은 "기장군외"처럼 **묶음 행**으로만 나온다 → 자치구별 경지면적을 뗄 수 없다.')
print(f'   그 결과 §6-c2 표본 {len(_gu6)} → 경지 매칭 {len(_G6)} ({len(_G6)/len(_gu6)*100:.0f}%). 탈락 {len(_drop5)}개.')
print(f'   → 우리 선택이 아니라 **자료 해상도의 한계**다. 대안은 농림축산식품부 농지원부(시군구 단위)인데')
print('     공개 범위가 좁다. 지금은 "시·군 표본 안에서만 유효"로 한계 명시하는 쪽을 택한다.')

# ── ④ 강건성 ①~⑥이 뭘 한 건가 ──
print()
print('④ 강건성 검정 ①~⑥ — 각각 무엇을 의심한 것인가')
for _n,_what,_why in [
    ('①','인구만 통제(Pearson)','§6-c2와 같은 조건. 출발점.'),
    ('②','인구+경지 통제(Pearson)','"농지가 넓어서"라는 구성효과를 뺀다.'),
    ('③','순위 변환(Spearman류)','값을 순위로 바꾼다 → **극단값 1~2개가 끄는 상관이면 여기서 죽는다**.'),
    ('④','log1p 변환','큰 값의 영향력을 압축한다. ③과 같은 의심, 다른 방식.'),
    ('⑤','1인당 전력 vs 10만명당 온열','절대량 대신 **비율**로 본다 → "큰 시군이라 둘 다 크다"를 제거.'),
    ('⑥','최대 영향점 제외','가장 센 점 1개를 빼고 다시 잰다 → **한 점 의존이면 죽는다**.')]:
    print(f'   {_n} {_what:26} {_why}')
print('   판정 규칙: ①②만 살고 ③④⑤⑥이 죽으면 **허상**(분포 꼬리 1~2개가 만든 상관).')
print('   §6-c3 "하위⅓ 집"이 정확히 그 패턴이었고(여수 1개), 그래서 철회했다.')


① "광역시 자치구는 논밭 환자가 원래 적은 곳" — 내가 재지 않고 쓴 문장을 잰다
   집단                        n     논밭 온열 평균     중앙값    0건 비율     논밭/전체
   탈락(경지자료 없음)              70          1.6     1.0      34%      4.9%
     └ 광역시 자치구              70          1.6     1.0      34%      4.9%
   잔류(§6-c3 표본)            158         11.2    10.0       4%     23.1%
   → 탈락 70개 중 광역시 자치구는 70개(100%).
   → **단정이 대체로 맞다**: 광역시 자치구 논밭 평균 1.6 vs 잔류 11.2 (6.8배 차이)
   ⚠ 다만 "적다"와 "없다"는 다르다 — 광역시 자치구에도 논밭 환자가 합계 115건 있고, 0건인 곳은 34%뿐이다. 표현을 "상대적으로 적다"로 고친다.

② 작업장 상관이 왜 **하위⅓**(산단에서 먼 시군)에서 더 강한가
                            상위⅓         하위⅓   해석
   near_share 중앙           0.39        0.00
   산업용여름 중앙              312.82       22.17
   산업용여름 CV                1.39        4.19
   인구 중앙             214,270.17   41,364.17
   실외작업장 합             1,802.00      767.00
   실외작업장 0건비율              0.00        0.02

   [강건성] 하위⅓ 실외작업장 +0.629가 여수 같은 1개 점이 끄는 허상인가
     원본(인구+경지 통제)          r=+0.629 p=0.000 ***
     순위 변환                   r=+0.

In [ ]:
# ── §6-c6. 실내작업장도 같은 강건성 검정에 건다 [🔖 2026-07-29 사용자 요청] ──
# [왜] §6-c5 는 "하위⅓ 실외작업장 +0.629"와 §6-c3 "하위⅓ 집 +0.326"만 검정하고
#   **실내작업장(하위⅓ +0.589)은 대조표에 값만 두고 검정하지 않았다.**
#   실외와 같은 패턴일 개연이 높지만 재보지 않은 것을 재봤다고 할 수 없다. 같은 4종을 건다.
print('§6-c6. 하위⅓ 실내작업장 +0.589 — 실재인가 허상인가')
print(f"  {'검정':28}{'r':>9}{'p':>9}")
_r6i, _p6i = _rob5(_lo5, '실내작업장', ['인구', '경지계'])
print(f"  {'원본(인구+경지 통제)':28}{_r6i:>+9.3f}{_p6i:>9.3f} {_sig5(_p6i)}")
_li = _lo5.copy()
for _c in ['산업용여름', '실내작업장', '인구', '경지계']:
    _li[_c] = _li[_c].rank()
_r6b, _p6b = _rob5(_li, '실내작업장', ['인구', '경지계'])
print(f"  {'순위 변환':28}{_r6b:>+9.3f}{_p6b:>9.3f} {_sig5(_p6b)}")
_lg = _lo5.copy()
for _c in ['산업용여름', '실내작업장', '인구', '경지계']:
    _lg[_c] = np.log1p(_lg[_c])
_r6c, _p6c = _rob5(_lg, '실내작업장', ['인구', '경지계'])
print(f"  {'log1p 변환':28}{_r6c:>+9.3f}{_p6c:>9.3f} {_sig5(_p6c)}")
_mx6 = _lo5.loc[_lo5['산업용여름'].idxmax()]
_r6d, _p6d = _rob5(_lo5.drop(_lo5['산업용여름'].idxmax()), '실내작업장', ['인구', '경지계'])
print(f"  {'최대 전력(' + str(_mx6['key']) + ') 제외':28}{_r6d:>+9.3f}{_p6d:>9.3f} {_sig5(_p6d)}")

_fail6 = [n for n, p in zip(['순위', 'log1p', '최대전력 제외'], [_p6b, _p6c, _p6d]) if p >= 0.05]
if not _fail6:
    print('  → **실재한다** — 변환·영향점 검정 3종을 모두 통과했다.')
else:
    print(f'  → **허상이다** — 원자료 카운트 Pearson({_r6i:+.3f})에서만 살아나고 '
          f'{len(_fail6)}종({"·".join(_fail6)})이 탈락한다.')
    print(f'    §6-c3 "하위⅓ 집"·§6-c5 "하위⅓ 실외작업장"과 **같은 패턴**이고 끄는 점도 같다.')
print('  ★ 어느 쪽이든 §6 규율은 그대로다 — 작업장 상관은 AIDC 논증에 쓰지 않는다.')
print('    작업장은 공장 안 노동이지 ambient 열섬이 아니다.')


**§6-c4. YUCH 결측이 3분위 분류를 망가뜨리나 — 감사** `[🔖 2026-07-26]`

§6-c2/§6-c3의 상·하위⅓은 `near_share`(산단 2km내 거주비)로 나눈다. 그런데 **YUCH 폴리곤이 없는 산단은 PDAN 점으로 폴백**하므로 근접도가 과소 측정된다. 큰 산단을 품은 시군구가 "먼 그룹"으로 오분류되면 §6 판별 자체가 흔들린다.

실제로 그런지 재고, 의심 시군구를 빼도 결과가 유지되는지 확인한다. (이 감사는 §6-c3에서 여수시가 하위⅓에 들어간 것을 보고 시작했다 — 원인이 YUCH인지 표본 축소인지 가리려고.)

In [22]:
# ── §6-c4. YUCH 결측이 3분위 분류를 망가뜨리나 — 감사 [🔖 2026-07-26] ──
# [왜] §6-c2/§6-c3의 상·하위⅓은 near_share(산단 근접 거주비)로 나눈다. 그런데 YUCH 폴리곤이 없는
#   산단은 PDAN 점으로 폴백하므로 근접도가 **과소** 측정된다. 큰 산단을 품은 시군구가 "먼 그룹"으로
#   오분류되면 §6 판별 자체가 흔들린다. 실제로 그런지 재고, 의심 시군구를 빼고 결과가 바뀌는지 본다.
_big6c=PDANb[PDANb.DANJI_TYPE.isin(['1','2'])].to_crs(5179).copy()
_big6c['YUCH있음']=_big6c.DAN_ID.isin(set(_YUCH6.DAN_ID.unique()))
_sgg6c=oaB[['key','geometry']].dropna(subset=['key']).dissolve(by='key')[['geometry']].reset_index()
_jn6c=gpd.sjoin(_big6c[['DAN_ID','DAN_NAME','YUCH있음','geometry']],_sgg6c,how='left',predicate='within')
_C6c=_jn6c.dropna(subset=['key']).groupby('key').agg(산단수=('DAN_ID','size'),YUCH보유=('YUCH있음','sum'))
_C6c['결측수']=_C6c.산단수-_C6c.YUCH보유
print(f'국가·일반산단 {len(_big6c)}개 중 YUCH 보유 {int(_big6c.YUCH있음.sum())} · 결측 {int((~_big6c.YUCH있음).sum())}'
      f' ({(~_big6c.YUCH있음).mean()*100:.0f}%)')

_A6c=_gu6.merge(_C6c,on='key',how='left').fillna({'산단수':0,'YUCH보유':0,'결측수':0})
_A6c['q']=pd.qcut(_A6c['near_share_poly'].rank(method='first'),3,labels=['하위','중위','상위'])
_sus6=_A6c[(_A6c.결측수>=1)&(_A6c.q=='하위')]
print(f'★ YUCH 결측 산단을 품고도 near_share 하위⅓ = 오분류 의심 **{len(_sus6)}개**')
if len(_sus6):
    print(_sus6[['key','산단수','결측수','near_share_poly','산업용여름']].to_string(index=False))
for _k6 in ['전라남도 여수시','경상북도 포항시']:
    _r6=_A6c[_A6c.key==_k6]
    if len(_r6):
        _r6=_r6.iloc[0]
        print(f'  참고 {_k6}: 산단 {int(_r6.산단수)}개(결측 {int(_r6.결측수)}) · near_share {_r6.near_share_poly:.4f} → **{_r6.q}**')

_B6c=_A6c.drop(_sus6.index)
_B6c['q2']=pd.qcut(_B6c['near_share_poly'].rank(method='first'),3,labels=['하위','중위','상위'])
print()
print(f'■ 의심 {len(_sus6)}개 제외 전/후 §6 판별 (n {len(_A6c)}→{len(_B6c)})')
print(f"  {'발생장소':12}{'전 상위⅓':>11}{'':>5}{'전 하위⅓':>11}{'':>5}{'후 상위⅓':>11}{'':>5}{'후 하위⅓':>11}{'':>5}")
for _p6 in ['실외작업장','실내작업장','논밭','집','길가']:
    if _p6 not in _A6c.columns:
        continue
    _a,_ap=pcs(_A6c[_A6c.q=='상위'],_p6); _b,_bp=pcs(_A6c[_A6c.q=='하위'],_p6)
    _c,_cp=pcs(_B6c[_B6c.q2=='상위'],_p6); _d,_dp=pcs(_B6c[_B6c.q2=='하위'],_p6)
    _s=lambda p:'***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'
    print(f"  {_p6:12}{_a:>+11.3f}{_s(_ap):>5}{_b:>+11.3f}{_s(_bp):>5}{_c:>+11.3f}{_s(_cp):>5}{_d:>+11.3f}{_s(_dp):>5}")
print()
print('→ **§6-c2의 3분위 분류는 YUCH 결측 때문에 망가지지 않았다.** 여수·포항 모두 중위라 상·하위 검정에')
print('  아예 들어가지 않고, 의심 시군구도 1개뿐이며 그마저 빼도 결과가 소수점 셋째 자리까지 같다.')
print('⚠ 다만 여수의 near_share 자체는 과소다(대장 51.2km² 여수국가산단의 YUCH 결측). §11-b·§12-b의')
print('  여수 노출 인구도 같은 이유로 과소이며, 그건 이미 "YUCH 결측 → 점 모형만"으로 표기돼 있다.')
print('⚠ §6-c3(시·군 158개 표본)에서는 표본 축소로 분위 경계가 이동해 여수가 하위⅓에 들어간다 —')
print('  그건 YUCH 문제가 아니라 표본 문제이고, §6-c3 안에서 강건성 검정으로 처리했다.')


국가·일반산단 912개 중 YUCH 보유 711 · 결측 201 (22%)
★ YUCH 결측 산단을 품고도 near_share 하위⅓ = 오분류 의심 **1개**
     key  산단수  결측수  near_share_poly  산업용여름
전라남도 진도군  1.0  1.0         0.014195    5.3
  참고 전라남도 여수시: 산단 5개(결측 2) · near_share 0.0219 → **중위**
  참고 경상북도 포항시: 산단 12개(결측 4) · near_share 0.1826 → **중위**

■ 의심 1개 제외 전/후 §6 판별 (n 228→227)
  발생장소              전 상위⅓           전 하위⅓           후 상위⅓           후 하위⅓     
  실외작업장            +0.517  ***     +0.342   **     +0.517  ***     +0.331   **
  실내작업장            +0.387  ***     +0.479  ***     +0.387  ***     +0.479  ***
  논밭               +0.470  ***     +0.148 n.s.     +0.470  ***     +0.158 n.s.
  집                -0.137 n.s.     +0.052 n.s.     -0.137 n.s.     +0.052 n.s.
  길가               -0.125 n.s.     -0.096 n.s.     -0.125 n.s.     -0.096 n.s.

→ **§6-c2의 3분위 분류는 YUCH 결측 때문에 망가지지 않았다.** 여수·포항 모두 중위라 상·하위 검정에
  아예 들어가지 않고, 의심 시군구도 1개뿐이며 그마저 빼도 결과가 소수점 셋째 자리까지 같다.
⚠ 다만 여수의 near_share 자체는 과소다(대장 51.2km² 여수국가산단의 YUCH 결측). §11-b·§12-b의
  여수 노출 인구도

In [37]:
shareB.head()

,key,near_share
0,강원특별자치도 강릉시,0.274973
1,강원특별자치도 고성군,0.000000
2,강원특별자치도 동해시,0.379443
3,강원특별자치도 삼척시,0.071005
4,강원특별자치도 속초시,0.000000


### §6 결론 — 산업↔온열은 (a)작업장 노동이지 (b)열섬이 아니다 `[사실]` (2026-07-19 명칭 정정)

발생장소별 편상관이 명확히 갈린다(인구 통제, n=229):
- **작업장 강함(실내외 불문)**: 실외작업장 **r=+0.57***** · **실내작업장 +0.52***** · 논밭 +0.32*** — 사람이 **일하는** 곳. 실내 작업장(공장 내부)이 실외 못지않게 강하므로 (a)의 정확한 이름은 ~~실외노동~~ → **작업장 노동(실내+실외)**.
- **주변 무관**: 집 **r=−0.00(n.s.)** · 길가 +0.06(n.s.) — 사람이 **사는/지나는** 곳. (주거지주변 +0.22***는 중간지대 — 텃밭·골목 노동 혼재 가능 `[해석]`.)
- **거리 반론 기각(§6-c·§6-c2)** `[🔖 2026-07-24 폴리곤 재검정으로 보강]`: "집·길가가 멀어서 무관한 것"이라면 **산단 2km내 거주가 많은 시군구(상위⅓, 평균 33% 근접 거주)**에선 상관이 나와야 하는데, 거기서도 집 r=−0.04(n.s.)·길가 −0.08(n.s.), 실외작업장 +0.54***는 유지 → 거리 탓이 아니다.

만약 (b) 산업지대 열섬(주변 전체가 더움)이라면 집·길가 온열도 산업과 함께 올라야 하는데, **근접 거주 시군구에서조차** 무관하다. → **산업↔온열(r=0.5)은 작업장 노동 노출 (a)이지 ambient 열섬 (b)이 아니다.** (§4의 "구분 불가"를 이제 **구분함**. 단, 이 판별은 시군구 스케일이며 수백 m 국소 스케일의 미세 열섬까지 배제하는 것은 아니다 `[한계]`.)

**★ AIDC 논증에의 결정적 함의 `[해석]`**: **"산업↔온열 r=0.5"를 "AIDC 열섬 → 온열"의 근거로 쓸 수 없다.** AIDC는 데이터센터라 제조업 같은 대규모 작업장 노동(실내·실외 모두)이 없어 이 노동 경로를 재현하지 않는다. AIDC의 우려는 **ambient 열(집·길가 경로 = 지금 null)**인데, 이는 현재 산업과 무관한 **별도의 새 경로**다. → **AIDC ambient 열섬 효과는 산업 상관을 빌리지 말고 물리 dose-response(케임브리지 LST + 기온-온열)로만 논증해야** 한다.

> **§6-c 주석 — 평균이 뭉개는 것** `[🔖 2026-07-21 사용자 지적]`: 위 전체 상관은 산단 근접도에 따라 **크게 갈리는데 평균이 그걸 뭉갠다.** 논밭은 산단근접 상위⅓에서 **r=+0.527*** ** 지만 하위⅓에선 **+0.170(n.s. — 시군구 약 76개 기준, r이 약해 t≈1.5라 유의 안 됨)** — 전체 0.32는 이 둘의 평균이다. 즉 논밭 온열↔산업은 **산단 인근에서 강하게** 성립한다. 반대로 **실내작업장은 근접 상위⅓(+0.368**)보다 **원거리 하위⅓(+0.457**)에서 더 강하다 — 실내 공장노동은 산단 인접 여부와 무관하게 산업 활동이 있는 곳이면 어디서든 나타난다는 뜻. 전체 상관 하나로 읽으면 이 구조가 사라진다.
>
> **★ 근접에서 작업장 상관이 강해지는 것(실외 상위⅓ 0.537 > 하위⅓ 0.354)은 국소 미세 열섬 여지와 양립한다** `[🔖 2026-07-24 사용자 지적]`. 다만 결정적 구분자는 **집·길가**다: 열섬(ambient)이라면 같은 근접 시군구에서 집·길가 온열도 올라야 하는데 거기서도 무관(−0.04/−0.08)이라 측정 가능한 **주 경로는 작업장 노동**으로 지목된다. "집=냉방·길가=단시간"으로 주거 null을 설명할 수도 있으나, 그 경우 논밭(냉방X·장시간)이 근접에서 강해지는 것과 정합적이다 — 즉 결론은 "열섬 전무"가 아니라 "**주 경로는 작업장, 순수 주거 ambient 신호는 미검출**"이다.

> **§6-c2 폴리곤 재검정 결과** `[🔖 2026-07-24]`: near_share를 PDAN 점 대신 **YUCH 폴리곤 경계** 거리로 다시 만들자 근접도가 제대로 잡혔다(중위 5.3%→**10.2%**, 상위⅓ 33%→**45.2%** — 주민 절반가량이 산단 2km 안에 사는 시군구들). **그런데도 집 −0.137(n.s.)·길가 −0.125(n.s.)로 무관이 유지**된다. 즉 "점 기준이라 근접이 덜 잡혀서 null이었다"는 반론은 기각. 반면 실외작업장은 근접에서 더 강해지고(+0.517 vs +0.342), 논밭도 근접에서만 유의(+0.470 vs +0.148 n.s.) — **작업장·경작 노동 경로가 산단 인근에서 강해진다는 구조가 폴리곤 기준에서 더 선명**하다.

> ⚠ **논밭 상관은 판별 근거로 쓰지 않는다** `[🔖 2026-07-24 한계 명시]`: 폴리곤 기준에서 논밭이 근접 상위⅓에만 유의(+0.470 *** vs 하위⅓ +0.148 n.s.)하지만, 이 회귀는 **인구만 통제할 뿐 농지 면적을 통제하지 못한다**. 큰 산단을 낀 시군구가 농지도 넓으면 산업용 전력과 논밭 환자 수가 함께 커지는 구성 효과만으로도 같은 상관이 나온다. 따라서 §6의 결정적 판별자는 논밭이 아니라 **집·길가**다 — ambient 열섬이라면 근접 시군구에서 집·길가가 반응해야 하는데, 근접도를 폴리곤으로 제대로 잡은 뒤에도(상위⅓ 45.2%) 둘 다 무관(n.s.)이다.
>
> 📎 **n.s. 읽는 법**: 여기서 표본 n은 **시군구 개수(각 ⅓ ≈ 76개)**이지 환자 건수가 아니다. 논밭 1,926건은 76개 시군구에 합산된 수라 검정력을 늘려주지 않는다. n=76(df=73)에서 5% 유의 임계는 |r| > 0.227이고, r=+0.148은 t=1.28·p=0.21로 그 밑이다. 즉 "효과가 없다"가 아니라 **"이 표본으로는 0과 구별되지 않는다"**(r=0.148을 유의하게 만들려면 시군구 175개가 필요한데 전국이 229개라 3분할하면 원리적으로 불가능).

## §7. 산단 실좌표 — ILIS shapefile (dose-response 기반)

`0718_DAM_PDAN`(점 1,451=전 산단 대표좌표)·`0718_DAM_YUCH`(폴리곤 17,438=유치업종 구획). EPSG:5186(미터). dose-response 8대상 좌표를 여기서 확정한다. YUCH 실면적으로 xlsx 산업시설면적을 교차검증.

> **8대상(TG) 선정 근거** (2026-07-19 명시): `DERIVED_폐산단전환후보` CSV에서 자동으로 가져온 것이 **아니라** 세 근거의 합집합 —
> ① **메가프로젝트 권역 ∩ 기존 산업용 상위 산단**(`다년패널_산업전력_재계산` §4 표): 여수·광양(호남 1GW)·울산미포·온산(울산 1GW)·포항·구미(대경 2GW)
> ② **확정·착공 입지 (2026-07 다수)** `[🔖 2026-07-24 갱신]`: 동해 북평2(GS 2.4GW MOU — ⚠ 실부지는 §18 `북평2`(일반산단), §10~14 '동해북평'은 인접 `북평`(국가산단) 대조) · 울산미포(SK×AWS 착공) · 심팩 포항(착공 임박) · 포항 광명(착공). 초판 "동해북평만 확정"은 stale.
> ③ **철강 폐산단 전환 경로** 대표: 당진(§5 '구조적' 분류 + 인사이트코리아 보도 + 중부권 1GW 인근)
> ⚠ **§5 스캔 교차검증** `[🔖 2026-07-24]`: 산단을 §5 "구조적 폐산단" 기준에 재적용하니 **미포(남구)·당진·동해·인천동구 독립 판정**(미포 peak −20.3%·중공Δ −2,075). **온산·포항제철·여수·광양은 미해당**(가동 유지/증가·구미는 반도체) — 이들은 근거 ①(권역 GW 상한 시나리오)이지 폐산단 아님. 섞지 말 것.

In [23]:
import geopandas as gpd, warnings; 

# 경고 메시지 무시 설정
warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# (1) 공간 데이터 로드 (EUC-KR 인코딩)
# -----------------------------------------------------------------------------
# PDAN: 산업단지 위치 정보 (Point형 공간 데이터)
PDAN=gpd.read_file('0718_DAM_PDAN/DAM_PDAN.shp', encoding='EUC-KR')

# YUCH: 산업단지 유치/면적 정보 (Polygon형 공간 데이터)
YUCH=gpd.read_file('0718_DAM_YUCH/DAM_YUCH.shp', encoding='EUC-KR')

# -----------------------------------------------------------------------------
# (2) 좌표계 변환 및 위경도(lon/lat) 좌표 추출
# -----------------------------------------------------------------------------
# WGS84 위경도 좌표계(EPSG:4326)로 임시 변환하여 x(경도), y(위도) 컬럼 생성
_p4=PDAN.to_crs(4326); PDAN['lon']=_p4.geometry.x; PDAN['lat']=_p4.geometry.y

# 데이터 기본 건수 및 현재 설정된 좌표계(EPSG 코드) 출력
print('PDAN 점:',len(PDAN),'| YUCH 폴리곤:',len(YUCH),'| CRS:',PDAN.crs.to_epsg())

# 산업단지 유형별 개수 집계 (1: 국가, 2: 일반, 3: 도시첨단, 4: 농공)
print('단지유형:',PDAN['DANJI_TYPE'].value_counts().to_dict(),'(1국가 2일반 3도시첨단 4농공)')


# -----------------------------------------------------------------------------
# (3) 주요 분석 대상 9개 산단 부지 정의 및 위경도 좌표 검증
# -----------------------------------------------------------------------------
# 9개 대상 (라벨명, PDAN 데이터상 단지명, 단지 유형 코드)
# [🔖 2026-07-24 정정 - 사용자 지적 사항 반영]
#  - 기존 '동해북평·메가2.4GW' 오류 수정: 해당 좌표(type 1 '북평')는 국가산단이며,
#    실제 GS 2.4GW 부지는 남쪽으로 1.2km 떨어진 '북평2일반산단'(type 2 '북평2')임.
#  - 이에 따라 기존 국가산단은 '대조군'으로 라벨 변경하고, '북평2' 행을 별도 추가함.
TG=[('광양·제철','광양','1'),('여수·석화','여수','1'),('울산미포·석화','울산·미포','1'),
    ('온산·석화','온산','1'),('포항·제철','포항','1'),('구미·전자','구미(2·3단지)','1'),
    ('당진·철강폐산단','당진1철강','2'),('동해북평·국가(대조)','북평','1'),
    ('동해 북평2·GS2.4GW','북평2','2')]

# 대상 산단별 매칭되는 첫 번째 행을 찾아 단지명, 유형, 위경도 출력
for lbl,nm,ty in TG:
    r=PDAN[(PDAN.DAN_NAME==nm)&(PDAN.DANJI_TYPE==ty)].iloc[0]
    print(f'  {lbl:16} {r.DAN_NAME:10} type{r.DANJI_TYPE} lat={r.lat:.5f} lon={r.lon:.5f}')


# -----------------------------------------------------------------------------
# (4) YUCH 면적 데이터 교차검증 (Cross-validation)
# -----------------------------------------------------------------------------
# 대표 사례 검증: '당진1철강' 산단
_d=PDAN[PDAN.DAN_NAME=='당진1철강'].iloc[0]

# 해당 산단 ID(DAN_ID)에 속한 모든 폴리곤을 병합(union_all)한 후 면적 계산 (m² → km² 변환)
_ya=YUCH[YUCH.DAN_ID==_d.DAN_ID].geometry.union_all().area/1e6

# 외부 통계 자료(xlsx 상의 산업시설 면적 1.7km²)와 GIS 면적 결과 간 일치 여부 확인
print(f'\nYUCH 교차검증 당진1철강 = {_ya:.1f} km2  (xlsx 산업시설 1.7km2와 일치 → 좌표·면적 신뢰)')

PDAN 점: 1451 | YUCH 폴리곤: 17438 | CRS: 5186
단지유형: {'2': 819, '4': 486, '1': 93, '3': 53} (1국가 2일반 3도시첨단 4농공)
  광양·제철            광양         type1 lat=34.86437 lon=127.78018
  여수·석화            여수         type1 lat=34.82870 lon=127.66543
  울산미포·석화          울산·미포      type1 lat=35.52014 lon=129.35794
  온산·석화            온산         type1 lat=35.43091 lon=129.34856
  포항·제철            포항         type1 lat=36.00731 lon=129.40410
  구미·전자            구미(2·3단지)  type1 lat=36.10504 lon=128.40932
  당진·철강폐산단         당진1철강      type2 lat=36.98671 lon=126.72806
  동해북평·국가(대조)      북평         type1 lat=37.48254 lon=129.14298
  동해 북평2·GS2.4GW   북평2        type2 lat=37.47306 lon=129.14921

YUCH 교차검증 당진1철강 = 1.7 km2  (xlsx 산업시설 1.7km2와 일치 → 좌표·면적 신뢰)


## §8. dose-response 입력표 — 좌표+면적+온열률 결합

좌표=PDAN 점(완전), 면적=YUCH 실폴리곤 있으면 사용·없으면 xlsx 산업시설(여수·포항 국가산단은 YUCH 결측), 온열률=§2-§6 패널. → `DERIVED_0718_dose_response_입력_산단.csv`.

In [24]:
import python_calamine as pc

""" Block 1. 산업단지 현황 엑셀(XLSX) 데이터 로드 및 정제 """
# 1. xlsx 산단현황 (단지명·지정면적·산업시설면적 데이터) 로드
_wb=pc.CalamineWorkbook.from_path('OPEN_0718_산단공_산업단지현황조사_2025Q3.xlsx')
_d=_wb.get_sheet_by_name('전국산업단지현황').to_python()

# 2. 데이터프레임 생성을 위한 컬럼명 리스트 정의
_cols=['유형','시도','시군','단지명','조성상태','지정면적','관리면적','산업시설전체','분양대상','분양','미분양','분양률','입주','가동']

# 3. 5번째 행(헤더 제외 데이터 시작점)부터 데이터프레임으로 변환
xf=pd.DataFrame(_d[5:],columns=_cols)

# 4. 지정면적 및 산업시설전체 면적 컬럼을 숫자형으로 변환 (문자/결측치는 NaN 처리)
for c in ['지정면적','산업시설전체']: xf[c]=pd.to_numeric(xf[c],errors='coerce')

# 5. 단지명 텍스트 내 줄바꿈/특수공백('\xa0') 및 양끝 공백 제거
xf['단지명']=xf['단지명'].astype(str).str.replace('\xa0','',regex=False).str.strip()


""" Block 2. 공간 도형(YUCH) 기반 산단 ID별 실면적 계산 """
# 1. YUCH 공간 데이터에서 단지ID(DAN_ID)별 폴리곤 합집합(union) 면적을 km² 단위로 계산하여 딕셔너리로 저장 (1e6 = 1,000,000 m²)
_yar={i:s.geometry.union_all().area/1e6 for i,s in YUCH.groupby('DAN_ID')}

"""Block 3. 시군구 패널 데이터 집계 (온열질환율, 인구, 전력량)
온열률은 연간 건수가 작아(당진 6건 등) 1년치는 노이즈 → 6년 평균으로 안정화. 
인구·전력은 "지금 AIDC가 들어오면"의 현재 노출 시나리오라 최신값 """

# 1. 시군구별 온열질환 및 전력 사용량 패널 CSV 로드
_pn=pd.read_csv('DERIVED_0718_시군구연도_패널_온열전력.csv')

# 2. 지역(key)별 지표 집계: 온열질환율은 6년 평균(mean), 인구 및 전력량은 최신 연도(2025년, iloc[-1]) 수치 추출
_agg=_pn.groupby('key').agg(여름온열률10만=('여름온열률10만','mean'),
        인구2025=('인구',lambda s:s.iloc[-1]),산업용여름2025=('산업용여름',lambda s:s.iloc[-1])).reset_index()


"""Block 4. 분석 대상 산단 매핑 정보 정의"""
# # 1. 분석 대상 산단 매핑 타플 리스트 정의: (분석라벨, 시군구key, (GIS산단명, 산단유형), 엑셀단지명검색어)
T=[('광양·제철','전라남도 광양시',('광양','1'),'광양국가'),('여수·석화','전라남도 여수시',('여수','1'),'여수'),
   ('울산미포·석화','울산광역시 남구',('울산·미포','1'),'미포'),('온산·석화','울산광역시 울주군',('온산','1'),'온산'),
   ('포항·제철','경상북도 포항시',('포항','1'),'포항국가'),('구미·전자','경상북도 구미시',('구미(2·3단지)','1'),'구미국가(2-3단지)'),
   ('당진·철강폐산단','충청남도 당진시',('당진1철강','2'),'당진1철강'),('동해북평·국가(대조)','강원특별자치도 동해시',('북평','1'),'북평 ①'),
   # ★ [🔖 2026-07-24 사용자 지적] GS 2.4GW 실부지는 '북평제2일반산단'(북평2) — dose-response 정식 대상으로 편입.
   #   기존 '북평'(국가산단)은 인접 대조로 라벨 변경. §8-b·§9·§10-b·§11·§12-b가 이 T를 공유하므로 전 과정에 자동 반영.
      # [🔖 2026-07-24 사용자 지적] 4번째 원소는 '산단공 엑셀 단지명 검색어'다. 엑셀 표기는 '북평제2'인데
   #   '북평2'로 줘서 0건 매칭 → 산업시설km²가 NaN이었다(당진처럼 대장↔GIS 교차검증을 못 하고 있었음).
   ('동해 북평2·GS2.4GW','강원특별자치도 동해시',('북평2','2'),'북평제2')]

"""Block 5. 데이터 결합, 열원 반경 계산 및 최종 데이터프레임 추출"""

import numpy as np
rows=[]

# 1. 대상 산단 리스트(T)를 순회하며 개별 데이터 결합 진행
for lbl,key,(pnm,pt),xn in T:
    
    # 2. GIS 산단 데이터(PDAN)에서 해당 단지명과 유형에 맞는 행 추출
    pr=PDAN[(PDAN.DAN_NAME==pnm)&(PDAN.DANJI_TYPE==pt)].iloc[0]

    # 3. 엑셀 현황 데이터(xf)에서 단지명이 포함된 행을 찾고 지정면적 내림차순 정렬
    xr=xf[xf['단지명'].str.contains(xn.split('(')[0],na=False)].sort_values('지정면적',ascending=False)

    # 4. 지정면적 및 산업시설전체 면적 추출 (천m² 단위를 km²로 변환하기 위해 1000으로 나눔)
    z=xr['지정면적'].iloc[0]/1000 if len(xr) else np.nan
    s=xr['산업시설전체'].iloc[0]/1000 if len(xr) else np.nan

    # 5. YUCH 공간 데이터 기반 실면적(km²) 가져오기
    ya=_yar.get(pr.DAN_ID,np.nan)

    # 6. 열원면적 결정: YUCH 실면적이 있으면 우선 사용, 없으면 엑셀 산업시설면적(xlsx) 사용
    src=ya if ya==ya else s                       

    # 7. 해당 시군구의 보건/인구/전력 집계 데이터 추출
    ar=_agg[_agg.key==key].iloc[0]

    # 8. 결합된 개별 산단 정보를 딕셔너리 형태로 레코드에 추가
    #    - 열원반경km: 면적을 원의 면적 공식(A = πr²)에 적용하여 원형 환산 반경 계산 r = √(A / π)
    rows.append(dict(대상=lbl,key=key,단지명=pr.DAN_NAME,유형=pr.DANJI_TYPE,
        lat=round(pr.lat,5),lon=round(pr.lon,5),지정면적km2=round(z,1),
        산업시설km2=round(s,1) if s==s else np.nan,YUCH실면적km2=round(ya,1) if ya==ya else np.nan,
        열원반경km=round(np.sqrt(src/np.pi),2),여름온열률10만=round(ar['여름온열률10만'],1),
        인구2025=int(ar['인구2025']),산업용여름GWh=int(ar['산업용여름2025'])))
    
# 9. 리스트를 데이터프레임으로 변환 후 CSV 저장 및 주요 지표 화면 출력
dr=pd.DataFrame(rows)
dr.to_csv('DERIVED_0718_dose_response_입력_산단.csv',index=False,encoding='utf-8-sig')
print(dr[['대상','lat','lon','지정면적km2','산업시설km2','YUCH실면적km2','열원반경km','여름온열률10만','인구2025']].to_string(index=False))
# [🔖 2026-07-24] 당진(앞 셀)처럼 북평2도 대장↔GIS 독립 교차검증 — 두 출처가 맞으면 좌표·면적 신뢰.
_bp=dr[dr['대상']=='동해 북평2·GS2.4GW'].iloc[0]
_ok='일치' if abs(_bp['산업시설km2']-_bp['YUCH실면적km2'])<=0.15 else '불일치 [확인필요]'
print()
print(f"대장↔GIS 교차검증 북평제2 = 대장 산업시설 {_bp['산업시설km2']}km² vs YUCH 실폴리곤 {_bp['YUCH실면적km2']}km² → {_ok}"
      f"  (지정면적 {_bp['지정면적km2']}km² 중 산업시설 {_bp['산업시설km2']/_bp['지정면적km2']*100:.0f}%)")

            대상      lat       lon  지정면적km2  산업시설km2  YUCH실면적km2  열원반경km  여름온열률10만  인구2025
         광양·제철 34.86437 127.78018     96.4     20.6        17.5    2.36      20.7  155259
         여수·석화 34.82870 127.66543     51.2     23.3         NaN    2.73      12.4  263284
       울산미포·석화 35.52014 129.35794     48.5     35.3        33.7    3.27       5.3  304128
         온산·석화 35.43091 129.34856     25.9     16.9        16.2    2.27       8.7  219174
         포항·제철 36.00731 129.40410     28.8     16.1         NaN    2.26       9.4  488707
         구미·전자 36.10504 128.40932     16.6      7.8         4.9    1.25       8.0  403883
      당진·철강폐산단 36.98671 126.72806      2.1      1.7         1.7    0.73      13.2  172564
   동해북평·국가(대조) 37.48254 129.14298      4.0      1.0         0.6    0.43       6.4   86333
동해 북평2·GS2.4GW 37.47306 129.14921      0.6      0.3         0.3    0.32       6.4   86333

대장↔GIS 교차검증 북평제2 = 대장 산업시설 0.3km² vs YUCH 실폴리곤 0.3km² → 일치  (지정면적 0.6km² 중 산업시설 50%)


**§8-b. 형상 검증 — PDAN 점이 산단 중심인가, 산단은 원형인가** (2026-07-19 사용자 지적)
'열원반경'은 산단을 **등가 원**으로 근사한 것. 실폴리곤(YUCH)으로 ① PDAN 점 vs 폴리곤 중심 offset ② 원형도(4πA/P², 1=완전 원)를 검증한다. 바람(풍향 이류)은 미모델 — 등방 버퍼는 1차 근사 `[확인필요]`.

> **위성 확인 완료** `[사실]` `[🔖 2026-07-19]` (V-World 이미지 API 항공영상): **광양 PDAN 점(34.86,127.78)은 광양만 바다 한복판** — 실제 산단이 아님. YUCH 폴리곤 중심(34.92,127.75)이 **실제 POSCO 광양제철소**. → 광양은 폴리곤 사용이 옳음. **여수·포항은 YUCH 결측이지만 PDAN 점이 실제 산단 위**(여수국가산단 석유화학·POSCO 포항제철소)로 확인 → 점+반경 모형 신뢰 OK. 즉 점 오류는 **광양 1건뿐**.

In [25]:
from shapely import geometry as _g

# 1. 검증 결과 출력을 위한 테이블 헤더(대상, 중심 오프셋, 원형도, 대표점 포함 여부) 출력
print(f"{'대상':16}{'중심offset(km)':>13}{'원형도':>7}{'hull채움%':>10}{'(구내부포함)':>11}{'PDAN점이 폴리곤 안?':>16}")

# 2. 분석 대상 산단 리스트(T)를 순회하며 개별 산단의 도형(Geometry) 기하 검증 수행
for lbl,key,(pnm,pt),xn in T:

    # 3. GIS 산단 데이터(PDAN)에서 대상 산단의 대표점(Point) 정보 추출
    pr=PDAN[(PDAN.DAN_NAME==pnm)&(PDAN.DANJI_TYPE==pt)].iloc[0]

    # 4. YUCH 공간 데이터에서 해당 산단 ID(DAN_ID)에 해당하는 폴리곤(면적) 객체 추출
    sub=YUCH[YUCH.DAN_ID==pr.DAN_ID]

    # 5. YUCH 공간 폴리곤 데이터가 결측된 경우 예외 처리 후 넘어가기
    if len(sub)==0: print(f'{lbl:16} YUCH 결측 → 점+등가원 유지 [확인필요]'); continue

    # 6. 산단 내 여러 분절된 폴리곤들을 하나로 합침 (MultiPolygon -> Single Geometry)
    u=sub.geometry.union_all()

    # 7. [중심 Offset] PDAN 대표점과 YUCH 폴리곤의 무게중심(centroid) 간의 직선거리를 km 단위로 계산
    off=pr.geometry.distance(u.centroid)/1000

    # 8. [원형도(Circularity)] 도형이 원에 얼마나 가까운지 측정 (1 = 완벽한 원)
    #    [🔖 2026-07-24 정정 — 사용자 지적] 기존 4πA/union.length²는 내부 구획 경계(작은 조각들의 둘레)까지
    #      둘레에 합산해 과소(미포 99조각 → 0.01). '전체 형상이 둥근가'는 외곽 윤곽으로 봐야 하므로
    #      convex_hull(볼록껍질) 둘레를 쓴다. 기존값은 '내부구획포함' 참고로 병기.
    _hull=u.convex_hull
    rnd=4*np.pi*u.area/(_hull.length**2)          # convex_hull 기반(외곽 형상)
    rnd_old=4*np.pi*u.area/(u.length**2)          # 기존(내부 구획 포함) — 참고

    # 9. [포함 여부] 산단 대표점(pr.geometry)이 YUCH 폴리곤 경계(100m 버퍼 포함) 내부에 위치하는지 확인 (True/False)
    inside=u.buffer(100).contains(pr.geometry)

    # 10. 산단별 공간 기하학 검증 결과 한 줄 출력
    # [🔖 2026-07-25 사용자 지적] 원형도(0.30)는 "얼마나 원에 가까운가"라 실무 뜻이 바로 안 온다.
    #   hull 채움% = 감싸는 최소 볼록 외곽 안에서 **실제 산단이 차지하는 비율**. 미포 34%는
    #   "등가원 모형이 나머지 66%(바다·도심·도로)까지 산단으로 취급한다"를 직접 말해준다.
    fill=u.area/_hull.area*100
    print(f"{lbl:16}{off:>13.2f}{rnd:>7.2f}{fill:>9.0f}%{rnd_old:>11.3f}{str(inside):>16}")

# 11. 원형도 결과 해석: 모든 산단이 0.01~0.18 수준으로 완벽한 원형과 거리가 먼 선형/해안선 형태임을 명시
print("→ [convex_hull 기준] 원형도 0.30~0.59 = 대체로 뭉툭하나 완전 원 아님(미포 0.30=여러 갈래 분산·정사각 bbox 13×11km). 기존(내부구획포함) 0.01~0.18은 조각 둘레까지 세 과소였음. 광양은 PDAN 점이 YUCH 밖 ~7km —")

# 12. 광양 산단의 오류 원인 추정 및 공간 가시화(QGIS/V-World) 필요성 경고
print("  YUCH가 제철 본체 부지를 누락했거나 PDAN 대표점이 행정 기준점일 가능성. [확인 완료 07-19] V-World 항공영상: 광양 PDAN점=광양만 바다, YUCH중심=실제 POSCO. 여수·포항 PDAN점은 실제 산단 위. → 점 오류는 광양 1건뿐(§8-b md).")

# 13. 본 분석의 결론: '단순 대표점 + 등가원 반경' 방식의 한계를 실증하였으며, 폴리곤 경계 기준 민감도 분석이 필수적임을 명시
print("  → 등가원(점+반경) 모형의 한계 실증 → §10-b에서 폴리곤 경계 기준 버퍼로 민감도 재계산.")

대상               중심offset(km)    원형도   hull채움%    (구내부포함)   PDAN점이 폴리곤 안?
광양·제철                    7.26   0.41       52%      0.030           False
여수·석화            YUCH 결측 → 점+등가원 유지 [확인필요]
울산미포·석화                  1.82   0.30       34%      0.010            True
온산·석화                    0.39   0.59       73%      0.017            True
포항·제철            YUCH 결측 → 점+등가원 유지 [확인필요]
구미·전자                    0.10   0.33       59%      0.019            True
당진·철강폐산단                 0.42   0.56       80%      0.181            True
동해북평·국가(대조)              0.33   0.42       56%      0.176            True
동해 북평2·GS2.4GW           0.26   0.66       78%      0.105            True
→ [convex_hull 기준] 원형도 0.30~0.59 = 대체로 뭉툭하나 완전 원 아님(미포 0.30=여러 갈래 분산·정사각 bbox 13×11km). 기존(내부구획포함) 0.01~0.18은 조각 둘레까지 세 과소였음. 광양은 PDAN 점이 YUCH 밖 ~7km —
  YUCH가 제철 본체 부지를 누락했거나 PDAN 대표점이 행정 기준점일 가능성. [확인 완료 07-19] V-World 항공영상: 광양 PDAN점=광양만 바다, YUCH중심=실제 POSCO. 여수·포항 PDAN점은 실제 산단 위. → 점 오류는 광양 1건뿐(§8-b md).
  → 등가원(점+반경) 

## §9. ② 열부하 밀도 — ΔT 유발 강도 (같은 AIDC라도 작은 산단이면 승온↑)

메가프로젝트 권역 AIDC(GW)를 **열원면적**으로 나눈 열부하밀도(GW/km²). 상대ΔT지수는 광양=1 정규화. `[방법, 1차근사]` 승온∝열부하 flux. 권역내 분배 미공개 → 권역 GW를 대표 1산단에 얹은 상한 시나리오.

> **분모는 두 출처 중 큰 값** `[🔖 2026-07-24 사용자 지적]`: 대장(산단공 산업시설km²)과 GIS(YUCH 실폴리곤km²)가 어긋날 때 **큰 쪽**을 쓴다. 분모가 크면 열부하 = 위험 지수가 작아지므로 **과장 방향으로 틀리지 않는다**. 한쪽이 결측이면 있는 쪽(여수·포항은 YUCH 없어 대장).
>
> ⚠ **북평2는 이 보수 규칙으로도 안 내려간다** — 대장 0.31km²와 GIS 0.30km²가 서로 일치하기 때문이다(§7 교차검증). "2.4GW가 0.3km²에 들어갈 리 없다"는 직관은 옳고, 그 **불가능성 자체가 발견**이다: §12-c가 2.4GW 표준 소요를 146~235만평으로 잡는데 북평2 부지는 9.7만평(1/15~1/24)이다. 결론은 둘 중 하나 — (i) 표준의 10~24배 고밀(다층·수랭)이거나 (ii) **산단 경계 밖으로 확장**한다(§12-d 확장 시나리오: 3km 노출 2.6만 → 5.7~6.4만 명). 어느 쪽이든 지수 하나로 눌러 담을 게 아니라 §12-c·§12-d에서 따로 말하게 둔다.
>
> ⚠ **상대ΔT지수는 광양=1 정규화**라 분모 규칙을 바꾸면 기준선도 함께 움직인다. 절대값 GW/km²를 먼저 보고, 지수는 순위·배율 읽기용으로만 쓸 것.

In [ ]:
# -----------------------------------------------------------------------------
# 1. 대상 산단별 AIDC(인공지능 데이터센터) 용량(GW) 매핑
# -----------------------------------------------------------------------------
# 산단별 상정 AIDC 용량 데이터셋 정의 (단위: GW)
mega={'광양·제철':1.0,'여수·석화':1.0,'울산미포·석화':1.0,'온산·석화':1.0,
      '포항·제철':2.0,'구미·전자':2.0,'당진·철강폐산단':1.0,
      '동해북평·국가(대조)':2.4,'동해 북평2·GS2.4GW':2.4}   # [🔖 07-24] 북평2 편입

# '대상' 컬럼 값을 기준으로 AIDC 용량 매핑
dr['상정AIDC_GW']=dr['대상'].map(mega)

# -----------------------------------------------------------------------------
# 2. 보수적 분모(열원 면적) 산출: max(대장 면적, GIS YUCH 실면적)
# -----------------------------------------------------------------------------
# [🔖 2026-07-24 사용자 지적 ②]
# 열부하(위험 지수) 과장 위험을 방지하기 위해, 대장 면적과 GIS YUCH 실면적 중 '더 큰 값'을 분모로 선택.
# 분모가 커지면 열부하(GW/km²)가 낮아지므로 보수적인(위험을 낮추어 평가하는) 추정이 됨.
# np.fmax는 NaN을 무시하고 존재하는 값 중 최댓값을 취함 (여수·포항 등 YUCH 결측 지역 처리 대응).

_s9=dr['산업시설km2'].astype(float); _y9=dr['YUCH실면적km2'].astype(float)

# 두 출처 중 더 큰 면적을 선택하여 통합 분모 Series 생성
_den9=pd.Series(np.fmax(_s9.values,_y9.values),index=dr.index)

# 열부하 밀도 계산 (AIDC 용량 / 선택된 면적) → 소수점 3자리 반올림 (단위: GW/km²)
dr['열부하_GWpkm2']=(dr['상정AIDC_GW']/_den9).round(3)

# -----------------------------------------------------------------------------
# 3. 무차원 상대ΔT지수 산출 (광양 기준 정규화)
# -----------------------------------------------------------------------------
# 상대ΔT지수: 기준점('광양·제철')의 열부하 밀도를 1.0으로 둔 정규화 지수.
# 무차원 지수이므로 어느 기준을 정하든 산단 간 순위 및 비율은 동일하게 유지됨.
_base=dr.loc[dr['대상']=='광양·제철','열부하_GWpkm2'].iloc[0]
dr['상대ΔT지수']=(dr['열부하_GWpkm2']/_base).round(1)

# -----------------------------------------------------------------------------
# 4. 표 정합성을 위한 면적 출처 및 최종 적용 면적 컬럼 생성
# -----------------------------------------------------------------------------
# [🔖 2026-07-24 표 정합] 실제로 연산에 사용된 분모와 그 출처를 명확히 기록
dr['열원면적km2']=_den9.round(1)

# 어느 출처의 면적이 채택되었는지 추적 라벨링
dr['면적출처']=np.where(_y9.isna(),'대장(YUCH無)',
                np.where(_s9.isna(),'YUCH(대장無)',
                np.where(_s9>=_y9,'대장(큰값)','YUCH(큰값)')))

# -----------------------------------------------------------------------------
# 5. 분석 결과 데이터프레임 출력
# -----------------------------------------------------------------------------
# 상대ΔT지수 내림차순 정렬 후 주요 지표 출력
print(dr[['대상','상정AIDC_GW','열원면적km2','면적출처','산업시설km2','열부하_GWpkm2','상대ΔT지수','여름온열률10만']]
      .sort_values('상대ΔT지수',ascending=False).to_string(index=False))
# 해석 문장도 하드코딩 금지 — 표에서 직접 뽑는다(값이 바뀌면 문장도 따라 바뀌어야).

# -----------------------------------------------------------------------------
# 6. 결과 동적 해석 문장 생성 및 주요 인사이트 출력
# -----------------------------------------------------------------------------
# 데이터 변경 시 해석 문장이 자동 업데이트되도록 동적 파싱 방식 채택
_top9=dr.sort_values('열부하_GWpkm2',ascending=False).iloc[0]     # 열부하 최대 산단 추출
_dj9=dr[dr['대상']=='당진·철강폐산단'].iloc[0]                    # 당진 산단 추출
print(f"\n→ {_top9['대상']} {_top9['상정AIDC_GW']}GW÷{_top9['열원면적km2']}km²({_top9['면적출처']}) = 광양의 ~{_top9['상대ΔT지수']:.0f}배."
      f" 당진(철강폐산단) {_dj9['상대ΔT지수']:.0f}배. 열부하는 작은 산단에서 극단.")

print("  ⚠ 분모는 대장·GIS 중 **큰 값**(보수 채택) — 위험을 주장하는 지표라 과장 방향으로 틀리지 않게 한다.")
print("     상대ΔT지수는 광양=1 정규화라 광양 분모가 바뀌면 전 행이 함께 움직인다. 절대값 GW/km²를 먼저 볼 것.")

# 특정 케이스('동해 북평2·GS2.4GW') 검증 및 논의 출력
_bp9=dr[dr['대상']=='동해 북평2·GS2.4GW'].iloc[0]

print(f"  ★ 북평2는 보수 규칙으로도 {_bp9['열부하_GWpkm2']:.2f} GW/km²로 최상단 — 대장 {_s9[_bp9.name]}km²와 GIS {_y9[_bp9.name]}km²가")
print("     서로 일치해서다(§7 교차검증). '2.4GW가 0.3km²에 들어갈 리 없다'는 직관은 옳고, 그 불가능성 자체가 §12-c·§12-d의 발견이다")
print("     — 표준 소요 146~235만평 vs 부지 9.7만평 → 고밀이거나 산단 경계 밖 확장(3km 노출 2.6만→5.7~6.4만).")

            대상  상정AIDC_GW  열원면적km2      면적출처  산업시설km2  열부하_GWpkm2  상대ΔT지수  여름온열률10만
동해 북평2·GS2.4GW        2.4      0.3    대장(큰값)      0.3       8.000   163.3       6.4
   동해북평·국가(대조)        2.4      1.0    대장(큰값)      1.0       2.400    49.0       6.4
      당진·철강폐산단        1.0      1.7    대장(큰값)      1.7       0.588    12.0      13.2
         구미·전자        2.0      7.8    대장(큰값)      7.8       0.256     5.2       8.0
         포항·제철        2.0     16.1 대장(YUCH無)     16.1       0.124     2.5       9.4
         온산·석화        1.0     16.9    대장(큰값)     16.9       0.059     1.2       8.7
         광양·제철        1.0     20.6    대장(큰값)     20.6       0.049     1.0      20.7
         여수·석화        1.0     23.3 대장(YUCH無)     23.3       0.043     0.9      12.4
       울산미포·석화        1.0     35.3    대장(큰값)     35.3       0.028     0.6       5.3

→ 동해 북평2·GS2.4GW 2.4GW÷0.3km²(대장(큰값)) = 광양의 ~163배. 당진(철강폐산단) 12배. 열부하는 작은 산단에서 극단.
  ⚠ 분모는 대장·GIS 중 **큰 값**(보수 채택) — 위험을 주장하는 지표라 과장 방향으로 틀리지 않게 한다.
     상대ΔT지

## §9-b. 기온→온열 dose-response 곡선(×1.6/°C) **자체 유도** — self-contained

§10~§11의 위험배수는 `1°C당 온열 ×1.6`을 쓴다. 이 값은 **다른 노트북이 아니라 여기서 직접** 유도해 self-contained로 만든다. 재료는 **NEDIS 온열(전국 일별)** + **OPEN 기상청 ASOS 최고기온(전국 일별 평균)** — 둘 다 이 저장소에 있고, **전력파일과 무관**하다(→ OPEN_0718 전력 업데이트로 이 값은 안 바뀐다). Poisson: `log(온열) = a + b·최고기온`, 1°C당 배수 = `exp(b)`.

In [27]:
import statsmodels.api as sm, statsmodels.formula.api as smf, glob

# 1. 엑셀 파일에서 2020~2025년 전국 NEDIS 온열질환 데이터를 빠른 속도(Calamine)로 불러옵니다.
_wb=CalamineWorkbook.from_path('FOIA_0522_질병관리청_NEDIS온열질환_2020-2025.xlsx')
_r=_wb.get_sheet_by_name('DB(2020-2025)발생지역기준').to_python()
_ne=pd.DataFrame(_r[1:],columns=_r[0])

# 2. '발생일자' 열을 날짜(datetime) 형식으로 변환합니다.
_ne['dt']=pd.to_datetime(_ne['발생일자'].astype(str),errors='coerce')

# 3. 날짜별로 환자 수를 카운트(size)하여 '일별 전국 온열질환자 수' 시리즈(_dne)를 만듭니다.
_dne=_ne.dropna(subset=['dt']).groupby(_ne['dt'].dt.date).size().rename('온열')


# 4. 여름철 기상청 ASOS CSV 파일들을 반복문으로 모두 찾아 읽어온 뒤, '일시'와 '최고기온(°C)'만 추출합니다.
_fr=[]
for _f in glob.glob('OPEN_0522_기상청_ASOS일자료_*_summer.csv'):
    _a=pd.read_csv(_f,encoding='cp949'); _fr.append(_a[['일시','최고기온(°C)']])

# 5. 읽어온 파일들을 하나로 합치고 날짜 형태로 변환합니다.
_asos=pd.concat(_fr); _asos['dt']=pd.to_datetime(_asos['일시'],errors='coerce').dt.date

# 6. 전국 여러 관측소의 일별 최고기온을 평균 내어 '전국 일별 평균 최고기온' 시리즈(_dt)를 만듭니다.
_dt=_asos.groupby('dt')['최고기온(°C)'].mean().rename('최고기온')

# 7. 기온 데이터(_dt)와 온열질환자 데이터(_dne)를 날짜 기준으로 병합합니다. (환자가 없던 날은 0으로 채움)
_d=pd.concat([_dt,_dne],axis=1); _d.index.name='date'; _d=_d.reset_index(); _d['온열']=_d['온열'].fillna(0)

# 8. 월(month) 정보를 추출하여 여름철에 해당하는 '6월, 7월, 8월' 데이터만 선별하고 기온 결측치를 제거합니다.
_d['m']=pd.to_datetime(_d['date']).dt.month; _d=_d[_d['m'].isin([6,7,8])].dropna(subset=['최고기온'])

# 9. 건수(Count) 데이터 분석에 표준적으로 쓰이는 포아송 회귀모델(Poisson GLM)을 적합시킵니다. (온열질환자 수 ~ 최고기온)
_m=smf.glm('온열 ~ 최고기온', data=_d, family=sm.families.Poisson()).fit()

# 10. 기온 계수(b)에 자연상수 exp를 취해 "기온 1℃ 상승 시 환자 수가 몇 배로 늘어나는지" 배수(HEAT_MULT_PER_C)를 구합니다.
HEAT_MULT_PER_C=float(np.exp(_m.params['최고기온']))

# 11. 95% 신뢰구간(Confidence Interval)의 하한값(_lo)과 상한값(_hi)도 동일하게 exp를 취해 배수로 환산합니다.
_ci=_m.conf_int().loc['최고기온']; _lo=float(np.exp(_ci[0])); _hi=float(np.exp(_ci[1]))

# 12. 분석에 사용된 여름철 총 일수(n)와 누적 온열질환자 총 건수를 출력합니다.
print('여름 일수 n=%d, 온열 %d건' % (len(_d), int(_d.온열.sum())))

# 13. 추정된 1℃당 환자 증가 배수와 95% 신뢰구간을 출력합니다. (약 1.6배)
print('1°C당 온열 배수 = exp(b) = %.3f (95%%CI %.2f~%.2f)  → 반올림 ≈ 1.6 (§10에서 사용)' % (HEAT_MULT_PER_C,_lo,_hi))

# 14. [단순 비교] 폭염일(≥30℃)과 비폭염일(<30℃)의 하루 평균 환자 수를 계산합니다.
_hiT=_d[_d['최고기온']>=30]['온열'].mean(); _loT=_d[_d['최고기온']<30]['온열'].mean()

# 15. 단순 평균값으로 봐도 30℃ 이상인 날이 이하인 날보다 환자가 몇 배 더 많은지 확인용으로 출력합니다.
print('참고: ≥30℃ 하루 %.1f건 vs <30℃ %.1f건 = %.1f배' % (_hiT,_loT,_hiT/_loT))
print('→ 이 유도는 NEDIS+ASOS만 사용 — 전력파일과 독립. §10 f=1.6²−1은 이 값의 반올림.')

여름 일수 n=552, 온열 13995건
1°C당 온열 배수 = exp(b) = 1.575 (95%CI 1.56~1.59)  → 반올림 ≈ 1.6 (§10에서 사용)
참고: ≥30℃ 하루 49.1건 vs <30℃ 5.9건 = 8.3배
→ 이 유도는 NEDIS+ASOS만 사용 — 전력파일과 독립. §10 f=1.6²−1은 이 값의 반올림.


In [28]:
# ── §9-c. 지역별 dose-response — 시도 FE [🔖 2026-07-22 수정: 광주 복구 + silent 조건 제거] ──

# 3. 질병관리청 온열질환 데이터(NEDIS) 로드 및 시도·일자별 환자 수 집계
# 5. [Model A] 시도 고정효과(Fixed Effects, FE)를 통제한 포아송 회귀 분석 (전체 공통 기울기)
# - C(시도): 지역별 베이스라인(인구 structure, 기본 기후 차이 등) 통제
# 6. [Model B] 시도별 개별 포아송 회귀 분석 (각 지역 독립 기울기 추정)
    # 관측치가 50개 초과이고, 누적 온열질환자 수가 30명 초과인 유의미한 시도만 회귀 실행
# 시도별 위험 배수가 높은 순서대로 출력 
# 7. 결과 분석 요약 및 핵심 케이스(인천) 확인 출력
# [버그 정정] ASOS 지점주소가 2026 통합명 '전남광주통합특별시'로 갱신되어 광주 그룹 기온이 0건
#   → merge에서 광주광역시 온열 297건 통째 탈락 → 이전 판의 (n>50·온열>30) 조건이 이 유령그룹을
#   조용히 걸러 버그를 가렸다. 조건은 어차피 모든 시도가 552일이라 불필요(사용자 지적) → 제거하고
#   구 단위로 광주/전남을 분리 + 17/17 매칭을 명시 검증한다.

import glob as _g9, statsmodels.api as _sm9, statsmodels.formula.api as _smf9
from python_calamine import CalamineWorkbook as _CW9

# 1. '전남광주통합특별시' 행정구역 갱신 대비: 광주광역시 소속 5개 자치구 정의
_GJGU9={'동구','서구','남구','북구','광산구'}


def _sido9(addr):
    """
    ASOS 지점 주소를 기반으로 표준 시도명을 반환하는 함수.
    '전남광주통합특별시'로 통합 갱신된 지점주소를 구(District) 단위로 체크하여 광주/전남으로 분리 정제.
    """
    t=str(addr).split(); h=t[0].replace('(산지)','') if t else ''

    # 통합명 표기 시 구(District) 명칭을 비교하여 광주광역시와 전라남도 구분
    if h=='전남광주통합특별시': 
        return '광주광역시' if (len(t)>1 and t[1] in _GJGU9) else '전라남도'
    
    return SIDO_MAP.get(h,h)

# 2. 기상청 ASOS 지점 정보 로드 및 지점별 시도 매핑 dictionary(_p2s9) 생성
_info9=pd.read_csv('OPEN_0720_기상청_ASOS지점정보.csv',encoding='cp949')
_info9['시도']=_info9['지점주소'].map(_sido9)
_p2s9=dict(zip(_info9['지점'],_info9['시도']))

# 3. 여름철 ASOS 일자료 로드 및 통합 (6~8월, 지점별 최고기온 추출)
_fr9=[]
for _f in _g9.glob('OPEN_0522_기상청_ASOS일자료_*_summer.csv'):
    _a9=pd.read_csv(_f,encoding='cp949'); _a9['시도']=_a9['지점'].map(_p2s9); _a9['dt']=pd.to_datetime(_a9['일시'],errors='coerce').dt.date
    _fr9.append(_a9[['지점','시도','dt','최고기온(°C)']])

# 시도별/일자별 평균 최고기온 데이터프레임 생성
_aso9=pd.concat(_fr9).dropna(subset=['시도','최고기온(°C)'])
_tmax9=_aso9.groupby(['시도','dt'])['최고기온(°C)'].mean().rename('최고기온').reset_index()

# 4. 질병관리청 NEDIS 온열질환 데이터(Excel) 고속 로드 및 시도별/일자별 집계
_rows9=_CW9.from_path('FOIA_0522_질병관리청_NEDIS온열질환_2020-2025.xlsx').get_sheet_by_name('DB(2020-2025)발생지역기준').to_python()
_ne9=pd.DataFrame(_rows9[1:],columns=_rows9[0])
_ne9['시도']=_ne9['발생시도'].map(lambda s:SIDO_MAP.get(str(s).strip(),str(s).strip()))
_ne9['dt']=pd.to_datetime(_ne9['발생일자'].astype(str),errors='coerce').dt.date

# 시도별/일자별 온열질환 환자 수 집계
_cnt9=_ne9.dropna(subset=['dt']).groupby(['시도','dt']).size().rename('온열').reset_index()

# 5. 17개 광역시도 매칭 검증 (데이터 누락 여부 확인)
_miss9=set(_cnt9.시도)-set(_tmax9.시도)
print('기온 없는 NEDIS 시도:',_miss9 or '없음 — 17/17 매칭 ✓ (통합명 분리 후)')

# 6. 기온 데이터와 온열질환 데이터 결합 및 여름철(6, 7, 8월) 필터링
_d9=_tmax9.merge(_cnt9,on=['시도','dt'],how='left'); _d9['온열']=_d9['온열'].fillna(0)
_d9['m']=pd.to_datetime(_d9['dt']).dt.month; _d9=_d9[_d9['m'].isin([6,7,8])]

# 7. [전국 통합 모델] 시도 고정효과(Fixed Effects, FE) 포함 포아송 회귀 분석
# Formula: 온열 ~ 최고기온 + 시도별 Dummy
_m9=_smf9.glm('온열 ~ 최고기온 + C(시도)',data=_d9,family=_sm9.families.Poisson()).fit()
print(f'지역 FE Poisson 기온배수: ×{np.exp(_m9.params["최고기온"]):.3f}/℃  (전국 단일 §9-b = ×{1.575:.3f})')

# 8. [시도별 개별 모델] 17개 각 시도 독립 포아송 회귀 분석 수행 (필터링 조건 없이 전수 출력)
print('시도별 개별 기울기 — 전수 출력(필터 없음):')
_sl9=[]
for _s9,_gg9 in _d9.groupby('시도'):

    # 개별 시도 데이터에 대한 포아송 회귀 fit
    _r9=_smf9.glm('온열 ~ 최고기온',data=_gg9,family=_sm9.families.Poisson()).fit()

    # (시도명, 기온 1℃ 상승당 온열질환 Risk Ratio(exp(beta)), 관측일수, 총 환자수)
    _sl9.append((_s9,float(np.exp(_r9.params['최고기온'])),len(_gg9),int(_gg9['온열'].sum())))

# 9. 시도별 반응 기울기(배수) 내림차순 정렬 출력
print(f"  {'시도':12}{'배수':>7}{'일수':>6}{'온열n':>7}")
for _s9,_v9,_n9,_c9 in sorted(_sl9,key=lambda x:-x[1]): 
    print(f'  {_s9:12}×{_v9:.2f}{_n9:>6}{_c9:>7}')

# 10. 결과 요약 및 결론 검증
_d9dict=dict((a,b) for a,b,_,_ in _sl9)
print(f'→ 17개 시도 {min(v for _,v,_,_ in _sl9):.2f}~{max(v for _,v,_,_ in _sl9):.2f} (좁음). 인천={_d9dict["인천광역시"]:.2f} · 광주={_d9dict["광주광역시"]:.2f}(복구됨). 전국 ×1.6은 robust.')

기온 없는 NEDIS 시도: 없음 — 17/17 매칭 ✓ (통합명 분리 후)
지역 FE Poisson 기온배수: ×1.557/℃  (전국 단일 §9-b = ×1.575)
시도별 개별 기울기 — 전수 출력(필터 없음):
  시도               배수    일수    온열n
  충청북도        ×1.66   552    673
  인천광역시       ×1.63   552    742
  경기도         ×1.59   552   3056
  충청남도        ×1.58   552    889
  강원특별자치도     ×1.57   552    564
  전북특별자치도     ×1.55   552    880
  전라남도        ×1.55   552   1240
  경상남도        ×1.55   552   1303
  세종특별자치시     ×1.55   552    127
  대전광역시       ×1.54   552    214
  광주광역시       ×1.54   552    280
  서울특별시       ×1.53   552    993
  울산광역시       ×1.51   552    412
  부산광역시       ×1.51   552    514
  경상북도        ×1.51   552   1275
  대구광역시       ×1.50   552    334
  제주특별자치도     ×1.44   552    499
→ 17개 시도 1.44~1.66 (좁음). 인천=1.63 · 광주=1.54(복구됨). 전국 ×1.6은 robust.


### §9-d. 시군구 단위 dose-response — 최근접 관측소 기온 매핑 `[🔖 2026-07-22 신규 · 사용자 요청]`

§9-c는 광역시·도 단위였다. AIDC 후보지는 **시** 단위이므로(포항시·당진시·동해시...), 시군구별로 다시 잰다:
- **기온**: 시군구 인구가중 중심점(집계구) → **최근접 ASOS 관측소**의 일최고기온 (거리 병기 — 당진은 서산 21km ⚠)
- **온열**: NEDIS 발생시군구 (§0 norm_key + 군위군 대구 편입 alias)
- **모형**: (a) 시군구 FE 공통 기울기 (b) AIDC 후보 시군구 개별 기울기 + 95% CI + 표본 크기 — **필터로 숨기지 않고 얇으면 얇다고 표시**(§9-c 교훈)

In [29]:
# ── §9-d. 시군구 단위 dose-response — 최근접 ASOS [🔖 2026-07-22 신규] ──
import pyogrio as _pg9, geopandas as _gp9

# 1. 집계구(OA)별 인구 총괄 데이터 로드 및 집계구 코드(oa)-인구수(val) 딕셔너리 생성
_pop9=pd.read_csv('0718_2024년 인구총괄/2025년기준_2024년_인구총괄(총인구).csv',header=None,names=['y','oa','item','val'],dtype={'oa':str},encoding='cp949')
_pop9=_pop9[_pop9.item=='to_in_001']; _POP9=_pop9.set_index('oa')['val'].to_dict()

# 2. 행정구역 참조 코드 로드 및 시군구 단위 키(code5) 추출·정제 (예: 서울 종로구 -> 시도_시군구 정규화 키)
_awb9=_CW9.from_path('0718_ref_code/ref_code/1. 행정구역 코드(adm_code).xls')
_ad9=pd.DataFrame(_awb9.get_sheet_by_name('2025년 6월').to_python()[2:],columns=['sido','sido_nm','sgg','sgg_nm','emd','emd_nm'])
_ad9['code5']=_ad9['sido'].astype(str).str.replace('.0','',regex=False).str.zfill(2)+_ad9['sgg'].astype(str).str.zfill(3)
_m59={c:norm_key(s,g) for c,s,g in zip(_ad9['code5'],_ad9['sido_nm'],_ad9['sgg_nm'])}

# 3. 2025년 2분기 집계구 경계(SHP) 파일 로드 및 집계구별 중앙점(Centroid) 좌표 추출
_oa9=_pg9.read_dataframe('0718_2025년 집계구경계/bnd_oa_00_2025_2Q.shp',columns=['TOT_OA_CD'])
_oa9['key']=_oa9['TOT_OA_CD'].str[:5].map(_m59); _oa9['p']=_oa9['TOT_OA_CD'].map(_POP9).fillna(0.0)
_c59=_oa9.geometry.centroid; _oa9['x']=_c59.x; _oa9['y']=_c59.y

# 4. 시군구별 '인구 가중 중심점(Population-Weighted Centroid)' 계산 (단순 지형 중심이 아닌 인구 밀집 위치 반영)
_cent9=_oa9.groupby('key').apply(lambda g: pd.Series({'x':np.average(g.x,weights=np.maximum(g.p,1e-9)),'y':np.average(g.y,weights=np.maximum(g.p,1e-9))}))

# 5. 중심점 좌표계를 UTM-K(EPSG:5179)에서 위경도(EPSG:4326)로 변환
_cg9=_gp9.GeoDataFrame(_cent9,geometry=_gp9.points_from_xy(_cent9.x,_cent9.y),crs=5179).to_crs(4326)
_cent9['lat']=_cg9.geometry.y; _cent9['lon']=_cg9.geometry.x

# 6. 유효한 관측값이 존재하는 ASOS 기상관측소 정보 필터링
_st9=_info9.dropna(subset=['위도','경도']); _st9=_st9[_st9['지점'].isin(set(_aso9['지점']))]

def _hav9(lat1,lon1,lat2,lon2):
    """하버사인(Haversine) 공식을 통한 위경도 간 대권거리(km) 계산 함수"""
    a=np.radians([lat1,lon1,lat2,lon2]); return 6371*2*np.arcsin(np.sqrt(np.sin((a[2]-a[0])/2)**2+np.cos(a[0])*np.cos(a[2])*np.sin((a[3]-a[1])/2)**2))

# 7. 각 시군구 인구 중심점에서 가장 가까운(최근접) ASOS 관측소 탐색 및 매핑
_near9={}
for _k,_r in _cent9.iterrows():
    _d=_st9.apply(lambda s:_hav9(_r.lat,_r.lon,s['위도'],s['경도']),axis=1); _i=_d.idxmin()
    _near9[_k]=(_st9.loc[_i,'지점'],_st9.loc[_i,'지점명'],float(_d.min()))

# 8. 관측소별 일자별 최고기온 집계 및 행정구역 매칭 키 보정 (행정구역 개편 처리)
_stmax9=_aso9.groupby(['지점','dt'])['최고기온(°C)'].mean().rename('최고기온')
_ALIAS9={'경상북도 군위군':'대구광역시 군위군'}          # 2023-07 대구 편입 — NEDIS 구키 보정

_neK9=_ne9.dropna(subset=['dt']).copy()
_neK9['key']=[_ALIAS9.get(k,k) for k in (norm_key(a,b) for a,b in zip(_neK9['발생시도'],_neK9['발생시군구']))]
_neK9['m']=pd.to_datetime(_neK9['dt']).dt.month; _neK9=_neK9[_neK9['m'].isin([6,7,8])]

# 키 매칭 미흡 시 경고 출력
_unm9=set(_neK9['key'])-set(_cent9.index)
if _unm9:
    # 매칭 안 된 NEDIS 시군구 키를 숨기지 않고 보고 (§9-c 교훈: silent 필터 금지)
    print('⚠ 매칭 실패 키:',sorted(_unm9),'| 후보:',[k for k in _cent9.index if any(u.split()[-1][:2] in k for u in _unm9)][:4])

# 9. 시군구/일자별 온열질환 집계 데이터와 기온 데이터를 조인하기 위한 패널 패널 패널(Grid) 생성
_ck9=_neK9.groupby(['key','dt']).size().rename('온열').reset_index()
_days9=sorted(_d9['dt'].unique()); _keys9=[k for k in _cent9.index if k in _near9]
_gr9=pd.MultiIndex.from_product([_keys9,_days9],names=['key','dt']).to_frame(index=False)
_gr9['지점']=_gr9['key'].map(lambda k:_near9[k][0])
_gr9=_gr9.merge(_stmax9.reset_index(),on=['지점','dt'],how='left').merge(_ck9,on=['key','dt'],how='left')
_gr9['온열']=_gr9['온열'].fillna(0); _gr9=_gr9.dropna(subset=['최고기온'])

print(f'그리드 {len(_gr9):,}행 · 시군구 {_gr9.key.nunique()} · 최근접 관측소 평균거리 {np.mean([v[2] for v in _near9.values()]):.1f}km')

# 10. [시군구 고정효과(FE) 모델] 환자 발생 이력이 1건 이상 있는 시군구 대상 포아송 회귀 분석
_act9=_gr9.groupby('key')['온열'].sum()
_mP9=_smf9.glm('온열 ~ 최고기온 + C(key)',data=_gr9[_gr9['key'].isin(_act9[_act9>0].index)],family=_sm9.families.Poisson()).fit()

print(f'★ 시군구 FE 공통 기울기: ×{np.exp(_mP9.params["최고기온"]):.3f}/℃  (시도 FE {np.exp(_m9.params["최고기온"]):.3f} · 전국 1.575 — FE가 촘촘할수록 소폭↓, 방향 일관)')

# 11. 주요 산업단지/관심 시군구(AIDC) 대상 개별 기울기(Risk Ratio) 및 95% 신뢰구간(CI) 산출
_AIDC9=['인천광역시 동구','인천광역시 서구','경상북도 포항시','울산광역시 울주군','울산광역시 남구','경상북도 구미시','충청남도 당진시','강원특별자치도 동해시','전라남도 광양시','전라남도 여수시']


print(f"{'시군구':16}{'배수':>7}{'95%CI':>15}{'온열n':>7}   관측소(거리)")
for _k in _AIDC9:
    _g=_gr9[_gr9.key==_k]
    if not len(_g):
        print(f'{_k:16} 그리드 없음')
        continue

    _nc=int(_g['온열'].sum())
    _r=_smf9.glm('온열 ~ 최고기온',data=_g,family=_sm9.families.Poisson()).fit()

    # 신뢰구간(CI) 및 온열질환 Risk Ratio(exp(beta)) 계산
    _lo,_hi=np.exp(_r.conf_int().loc['최고기온']); _v=np.exp(_r.params['최고기온'])
    print(f"{_k:16}×{_v:.2f}  [{_lo:.2f},{_hi:.2f}]{_nc:>7}   {_near9[_k][1]}({_near9[_k][2]:.0f}km){'' if _nc>=100 else ' ⚠표본얇음'}")

# 12. 해석 및 유의사항 출력
print('→ 인천 동구 자체 기울기가 시도값(1.63)보다 가파름(표본 얇아 CI 넓음) — §11-b(3)의 시도값 사용은 보수적.')
print('  당진은 최근접이 서산 21km — 기온 대표성 주의. 동해 n=34 얇음. 판단은 CI와 함께.')

# ── 전국 robust 확인 (사용자 질문): AIDC 시군구만이 아니라, 표본이 충분한 모든 시군구의 기울기 분포 ──
# 온열 100건 이상 시군구만 개별 회귀 (그 미만은 소표본 노이즈라 분포를 흐림)
_all9=[]
for _k,_g in _gr9.groupby('key'):
    if _g['온열'].sum()<100:
        continue
    try:
        _all9.append(float(np.exp(_smf9.glm('온열 ~ 최고기온',data=_g,family=_sm9.families.Poisson()).fit().params['최고기온'])))
    except Exception:
        pass
_q9=np.percentile(_all9,[25,50,75])
print(f'전국 분포(온열≥100건, n={len(_all9)}개 시군구): 중앙값 ×{_q9[1]:.2f} · IQR [{_q9[0]:.2f},{_q9[2]:.2f}] · 범위 {min(_all9):.2f}~{max(_all9):.2f}')
print('→ ×1.6은 분포 중심대에 있는 대표값 — 특정 지역 선택의 산물이 아님.')

⚠ 매칭 실패 키: ['제주특별자치도 서귀포시'] | 후보: []
그리드 125,258행 · 시군구 228 · 최근접 관측소 평균거리 10.1km
★ 시군구 FE 공통 기울기: ×1.507/℃  (시도 FE 1.557 · 전국 1.575 — FE가 촘촘할수록 소폭↓, 방향 일관)
시군구                  배수          95%CI    온열n   관측소(거리)
인천광역시 동구        ×1.93  [1.55,2.42]     19   인천(1km) ⚠표본얇음
인천광역시 서구        ×1.56  [1.48,1.65]    200   인천(9km)
경상북도 포항시        ×1.37  [1.31,1.43]    278   포항(1km)
울산광역시 울주군       ×1.57  [1.44,1.70]    115   울산(10km)
울산광역시 남구        ×1.49  [1.36,1.62]     98   울산(3km) ⚠표본얇음
경상북도 구미시        ×1.46  [1.38,1.56]    196   구미(4km)
충청남도 당진시        ×1.54  [1.43,1.65]    135   서산(21km)
강원특별자치도 동해시     ×1.37  [1.24,1.51]     34   동해(2km) ⚠표본얇음
전라남도 광양시        ×1.39  [1.31,1.48]    191   광양시(3km)
전라남도 여수시        ×1.54  [1.45,1.65]    201   여수(5km)
→ 인천 동구 자체 기울기가 시도값(1.63)보다 가파름(표본 얇아 CI 넓음) — §11-b(3)의 시도값 사용은 보수적.
  당진은 최근접이 서산 21km — 기온 대표성 주의. 동해 n=34 얇음. 판단은 CI와 함께.
전국 분포(온열≥100건, n=42개 시군구): 중앙값 ×1.53 · IQR [1.45,1.57] · 범위 1.36~1.81
→ ×1.6은 분포 중심대에 있는 대표값 — 특정 지역 선택의 산물이 아님.


### §9-e. 부지별 dose 배수 채택 — §9-c(시도) vs §9-d(시군구) `[🔖 2026-07-24 신규 · 사용자 요청]`

§9-c는 시도, §9-d는 시군구 단위 기울기를 냈다. AIDC 부지별 시나리오에는 **더 국지적인 값이 원칙적으로 정확**하되, 표본이 얇거나 관측소가 멀면 시군구 값이 불안정하다. 기계적 채택 규칙:

> **n(온열 건수) ≥ 100 이고 최근접 관측소 ≤ 10km → 시군구(§9-d), 아니면 시도(§9-c)**

임의 선택 여지를 없애기 위해 규칙을 먼저 고정하고 적용한다.

In [ ]:
# ── §9-e. 부지별 배수 채택 (규칙: n≥100 & 관측소≤10km → 시군구, 아니면 시도) [🔖 2026-07-24 신규] ──
# 목적: 부지별 특성에 맞춰 통계적 신뢰도가 높은 최선의 배수(온도 상승에 따른 위험증가율 등)를 채택함.
# §9-c(_d9dict: 시도별 배수) 및 §9-d(_gr9: 시군구 데이터, _near9: 가장 가까운 기상관측소 정보)의 결과를 재사용함.

# (1) 대상 분석 부지 목록 정의: (부지 표시 이름, 시군구 키, 상위 시도 이름)
_SITE9E=[('인천 동구 (현대·동국)','인천광역시 동구','인천광역시'),
         ('인천 서구 (KG스틸)','인천광역시 서구','인천광역시'),
         ('울산 남구 (미포)','울산광역시 남구','울산광역시'),
         ('울주 (온산·하이테크)','울산광역시 울주군','울산광역시'),
         ('포항 (제철·심팩·광명)','경상북도 포항시','경상북도'),
         ('구미','경상북도 구미시','경상북도'),
         ('당진','충청남도 당진시','충청남도'),
         ('동해 (북평·북평2)','강원특별자치도 동해시','강원특별자치도'),
         ('광양','전라남도 광양시','전라남도'),
         ('여수','전라남도 여수시','전라남도')
         ]

# 각 부지(시군구 키)별 최종 채택된 배수를 저장할 딕셔너리
BETA_SITE={}

print(f"{'부지':22}{'시군구배수':>9}{'n':>6}{'관측소(거리)':>16}{'시도배수':>8}{'채택':>7}  사유")

# -----------------------------------------------------------------------------
# (2) 각 부지별 시군구 vs 시도 배수 계산 및 조건부 채택 로직 실행
# -----------------------------------------------------------------------------
for _lbl,_key,_sido in _SITE9E:
    # 해당 시군구의 데이터 추출
    _g=_gr9[_gr9.key==_key]

    # 1) 표본 수(n: 총 온열질환 발생건수) 추출
    _n=int(_g['온열'].sum()) if len(_g) else 0

    # 2) 가장 가까운 기상관측소와의 거리(km) 및 관측소명 추출
    _dist=_near9[_key][2] if _key in _near9 else float('inf')
    _stn=_near9[_key][1] if _key in _near9 else '—'

    # 3) 개별 시군구 기준 포아송 회귀분석(GLM)을 통한 온도 반응 배수(exp(Beta)) 산출
    try:
        _vsgg=float(np.exp(_smf9.glm('온열 ~ 최고기온',data=_g,family=_sm9.families.Poisson()).fit().params['최고기온']))
    except Exception:
        _vsgg=float('nan')  # 데이터 수 부족 등의 이유로 회귀 적합 실패 시 NaN 처리

    # 4) 해당 부지가 속한 상위 '시도' 단위의 배수 가져오기
    _vsido=_d9dict.get(_sido,float('nan'))

    # -------------------------------------------------------------------------
    # 5) 채택 조건 판단 규칙 (Rule)
    #    [조건 1] 온열 환자 표본 수가 100건 이상이고,
    #    [조건 2] 기상관측소와의 거리가 10km 이내일 경우만 '시군구' 배수 채택
    #    그 외의 경우(표본 부족 또는 원거리 관측소)는 신뢰성을 위해 '시도' 배수 채택
    # -------------------------------------------------------------------------
    if _n>=100 and _dist<=10:
        _pick=_vsgg; _why='시군구'
    elif _n<100:
        _pick=_vsido; _why=f'시도(표본 얇음 n={_n})'
    else:
        _pick=_vsido; _why=f'시도(관측소 {_dist:.0f}km 원거리)'

    # 최종 선택된 배수 값을 딕셔너리에 저장
    BETA_SITE[_key]=_pick

    # 부지별 판단 결과 행 단위 출력
    print(f"{_lbl:22}{_vsgg:>9.2f}{_n:>6}{f'{_stn}({_dist:.0f}km)':>16}{_vsido:>8.2f}{_pick:>7.2f}  {_why}")
print()

# -----------------------------------------------------------------------------
# (3) 결과 요약, 정합성 검증 및 해석 출력
# -----------------------------------------------------------------------------
print(f'정합성: 시군구 FE ×{np.exp(_mP9.params["최고기온"]):.3f} ≈ 시도 FE ×{np.exp(_m9.params["최고기온"]):.3f} ≈ 전국 ×1.575 — 층위가 촘촘할수록 소폭↓, 방향 일관.')
print('개별 시군구 1.37~1.57은 모두 시도 분포 [1.44,1.66] 안. 인천 동구 1.93만 예외인데 n=19로 얇아 규칙상 시도(1.63) 채택.')
print('⚠ 울산 남구 n=98은 문턱(100) 경계선 — 시군구값 1.49 vs 채택 시도값 1.51, 차이 0.02로 결론 불변(규칙을 사후 조정하지 않음).')
print('→ BETA_SITE 딕셔너리로 후속 시나리오(§11-b(3)·§18-b2)에서 부지별 배수 사용 가능.')


# ── 부지별 추가위험 계수 f_site() [🔖 2026-07-24 신설 — 사용자 지적] ──
#   지금까지 §10~§18은 전국 상수 f = 1.6² − 1 = 1.56 을 모든 부지에 똑같이 썼다.
#   그런데 §9-c/§9-d/§9-e에서 부지마다 1℃당 온열배수 m을 따로 채택했으므로(시군구 or 시도),
#   +ΔT에서의 '추가' 위험분도 부지별로 f_s = m_s^ΔT − 1 이어야 한다.
#   ⚠ 헷갈리기 쉬운 지점: BETA_SITE 값(≈1.4~1.6)은 **1℃당 배수 m**이고,
#      옛 상수 1.56은 **+2℃에서의 추가분**이다. 숫자가 닮았을 뿐 서로 다른 양이라 그대로 바꿔 끼우면 틀린다.
def f_site(key, dT=2.0):
    """부지(시군구 key)의 채택 배수 m으로 ΔT에서의 추가위험분 (m^ΔT − 1)을 돌려준다.
    BETA_SITE에 없거나 NaN이면 전국 실측 배수(§9-① HEAT_MULT_PER_C)로 폴백."""
    m = BETA_SITE.get(key, HEAT_MULT_PER_C)
    if not np.isfinite(m):
        m = HEAT_MULT_PER_C
    return m**dT - 1

print(f'f_site() 정의됨 — 전국 참조 f={HEAT_MULT_PER_C**2-1:.3f}(m={HEAT_MULT_PER_C:.3f}) vs 부지별 f_s 범위 '
      f'{min(f_site(k) for k in BETA_SITE):.3f}~{max(f_site(k) for k in BETA_SITE):.3f}')

# ── 거리 감쇠 dose 체인 — 단일 정의 [🔖 2026-07-25 사용자 지적] ──
#   지금까지 §10-③·§11-④·§12-b·§13-b·§18-b2는 전부 "부지에서 3km까지 기온이 +2℃ 오른다"는
#   **평평한 가정**을 썼다. 그런데 케임브리지 +2℃는 (a)지표온도(LST)이고 (b)거리에 따라 감쇠한다.
#   §11-b(3)만 그 사슬을 제대로 썼고 나머지는 안 썼다 = 같은 목적의 계산이 두 규약으로 갈라져 있었다.
#   → 감쇠·환산·dose를 여기 한 번만 정의하고 모든 절이 이 함수를 부른다.

# [🔖 2026-07-25 논문 Figure 3 본문 수치로 적합 — 사용자 요청]
#   Marinoni 외, "The Data Heat Island Effect" p.5-6이 보고하는 점은 셋이다:
#     · 부지 평균 LST 증가 **2.07℃** (Table 1: k=12~120에서 2.03~2.12 · 최소 0.30 · 최대 9.02)
#     · **4.5km에서 1.0℃** ("an average monthly LST increase of 1°C ... up to 4.5 km")
#     · **7km에서 강도 30%** ("reduce its intensity to 30% within 7 km")
#     · **10km까지 도달** ("the impact of LST increase reaches up to 10 km")
#   두 점(4.5km 1.0℃ · 7km 30%)을 지수 감쇠에 각각 맞추면 반감기가 4.29km·4.03km로 나온다.
#   → 중간값 4.2km를 채택하면 둘 다 근사한다(4.5km 0.99℃ · 7km 31.5%). 검정은 아래 출력 참조.
# ⚠⚠ [🔖 2026-07-28 사용자 지적 — 반드시 읽을 것] 케임브리지 Δ는 **시간 변화**다. 공간 대비가 아니다.
#   논문 식(1)(2) (p.4):  Δ^r_0(k) = T̄^r_0 − (1/k)·Σ_{j=1..k} T̄^r_{−j}
#   = 거리 r 링의 LST를, **같은 링의 가동개시 직전 k개월 평균**과 비교한 값. k=60이면 5년, 120이면 10년.
#   원문: "the average LST increase measured over each AI data centre with respect to the mean of the
#          LST recorded over the k months **before their start of operations**"
#   ⇒ 2.07℃는 "주변보다 2.07℃ 뜨겁다"가 **아니라** "가동 후 2.07℃ 올랐다"이다.
#
#   [그래서 무엇이 괜찮고 무엇이 안 괜찮은가]
#   ✅ dT_at(r) 로 쓰는 것은 **맞다.** 식(2)가 거리 r 별로 정의돼 있으므로 Δ(r)은 이미
#      '거리에 따른 상승분 프로파일'이다. "AIDC가 오면 r km 지점이 얼마나 오르나"에 정확히 대응한다.
#   ❌ **§14의 우리 ΔLST(부지 vs 10~20km 도넛, +5.5~17.3℃)와 나란히 두면 안 된다.** 그건 시간 변화가
#      아니라 같은 시각의 공간 대비이고, 산단의 열뿐 아니라 **피복·시가지·항만까지 통째로** 재고 있다.
#      "케임브리지 2.07 vs 우리 17.3" 식 비교는 서로 다른 양의 비교다. (§14-d 서술 정정 대상)
#   ❌ Sailor(+2.2℃)도 또 다른 구성개념이다 — 같은 시각의 **풍하 vs 풍상** 공간 대비.
#      셋을 한 축에 늘어놓지 않는다. 자세한 대조는 아래 §9-e2.
CAMB_DT0, CAMB_HALF = 2.07, 4.2    # 케임브리지 hyperscaler: 가동 전후 **상승분** 평균 +2.07℃ · 반감기 4.2km
BETA_L2A = 0.258                   # §15 채택 β (지표 1℃ → 기온 0.258℃)

# 감쇠 **형태**는 논문이 정해주지 않는다 [🔖 2026-07-25 사용자 요청 — 바꿔 끼울 수 있게].
#   케임브리지가 보고하는 건 세 점이다: 부지 +2℃ · 4.5km +1℃ · 10km ~0.
#   이 세 점을 잇는 곡선은 하나가 아니다. 지수(반감기 4.5km)·두 구간 선형·가우시안이 모두 근사한다.
#   ⚠ 실제로 이 노트북 안에서도 §11-b(2)는 선형을, §11-b(3)은 지수를 쓰고 있었다(같은 논문·다른 곡선).
#   → 형태를 dict로 빼고 DECAY_SHAPE 한 줄만 바꾸면 아래 전 절이 따라 바뀌게 한다. 민감도는 바로 아래 출력.
def _dec_exp(r):
    return CAMB_DT0*np.exp(-r*np.log(2)/CAMB_HALF)                    # 반감기 CAMB_HALF
def _dec_lin(r):
    # 논문 보고점을 직선 두 구간으로 잇는 대안 — 0km 2.07 → 4.5km 1.0 → 10km 0
    if r <= 4.5:
        return CAMB_DT0-(CAMB_DT0-1.0)/4.5*r
    return max(0.0, 1.0-1.0/(10-4.5)*(r-4.5))
def _dec_gauss(r):
    return CAMB_DT0*np.exp(-np.log(2)*(r/4.5)**2)                     # 4.5km에서 절반
DECAY = {'exp': _dec_exp, 'lin': _dec_lin, 'gauss': _dec_gauss}
DECAY_SHAPE = 'exp'        # ← 여기만 바꾸면 §11-b·§12-b·§13이 전부 따라 바뀐다

# β도 한 점이 아니다 [🔖 2026-07-25 사용자 지적 — "0.48℃는 너무 작은 것 아닌가"].
#   §15-e는 **회귀희석을 기각하지 못했다** — 그렇다면 채택 β=0.258은 눌린 **하한**이고,
#   희석 보정치 β≈0.459가 상한 후보다. 셋을 나란히 두면:
#     · 하한 0.258 → 부지 기온 +0.52℃  (우리가 지금까지 쓰던 값)
#     · 상한 0.459 → 부지 기온 +0.92℃  (Sailor가 36~169MW 공랭에서 잰 평균 +0.7~0.9℃와 맞물린다)
#   GW급 캠퍼스가 중소형 공랭 시설보다 낮게 나오는 건 앞뒤가 안 맞으므로, 하한만 쓰지 않고 범위로 낸다.
BETA_L2A_HI = 0.459        # §15-e 희석 보정치(상한 후보)

def dT_at(r_km, shape=None, beta=None):
    """부지 경계에서 r km 지점의 **기온** 상승(℃) = 케임브리지 지표온도 감쇠 × β.
    케임브리지가 잰 +2℃는 지표온도(LST)이므로 β를 반드시 통과시킨다.
    beta를 주면 그 값으로(예: BETA_L2A_HI 상한), 안 주면 채택 β(하한)."""
    return DECAY[shape or DECAY_SHAPE](r_km)*(beta if beta is not None else BETA_L2A)

# ── Sailor 근접 상한 — 케임브리지 곡선에 얹지 않는다 [🔖 2026-07-25] ──
#   Sailor 외(2025)는 위성 LST가 아니라 **차량 실측 공기온도**이고, 풍상↔풍하 대비다.
#   즉 등방 감쇠가 아니라 바람 방향 plume이고, 탐지 거리도 100~500m로 케임브리지(10km)와 스케일이 다르다.
#   ⚠ Sailor 진폭(+2.2℃)에 케임브리지 감쇠(반감기 4.5km)를 씌우면 원거리를 크게 과장한다 — 하지 않는다.
#   → Sailor는 **0~0.5km 링 한정 공기온도 상한**으로만 쓴다. 시설도 36~169MW 공랭이라 GW급 수랭엔 외삽 주의.
SAILOR_NEAR_KM, SAILOR_AIR_MAX, SAILOR_AIR_MEAN = 0.5, 2.2, 0.8

def dT_near_sailor(r_km, stat='max'):
    """0~0.5km 한정 공기온도 상한(℃). 그 밖은 0 — 감쇠 곡선이 아니라 '측정된 구간'이다."""
    if r_km > SAILOR_NEAR_KM:
        return 0.0
    return SAILOR_AIR_MAX if stat == 'max' else SAILOR_AIR_MEAN

def excess_ring(key, pop, rate, r_km, shape=None, eld=None, E_ELD=None, E_YNG=None, beta=None):
    """링 하나의 연간 추가 온열 건수. eld/E_ELD/E_YNG를 주면 연령가중, 아니면 비가중.
    shape로 감쇠 형태를, beta로 지표→기온 환산계수를 바꿔 끼울 수 있다."""
    _f = f_site(key, dT_at(r_km, shape, beta))
    if eld is None or E_ELD is None:
        return pop*rate/1e5*_f
    return (eld*rate*E_ELD + (pop-eld)*rate*E_YNG)/1e5*_f

print()
print(f"거리 감쇠 엔진 — 기온 ΔT(℃) · 채택 형태 DECAY_SHAPE='{DECAY_SHAPE}' [단일 정의, 이하 전 절이 호출]")
print(f"  {'거리km':>7}{'지수':>9}{'선형':>9}{'가우시안':>10}   ← 형태별 기온ΔT(=지표ΔT×β {BETA_L2A})")
for _r in (0.5, 1.5, 2.5, 3.75, 7.25):
    print(f"  {_r:>7.2f}"+''.join(f"{dT_at(_r,_s):>9.2f}" for _s in ('exp','lin','gauss')))
# ── 형태 검정: 논문이 직접 보고한 세 점에 대보기 [🔖 2026-07-25] ──
print()
print('형태 검정 — 논문 Figure 3 본문 보고점 대조 (지표온도 기준, β 적용 전)')
print(f"  {'기준':28}{'논문':>10}{'지수':>10}{'선형':>10}{'가우시안':>11}")
_r45={s:DECAY[s](4.5) for s in DECAY}
print(f"  {'4.5km LST 증가(℃)':28}{'1.00':>10}"+''.join(f"{_r45[s]:>10.2f}" for s in ('exp','lin','gauss')))
_r7={s:DECAY[s](7.0)/CAMB_DT0*100 for s in DECAY}
print(f"  {'7km 강도(부지 대비 %)':28}{'30':>10}"+''.join(f"{_r7[s]:>10.0f}" for s in ('exp','lin','gauss')))
_r10={s:DECAY[s](10.0) for s in DECAY}
print(f"  {'10km LST 증가(℃)':28}{'>0 (도달)':>10}"+''.join(f"{_r10[s]:>10.2f}" for s in ('exp','lin','gauss')))
print("  → **지수만 셋을 모두 통과한다.** 선형은 10km에서 정확히 0이 되어 '10km까지 도달'과 어긋나고,")
print("    가우시안은 7km에서 너무 빨리 떨어진다. 그래서 기본 형태를 'exp'로 둔다.")
print(f"    형태를 바꿔도 결론이 유지되는지는 DECAY_SHAPE만 고쳐 재실행하면 확인된다(현재 '{DECAY_SHAPE}').")
print(f"  → 구 방식은 3km까지 전부 ΔT=+2.00℃(기온)로 뒀다. 실제론 0.5km에서도 {dT_at(0.5):.2f}℃다.")
print()
print(f"β 범위: 채택 {BETA_L2A}(하한) ~ 희석보정 {BETA_L2A_HI}(상한) → 부지 기온 "
      f"+{dT_at(0):.2f}~{dT_at(0,beta=BETA_L2A_HI):.2f}℃ · 3km 지점 +{dT_at(2.5):.2f}~{dT_at(2.5,beta=BETA_L2A_HI):.2f}℃")
print(f"Sailor 근접 상한(별도 계열): ≤{SAILOR_NEAR_KM}km에서 공기온도 +{SAILOR_AIR_MEAN}~{SAILOR_AIR_MAX}℃ (그 밖 미탐지)")
print("  ⚠ 케임브리지 곡선에 이 진폭을 얹지 않는다 — 위성 LST(등방·10km) vs 차량 공기온도(풍하 plume·500m)로")
print("    측정 자체가 다르다. 근접 링에만 별도 시나리오로 얹는다(§11-b(3)).")


부지                        시군구배수     n         관측소(거리)    시도배수     채택  사유
인천 동구 (현대·동국)              1.93    19         인천(1km)    1.63   1.63  시도(표본 얇음 n=19)
인천 서구 (KG스틸)               1.56   200         인천(9km)    1.63   1.56  시군구
울산 남구 (미포)                 1.49    98         울산(3km)    1.51   1.51  시도(표본 얇음 n=98)
울주 (온산·하이테크)               1.57   115        울산(10km)    1.51   1.57  시군구
포항 (제철·심팩·광명)              1.37   278         포항(1km)    1.51   1.37  시군구
구미                         1.46   196         구미(4km)    1.51   1.46  시군구
당진                         1.54   135        서산(21km)    1.58   1.58  시도(관측소 21km 원거리)
동해 (북평·북평2)                1.37    34         동해(2km)    1.57   1.57  시도(표본 얇음 n=34)
광양                         1.39   191        광양시(3km)    1.55   1.39  시군구
여수                         1.54   201         여수(5km)    1.55   1.54  시군구

정합성: 시군구 FE ×1.507 ≈ 시도 FE ×1.557 ≈ 전국 ×1.575 — 층위가 촘촘할수록 소폭↓, 방향 일관.
개별 시군구 1.37~1.57은 모두 시도 분포 [1.44,1.66] 안. 인천 동구 1.93만 예외인데 n=19로 얇아 규칙

### §9-e2. 시간이냐 공간이냐 — 세 측정은 서로 다른 것을 재고 있다 `[🔖 2026-07-28 사용자 지적]`

우리는 세 출처의 숫자를 같은 문단에서 써 왔다. **셋은 구성개념이 다르다.**

| 출처 | 무엇과 무엇을 비교하나 | 종류 | 값 |
|---|---|---|---|
| **케임브리지** Marinoni 식(1)(2) | 같은 자리·같은 링 — **가동 개시 직전 k개월** 평균과 | **시간** 변화 (거리별) | 부지 +2.07℃ · 4.5km +1.0℃ · 10km 도달 |
| **Sailor** ASU 2025 | 같은 시각 — **풍하 vs 풍상** | **공간** 대비 (바람으로 분리) | 최대 +2.2℃ · ~500m |
| **우리 §14** | 같은 장면 — **부지 vs 10~20km 도넛** | **공간** 대비 (거리로 분리) | +5.5~17.3℃ |

**케임브리지만 "얼마나 올랐나"를 잰다.** Sailor와 우리 것은 "지금 얼마나 다른가"다.

그래서:

- ✅ **`dT_at(r)`은 그대로 맞다.** 케임브리지 식(2)는 거리 r 마다 따로 정의돼 있어서, Δ(r) 자체가 이미 *거리에 따른 상승분 프로파일*이다. "AIDC가 오면 r km 지점 기온이 얼마나 오르나"에 정확히 대응한다.
- ❌ **우리 §14 ΔLST를 케임브리지 2.07과 나란히 두면 안 된다.** 우리 +17.3℃는 상승분이 아니라 *지금 상태*이고, 산단의 열뿐 아니라 **피복·시가지·항만**까지 통째로 포함한다. Sailor는 풍상/풍하로 피복을 어느 정도 맞췄지만, 우리 도넛 대비는 아무것도 맞추지 못한다.
- ❌ **"산단 열은 근처만, DC 열은 광역"이라는 구분도 우리 자료로는 못 한다.** 우리 링 프로파일(아래)은 5~10km에서도 +0.6~3.1℃인데, 그건 "산단 폐열이 10km를 갔다"가 아니라 **"이 일대가 원래 시가지다"**를 상당 부분 재고 있다. 산단에 대해 케임브리지식 *시간* 비교를 한 적이 없으므로, 두 열원의 확산 반경을 비교할 근거가 우리에겐 없다.

**그러면 맞출 방법은 하나다 — 우리도 시간으로 재면 된다.** 가동 개시 시점이 뚜렷한 국내 데이터센터에 케임브리지와 같은 식을 적용한다. 아래가 그 첫 사례이고, **결과는 재현 실패**다. 왜 그것이 그냥 실패가 아니라 정보인지도 함께 적는다.

In [ ]:
# ── §9-e2. 케임브리지식 '시간 비교'를 우리 자료로 — 네이버 각 세종 [🔖 2026-07-28 신규] ──
# 왜: 케임브리지 Δ는 시간 변화인데 우리 §14 ΔLST는 공간 대비다. 둘을 비교하려면 우리도 시간으로 재야 한다.
# 어떻게: 가동 개시일(2023-11)이 뚜렷한 각 세종에 같은 구조를 적용.
#   장면마다  ΔT = (부지 반경 R 안 평균 LST) − (같은 장면 10~20km 도넛 중앙값)
#   ← 도넛을 빼는 것은 케임브리지엔 없는 단계지만, 장면마다 다른 대기·계절을 상쇄하려면 필요하다.
#     (케임브리지는 월 평균 20년 시계열이라 그 교란이 평균으로 씻긴다. 우리는 장면이 몇 장뿐이다.)
#   그다음 시기를 셋으로 나눈다 — 케임브리지 k개월 창은 착공 이전까지 거슬러 가므로 **공사 효과를 품는다**.
#   그 사실을 드러내려고 우리는 P1(빈 땅) / P2(준공 직전) / P3(가동)로 쪼갠다.
_GS = pd.read_csv('DERIVED_0728_각세종_전후_LST.csv', encoding='utf-8-sig')
_GS['촬영일'] = pd.to_datetime(_GS['촬영일'])
_GS['월'] = _GS['촬영일'].dt.month
print('각 세종 (36.508N 127.341E · 가동 개시 2023-11) — 장면', len(_GS), '장')

for _R, _mf in ((300, 0.55), (500, 0.55)):
    _q = _GS[(_GS[f'유효비{_R}'] >= _mf) & _GS[f'ΔT{_R}'].notna() & (_GS['월'] == 6)]
    _g = _q.groupby('기')[f'ΔT{_R}'].agg(['count', 'mean'])
    print(f'\n【반경 {_R}m · 6월 장면만 · 유효화소 ≥{_mf:.0%}】')
    for _k in ['P1 빈 땅', 'P2 준공직전', 'P3 가동']:
        if _k in _g.index:
            print(f'   {_k:12}n={int(_g.loc[_k,"count"])}   ΔT {_g.loc[_k,"mean"]:+.2f}℃')
    if all(k in _g.index for k in ('P1 빈 땅', 'P2 준공직전', 'P3 가동')):
        _con = _g.loc['P2 준공직전', 'mean'] - _g.loc['P1 빈 땅', 'mean']
        _ops = _g.loc['P3 가동', 'mean'] - _g.loc['P2 준공직전', 'mean']
        _tot = _g.loc['P3 가동', 'mean'] - _g.loc['P1 빈 땅', 'mean']
        print(f'   공사(P1→P2) {_con:+.2f}℃   가동(P2→P3) {_ops:+.2f}℃   **전체(P1→P3) {_tot:+.2f}℃**')

# ── 케임브리지 2.07은 공사인가 가동인가 — Table 1이 답한다 [🔖 2026-07-28 밤 정정] ──
# 논문 Table 1 (p.6): Δ⁰(k) 를 기준선 창 k 를 바꿔가며 낸 값.
_T1 = {12: 2.03, 24: 2.05, 36: 2.06, 120: 2.12}
print()
print('━━ 케임브리지 2.07℃ 는 공사 효과인가 가동 효과인가 ━━')
print('   논문 Table 1 — 기준선 창 k(개월)를 바꿔가며 잰 Δ⁰(k)')
print('   ' + ''.join(f'k={k:<6}' for k in _T1) + '   ← 기준선을 몇 개월 전까지 잡느냐')
print('   ' + ''.join(f'{v:<8.2f}' for v in _T1.values()) + f'   폭 {max(_T1.values())-min(_T1.values()):+.2f}℃')
print()
print('   [추론] 공사(피복 변화)가 주범이라면 k=120(10년 창 — 빈 땅 시절 포함)이 k=12(마지막 1년 —')
print('   이미 다 지어진 상태)보다 **훨씬 커야 한다.** 실제로는 2.03 → 2.12, 겨우 +0.09℃(4%)다.')
print('   게다가 Figure 2 는 i=0(가동 개시)에서 **계단**이다(원문: "a clear increase of LST coinciding')
print('   with the start of operations" · "This apparent step function").')
print('   → **케임브리지의 2.07℃ 는 공사가 아니라 가동 효과다.**')

# ── 그러면 대응 비교가 바뀐다 ──
_q5 = _GS[(_GS['유효비500'] >= 0.55) & _GS['ΔT500'].notna() & (_GS['월'] == 6)]
_g5 = _q5.groupby('기')['ΔT500'].mean()
_ops5 = _g5.get('P3 가동', np.nan) - _g5.get('P2 준공직전', np.nan)
_tot5 = _g5.get('P3 가동', np.nan) - _g5.get('P1 빈 땅', np.nan)
print()
print('━━ 그러면 각 세종의 어느 값과 맞대야 하나 ━━')
print(f"   {'비교':34}{'값':>9}   판정")
print(f"   {'케임브리지 k=12 (가동 12개월 전 대비)':34}{_T1[12]:>8.2f}℃")
print(f"   {'각 세종 P2→P3 (준공직전→가동) ← 대응':34}{_ops5:>8.2f}℃   ✗ 재현 실패")
print(f"   {'각 세종 P1→P3 (빈 땅→가동)':34}{_tot5:>8.2f}℃   (총합은 비슷하나 대응 아님)")
print()
print('   ⚠ **총합 +1.78 이 2.07 과 가까운 것은 우연이다.** 분해가 정반대다 —')
print(f'     우리: 공사 {_g5.get("P2 준공직전",np.nan)-_g5.get("P1 빈 땅",np.nan):+.2f} + 가동 {_ops5:+.2f}')
print(f'     케임브리지: 공사 ≈ +0.09(k 차이 전부) + 가동 ≈ +2.03')
print('     앞선 판(2026-07-28 저녁)에서 "재현된다"고 쓴 것을 **철회한다.**')
print()
print(f'   이 실패는 힘이 있다 — 우리 최소검출가능효과가 약 1.7℃ 이므로 2.03℃ 짜리 효과였다면')
print(f'   **약 80% 확률로 잡혔어야 한다.** 못 잡았다는 것은 그냥 표본이 작아서가 아니다.')

print()
print('★ 그래서 "기존 산단에 2.07 을 얹어도 되나" — 층을 나눠 답한다')
print('   ① 메커니즘은 옮겨간다. 원인이 가동 폐열이면 옆이 논밭이든 제철소든 W 는 W 다.')
print('      → 우리가 걱정하던 "폐공장은 이미 포장돼 공사 몫이 없다"는 논점은 **사라진다.**')
print('   ② 그러나 표본 틀이 다르다. 그것도 논문이 **의도적으로** 그렇게 만들었다(p.3):')
print('      11,000+ 곳 중 **8,472곳** 만 쓴 이유 — "Considering only the AI data centres located')
print('      outside of highly dense regions allows us to provide a solid connection between the LST')
print('      trends that we can measure and the presence of AI data centres in the area."')
print('      즉 **밀집 시가지는 귀속이 안 되니까 뺐다.** 산단 입지는 그들이 뺀 바로 그 케이스다.')
print('   ③ 반응 함수가 다를 수 있다. 같은 W 를 부어도 표면이 다르면 ΔT 가 다르다.')
print('      이미 뜨거운 불투수면은 복사 방출이 T⁴ 로 이미 크므로 같은 열속에 **덜** 오르는 쪽이')
print('      물리적으로 예상되는 방향이다. ⚠ 다만 이건 추론이고 측정이 아니다.')
print(f'   ④ 평균 뒤의 산포가 크다 — 최소 {0.30}℃ ~ 최대 {9.02}℃, 95퍼센타일 1.5~2.4℃.')
print('      특정 부지에 평균을 얹는 것 자체가 이미 강한 가정이다.')
print()
print('   [권고] dT_at() 의 2.07 은 유지하되 **성격을 바꿔 표기한다** —')
print('     "AIDC 가 오면 이만큼 오른다"(확정 예측) ✗')
print('     "밀집 시가지 **밖** hyperscaler 8,472곳의 평균 상승분이며, 산단 입지는 표본 밖이고,')
print('      국내 사례 1건(각 세종)에서는 재현되지 않았다" ✓')

print()
print('⚠ 우리가 못 잡은 이유 후보 — 이제 이게 더 중요해졌다')
for _i, _l in enumerate([
    '부하 — 가동 1~2년차 실제 IT 부하가 설계용량의 얼마인지 모른다',
    '시각 — Landsat 은 오전 11시경. **논문은 MODIS 를 쓰면서 주간/야간을 밝히지 않았다.**'
    ' 폐열 신호는 보통 밤에 더 뚜렷하다(태양복사가 덮지 않으므로). 주간만 보면 놓칠 수 있다',
    '냉각 방식 미확인 — 열이 물(잠열)로 나가면 LST 는 원래 덜 오른다(§12-c 열 수지와 같은 방향)',
    '센서·집계 — MODIS 500m 월평균 20년 vs Landsat 30m 개별장면 6월 7장',
    'P1 이 유효한 6월 장면 **한 장**뿐 — 추정이 아니라 관측 한 건'], 1):
    print(f'     ({_i}) {_l}')
print('   → 다음 수: **MODIS 야간 LST 산출물로 각 세종을 다시 본다.** 무료이고, 위 (2)를 직접 시험한다.')

# ── §9-e3. 곡선을 기존 산단에 그대로 얹어도 되나 [🔖 2026-07-28 밤 · 사용자 질문] ──
# dT_at(r) = 2.07 · exp(−r·ln2/4.2) · β 에는 케임브리지 표본이 **두 번** 들어간다:
#   진폭 2.07℃  그리고  **반감기 4.2km**.  지금까지 우리는 진폭만 의심했다. 모양이 더 크게 흔든다.
print()
print('━' * 78)
print('§9-e3. 감쇠 곡선을 기존 산단에 얹어도 되나 — 진폭과 모양을 나눠 본다')
print('━' * 78)

# ── (1) 꼬리가 결과를 지배한다 ──
# §11-b(3) 인천 링별 결과를 그대로 가져와, 곡선의 어느 구간이 답을 만드는지 본다.
_RING_E3 = [('0-1km', 0.4), ('1-3km', 2.1), ('3-4.5km', 2.6), ('4.5-10km', 8.3)]
_totE3 = sum(v for _, v in _RING_E3)
print('(1) 우리 추가환자 추정의 어느 구간에서 나오나 (§11-b(3) 인천)')
for _lb, _v in _RING_E3:
    print(f'    {_lb:10}{_v:>6.1f}건{_v/_totE3*100:>7.0f}%' + ('   ← 곡선의 꼬리' if _lb.startswith('4.5') else ''))
print(f'    {"합계":10}{_totE3:>6.1f}건')
print(f'    → **{_RING_E3[-1][1]/_totE3*100:.0f}% 가 4.5~10km 한 구간에서 나온다.** 곡선의 꼬리가 답을 지배한다.')
print()
print('    ⚠ [🔖 2026-07-28 정정 — 사용자 지적] 앞선 판에서 이 구간을 "논문이 외삽에 가장 가까운 곳"이라')
print('      적었는데 **틀렸다.** 케임브리지는 4.5km(+1.0℃)·7km(30%)·10km(도달)를 **직접 보고**한다.')
print('      그 사이는 외삽이 아니라 **보간**이다. 외삽은 **반대쪽 끝**에 있다 —')
print('      §15-g 가 불투수 관측소 **n=0** 을 확인했으므로(AWS·ASOS 양쪽), 지표→기온 환산 β 를')
print('      부지(불투수·고온)에 적용하는 것이 **근거리 외삽**이다. 두 외삽을 헷갈리지 않는다:')
print(f"      {'구간':14}{'케임브리지 곡선':>18}{'β 환산':>16}")
print(f"      {'부지 0~1km':14}{'보고점(2.07)':>18}{'★ 외삽 (관측소 n=0)':>16}")
print(f"      {'1~4.5km':14}{'보간':>18}{'외삽':>16}")
print(f"      {'4.5~10km':14}{'보고점 사이 보간':>18}{'주거·녹지 관측 범위 안':>16}")
print(f"      {'10km 밖':14}{'★ 외삽':>18}{'—':>16}")
print('      → 꼬리가 위험한 이유는 외삽이어서가 아니라 (a) **총량의 62% 를 만들고**')
print('        (b) 보고점 사이를 잇는 **형태 가정**에 가장 민감하며 (c) 도시 혼합이 가장 크게 줄일 곳이어서다.')

# ── (2) 진폭 — 같은 W 라도 뜨거운 표면에서는 덜 오른다 ──
# 지표 에너지수지 선형화:  ΔT ≈ Q / (4εσT₀³ + ρ·c_p·C_H·U + 잠열항)
# 산단은 T₀ 가 이미 높다 → 분모의 복사항이 커진다 → 같은 Q 에 ΔT 가 작아진다.
_SIG = 5.670374419e-8
print()
print('(2) 진폭 — 이미 뜨거운 표면은 같은 열속에 **덜** 오른다 (복사항 ∝ T₀³)')
print(f"    {'기준 표면온도':16}{'4εσT₀³':>12}{'상대 반응':>10}")
_T0ref = 300.0
for _lb, _T0 in [('논밭·시골 300K', 300.0), ('산단 +10℃ 310K', 310.0), ('산단 +20℃ 320K', 320.0)]:
    _d = 4 * 0.97 * _SIG * _T0 ** 3
    _d0 = 4 * 0.97 * _SIG * _T0ref ** 3
    print(f'    {_lb:16}{_d:>11.2f}{_d0/_d:>10.2f}×')
print('    → 복사항만 따져도 산단에서는 **10~20% 덜** 오른다. 거칠기가 커서 난류 교환도 더 좋다(추가 감쇠).')
print('    ⚠ 이건 물리 추론이고 우리 측정이 아니다. 방향만 신뢰하고 크기는 쓰지 않는다.')

# ── (3) 모양 — 반감기도 그 표본의 지표 위에서 잰 것이다 ──
print()
print('(3) 모양 — 반감기 4.2km 는 **논밭·숲 위에서** 잰 값이다')
print('    [용어] 여기서 "모양"은 **거리에 따라 온도 상승분이 어떤 곡선으로 줄어드는가**를 말한다.')
print('    논문이 준 것은 점 서넛(부지 2.07 · 4.5km 1.0 · 7km 30% · 10km 도달)뿐이고,')
print('    그 점들을 **무슨 함수로 잇느냐**는 우리가 골랐다(지수·선형·가우시안 중 지수 — §9-e 형태 검정).')
print('    지수를 고르면 남는 자유도가 하나, **반감기**다. "몇 km 갈 때마다 절반이 되나".')
print('    4.2km 는 논문 보고점에 맞춘 값이지 우리가 국내에서 잰 값이 아니다.')
print('    감쇠는 대기 혼합이고, 혼합은 그 위를 덮은 지표에 달렸다. 거친 지표(도시·산단·항만)는')
print('    경계층이 두껍고 난류가 강해 **더 빨리 희석된다** → 반감기가 **짧아질** 가능성이 크다.')
print()
print(f"    {'반감기':>8}{'0.5km':>9}{'3km':>9}{'7.25km':>9}{'10km':>9}   추가환자 상대")
for _hl in (4.2, 3.0, 2.0):
    _row = ''.join(f'{CAMB_DT0*np.exp(-_r*np.log(2)/_hl)*BETA_L2A:>8.2f}℃' for _r in (0.5, 3, 7.25, 10))
    # 링 인구 가중으로 대략의 총량 비교 (인천 §11-b(3) 링 인구)
    _pops = [(0.5, 28691), (2.0, 182420), (3.75, 314180), (7.25, 1836496)]
    _rel = sum(p * (1.63 ** (CAMB_DT0 * np.exp(-r * np.log(2) / _hl) * BETA_L2A) - 1) for r, p in _pops)
    _base = sum(p * (1.63 ** (CAMB_DT0 * np.exp(-r * np.log(2) / 4.2) * BETA_L2A) - 1) for r, p in _pops)
    print(f'    {_hl:>6.1f}km{_row}{_rel/_base:>13.2f}×')
print('    → 반감기가 4.2 → 3.0km 로만 짧아져도 추가환자 추정이 3할 가까이 준다.')
print('      **곡선의 모양은 진폭만큼이나 결과를 좌우한다.** 그런데 우리는 모양을 검증한 적이 없다.')

# ── (4) 표본 틀 ──
print()
print('(4) 표본 틀 — 논문이 **의도적으로** 밀집 지역을 뺐다 (p.3)')
print('    "Considering only the AI data centres located outside of highly dense regions allows us')
print('     to provide a solid connection between the LST trends ... and the presence of AI data centres"')
print('    11,000+ 곳 중 8,472곳만 쓴 이유다 — 밀집 지역은 귀속이 안 되니까.')
print('    ⇒ **산단 입지는 그들이 뺀 바로 그 케이스다.** "다를까?"의 답은 "다를 수 있고, 그걸 잰 측정이 없다".')

# ── (5) 열원 교체 ──
print()
print('(5) 열원 교체 — 폐공장이냐 가동 산단이냐로 갈린다 (§11-b(2) 기존 단서)')
print('    폐공장: 그 자리 열이 이미 0 → 순수 더하기에 가깝다.')
print('    가동 산단: 공정열이 빠지고 DC열이 들어온다 → 순변화는 +2.07 이 아니다.')

# ── (6) 종합 ──
print()
print('★ 종합 — 네 갈래 중 셋이 **과대** 쪽을 가리킨다')
print(f"    {'요인':28}{'방향':>8}   근거")
for _lb, _dir, _why in [
    ('반응함수 (뜨거운 표면 T₀³)', '과대', '복사항만으로 10~20% (물리 추론)'),
    ('도시 혼합 (반감기 단축)', '과대', '거친 지표 = 빠른 희석 (물리 추론)'),
    ('열원 교체 (가동 산단)', '과대', '§11-b(2) — 폐공장엔 해당 없음'),
    ('규모 (GW급 > 각 세종)', '과소', '진폭에만 걸리고 모양엔 안 걸린다')]:
    print(f'    {_lb:28}{_dir:>8}   {_why}')
print()
print('    [권고] 곡선은 그대로 쓰되 **결과를 "상한"으로 표기한다.** 값을 임의로 깎지 않는다 —')
print('      깎을 근거(반감기 얼마?)가 우리에게 없기 때문이다. 대신 성격을 바꿔 적는다:')
print('      ✗ "AIDC 가 오면 이만큼 환자가 는다"')
print('      ✓ "밀집 시가지 밖에서 잰 곡선을 산단에 그대로 얹었을 때의 값 = 상한.')
print('         반응함수·도시 혼합·열원 교체 셋이 모두 이 값을 낮추는 방향이다."')
print('    [검증안] 반감기는 우리 자료로 시험할 수 있다 — §14 링 프로파일은 공간 대비라 안 되지만,')
print('      가동 개시 시점이 있는 국내 DC 를 **링별 시간 비교**로 재면 모양이 직접 나온다(각 세종 확장).')


각 세종 (36.508N 127.341E · 가동 개시 2023-11) — 장면 18 장

【반경 300m · 6월 장면만 · 유효화소 ≥55%】
   P1 빈 땅      n=1   ΔT -2.98℃
   P2 준공직전     n=3   ΔT +1.66℃
   P3 가동       n=4   ΔT +0.61℃
   공사(P1→P2) +4.63℃   가동(P2→P3) -1.05℃   **전체(P1→P3) +3.58℃**

【반경 500m · 6월 장면만 · 유효화소 ≥55%】
   P1 빈 땅      n=1   ΔT -2.93℃
   P2 준공직전     n=3   ΔT -0.40℃
   P3 가동       n=4   ΔT -1.15℃
   공사(P1→P2) +2.53℃   가동(P2→P3) -0.75℃   **전체(P1→P3) +1.78℃**

━━ 케임브리지 2.07℃ 는 공사 효과인가 가동 효과인가 ━━
   논문 Table 1 — 기준선 창 k(개월)를 바꿔가며 잰 Δ⁰(k)
   k=12    k=24    k=36    k=120      ← 기준선을 몇 개월 전까지 잡느냐
   2.03    2.05    2.06    2.12       폭 +0.09℃

   [추론] 공사(피복 변화)가 주범이라면 k=120(10년 창 — 빈 땅 시절 포함)이 k=12(마지막 1년 —
   이미 다 지어진 상태)보다 **훨씬 커야 한다.** 실제로는 2.03 → 2.12, 겨우 +0.09℃(4%)다.
   게다가 Figure 2 는 i=0(가동 개시)에서 **계단**이다(원문: "a clear increase of LST coinciding
   with the start of operations" · "This apparent step function").
   → **케임브리지의 2.07℃ 는 공사가 아니라 가동 효과다.**

━━ 그러면 각 세종의 어느 값과 맞대야 하나 ━━
   비교                                      

### §9-e4. 감쇠 곡선의 '모양'을 국내에서 처음 잰다 `[🔖 2026-07-28 밤 · 사용자 요청]`

반감기 4.2km 는 케임브리지 보고점에 맞춘 값이고 **우리가 국내에서 잰 적이 없다.** 진폭(2.07)은 §9-e2에서 각 세종으로 시험해 봤지만 모양은 손도 못 댔다.

같은 자리·가동 전후를 **거리 링마다** 재면 모양이 직접 나온다. 링을 여섯으로 나누고 각 링에서 (링 평균 LST − 같은 장면 10~20km 도넛 중앙값)을 구한 뒤, 시기별로 비교한다.

⚠ **이 설계가 못 보는 것 하나를 먼저 밝힌다.** 우리는 도넛을 빼서 장면마다 다른 대기·계절을 상쇄한다(장면이 몇 장뿐이라 필요하다). 그 대가로 **0~20km 전체가 똑같이 더워지는 변화에는 눈이 먼다.** 케임브리지 설계(자기 과거와 비교)는 그것을 볼 수 있다. 다만 점 열원이 20km를 균일하게 데우는 일은 물리적으로 어렵고, 그렇다면 기울기가 남아야 하는데 — 아래에서 그 기울기를 찾는다.

In [ ]:
# ── §9-e4. 각 세종 링별 시간 비교 — 모양 측정 [🔖 2026-07-28 신규] ──
from scipy import stats as _st9e4
_R9 = pd.read_csv('DERIVED_0728_각세종_링별_전후_LST.csv', encoding='utf-8-sig')
_R9['촬영일'] = pd.to_datetime(_R9['촬영일'])
_J9 = _R9[_R9['촬영일'].dt.month == 6]                       # 계절 구성 맞춤
_L9 = ['0-0.5km', '0.5-1km', '1-2km', '2-3km', '3-5km', '5-10km']
_M9r = {'0-0.5km': .25, '0.5-1km': .75, '1-2km': 1.5, '2-3km': 2.5, '3-5km': 4.0, '5-10km': 7.5}
_camb9 = lambda r: CAMB_DT0 * np.exp(-r * np.log(2) / CAMB_HALF)     # 지표온도 기준(β 적용 전)

print(f'각 세종 6월 장면 {len(_J9)}장 — P1 {sum(_J9.기=="P1 빈 땅")} · P2 {sum(_J9.기=="P2 준공직전")} · P3 {sum(_J9.기=="P3 가동")}')
print()
print('가동 전후 링별 변화 vs 케임브리지 예측  (모두 지표온도 ℃, β 적용 전)')
print(f"{'링':>9}{'중앙r':>7}{'케임브리지':>11}{'우리 P2→P3':>13}{'95% CI':>18}{'판정':>10}")
print('─' * 70)
_rej9 = _tot9 = 0
for _lab in _L9:
    _x = _J9.loc[_J9.기 == 'P2 준공직전', _lab].dropna()
    _y = _J9.loc[_J9.기 == 'P3 가동', _lab].dropna()
    _pred = _camb9(_M9r[_lab])
    if len(_x) < 2 or len(_y) < 2:
        print(f'{_lab:>9}{_M9r[_lab]:>7.2f}{_pred:>10.2f}℃{"표본부족":>13}')
        continue
    _d = _y.mean() - _x.mean()
    _sp = np.sqrt(((len(_x)-1)*_x.var(ddof=1) + (len(_y)-1)*_y.var(ddof=1)) / (len(_x)+len(_y)-2))
    _se = _sp * np.sqrt(1/len(_x) + 1/len(_y))
    _tc = _st9e4.t.ppf(0.975, len(_x)+len(_y)-2)
    _lo, _hi = _d - _tc*_se, _d + _tc*_se
    _v = '예측 기각' if _pred > _hi else ('구분 못함' if _pred >= _lo else '—')
    _tot9 += 1; _rej9 += (_v == '예측 기각')
    print(f'{_lab:>9}{_M9r[_lab]:>7.2f}{_pred:>10.2f}℃{_d:>+12.2f}℃  [{_lo:>+5.2f},{_hi:>+5.2f}]{_v:>10}')
print()
print(f'→ 케임브리지 곡선은 링마다 +2.0 → +0.6℃ 를 예측한다. 우리 측정은 어느 링에서도 그렇지 않고')
print(f'  **{_rej9}/{_tot9} 링에서 95% 신뢰구간이 예측값을 배제**한다.')

print()
print('━━ P1→P3 (빈 땅 → 가동) 는 어디까지 퍼지나 ━━')
_G9 = _J9.groupby('기')[_L9].mean()
for _lab in _L9:
    if 'P1 빈 땅' in _G9.index and np.isfinite(_G9.loc['P1 빈 땅', _lab]) and np.isfinite(_G9.loc['P3 가동', _lab]):
        print(f'   {_lab:>9}{_G9.loc["P3 가동",_lab]-_G9.loc["P1 빈 땅",_lab]:>+9.2f}℃')
print('   → 변화가 **0~0.5km 한 링에만** 있고 그 밖에서는 사라지거나 음수다.')
print('     대기로 퍼지는 열이라면 0.5~1km·1~2km 로 완만히 이어져야 하는데 **끊긴다.**')
print('     이것은 열의 확산이 아니라 **건물 발자국**(피복이 바뀐 면적)의 서명이다.')

print()
print('★ 정리 — 진폭도 모양도 이 사례에서는 재현되지 않았다')
print(f"   {'':22}{'케임브리지':>12}{'각 세종':>12}")
print(f"   {'부지 상승분':22}{CAMB_DT0:>11.2f}℃{-0.75:>11.2f}℃   (§9-e2 · P2→P3 0~0.5km)")
print(f"   {'4.5km 부근':22}{1.0:>11.2f}℃{0.14:>11.2f}℃   (3~5km 링)")
print(f"   {'도달 거리':22}{'10km':>12}{'0.5km 이내':>12}")
print()
print('⚠ 그러나 이것으로 케임브리지가 틀렸다고 말하지 않는다. 우리가 반박한 것은')
print('   **"이 곡선을 각 세종에 그대로 적용하는 것"**이지 8,472곳 전체의 평균이 아니다. 남는 차이:')
for _i, _l in enumerate([
    '표본 1곳 vs 8,472곳 — 개별 시설 산포가 크다(논문 최소 0.30 ~ 최대 9.02℃)',
    '부하 — 가동 1~2년차 실제 IT 부하 미확인. 만재가 아니면 진폭이 작은 게 당연하다',
    '시각 — Landsat 오전 11시 · 논문은 MODIS 주야 미명시. 폐열은 밤에 더 뚜렷하다',
    '설계 — 우리는 도넛을 빼서 **전역 균일 상승에는 눈이 먼다**(위 머리말 경고)',
    '냉각 — 잠열로 나가면 LST 는 원래 덜 오른다(§12-c)'], 1):
    print(f'     ({_i}) {_l}')
print()
print('   [그래서 dose 엔진은?] 값을 바꾸지 않는다 — 1곳 결과로 8,472곳 평균을 대체할 수 없다.')
print('   대신 **결과 표기를 "상한"으로 바꾸고**(§9-e3 권고) 이 반증 시도를 한계로 병기한다.')
print('   [다음] 야간 MODIS 로 같은 링 분석 · 가동률 정보공개청구 · 국내 DC 2~3곳 추가.')


각 세종 6월 장면 11장 — P1 2 · P2 4 · P3 5

가동 전후 링별 변화 vs 케임브리지 예측  (모두 지표온도 ℃, β 적용 전)
        링    중앙r      케임브리지     우리 P2→P3            95% CI        판정
──────────────────────────────────────────────────────────────────────
  0-0.5km   0.25      1.99℃       -0.75℃  [-2.09,+0.59]     예측 기각
  0.5-1km   0.75      1.83℃         표본부족
    1-2km   1.50      1.62℃       -0.43℃  [-1.66,+0.80]     예측 기각
    2-3km   2.50      1.37℃       -0.03℃  [-0.84,+0.79]     예측 기각
    3-5km   4.00      1.07℃       +0.14℃  [-0.59,+0.86]     예측 기각
   5-10km   7.50      0.60℃       +0.00℃  [-0.65,+0.66]     구분 못함

→ 케임브리지 곡선은 링마다 +2.0 → +0.6℃ 를 예측한다. 우리 측정은 어느 링에서도 그렇지 않고
  **4/5 링에서 95% 신뢰구간이 예측값을 배제**한다.

━━ P1→P3 (빈 땅 → 가동) 는 어디까지 퍼지나 ━━
     0-0.5km    +1.78℃
       2-3km    -1.54℃
       3-5km    -0.76℃
      5-10km    -0.06℃
   → 변화가 **0~0.5km 한 링에만** 있고 그 밖에서는 사라지거나 음수다.
     대기로 퍼지는 열이라면 0.5~1km·1~2km 로 완만히 이어져야 하는데 **끊긴다.**
     이것은 열의 확산이 아니라 **건물 발자국**(피복이 바뀐 면적)의 서명이다.

★ 정리 — 진폭도 모양도 이 사례에서는 재현되지 않았다

### §9-f. 습구온도 — 습도가 온열질환을 추가로 설명하는가 `[🔖 2026-07-25 신규]`

§12-c에서 냉각 방식이 갈리는 걸 확인했다. **건식(드라이쿨러)은 전량 현열**로 버려 주변 기온을 올리고, **증발식(냉각탑)은 잠열**로 빠져 기온은 덜 오르되 **습도를 올린다**.

그렇다면 "냉각탑을 쓰니 기온이 덜 오른다"가 곧 "덜 위험하다"인가? 온열질환이 습도에 민감하다면 아니다. 그 전제를 **우리 자료로 확인**한다 — ASOS 일자료에 `평균 상대습도(%)`가 있다.

네 모형을 AIC로 비교한다: ① 최고기온만(현행 §9-①) ② 습구온도(최고기온 기준) ③ 습구온도(평균기온 기준·짝 일치) ④ 최고기온 + 상대습도.

> ⚠ **일평균 습도와 일최고기온은 시각이 어긋난다**(습도는 기온이 최고일 때 가장 낮다). 그래서 짝이 맞는 ③도 함께 돌린다. 정밀하게 하려면 ASOS **시간자료를 습도 포함해 재다운로드**해야 한다(현재 보유한 시간자료엔 기온만 있다).

$$AIC = 2k - 2\ln(L)$$

In [ ]:
# ── §9-f. 습구온도 dose-response — 습도가 온열질환을 추가로 설명하는가 [🔖 2026-07-25 신규] ──
# [왜] §12-c에서 확인했듯 냉각 방식이 갈린다: **건식(드라이쿨러)은 전량 현열**로 기온을 올리고,
#   **증발식(냉각탑)은 잠열**로 빠져 기온은 덜 오르되 **습도를 올린다**. 온열질환이 습도에 민감하다면
#   "냉각탑을 쓰니 기온이 덜 오른다"는 안심의 근거가 못 된다. 그걸 우리 자료로 확인한다.
# [자료] ASOS 일자료에 `평균 상대습도(%)`가 있다(시간자료엔 없음 — 필요하면 습도 포함 재다운로드).
# ⚠ 한계: 일평균 습도와 일최고기온은 **시각이 어긋난다**(습도는 기온이 최고일 때 가장 낮다).
#   그래서 짝이 맞는 (평균기온, 평균습도) 조합도 함께 돌려 비교한다.
import statsmodels.api as _sm9f, statsmodels.formula.api as _smf9f, glob as _g9f

# -----------------------------------------------------------------------------
# 1. ASOS 종관기상관측 일자료 파일 로드 및 전처리
# -----------------------------------------------------------------------------
_fr9f=[]
# 지정된 패턴의 모든 기상 데이터 CSV 파일 탐색 및 결합
for _f in _g9f.glob('OPEN_0522_기상청_ASOS일자료_*_summer.csv'):
    _a=pd.read_csv(_f,encoding='cp949')
    _need=['일시','최고기온(°C)','평균기온(°C)','평균 상대습도(%)','일강수량(mm)','합계 일사량(MJ/m2)','평균 풍속(m/s)']
    # 필수 칼럼 존재 여부 검증
    if not all(c in _a.columns for c in _need):
        continue
    _fr9f.append(_a[_need])

_A9f=pd.concat(_fr9f,ignore_index=True)
_A9f['dt']=pd.to_datetime(_A9f['일시'],errors='coerce').dt.date

# 날짜(dt)별 관측치 전국 평균 집계
_W9f=_A9f.groupby('dt').agg(Tmax=('최고기온(°C)','mean'),Tavg=('평균기온(°C)','mean'),
                            RH=('평균 상대습도(%)','mean'),RN=('일강수량(mm)','mean'),
                            SUN=('합계 일사량(MJ/m2)','mean'),WS=('평균 풍속(m/s)','mean')).reset_index()

# 강수량 결측치는 무강수(0mm)로 보정
_W9f['RN']=_W9f['RN'].fillna(0)

# -----------------------------------------------------------------------------
# 2. 습구온도(Wet-Bulb Temperature, Tw) 산출 함수 정의
# -----------------------------------------------------------------------------
def _wetbulb(T,RH):
    """Stull(2011) 습구온도 근사 (℃). 유효범위 RH 5~99% · T −20~50℃."""
    return (T*np.arctan(0.151977*np.sqrt(RH+8.313659))+np.arctan(T+RH)-np.arctan(RH-1.676331)
            +0.00391838*RH**1.5*np.arctan(0.023101*RH)-4.686035)

# 기온-습도 지표 계산
_W9f['Tw_max']=_wetbulb(_W9f['Tmax'],_W9f['RH'])     # 시각 불일치 있음(참고)
_W9f['Tw_avg']=_wetbulb(_W9f['Tavg'],_W9f['RH'])     # 짝이 맞는 조합

# -----------------------------------------------------------------------------
# 3. 질병관리청 온열질환 응급실감시체계 데이터 병합
# -----------------------------------------------------------------------------
_wb9f=CalamineWorkbook.from_path('FOIA_0522_질병관리청_NEDIS온열질환_2020-2025.xlsx')
_rw9f=_wb9f.get_sheet_by_name('DB(2020-2025)발생지역기준').to_python()
_ne9f=pd.DataFrame(_rw9f[1:],columns=_rw9f[0])
_ne9f['dt']=pd.to_datetime(_ne9f['발생일자'].astype(str),errors='coerce')

# 전국 일별 온열질환 환자 발생 건수 집계
_cnt9f=_ne9f.dropna(subset=['dt']).groupby(_ne9f['dt'].dt.date).size().rename('온열')

# 기상 데이터와 온열질환 데이터 날짜 기준 매칭 (Left Join)
_M9f=_W9f.merge(_cnt9f,left_on='dt',right_index=True,how='left')
_M9f['온열']=_M9f['온열'].fillna(0)

# 여름철(6, 7, 8월) 데이터 추출 및 결측치 정제
_M9f['m']=pd.to_datetime(_M9f['dt']).dt.month
_M9f=_M9f[_M9f['m'].isin([6,7,8])].dropna(subset=['Tmax','Tavg','RH','RN','SUN','WS'])

print(f'여름 일수 n={len(_M9f)} · 온열 {int(_M9f.온열.sum()):,}건 · 평균 상대습도 {_M9f.RH.mean():.1f}%')
print(f'습구온도(평균기온 기준) 평균 {_M9f.Tw_avg.mean():.1f}℃ · 건구-습구 차 {(_M9f.Tavg-_M9f.Tw_avg).mean():.1f}℃')
print()

# -----------------------------------------------------------------------------
# 4. 포아송 일반화선형모형(Poisson GLM) 구축 및 비교
# -----------------------------------------------------------------------------
# 카운트 데이터(온열질환자 수)의 특성을 반영하여 Poisson Family GLM 적용
_MODELS=[('① 최고기온만 (현행 §9-①)','온열 ~ Tmax'),
         ('② 습구온도(최고기온 기준)','온열 ~ Tw_max'),
         ('③ 습구온도(평균기온 기준·짝 일치)','온열 ~ Tw_avg'),
         ('④ 최고기온 + 상대습도','온열 ~ Tmax + RH'),
         # [🔖 2026-07-26 감사 추가] 처음엔 "강수·일사 미통제 — 습도 계수는 교란 가능"이라고 한계만
         #   적고 넘어갔다. 실제로 통제해보니 **방향이 반대**였다 — 부풀려진 게 아니라 눌려 있었다.
         #   습도-일사 상관이 −0.8이라(습한 날=흐린 날) 일사의 음의 경로가 습도 효과를 가리고 있었다.
         ('⑤ ④ + 강수','온열 ~ Tmax + RH + RN'),
         ('⑥ ④ + 강수 + 일사','온열 ~ Tmax + RH + RN + SUN'),
         ('⑦ ④ + 강수 + 일사 + 풍속','온열 ~ Tmax + RH + RN + SUN + WS')]

print(f"{'모형':30}{'AIC':>12}{'주항 계수':>11}{'1℃당 배수':>11}{'p':>10}")
_fit9f={}
for _lab,_fml in _MODELS:
    # GLM 모형 적합 (Poisson 가정)
    _m=_smf9f.glm(_fml,data=_M9f,family=_sm9f.families.Poisson()).fit()
    _fit9f[_lab]=_m
    # 절편(Intercept)을 제외한 첫 번째 독립변수(주항) 계수 추출
    _key=[k for k in _m.params.index if k!='Intercept'][0]
    # AIC, 계수값, 위험비(Relative Risk = exp(beta)), p-value 출력
    print(f"{_lab:30}{_m.aic:>12.0f}{_m.params[_key]:>11.4f}{np.exp(_m.params[_key]):>11.3f}{_m.pvalues[_key]:>10.1e}")

# 적합도 최적 모형(AIC가 가장 작은 모형) 도출
_best=min(_fit9f,key=lambda k:_fit9f[k].aic)
print()
print(f'→ AIC 최소 = **{_best}**')

# -----------------------------------------------------------------------------
# 5. 습도의 독립적 기여도 및 교란효과(Confounding) 진단
# -----------------------------------------------------------------------------
print(f'   교란 진단: 습도-일사 상관 {_M9f.RH.corr(_M9f.SUN):+.2f} · 습도-강수 {_M9f.RH.corr(_M9f.RN):+.2f}')

_m7=_fit9f['⑦ ④ + 강수 + 일사 + 풍속']
print(f'   ⑦(전부 통제) 습도 계수 {_m7.params["RH"]:+.4f} → 10%p당 ×{np.exp(_m7.params["RH"]*10):.3f} (④의 {np.exp(_m7.params["RH"]*10)/np.exp(_fit9f["④ 최고기온 + 상대습도"].params["RH"]*10):.2f}배)')

_m4=_fit9f['④ 최고기온 + 상대습도']
_bRH=_m4.params['RH']; _pRH=_m4.pvalues['RH']
print(f'→ ④에서 습도의 독립 계수 = {_bRH:+.4f} (p={_pRH:.1e}) → 상대습도 10%p 증가당 온열 ×{np.exp(_bRH*10):.3f}')
print(f'   같은 모형에서 기온 계수 {_m4.params["Tmax"]:+.4f} → 1℃당 ×{np.exp(_m4.params["Tmax"]):.3f}')
print()

# -----------------------------------------------------------------------------
# 6. 통계 분석 결과 해석 및 한계 제시
# -----------------------------------------------------------------------------
print('[해석] 이 결과가 §12-c 냉각방식 논증의 근거다:')
if _pRH<0.05 and _bRH>0:
    print('  습도가 기온과 **독립적으로** 온열질환을 늘린다 → 증발식 냉각탑이 기온을 덜 올린다는 게')
    print('  곧 "덜 위험하다"가 아니다. 냉각 방식 비교는 기온만이 아니라 습도까지 봐야 한다.')
elif _pRH<0.05 and _bRH<0:
    print('  습도 계수가 **음(−)**이다 — 우리 여름 자료에서는 습한 날에 온열이 오히려 적었다.')
    print('  ⚠ 강수·흐림과 교란됐을 수 있다(비 오는 날 습도↑·일사↓·야외활동↓). 인과로 읽지 말 것.')
    print('  → 냉각탑의 습도 상승 위험은 이 자료로는 지지되지 않는다. 주장하려면 다른 근거가 필요하다.')
else:
    print('  습도의 독립 기여가 유의하지 않다 → 이 자료로는 냉각 방식별 위험 차이를 말할 수 없다.')
print('⚠ 공통 한계: (a) 일평균 습도 × 일최고기온은 시각 불일치 → **§9-g에서 시간자료로 해소했다**(2026-07-27).')
print('  (b) 전국 일별 집계라 지역 편차를 못 본다.')
print('  (c) [정정 07-26] 강수·일사를 통제하니 습도 계수가 오히려 **커졌다** — 교란은 억제 방향이었다(⑤~⑦).')
print('  (d) 냉각탑이 실제로 주변 습도를 몇 %p 올리는지는 별개 문제다(우리 자료에 없음 — plume 모형 필요).')

# ── [🔖 2026-07-27 사용자 질문] "상대습도가 전국·일 단위로 집계된 거야?" ──
print()
print('[집계 수준] 그렇다. 그리고 그건 자료 한계가 아니라 **우리가 고른 설계**다 — 그래서 적어둔다')
print('  · 습도: 지점별 값이 있는데 **날짜별로 지점 평균**을 냈다(전국 대표값 1개/일).')
print('  · 온열: NEDIS는 발생 **시군구**를 갖고 있는데 여기선 **날짜로만** 집계했다(전국 1개/일).')
print('  · 왜: §9-f의 질문이 "습도가 기온과 **별개로** 설명력을 갖나"라서, 지역 구조를 빼고')
print('    가장 단순한 설정에서 먼저 보려 했다. §9-c(시도 FE)·§9-d(시군구 FE)는 이미 지역을 쓴다.')
print('  ⚠ 대가: (i) 지역 편차를 못 본다 (ii) 전국 평균 습도는 실제 누구도 겪지 않는 값이다')
print('    (iii) 표본이 552일뿐이라 검정력이 낮다(시군구×일이면 수십만 행).')
print('  → **가능한 개선**: 시군구를 §9-d의 최근접 관측소에 붙이면 (시군구×일) 패널로 같은 검정을 할 수 있다.')
print('    자료는 이미 다 있다(ASOS 시간자료 97지점 + NEDIS 시군구). 아직 안 했을 뿐이다 — 미결 과제로 남긴다.')

여름 일수 n=552 · 온열 13,995건 · 평균 상대습도 79.9%
습구온도(평균기온 기준) 평균 22.2℃ · 건구-습구 차 2.7℃

모형                                     AIC      주항 계수     1℃당 배수         p
① 최고기온만 (현행 §9-①)                     5143     0.4542      1.575   0.0e+00
② 습구온도(최고기온 기준)                       6578     0.4968      1.643   0.0e+00
③ 습구온도(평균기온 기준·짝 일치)                 10873     0.4229      1.526   0.0e+00
④ 최고기온 + 상대습도                         5069     0.4683      1.597   0.0e+00
⑤ ④ + 강수                              5070     0.4669      1.595   0.0e+00
⑥ ④ + 강수 + 일사                         4945     0.4388      1.551   0.0e+00
⑦ ④ + 강수 + 일사 + 풍속                    4943     0.4392      1.551   0.0e+00

→ AIC 최소 = **⑦ ④ + 강수 + 일사 + 풍속**
   교란 진단: 습도-일사 상관 -0.81 · 습도-강수 +0.61
   ⑦(전부 통제) 습도 계수 +0.0371 → 10%p당 ×1.449 (④의 1.26배)
→ ④에서 습도의 독립 계수 = +0.0141 (p=1.8e-17) → 상대습도 10%p 증가당 온열 ×1.152
   같은 모형에서 기온 계수 +0.4683 → 1℃당 ×1.597

[해석] 이 결과가 §12-c 냉각방식 논증의 근거다:
  습도가 기온과 **독립적으로** 온열질환을 늘린다 → 증발식 냉각탑이 기온을 덜 올린다는 게
  곧 "덜

### §9-g. 시간자료로 정밀화 — 시각 불일치 해소 `[🔖 2026-07-27]`

§9-f는 **일평균 습도 × 일최고기온**이라는 어긋난 짝을 썼다. 습도는 기온이 최고일 때 가장 낮으므로 그 조합은 습구온도를 과대 추정할 수 있다 — §9-f 한계 (a)로 적어둔 것이다.

기상청 ASOS **시간자료**(2026-07-26 API 수집, 97지점 × 2020~2025 여름)로 짝을 맞춘다:
- **일 최고 습구온도** — 시간별 Tw의 일 최댓값. 생리적으로 의미 있는 일 노출 지표
- **일최고기온이 난 그 시각의 습도** — 기온과 시각이 일치하는 습도

`[🧰 자료]` `OPEN_0726_기상청_ASOS시간자료_여름_2020-2025.csv` · 수집 스크립트 `scratchpad/dl_asos_hourly.py` · API 키는 gitignore되는 memory 폴더.

In [34]:
# ── §9-g. 시간자료로 정밀화 — §9-f의 '시각 불일치'를 없앤다 [🔖 2026-07-27 신규] ──
# [문제] §9-f는 **일평균 습도 × 일최고기온**이라는 어긋난 짝을 썼다. 습도는 기온이 최고일 때 가장 낮으므로
#   그 조합은 습구온도를 과대, 습도의 효과를 왜곡할 수 있다. §9-f 한계 (a)로 적어둔 것이다.
# [해결] 기상청 ASOS **시간자료**(2026-07-26 API 수집, 97지점×2020~2025 여름)로 짝을 맞춘다:
#   · Tw_h = 시간별 습구온도(Stull 2011) → **일 최고 습구온도**(생리적으로 의미 있는 일 노출 지표)
#   · **일최고기온이 난 그 시각의 습도** → 기온과 시각이 일치하는 습도

# -----------------------------------------------------------------------------
# 1. 기상청 ASOS 시간자료 로드 및 기본 전처리
# -----------------------------------------------------------------------------
# 97개 관측 지점의 2020~2025년 여름철(6~8월) 시간별 관측 데이터 읽기
_H9g=pd.read_csv('OPEN_0726_기상청_ASOS시간자료_여름_2020-2025.csv',encoding='utf-8-sig',
                 dtype={'stnId':str},low_memory=False)

# 주요 기상 변수 수치형 변환 및 결측치 처리 (ta:기온, hm:상대습도, rn:강수량, ws:풍속, icsr:일사량)
for _c in ['ta','hm','rn','ws','icsr']:
    _H9g[_c]=pd.to_numeric(_H9g[_c],errors='coerce')

_H9g['dt']=pd.to_datetime(_H9g['tm'],errors='coerce')
_H9g=_H9g.dropna(subset=['dt','ta','hm'])
_H9g['d']=_H9g['dt'].dt.date
_H9g=_H9g[_H9g['dt'].dt.month.isin([6,7,8])]
print(f'시간자료 {len(_H9g):,}행 · 지점 {_H9g.stnId.nunique()}개 · {_H9g.d.min()}~{_H9g.d.max()}')

# -----------------------------------------------------------------------------
# 2. 시간 단위 습구온도(Tw) 계산 및 시각 정합 지표 추출
# -----------------------------------------------------------------------------
# §9-f에서 정의한 Stull(2011) 공식을 적용하여 시간별 습구온도 계산
_H9g['Tw']=_wetbulb(_H9g['ta'].values,_H9g['hm'].values)     # §9-f에서 정의한 Stull 근사 재사용

# [핵심] 각 관측지점(stnId)·일자(d)별로 '일최고기온'이 발생한 시각의 행 인덱스(idxmax) 추출
_ix=_H9g.groupby(['stnId','d'])['ta'].idxmax()

# 최고기온 시각의 동시 관측치(기온, 상대습도, 습구온도) 동기화 추출
_P=_H9g.loc[_ix,['stnId','d','ta','hm','Tw']].rename(columns={'ta':'Tmax','hm':'RH_at_Tmax','Tw':'Tw_at_Tmax'})

# 지점-일 단위 종합 기상 집계 (일 최고 습구온도, 일평균 습도, 일강수량 합계, 일사량 합계, 평균 풍속)
_P=_P.merge(_H9g.groupby(['stnId','d']).agg(TwMax=('Tw','max'),RHmean=('hm','mean'),
            RN=('rn','sum'),ICSR=('icsr','sum'),WS=('ws','mean')).reset_index(),on=['stnId','d'])

# -----------------------------------------------------------------------------
# 3. 전국 일별 평균 지표로 공간 집계 및 전처리
# -----------------------------------------------------------------------------
_N=_P.groupby('d').agg(Tmax=('Tmax','mean'),
                       RH_at_Tmax=('RH_at_Tmax','mean'), # 최고기온 시각의 전국 평균 습도
                       TwMax=('TwMax','mean'),           # 일 최고 습구온도의 전국 평균
                       RHmean=('RHmean','mean'),         # 일평균 습도의 전국 평균 (비교 대조군)
                       RN=('RN','mean'),
                       ICSR=('ICSR','mean'),
                       WS=('WS','mean')).reset_index()

# 결측치 보정 (강수량=0, 일사량=중앙값 대체)
_N['RN']=_N['RN'].fillna(0); _N['ICSR']=_N['ICSR'].fillna(_N['ICSR'].median())
print(f'전국 일별 {len(_N)}일 · 일최고기온 시각 습도 {_N.RH_at_Tmax.mean():.1f}% vs 일평균 습도 {_N.RHmean.mean():.1f}%'
      f' (차이 {_N.RH_at_Tmax.mean()-_N.RHmean.mean():+.1f}%p)')
print(f'  일 최고 습구온도 {_N.TwMax.mean():.1f}℃ · 일최고기온 {_N.Tmax.mean():.1f}℃ · 건구-습구 차 {(_N.Tmax-_N.TwMax).mean():.1f}℃')

# -----------------------------------------------------------------------------
# 4. 온열질환 환자 발생 데이터 병합
# -----------------------------------------------------------------------------
_M9g=_N.merge(_cnt9f,left_on='d',right_index=True,how='left')
_M9g['온열']=_M9g['온열'].fillna(0)

# -----------------------------------------------------------------------------
# 5. [모형 비교] 시각 동기화 여부 및 지표 조합별 포아송 회귀(Poisson GLM)
# -----------------------------------------------------------------------------
print()
print(f"{'모형':44}{'AIC':>10}{'주항 배수':>10}{'습도 계수':>10}{'10%p당':>9}")

# 비교 모형 5종 세트
_MG=[('Ⓐ 일최고기온만 (§9-① 현행)','온열 ~ Tmax',None),
     ('Ⓑ **일 최고 습구온도**','온열 ~ TwMax',None),
     ('Ⓒ 기온 + 습도(**시각 일치**)','온열 ~ Tmax + RH_at_Tmax','RH_at_Tmax'),
     ('Ⓓ 기온 + 습도(일평균·§9-f 방식)','온열 ~ Tmax + RHmean','RHmean'),
     ('Ⓔ Ⓒ + 강수·일사·풍속','온열 ~ Tmax + RH_at_Tmax + RN + ICSR + WS','RH_at_Tmax')]
_fitg={}
for _lab,_f,_hk in _MG:
    _m=_smf9f.glm(_f,data=_M9g,family=_sm9f.families.Poisson()).fit()
    _fitg[_lab]=_m
    _main=[k for k in _m.params.index if k not in ('Intercept',)][0]
    _hb=f'{_m.params[_hk]:+.4f}' if _hk else '—'
    _hx=f'{np.exp(_m.params[_hk]*10):.3f}' if _hk else '—'
    print(f'{_lab:44}{_m.aic:>10.0f}{np.exp(_m.params[_main]):>10.3f}{_hb:>10}{_hx:>9}')

# -----------------------------------------------------------------------------
# 6. 결과 평가 및 시각 정합성에 따른 바이アス 정정 분석
# -----------------------------------------------------------------------------
_bestg=min(_fitg,key=lambda k:_fitg[k].aic)
print()
print(f'→ AIC 최소 = **{_bestg}**  ※ **채택 아님** — 왜 안 쓰는지는 §9-h. AIC는 적합도일 뿐 반사실을 정해주지 않는다.')

_cC=_fitg['Ⓒ 기온 + 습도(**시각 일치**)'].params['RH_at_Tmax']
_cD=_fitg['Ⓓ 기온 + 습도(일평균·§9-f 방식)'].params['RHmean']

print(f'→ 시각을 맞추면 습도 계수가 {np.exp(_cD*10):.3f} → {np.exp(_cC*10):.3f} (10%p당)로 이동한다.')
print(f'  §9-f가 적어둔 한계 (a)의 크기가 이것이다 — 방향은 {"과대" if _cD>_cC else "과소"}였다.')
print()

# -----------------------------------------------------------------------------
# 7. 정책/방법론적 해석 및 기술적 한계 명시
# -----------------------------------------------------------------------------
print('[해석] §12-c 냉각방식 논증에 주는 답:')
print(f'  · 습도의 독립 기여는 시각을 맞춘 뒤에도 남는다(Ⓒ 10%p당 ×{np.exp(_cC*10):.3f}).')
print('    → 증발식 냉각탑이 기온을 덜 올린다는 게 곧 "덜 위험하다"가 아니다.')
print('  · 다만 **일 최고 습구온도 하나(Ⓑ)**가 기온+습도 조합보다 나은지는 AIC가 답한다 — 위 표 참조.')

print('⚠ 남는 한계: (b) 전국 일별 집계라 지역 편차를 못 본다. (d) 냉각탑이 주변 습도를 몇 %p 올리는지는')
print('  여전히 우리 자료 밖이다(plume 모형 필요 — AERMOD/SACTI 계열).')


시간자료 1,271,755행 · 지점 97개 · 2020-06-01~2025-08-31
전국 일별 552일 · 일최고기온 시각 습도 63.3% vs 일평균 습도 79.9% (차이 -16.6%p)
  일 최고 습구온도 23.8℃ · 일최고기온 29.0℃ · 건구-습구 차 5.2℃

모형                                                 AIC     주항 배수     습도 계수    10%p당
Ⓐ 일최고기온만 (§9-① 현행)                                5112     1.576         —        —
Ⓑ **일 최고 습구온도**                                   9400     1.631         —        —
Ⓒ 기온 + 습도(**시각 일치**)                              5020     1.606   +0.0117    1.124
Ⓓ 기온 + 습도(일평균·§9-f 방식)                            5039     1.598   +0.0141    1.152
Ⓔ Ⓒ + 강수·일사·풍속                                    4691     1.511   +0.0352    1.421

→ AIC 최소 = **Ⓔ Ⓒ + 강수·일사·풍속**  ※ **채택 아님** — 왜 안 쓰는지는 §9-h. AIC는 적합도일 뿐 반사실을 정해주지 않는다.
→ 시각을 맞추면 습도 계수가 1.152 → 1.124 (10%p당)로 이동한다.
  §9-f가 적어둔 한계 (a)의 크기가 이것이다 — 방향은 과대였다.

[해석] §12-c 냉각방식 논증에 주는 답:
  · 습도의 독립 기여는 시각을 맞춘 뒤에도 남는다(Ⓒ 10%p당 ×1.124).
    → 증발식 냉각탑이 기온을 덜 올린다는 게 곧 "덜 위험하다"가 아니다.
  · 다만 **일 최고 습구온도 하나(Ⓑ)**가 기온+습도 조합보다 나은지는 A

### §9-i. (시군구 × 일) 패널 — 전국 평균이 습도 신호를 지우고 있었나 `[🔖 2026-07-27 사용자 지적]`

> "지점별 습도가 있으면 그렇게 접근했어야 하지 않아?"

**맞는 지적이다.** §9-f·§9-g는 습도를 **전국·일 평균**으로 뭉갰다. 지점별 습도(97지점)도 있고 NEDIS 발생 **시군구**도 있는데 둘 다 날짜로만 집계한 것이다. 자료 한계가 아니라 설계 선택이었고, 그 선택이 무엇을 지웠는지 여기서 잰다.

재료는 이미 다 있다 — §9-d의 `_near9`(시군구→최근접 ASOS) + §9-g의 `_P`(지점×일) + `_neK9`(NEDIS 시군구×일).

In [ ]:
# ── §9-i. (시군구 × 일) 패널 — 전국 평균이 습도 신호를 지우고 있었나 [🔖 2026-07-27 사용자 지적] ──
# [지적] "지점별 습도가 있으면 그렇게 접근했어야 하지 않아?" — 맞다. §9-f·§9-g는 전국·일 평균이었다.
#   습도는 지점별로 있고 NEDIS는 시군구별로 있는데 둘 다 날짜로 뭉갰다. 지역 단위로 다시 본다.
# [재료] 전부 이미 있다 — §9-d의 `_near9`(시군구→최근접 ASOS) + §9-g의 `_P`(지점×일 기상) + `_neK9`(NEDIS 시군구×일).
_P9i=_P.copy(); _P9i['지점']=_P9i['stnId'].astype(int)
_near9i=pd.DataFrame([{'key':k,'지점':v[0],'dist':v[2]} for k,v in _near9.items()])
_dt9i=pd.to_datetime(_neK9['dt'])                     # §9-d의 'dt'는 object dtype이라 .dt 직접 접근 불가
_cnt9i=_neK9.groupby(['key',_dt9i.dt.date]).size().rename('온열').reset_index()
_cnt9i.columns=['key','d','온열']

_days9i=sorted(_P9i['d'].unique())
_keys9i=[k for k in _near9i['key'] if k in set(_cnt9i['key'])]
_G9i=pd.MultiIndex.from_product([_keys9i,_days9i],names=['key','d']).to_frame(index=False)
_G9i=_G9i.merge(_near9i,on='key',how='left').merge(
    _P9i[['지점','d','Tmax','RH_at_Tmax','TwMax','RHmean']],on=['지점','d'],how='left').merge(
    _cnt9i,on=['key','d'],how='left')
_G9i['온열']=_G9i['온열'].fillna(0)
_G9i=_G9i.dropna(subset=['Tmax','RH_at_Tmax'])
print(f'패널 {len(_G9i):,}행 · 시군구 {_G9i.key.nunique()} · 일 {_G9i.d.nunique()} · 온열 {int(_G9i.온열.sum()):,}건')

# 표본 얇은 시군구 제외 — §9-e와 같은 문턱(n≥100). 얇은 곳은 FE가 불안정하다.
_act9i=_G9i.groupby('key')['온열'].sum(); _GG9i=_G9i[_G9i['key'].isin(_act9i[_act9i>=100].index)].copy()
print(f'  n≥100 시군구만: {len(_GG9i):,}행 · {_GG9i.key.nunique()}개 · 온열 {int(_GG9i.온열.sum()):,}건'
      f' ({_GG9i.온열.sum()/_G9i.온열.sum()*100:.0f}%) · 관측소 거리 중앙 {_GG9i.dist.median():.1f}km')
print(f'  ★ 지역 간 습도 편차 SD {_GG9i.groupby("key").RH_at_Tmax.mean().std():.1f}%p'
      f' — **전국 평균이 지우던 것이 이것이다**')

print()
print(f"{'모형':42}{'AIC':>10}{'기온배수':>9}{'습도계수':>10}{'10%p당':>9}")
_M9I=[('ⓘ1 기온만 + 시군구FE','온열 ~ Tmax + C(key)',None,'Tmax'),
      ('ⓘ2 기온 + 습도(시각일치) + 시군구FE','온열 ~ Tmax + RH_at_Tmax + C(key)','RH_at_Tmax','Tmax'),
      ('ⓘ3 기온 + 습도(일평균) + 시군구FE','온열 ~ Tmax + RHmean + C(key)','RHmean','Tmax'),
      ('ⓘ4 습구온도만 + 시군구FE','온열 ~ TwMax + C(key)',None,'TwMax')]
_f9i={}
for _lb,_f,_hk,_mn in _M9I:
    _m=_smf9f.glm(_f,data=_GG9i,family=_sm9f.families.Poisson()).fit(); _f9i[_lb]=_m
    _hb=f'{_m.params[_hk]:+.4f}' if _hk else '—'
    _hx=f'{np.exp(_m.params[_hk]*10):.3f}' if _hk else '—'
    print(f'{_lb:42}{_m.aic:>10.0f}{np.exp(_m.params[_mn]):>9.3f}{_hb:>10}{_hx:>9}')
_best9i=min(_f9i,key=lambda k:_f9i[k].aic)
_c9i=_f9i['ⓘ2 기온 + 습도(시각일치) + 시군구FE']
_EQ9i=float(_c9i.params['Tmax']/_c9i.params['RH_at_Tmax'])
print()
print(f'→ AIC 최소 = **{_best9i}**')
print(f'→ **습도 등가가 바뀐다**: 기온 1℃ = 습도 {_EQ9i:.1f}%p')
# [🔖 2026-07-28 정정 — 사용자 지적] `_EQ12`·`_EQphys` 는 **§12-c2(이 셀보다 뒤)** 에서 정의된다.
#   즉 여기서 부르는 것은 전방 참조다. 기존 코드는 `_EQ12 if "_EQ12" in dir() else 40.6` 으로 숨겨
#   죽지는 않았지만, **같은 셀이 실행 순서에 따라 다른 숫자를 찍었다** —
#   위에서부터 처음 돌리면 40.6(하드코딩), §12-c2를 돌린 뒤 이 셀만 재실행하면 계산값.
#   → 이 절이 인용하는 것은 '§9-g에서 이미 보고된 값'이므로 **상수로 못박고**,
#     뒤에서 계산된 값과 어긋나면 조용히 넘어가지 말고 경고한다.
_EQ12_REF, _EQphys_REF = 40.6, 5.8        # §9-g 전국·일 등가(%p/℃) · Stull 물리 등가(%p/℃)
for _nm5, _ref5 in (('_EQ12', _EQ12_REF), ('_EQphys', _EQphys_REF)):
    if _nm5 in dir() and abs(eval(_nm5) - _ref5) > 0.5:
        print(f'⚠ §12-c2의 {_nm5}={eval(_nm5):.2f} 가 이 절의 상수 {_ref5}와 어긋난다 — 상수를 갱신할 것.')
print(f'   §9-g 전국·일 {_EQ12_REF:.1f}%p → §9-i 시군구×일 **{_EQ9i:.1f}%p** (Stull 물리 {_EQphys_REF:.1f}%p)')
print(f'   → 전국 평균이 습도 신호를 실제로 지우고 있었다. 물리와의 괴리가'
      f' {_EQ12_REF/_EQphys_REF:.0f}배 → {_EQ9i/_EQphys_REF:.0f}배로 줄었다.')
print(f'   → 표본 {len(_GG9i):,}행 (§9-g는 552행 — {len(_GG9i)/552:.0f}배)')
print()
print('[유지되는 것] ⓘ4 습구온도 단일지표는 여기서도 최악이다 → **기온·습도를 따로 두는 결정은 불변**.')
print(f'[정합성] ⓘ1 기온 배수 {np.exp(_f9i["ⓘ1 기온만 + 시군구FE"].params["Tmax"]):.3f}'
      f' ≈ §9-d 시군구 FE {np.exp(_mP9.params["최고기온"]):.3f} — 별개 경로로 같은 값(교차검증).')
print('⚠ 남는 한계: (a) n≥100 문턱으로 41개 시군구만 남아 도시 편중 (b) 최근접 관측소 중앙 8.6km —')
print('   시군구 안에서도 습도는 다르다 (c) 발생지 기준 온열 vs 거주지 기준 인구 불일치는 그대로.')
print('⚠ HEAT_MULT_PER_C는 **바꾸지 않는다** — §9-h 결정(사용자)이 유효하고, 여기 ⓘ1 1.494는')
print('   §9-d가 이미 보고한 시군구 FE 1.51과 같은 이야기다(FE가 촘촘할수록 소폭↓).')


패널 125,276행 · 시군구 228 · 일 552 · 온열 13,841건
  n≥100 시군구만: 23,184행 · 42개 · 온열 6,744건 (49%) · 관측소 거리 중앙 8.1km
  ★ 지역 간 습도 편차 SD 3.7%p — **전국 평균이 지우던 것이 이것이다**

모형                                               AIC     기온배수      습도계수    10%p당
ⓘ1 기온만 + 시군구FE                                 26117    1.494         —        —
ⓘ2 기온 + 습도(시각일치) + 시군구FE                       25730    1.599   +0.0273    1.314
ⓘ3 기온 + 습도(일평균) + 시군구FE                        25887    1.557   +0.0260    1.298
ⓘ4 습구온도만 + 시군구FE                               27920    1.606         —        —

→ AIC 최소 = **ⓘ2 기온 + 습도(시각일치) + 시군구FE**
→ **습도 등가가 바뀐다**: 기온 1℃ = 습도 17.2%p
   §9-g 전국·일 40.6%p → §9-i 시군구×일 **17.2%p** (Stull 물리 5.8%p)
   → 전국 평균이 습도 신호를 실제로 지우고 있었다. 물리와의 괴리가 7배 → 3배로 줄었다.
   → 표본 23,184행 (§9-g는 552행 — 42배)

[유지되는 것] ⓘ4 습구온도 단일지표는 여기서도 최악이다 → **기온·습도를 따로 두는 결정은 불변**.
[정합성] ⓘ1 기온 배수 1.494 ≈ §9-d 시군구 FE 1.507 — 별개 경로로 같은 값(교차검증).
⚠ 남는 한계: (a) n≥100 문턱으로 41개 시군구만 남아 도시 편중 (b) 최근접 관측소 중앙 8.6km —
   시군구 안에서도 습도는 다르다 (c

### §9-j. 플룸 모형 주입구 `[🔖 2026-07-27 사용자 요청]`

> "plume 모형이 확인되면 열적 영향을 확인할 수 있도록 `dT_at` 같은 함수를 도입해줘."

지금 우리 ΔT는 **등방**(사방 균일)이다. 실제 폐열은 바람을 타고 한쪽으로 간다. §11-b(2)에서 확인했듯 **등방은 보수적이지 않다** — 열이 몰리면 dose가 볼록해서 추가 환자가 오히려 늘어난다.

그런데 방위각별 ΔT는 우리가 계산할 수 없다. 플룸 모형이 필요하고, 그건 배출량을 아는 쪽(사업자)만 할 수 있다 — 그게 §17 "모델링 제출 의무화" 요구의 근거다.

**모형이 들어왔을 때 한 곳만 바꾸면 전 계산이 따라오는 자리**를 미리 만든다.

| 함수 | 하는 일 |
|---|---|
| `register_plume(fn·table, meta)` | 플룸 결과 등록. **출처·통계량 meta 없으면 거부** |
| `dT_field(r_km, bearing_deg)` | 수용점 Δ기온. 플룸 없으면 `dT_at`으로 폴백 |
| `pop_ring_sector(geom, r0, r1, n_sec)` | 링을 부채꼴로 쪼개 인구 집계 |
| `excess_plume(...)` | **부채꼴마다 dose를 따로 적용해 합산** (링 평균에 한 번 적용하는 것과 다르다) |

> **설계 원칙**: 플룸이 없으면 현행 등방 결과를 **정확히 재현**해야 한다. 셀 안의 회귀 게이트(`assert`)가 이를 강제한다.

In [ ]:
# ── §9-j. 플룸 모형 주입구 — 모형이 들어오면 여기만 바꾼다 [🔖 2026-07-27 사용자 요청] ──
# [왜] 지금 우리 ΔT는 **등방**(사방 균일)이다. 실제 폐열은 바람을 타고 한쪽으로 간다.
#   §11-b(2)에서 확인한 대로 등방은 보수적이지 않다 — 열이 몰리면 dose가 볼록해 추가 환자가 **늘어난다**.
#   그런데 방위각별 ΔT는 우리가 계산할 수 없다(플룸 모형 필요·§17 제출 의무화 요구의 근거).
# [그래서] 모형이 들어왔을 때 **한 곳만 바꾸면 전 계산이 따라오는 자리**를 미리 만든다.
#   설계 원칙: 플룸이 **없으면 현행 등방 결과를 정확히 재현**해야 한다(아래 회귀 게이트로 강제).
PLUME = None          # ← 여기에 플룸 결과를 등록하면 dT_field()가 방위각을 쓴다. None이면 등방.

def register_plume(fn=None, table=None, *, meta):
    """플룸 모형 결과 등록.
    fn    : callable(r_km, bearing_deg) -> Δ기온(℃)   ← 연속 모형
    table : [(r_km, bearing_deg, dT_C), ...]          ← 격자 출력(최근접 이웃)
    meta  : model·source·date·stat 필수. stat 은 'summer_mean'|'p95'|'max'.
            **무엇의 통계량인지 없으면 등록 거부** — §11-b(2)의 링평균↔풍하최대 혼동 재발 방지.
    """
    global PLUME
    _miss=[k for k in ['model','source','date','stat'] if k not in meta]
    if _miss:
        raise ValueError(f'meta 누락 {_miss} — 출처 없는 모형은 등록하지 않는다')
    if meta['stat'] not in ('summer_mean','p95','max'):
        raise ValueError("meta['stat']은 summer_mean|p95|max 중 하나여야 한다")
    if fn is None and table is None:
        raise ValueError('fn 또는 table 중 하나는 필요하다')
    if fn is None:
        _t=list(table)
        def fn(r_km,bearing_deg):
            return min(_t,key=lambda e:(e[0]-r_km)**2
                       +(((e[1]-bearing_deg+180)%360-180)/57.3*max(r_km,.1))**2)[2]
    PLUME={'fn':fn,'meta':dict(meta)}
    print(f"[플룸 등록] {meta['model']} · 출처 {meta['source']} · {meta['date']} · 통계량 {meta['stat']}")
    return PLUME

def dT_field(r_km, bearing_deg=None, *, shape=None, beta=None):
    """수용점의 Δ기온(℃). 플룸이 등록돼 있으면 방위각을 쓰고, 없으면 등방 dT_at으로 폴백.
    bearing_deg = 배출원에서 수용점을 본 방위(0=북, 시계방향). 등방일 때는 무시된다.
    ⚠ 플룸 결과는 **이미 기온**으로 간주한다 — β를 다시 곱하지 않는다."""
    if PLUME is None or bearing_deg is None:
        return dT_at(r_km, shape, beta)
    return float(PLUME['fn'](r_km, bearing_deg))

print('§9-j. 플룸 주입구 — 인터페이스 등록')
print(f'  현재 PLUME = {PLUME} → dT_field()는 등방 dT_at으로 폴백한다')
print(f'  폴백 확인: dT_field(2.0) = {dT_field(2.0):.3f}℃ == dT_at(2.0) = {dT_at(2.0):.3f}℃')
assert abs(dT_field(2.0)-dT_at(2.0))<1e-12, '폴백 불일치'
print('  ✓ 플룸 미등록 시 현행 등방 경로와 동일')
print('  ※ 링 집계(pop_ring_sector·excess_plume)와 회귀 게이트는 **§12-e**에 있다 —')
print('     그쪽이 pop_in()·인구 자료가 정의된 뒤라서다(여기서 쓰면 전방 참조가 된다).')
print()
print('[사용법] 사업자·연구자가 플룸 모형을 내놓으면:')
print("  register_plume(table=[(r_km, bearing_deg, dT_C), ...],")
print("                 meta={'model':'AERMOD v23132','source':'사업자 제출','date':'2026-XX-XX',")
print("                       'heat_MW':2400,'stack_h_m':25,'stability':'C','wind_ms':2.0,'stat':'summer_mean'})")
print('  → 이후 excess_plume()이 방위각별 ΔT를 쓴다. §11-b·§12-b는 호출만 바꾸면 된다.')
print('  ⚠ meta 필수 4종(model·source·date·stat) 없으면 **등록 거부**한다. 출처 없는 모형은 안 받는다.')
print("  ⚠ stat이 무엇의 통계량인지 반드시 받는다 — '여름 평균'과 '최대'를 섞으면 §11-b(2)에서 겪은")
print('     케임브리지(링평균) vs Sailor(풍하최대) 혼동이 그대로 재발한다.')
print()
print('[남는 한계 — 모형이 와도 안 풀리는 것]')
print('  (a) 방위는 형상 **대표점** 기준 근사다. 대형 산단은 부지 안에서도 방위가 갈린다.')
print('  (b) 플룸 모형은 ΔT를 주지만 **우리 β(지표→기온)를 대체하지 않는다** — 모형이 기온을 직접 주면 β를 빼야 한다.')
print('      register_plume 결과는 **이미 기온**으로 간주한다(dT_field가 β를 다시 곱하지 않는다).')
print('  (c) 계절 내 풍향 분포는 별도다. 한 조건의 플룸 한 장으로 연간 노출을 대신할 수 없다.')
print('  (d) 부채꼴 수 n_sec는 모형 격자 해상도에 맞춰야 한다 — 12는 30°로 거칠다.')


§9-j. 플룸 주입구 — 인터페이스 등록
  현재 PLUME = None → dT_field()는 등방 dT_at으로 폴백한다
  폴백 확인: dT_field(2.0) = 0.384℃ == dT_at(2.0) = 0.384℃
  ✓ 플룸 미등록 시 현행 등방 경로와 동일
  ※ 링 집계(pop_ring_sector·excess_plume)와 회귀 게이트는 **§12-e**에 있다 —
     그쪽이 pop_in()·인구 자료가 정의된 뒤라서다(여기서 쓰면 전방 참조가 된다).

[사용법] 사업자·연구자가 플룸 모형을 내놓으면:
  register_plume(table=[(r_km, bearing_deg, dT_C), ...],
                 meta={'model':'AERMOD v23132','source':'사업자 제출','date':'2026-XX-XX',
                       'heat_MW':2400,'stack_h_m':25,'stability':'C','wind_ms':2.0,'stat':'summer_mean'})
  → 이후 excess_plume()이 방위각별 ΔT를 쓴다. §11-b·§12-b는 호출만 바꾸면 된다.
  ⚠ meta 필수 4종(model·source·date·stat) 없으면 **등록 거부**한다. 출처 없는 모형은 안 받는다.
  ⚠ stat이 무엇의 통계량인지 반드시 받는다 — '여름 평균'과 '최대'를 섞으면 §11-b(2)에서 겪은
     케임브리지(링평균) vs Sailor(풍하최대) 혼동이 그대로 재발한다.

[남는 한계 — 모형이 와도 안 풀리는 것]
  (a) 방위는 형상 **대표점** 기준 근사다. 대형 산단은 부지 안에서도 방위가 갈린다.
  (b) 플룸 모형은 ΔT를 주지만 **우리 β(지표→기온)를 대체하지 않는다** — 모형이 기온을 직접 주면 β를 빼야 한다.
      register_plume 결과는 **이미 기온**으로 간주한다(dT_field가 β를 다시

### §9-h. 모형 선택 결정 — AIC가 최소여도 Ⓔ를 쓰지 않는다 `[🔖 2026-07-27 사용자 결정]`

§9-g에서 **Ⓔ(기온+습도+강수·일사·풍속)가 AIC 최소**였다. 그러면 dose 엔진의 밑(`HEAT_MULT_PER_C=1.575`)을 Ⓔ의 1.511로 갈아야 하는가?

**아니다.** 결정과 근거를 남긴다.

| | |
|---|---|
| **결정** | dose 엔진은 **일최고기온 단일항(Ⓐ)** 유지. Ⓔ·습구온도·상호작용은 **기록만** |
| **근거 ①** | 우리는 태양광 발전량이 아니라 **온열질환자에 대한 열적 영향**을 구한다. 더운 날이 맑지 않을 수(일사량이 작을 수) 있지만 우린 열에 집중한다. |
| **근거 ②** | **산단이 AIDC로 바뀔 때 강수·일사·풍속이 어떻게 변할지 통제도 예측도 못 한다.** 통제변수는 **반사실 값을 알 때만** 쓸 수 있다 |
| **근거 ③** | 기온·습구온도는 다르다 — plume 모형·냉각방식으로 알 수 있을지도 모른다(§12-c). 알기 전까지는 넣지 않는다 |

> ⚠ **방향을 숨기지 않는다.** 유지하기로 한 Ⓐ가 곧 **추정치를 10~13% 크게** 하는 쪽이다. 그래서 근거를 결과가 아니라 ①②③(반사실을 아느냐)에 두었다.

In [ ]:
# ── §9-h. 모형 선택 결정 — AIC가 최소여도 Ⓔ를 쓰지 않는다 [🔖 2026-07-27 사용자 결정] ──
# §9-g에서 Ⓔ(기온+습도+강수·일사·풍속)가 AIC 최소였다. 그러면 f_site의 밑(HEAT_MULT_PER_C=1.575)을
# Ⓔ의 1.511로 갈아야 하는가? **아니다.** 사용자 판단(2026-07-27)과 그 근거를 여기 남긴다.
#
# [결정] dose 엔진은 **일최고기온 단일항(Ⓐ)** 을 유지한다. Ⓔ·습구온도·상호작용은 **기록만** 한다.
# [사용자 근거]
#   ① 우리는 태양광 발전량이 아니라 **온열질환자에 대한 열적 영향**을 구한다. 더운 날은 대개 맑은 날이고,
#      그 둘이 함께 오는 것이 곧 현실의 폭염이다. 일사를 떼어내면 '현실에 없는 하루'의 계수가 된다.
#   ② **산단이 AIDC로 바뀔 때 강수·일사·풍속이 어떻게 변할지 우리는 통제도 예측도 못 한다.**
#      통제변수는 **반사실 값을 알 때만** 쓸 수 있다. 값을 못 정하면서 회귀에 넣는 것은 통제가 아니라
#      "그대로 있을 것"이라는 또 하나의 미검증 가정이다.
#   ③ 기온·습구온도는 다르다 — plume 모형·냉각방식으로 **알 수 있을지도 모른다**(§12-c).
#      그래서 그쪽은 계속 조사하되, 알기 전까지는 모형에 넣지 않는다.
print('[결정] dose 엔진 = 일최고기온 단일항(Ⓐ). HEAT_MULT_PER_C = %.3f 유지.' % HEAT_MULT_PER_C)
print('       Ⓔ(1.511)·Ⓑ(습구)·상호작용 = 기록 보존, 미채택.')
print()

# ── 근거 1. "AIC 최소면 예측도 항상 낮은가" — 아니다 (사용자 질문 2026-07-27) ──
_M9h=_M9g.copy()
_fA9h=_fitg['Ⓐ 일최고기온만 (§9-① 현행)']; _fE9h=_fitg['Ⓔ Ⓒ + 강수·일사·풍속']
_M9h['pA']=_fA9h.predict(_M9h); _M9h['pE']=_fE9h.predict(_M9h)
_qI=_M9h.ICSR.quantile([1/3,2/3]).values
_hot=_M9h[_M9h.Tmax>=_M9h.Tmax.quantile(.75)]
print('근거 1 — AIC가 낮다고 예측이 항상 낮지는 않다')
print(f'  전체 {len(_M9h)}일 중 Ⓔ 예측 > Ⓐ 예측: {int((_M9h.pE>_M9h.pA).sum())}일 ({(_M9h.pE>_M9h.pA).mean()*100:.0f}%)')
for _lab,_s in [('일사 하위⅓',_M9h[_M9h.ICSR<=_qI[0]]),('일사 중위⅓',_M9h[(_M9h.ICSR>_qI[0])&(_M9h.ICSR<_qI[1])]),
                ('일사 상위⅓',_M9h[_M9h.ICSR>=_qI[1]])]:
    print(f'    {_lab} (n={len(_s):3d}) 실측 {_s.온열.mean():5.1f} · Ⓐ {_s.pA.mean():5.1f} · Ⓔ {_s.pE.mean():5.1f}  (Ⓔ−Ⓐ {_s.pE.mean()-_s.pA.mean():+5.1f})')
_hs=_M9h[(_M9h.Tmax>=_M9h.Tmax.quantile(.75))&(_M9h.ICSR>=_qI[1])]
print(f'  ★ **덥고 맑은 날**(n={len(_hs)}, 폭염의 전형): Ⓐ {_hs.pA.mean():.1f} vs Ⓔ {_hs.pE.mean():.1f} → **Ⓔ가 하루 {_hs.pE.mean()-_hs.pA.mean():+.1f}건 더 예측**한다.')
print('    → AIC는 552일 전체의 적합도일 뿐, 특정 조건에서 어느 쪽이 크게 예측하는지는 따로 봐야 한다.')

# ── 근거 2. 다만 우리 공식이 쓰는 건 '수준'이 아니라 '배수'다 ──
_mA9h=float(np.exp(_fA9h.params['Tmax'])); _mE9h=float(np.exp(_fE9h.params['Tmax']))
print()
print('근거 2 — 그런데 f_s = m^ΔT − 1 은 관측된 실제 환자 수에 곱하는 **배수**다(모형의 예측 수준을 쓰지 않는다)')
print(f"  {'ΔT':>7}{'Ⓐ f_s':>10}{'Ⓔ f_s':>10}{'Ⓔ 채택 시':>11}")
for _dT in (0.26,0.5,1.0,2.0):
    _fa,_fe=HEAT_MULT_PER_C**_dT-1,_mE9h**_dT-1
    print(f'  {_dT:>6}℃{_fa:>+10.3f}{_fe:>+10.3f}{(_fe/_fa-1)*100:>+10.1f}%')
print('  → 배수로 보면 Ⓔ는 **언제나 더 작다**. 즉 Ⓔ 채택은 §10~§18 추정치를 10~13% 낮춘다.')
print('  ⚠ **방향을 숨기지 않는다**: 우리가 유지하기로 한 Ⓐ가 곧 **더 큰 숫자**를 주는 쪽이다.')
print('    그래서 근거를 결과가 아니라 위 ①②③(반사실을 아느냐)에 두었다. 반박 시 이 문단을 먼저 보일 것.')

# ── 근거 3. 통제변수의 크기 — 우리가 정할 수 없는 값이 얼마나 큰 일을 하는가 ──
print()
print('근거 3 — Ⓔ가 통제하는 변수들은 작지 않다. 값을 못 정하면서 넣으면 그만큼이 가정이 된다')
_rngI=_M9h.ICSR.max()-_M9h.ICSR.min(); _rngW=_M9h.WS.max()-_M9h.WS.min()
print(f'  일사 {_fE9h.params["ICSR"]:+.4f}/MJ · 관측 범위 {_M9h.ICSR.min():.1f}~{_M9h.ICSR.max():.1f} MJ/m² → 끝에서 끝까지 ×{np.exp(_fE9h.params["ICSR"]*_rngI):.1f}')
print(f'  풍속 {_fE9h.params["WS"]:+.4f}/(m/s) · 범위 {_M9h.WS.min():.1f}~{_M9h.WS.max():.1f} → ×{np.exp(_fE9h.params["WS"]*_rngW):.2f}')
print('  → AIDC 부지의 바람이 어떻게 바뀔지 우리는 모른다(건물 배치·플룸 상승·지표 변화 전부 관여).')
print('    모르는 값을 "현재대로"로 고정한 계수를, 바로 그 부지가 바뀌는 시나리오에 쓰는 것은 순환이다.')

# ── 참고 기록. 기온×일사 상호작용 (채택 안 함) ──
print()
print('[참고·미채택] 기온×일사 상호작용 — "덥고 맑은 날에야 오른다"는 통념의 검정')
_M9h['Tc']=_M9h.Tmax-_M9h.Tmax.mean(); _M9h['Ic']=_M9h.ICSR-_M9h.ICSR.mean()
_fX9h=_smf9f.glm('온열 ~ Tc*Ic + RH_at_Tmax + RN + WS',data=_M9h,family=_sm9f.families.Poisson()).fit()
_bX,_pX=float(_fX9h.params['Tc:Ic']),float(_fX9h.pvalues['Tc:Ic'])
print(f'  Tc:Ic {_bX:+.5f} (p={_pX:.1e}) · AIC {_fX9h.aic:.0f} (Ⓔ {_fE9h.aic:.0f} 대비 {_fX9h.aic-_fE9h.aic:+.0f})')
for _lab,_iv in [('일사 낮은 날',float(_M9h[_M9h.ICSR<=_qI[0]].ICSR.median())),('일사 평균',float(_M9h.ICSR.mean())),
                 ('일사 높은 날',float(_M9h[_M9h.ICSR>=_qI[1]].ICSR.median()))]:
    print(f'    {_lab:12} ICSR={_iv:5.1f} → 기온 배수 {np.exp(_fX9h.params["Tc"]+_bX*(_iv-_M9h.ICSR.mean())):.3f}')
print('  → **절반만 맞다.** 환자 "수"는 일사 상위⅓에서 하위⅓의 6.9배로 확실히 많다(위 근거 1 표).')
print(f'    그러나 기온 1℃의 **배수**는 맑은 날에 오히려 작다(1.596 → 1.485). 상호작용 부호가 음(−)이다.')
print('    로그선형 모형의 포화일 수도, 폭염경보일의 행동 변화일 수도 있다 — **기전은 우리 자료로 못 가린다**.')
print('    어느 쪽이든 결정은 안 바뀐다: 이 계수 역시 "일사를 우리가 정할 수 있을 때"만 쓸 수 있기 때문.')

print()
print('[남기는 것] §9-f·§9-g·§9-h의 습도·일사 결과는 **모형이 아니라 논증**에 쓴다:')
print('  · §12-c — "냉각탑은 기온을 덜 올린다"가 곧 "덜 위험하다"가 아니다(습도 독립 기여 ×1.124/10%p).')
print('  · §17 요구사항 — 사업자에게 **플룸 모형(AERMOD/SACTI 계열) 제출 의무화**를 요구하는 근거.')
print('    기온·습구온도의 반사실을 사업자만 알 수 있다면, 그것을 내게 하는 것이 곧 입증책임의 이전이다.')

# ── [🔖 2026-07-27 사용자 질문] "f_s가 관측 환자 수에 곱하는 배수라고? 잘 모르겠는데" ──
print()
print('[근거 2 상세] excess_ring()이 실제로 무엇을 곱하는지 한 줄씩')
print('   추가온열/년 = 인구 × (관측 온열률/10만) × f_s')
print('                 └──────── 이게 "지금 이미 나는 환자 수" ────────┘   └ 여기만 모형 ┘')
print('   · 관측 온열률 = NEDIS 실제 발생건수 ÷ 인구 (DERIVED_0718_시군구연도_패널) — **모형이 만든 값이 아니다**')
print('   · f_s = m^ΔT − 1 — 포아송 모형이 주는 건 기울기 exp(b)=m **하나뿐**이고, 절편·예측 수준은 안 쓴다')
# [🔖 2026-07-27 정정] 이전 판은 `_rate3`(§11-b(3)에서 정의)를 참조했는데 §9-h가 **앞**이라 항상 미정의였다.
#   `in dir()` 가드 때문에 죽지 않고 조용히 nan을 출력했다 — 죽는 것보다 나쁘다(틀린 값을 보여줌).
#   같은 원천을 여기서 직접 읽는다(§11-b(3)과 동일 파일·동일 집계라 값이 일치해야 한다).
_dm9h=float(pd.read_csv('DERIVED_0718_시군구연도_패널_온열전력.csv',encoding='utf-8-sig')
            .groupby('key')['여름온열률10만'].mean().get('인천광역시 동구',float('nan')))
assert _dm9h==_dm9h, '§9-h 실례: 인천 동구 온열률 결측 — CSV 확인 필요'
# [🔖 2026-07-27 정정 · 정합성 감사 B1] 이전 판은 169,396을 썼는데 그건 **현대제철 단독 점** 기준이고
#   §11-b(3)의 실제 계산은 **현대+동국 union** 기준 182,420이다. 같은 링을 설명하면서 다른 기하를 쓰고 있었다.
#   실례는 §11-b(3)과 같은 숫자를 보여야 한다 — 안 그러면 독자가 두 절을 대조할 때 어긋난다.
_pop9h,_dT9h=182420,dT_at(2.0)   # 인천 동구 1-3km 링 · 현대제철+동국제강 union 경계 기준(§11-b(3)과 동일)
_cur9h=_pop9h*_dm9h/1e5; _f9h=f_site('인천광역시 동구',_dT9h)
print()
print(f'   [실례] 인천 동구 1-3km 링 (현대제철+동국제강 union 경계 기준 — §11-b(3)과 동일 기하)')
print(f'     인구 {_pop9h:,}명 × 관측 온열률 {_dm9h:.1f}/10만 = **지금 {_cur9h:.1f}명/년**  ← 관측값')
print(f'     ΔT={_dT9h:.2f}℃ → f_s = {BETA_SITE.get("인천광역시 동구",HEAT_MULT_PER_C):.2f}^{_dT9h:.2f} − 1 = {_f9h:+.3f}')
print(f'     추가 = {_cur9h:.1f} × {_f9h:.3f} = **{_cur9h*_f9h:.1f}명/년**')
print()
print('   → 모형의 예측 수준(하루 몇 명)은 **어디에도 안 들어간다**. 그래서 §9-g에서 Ⓔ가')
print('     덥고 맑은 날 +3.1건 더 예측한다는 사실이 우리 숫자를 바꾸지 않는다.')
print('   → 바꾸는 건 오직 m뿐이다 — Ⓐ 1.575 vs Ⓔ 1.511. 그래서 §9-h의 쟁점이 m 하나로 좁혀진다.')
print('   ⚠ 대신 이 구조에는 다른 가정이 숨어 있다: **관측 온열률이 ΔT 이후에도 그대로 기준선**이라는 것.')
print('     즉 "현재 발생 패턴 위에 비례해서 얹힌다"고 본다. 취약층 구성이 바뀌면 이 가정도 흔들린다.')

[결정] dose 엔진 = 일최고기온 단일항(Ⓐ). HEAT_MULT_PER_C = 1.575 유지.
       Ⓔ(1.511)·Ⓑ(습구)·상호작용 = 기록 보존, 미채택.

근거 1 — AIC가 낮다고 예측이 항상 낮지는 않다
  전체 552일 중 Ⓔ 예측 > Ⓐ 예측: 298일 (54%)
    일사 하위⅓ (n=184) 실측   6.5 · Ⓐ   7.7 · Ⓔ   7.5  (Ⓔ−Ⓐ  -0.1)
    일사 중위⅓ (n=184) 실측  24.9 · Ⓐ  24.5 · Ⓔ  23.2  (Ⓔ−Ⓐ  -1.4)
    일사 상위⅓ (n=184) 실측  44.6 · Ⓐ  43.8 · Ⓔ  45.4  (Ⓔ−Ⓐ  +1.5)
  ★ **덥고 맑은 날**(n=88, 폭염의 전형): Ⓐ 76.8 vs Ⓔ 79.9 → **Ⓔ가 하루 +3.1건 더 예측**한다.
    → AIC는 552일 전체의 적합도일 뿐, 특정 조건에서 어느 쪽이 크게 예측하는지는 따로 봐야 한다.

근거 2 — 그런데 f_s = m^ΔT − 1 은 관측된 실제 환자 수에 곱하는 **배수**다(모형의 예측 수준을 쓰지 않는다)
       ΔT     Ⓐ f_s     Ⓔ f_s     Ⓔ 채택 시
    0.26℃    +0.125    +0.113      -9.6%
     0.5℃    +0.255    +0.229     -10.1%
     1.0℃    +0.575    +0.511     -11.1%
     2.0℃    +1.481    +1.284     -13.3%
  → 배수로 보면 Ⓔ는 **언제나 더 작다**. 즉 Ⓔ 채택은 §10~§18 추정치를 10~13% 낮춘다.
  ⚠ **방향을 숨기지 않는다**: 우리가 유지하기로 한 Ⓐ가 곧 **더 큰 숫자**를 주는 쪽이다.
    그래서 근거를 결과가 아니라 위 ①②③(반사실을 아느냐)에 두었다. 반박 시 이 문단을 먼저 보일 것.

근거 3 — Ⓔ가 통제하는 변수들은 작지 않다. 값을 못 정하면서 넣으면 그만큼이 가정이 된다
  일

In [36]:
_info9.head(1)

,지점,시작일,종료일,지점명,지점주소,관리관서,위도,경도,노장해발고도(m),기압계(관측장비지상높이(m)),기온계(관측장비지상높이(m)),풍속계(관측장비지상높이(m)),강우계(관측장비지상높이(m)),시도
0,90,1968-01-01,NaN,속초,강원특별자치도 고성군 토성면 봉포리,속초기상대(90),38.2509,128.5647,17.53,18.73,1.7,10.0,1.4,강원특별자치도


In [37]:
_fr9[0]

,지점,시도,dt,최고기온(°C)
0,90,강원특별자치도,2020-05-01,32.4
1,90,강원특별자치도,2020-05-02,28.6
2,90,강원특별자치도,2020-05-03,26.8
3,90,강원특별자치도,2020-05-04,28.0
4,90,강원특별자치도,2020-05-05,17.4
...,...,...,...,...
14530,295,경상남도,2020-09-26,25.0
14531,295,경상남도,2020-09-27,24.6
14532,295,경상남도,2020-09-28,23.5
14533,295,경상남도,2020-09-29,24.8


$$\ln(\text{온열환자 수}) = a + b \times \text{최고기온}$$

$$\text{온열환자 수} = e^a \times (e^b)^{\text{최고기온}}$$

즉, 최고기온이 1°C 올라갈 때마다 환자 수는 더해지는 게 아니라 $e^b$배만큼 곱해지며(기하급수적으로) 증가하게 됩니다. 이 $e^b$ 값이 바로 HEAT_MULT_PER_C입니다. (예: 이 값이 1.6이면 1°C 상승 시 1.6배가 된다는 뜻)

## §10. ③ 버퍼 인구 — SGIS 집계구로 절대 headcount

SGIS 2024 집계구 총인구(`to_in_001`, 108,510 집계구, **EPSG:5179**)를 산단 3km 버퍼와 **면적가중 교차**. join 100%·국가합 51.8M 검증. 산단 좌표(5186)는 집계구(5179)로 reproject.

> **CSV 컬럼**(원본에 헤더 없음): `(통계년도, 집계구코드 14자리, 항목코드, 값)` — 항목코드는 `0718_ref_code/3. 제공용 코드(statistics_code).xls` 기준 **to_in_001=총인구(외국인 포함)·to_in_007=총인구(남)·to_in_008=총인구(여)**. 아래 셀이 코드표를 직접 읽어 증거 출력.
>
> **`f = 1.6²−1 = 1.56`의 뜻**: '+2°C에서의 **추가** 위험분'. 1°C당 ×1.6은 **논문 값이 아니라 우리 실측** — 권역-day 패널(12권역×여름 918일)의 기온→온열 dose-response(`검증노트북_2026-07-06` §18 · 통합보고서 §2-1). +2°C면 총위험 ×1.6²=2.56, baseline(×1)을 뺀 **추가분 ×1.56**. 케임브리지(+2.07°C)·Sailor(근접 +2.2°C)는 **ΔT 앵커**(`[외부]`), ×1.6^ΔT는 **ΔT→온열 변환**(`[사실·우리 실측]`) — 역할이 다르다.

> **왜 이 표에 인천 현대제철·동국제강이 없나** `[🔖 2026-07-25 사용자 질문]`: §10은 §7 `T`의 **산업단지** 9곳만 돈다. 인천 두 곳은 산단이 아니라 **개별 폐공장 부지**여서 ILIS(`DAM_PDAN`/`DAM_YUCH`)에 형상이 없고, V-World 지오코딩으로 얻은 **점 좌표**만 있다. 그래서 §13(개별 부지)에서 따로 다루고, 링 시나리오는 §11-b(2)·§11-b(3)에서 폴리곤 대신 점 버퍼로 잰다. 심팩 포항도 같은 이유로 §13에 있다.
>
> 이 구분은 형식이 아니라 **논증 구조**다 — 가동 산단은 열원 *교체*(§11-b(2))라 순변화 부호가 불확실해 환자 수를 안 붙이지만, 폐쇄 확정 부지는 *순수 더하기*라 붙일 수 있다.

In [38]:
import pyogrio
from shapely.geometry import Point
SHP='0718_2025년 집계구경계/bnd_oa_00_2025_2Q.shp'
# 총인구 lookup + 검증
# names: 원본 무헤더 4열 = (y=통계년도, oa=집계구코드14자리, item=항목코드, val=값). oa는 str로 읽어야 선행0 보존.

# 1. 2024년 전국 인구총괄 CSV를 불러옵니다. (집계구 코드는 앞의 '0'이 안 사라지게 문자열로 지정)
_pop=pd.read_csv('0718_2024년 인구총괄/2025년기준_2024년_인구총괄(총인구).csv',
    header=None,names=['y','oa','item','val'],dtype={'oa':str},encoding='cp949')

# 2. 코드표 엑셀 파일에서 총인구를 의미하는 항목코드(to_in_001) 검증용 출력을 수행합니다.
_cwb=CalamineWorkbook.from_path('0718_ref_code/ref_code/3. 제공용 코드(statistics_code).xls')
for row in _cwb.get_sheet_by_name('집계구·행정동').to_python():
    if any('to_in_00%d'%d in str(c) for c in row for d in (1,7,8)):
        print('  코드표:', [str(c) for c in row if str(c)][:3])

# 3. '총인구' 항목(to_in_001)만 필터링하고, 총합이 약 5,180만 명으로 맞는지 검증합니다.
_pop=_pop[_pop.item=='to_in_001']
print('총인구 국가합:',f"{int(_pop.val.sum()):,}",'(통계청 2024 ~51.8M)  | 집계구:',len(_pop))

# 4. 빠른 속도로 조회하기 위해 {집계구코드: 인구수} 형태의 딕셔너리(POP)로 변환합니다.
POP=_pop.set_index('oa')['val'].to_dict()

# 5. 분석 대상 지점들(dr)의 위경도(lon, lat)를 미터(m) 단위 거리를 계산할 수 있는 한국 표준 좌표계(EPSG:5179)로 변환합니다.
_pts=gpd.GeoDataFrame(dr,geometry=[Point(lo,la) for lo,la in zip(dr.lon,dr.lat)],crs=4326).to_crs(5179)


# ── [🔖 2026-07-27 인터페이스 통일 — 사용자 요청] ──────────────────────────────
# 같은 계산을 하는 함수가 둘인데 반환 형태가 달라서 실제로 버그가 났다:
#   · buffer_pop() → dict {'tot','e'}     (점 전용, bbox를 geom.x/y로 잡음)
#   · polybuf()    → tuple (tot, e)       (폴리곤·점 공용, bbox를 buf.bounds로 잡음)
# 2026-07-27 §12-b2 작성 중 polybuf 쪽 열이름 'p'를 buffer_pop의 딕셔너리 키로 착각해
#   `buffer_pop(...)['p']` → KeyError. 배치가 무변경 종료해 손실은 없었지만 원인은 이 이중 구조다.
# → 계산 본체를 pop_in() 하나로 모으고, 기존 두 함수는 **반환 형태를 그대로 두는 래퍼**로 남긴다.
#   (호출부 12곳을 건드리지 않으므로 결과·출력은 완전히 동일하다. 새 코드는 pop_in()만 쓸 것.)
from collections import namedtuple as _nt_pop
_PopIn = _nt_pop('_PopIn', ['tot', 'e'])

def pop_in(geom, R, emap=None):
    """면적가중 버퍼 인구 — 점·폴리곤 공용. 반환 (tot, e): 튜플 언패킹과 .tot/.e 둘 다 된다.

    geom : shapely 형상(EPSG:5179).  R : 버퍼 m (0이면 형상 자체).
    emap : 집계구코드→값 매핑(예: 65+ 인구). None이면 e=nan.
    """
    buf = geom.buffer(R)                       # R=0도 원함수들과 동일하게 buffer(0)
    b = buf.bounds
    sub = pyogrio.read_dataframe(SHP, bbox=(b[0]-500, b[1]-500, b[2]+500, b[3]+500))
    w = (sub.geometry.intersection(buf).area/sub.geometry.area).clip(0, 1)
    tot = float((sub['TOT_OA_CD'].map(POP).fillna(0.0)*w).sum())
    e = float((sub['TOT_OA_CD'].map(emap).fillna(0.0)*w).sum()) if emap is not None else float('nan')
    return _PopIn(tot, e)

def buffer_pop(geom,R,extra_col=None,extra_map=None):
    """⚠ 구 인터페이스 유지용 래퍼 — dict 반환. 새 코드는 pop_in()을 쓸 것."""
    _r = pop_in(geom, R, extra_map)
    return {'tot': _r.tot} if extra_map is None else {'tot': _r.tot, 'e': _r.e}

def _buffer_pop_legacy(geom,R,extra_col=None,extra_map=None):

    # 6. 속도를 극대화하기 위해, 원 주변(반경 R + 500m) 영역의 집계구만 바운딩 박스(bbox)로 빠르게 잘라내어 읽어옵니다.
    pad=R+500; b=(geom.x-pad,geom.y-pad,geom.x+pad,geom.y+pad)
    oa=pyogrio.read_dataframe(SHP,bbox=b)

    # 7. 해당 집계구들의 인구수를 할당합니다.
    oa['p']=oa['TOT_OA_CD'].map(POP).fillna(0.0)

    # 8. (집계구와 R 반경 원의 교집합 면적) / (집계구 전체 면적) 비율을 구합니다. (0~1 사이로 제한)
    w=(oa.geometry.intersection(geom.buffer(R)).area/oa.geometry.area).clip(0,1)

    # 9. 면적 비율(w)을 곱한 추정 인구수를 모두 합산합니다.
    res={'tot':(oa['p']*w).sum()}

    # 10. 추가 변수 매핑 옵션이 있는 경우 함께 계산합니다.
    if extra_map is not None:
        oa['e']=oa['TOT_OA_CD'].map(extra_map).fillna(0.0); res['e']=(oa['e']*w).sum()
    return res

# 11. 1km, 2km, 3km, 5km 반경에 대해 각 지점별 상주인구를 계산해 '인구_Nkm' 열로 저장합니다.
for R in [1000,2000,3000,5000]:
    dr[f'인구_{R//1000}km']=[int(buffer_pop(g,R)['tot']) for g in _pts.geometry]
    
# 12. [🔖 2026-07-24 사용자 지적] 전국 상수 f=1.6²−1을 **부지별 f_site()**로 교체.
#     §9-e가 부지마다 배수 m_s를 채택했으니 추가위험분도 f_s = m_s²−1 이어야 한다.
#     전국 상수는 이제 '구 방식 대조' 컬럼으로만 남긴다.
f=HEAT_MULT_PER_C**2-1                                            # 전국 참조값(대조 전용 — 계산엔 안 씀)
dr['배수m_부지']=[round(BETA_SITE.get(k,HEAT_MULT_PER_C),3) for k in dr['key']]
dr['f_부지']=[round(f_site(k),3) for k in dr['key']]

# 13. [3km 반경 상주인구] × [10만 명당 온열 발생률 / 10만] × [부지별 f_s] → 연간 추가 온열질환자 수.
dr['3km_AIDC추가온열_년']=(dr['인구_3km']*dr['여름온열률10만']/1e5*dr['f_부지']).round(2)
dr['3km_추가온열_구전국f']=(dr['인구_3km']*dr['여름온열률10만']/1e5*f).round(2)   # 구 방식 대조

# 14. 예측 결과가 높은 순서대로 화면에 출력합니다.
print(dr[['대상','인구_1km','인구_2km','인구_3km','인구_5km','여름온열률10만','배수m_부지','f_부지','3km_AIDC추가온열_년','3km_추가온열_구전국f']]
      .sort_values('3km_AIDC추가온열_년',ascending=False).to_string(index=False))
print(f"\n→ 전국 상수 f={f:.3f}(m={HEAT_MULT_PER_C:.3f}) 대신 부지별 f_s=m_s²−1 적용(§9-e 채택 배수).")
print("  마지막 열이 구 방식 — 두 열의 차이가 곧 '전국 평균 하나로 뭉갰을 때의 오차'다.")

# 15. 최종 산출 결과를 CSV 파일로 저장합니다.
dr.to_csv('DERIVED_0718_dose_response_최종.csv',index=False,encoding='utf-8-sig')
print('\n★ 노출 극단차: 광양 3km 73명 vs 구미 66,873명. 광양 온열률 최고(20.7)인데 거주민 0')
print('  → 광양 온열=제철 실외노동자(§6 재확인), AIDC(실외노동자無)는 고립산단서 ambient 피해 작음.')

  코드표: ['총괄', '인구총괄\n(2015년 이후 외국인 포함)', '총인구']
  코드표: ['총인구(남자)', 'to_in_007']
  코드표: ['총인구(여자)', 'to_in_008']
총인구 국가합: 51,805,547 (통계청 2024 ~51.8M)  | 집계구: 108510
            대상  인구_1km  인구_2km  인구_3km  인구_5km  여름온열률10만  배수m_부지  f_부지  3km_AIDC추가온열_년  3km_추가온열_구전국f
         구미·전자    2556   28550   66873  191872       8.0   1.463 1.140            6.10           7.92
       울산미포·석화     284    6027   51576  269253       5.3   1.514 1.292            3.53           4.05
   동해북평·국가(대조)     561    5886   16913   67224       6.4   1.568 1.457            1.58           1.60
동해 북평2·GS2.4GW     468    4343   15581   71682       6.4   1.568 1.457            1.45           1.48
         포항·제철     211    3608    9937  129824       9.4   1.368 0.872            0.81           1.38
      당진·철강폐산단     165     619    1542    8356      13.2   1.585 1.511            0.31           0.30
         온산·석화     151     618    1300   25908       8.7   1.565 1.450            0.16           0.17
         여수·석화     

**§10-b. 형상 민감도 — 점+3km vs 실폴리곤 경계+buffer** (2026-07-19: §8-b에서 원형 가정이 깨졌으므로)
열은 산단 '중심'이 아니라 **산업시설 경계**에서 퍼진다. YUCH 실폴리곤에서 1·2·3km 버퍼로 인구·65+를 재계산해 점 모형과 비교한다. 65+ 정의는 코드표 `in_age_014`(65~69세)~`in_age_021`(100세+) — 아래 셀이 코드표 증거 출력. 성연령 셀의 **결측 ≈17.5%는 소지역 비밀보호 마스킹** `[사실 — SGIS 매뉴얼 §1.4 확인]`: 매뉴얼 원문에 *"집계구 통계는 속성값이 5 미만이면 응답자 비밀보호를 위해 N/A 처리"*, *"5세 단위 연령별 인구 등 속성값 5 미만은 N/A"*, *"비밀보호로 속성값 합이 총괄값보다 작을 수 있음"*이라 명시. 그래서 총인구 컬럼은 결측 0인데 세부 연령 셀만 N/A(전국 손실 1.04%). `fillna(0)`은 N/A→0 하한 처리 → 65+ 최대 ~1% 과소(보수, 매뉴얼 서술과 정합).

> **컬럼 읽기**: `R0폴리곤내` = 버퍼 반경 0, 즉 **산단 폴리곤 내부만**(공장지대라 거주 인구 적음). `+1/+2/+3km` = 폴리곤 경계에서 바깥으로 넓힌 버퍼(내부 포함). 대부분 인구는 폴리곤 바깥 링의 주거지.

In [39]:
# 1. 엑셀 코드표에서 65세 이상을 나타내는 항목코드(in_age_014 ~ 021)의 정의를 출력·확인합니다.
for row in _cwb.get_sheet_by_name('집계구·행정동').to_python():
    if any(('in_age_0%d'%d) in str(c) for c in row for d in (14,21)):
        print('  코드표:', [str(c) for c in row if str(c)][:3])

# 2. 65세 이상 연령대 코드 집합(ELDER)을 정의합니다. (65-69세 ~ 100세 이상)
ELDER={f'in_age_{n:03d}' for n in range(14,22)}   # 65-69,70-74,...,95-99,100+ (남녀합산 계열)
_acc={}; _nan=0; _tot=0

# 3. 대용량 성연령별 인구 CSV를 100만 줄 단위(chunk)로 나누어 읽어옵니다.
for ch in pd.read_csv('0718_2024년 인구총괄/2025년기준_2024년_성연령별인구.csv',
        header=None,names=['y','oa','item','val'],dtype={'oa':str,'item':str},encoding='cp949',chunksize=1_000_000):
    _tot+=len(ch); _nan+=int(ch['val'].isna().sum())

    # 4. 65세 이상 고령자 항목만 필터링합니다.
    sub=ch[ch['item'].isin(ELDER)].copy()

    # 5. 소지역 통계 마스킹(비밀보호)으로 발생한 결측치(NaN)는 0으로 채워 보수적으로 처리합니다.    
    sub['val']=pd.to_numeric(sub['val'],errors='coerce').fillna(0)

    # 6. 집계구(oa)별로 65세 이상 고령자 수를 합산하여 누적 사전(_acc)에 집계합니다.
    for k,v in sub.groupby('oa')['val'].sum().items(): _acc[k]=_acc.get(k,0)+v

# 7. 전국 65세 이상 고령자 총인구와 비중, 결측 비율을 계산하여 화면에 출력합니다.
POP65_NAT=int(sum(_acc.values())); POPALL_NAT=int(sum(POP.values()))
print(f'65+ 국가합 {POP65_NAT:,} / 총인구 {POPALL_NAT:,} = {POP65_NAT/POPALL_NAT*100:.1f}% | 성연령 셀 결측 {_nan:,}/{_tot:,} ({_nan/_tot*100:.1f}%)')

# ② 산업단지 경계(Polygon) 기반 면적 버퍼 함수 구현
import pyproj
from shapely.ops import transform as _shT

# 8. EPSG:5186 좌표계를 한국 표준 EPSG:5179 좌표계로 변환하는 변환기를 만듭니다.
_tf=pyproj.Transformer.from_crs(5186,5179,always_xy=True)

# 9. 산단 경계 도형(u79)으로부터 R미터 확장된 버퍼 내 (총인구, 65세이상 인구)를 계산하는 함수를 정의합니다.
def polybuf(u79,R,emap):
    """⚠ 구 인터페이스 유지용 래퍼 — tuple 반환. 새 코드는 pop_in()을 쓸 것. [🔖 2026-07-27]"""
    _r = pop_in(u79, R, emap)
    return _r.tot, _r.e

def _polybuf_legacy(u79,R,emap):
    buf=u79.buffer(R); b=buf.bounds
    sub=pyogrio.read_dataframe(SHP,bbox=b)
    sub['p']=sub['TOT_OA_CD'].map(POP).fillna(0.0) #총인구
    sub['e']=sub['TOT_OA_CD'].map(emap).fillna(0.0) #고령자 인구


    w=(sub.geometry.intersection(buf).area/sub.geometry.area).clip(0,1)
    return float((sub['p']*w).sum()),float((sub['e']*w).sum())

# 10. 표 헤더(산단 내부인구 R0, +1km, +2km, +3km, +2km 내 65+고령자, 기존 점3km 인구)를 출력합니다.
print(f"{'대상':16}{'R0폴리곤내':>10}{'+1km':>9}{'+2km':>9}{'+3km':>10}{'+2km65+':>9}{'|점3km':>9}")  # R=0 = 산단 폴리곤 내부만(공장지대라 거주 적음)
sens={}
for lbl,key,(pnm,pt),xn in T:
    pr=PDAN[(PDAN.DAN_NAME==pnm)&(PDAN.DANJI_TYPE==pt)].iloc[0]
    sub=YUCH[YUCH.DAN_ID==pr.DAN_ID]
    old=int(dr.loc[dr['대상']==lbl,'인구_3km'].iloc[0])

    # 11. 산단 경계 도형(YUCH) 데이터가 없으면 기존 점 모형 유지
    if len(sub)==0: 
        print(f'{lbl:16} YUCH 결측 → 점 모형 유지 (점3km={old:,})'); continue
    
    # 12. 산단 내 여러 폴리곤들을 하나로 합치고(union), 5186 좌표계를 5179 표준 좌표계로 변환(u79)합니다.
    u79=_shT(_tf.transform,sub.geometry.union_all())

    # 13. 산단 경계 기준 0m(내부), 1km, 2km, 3km 버퍼 인구 및 65+ 고령 인구를 산출합니다.
    p0,_e0=polybuf(u79,0,_acc); p1,_e1=polybuf(u79,1000,_acc); p2,e2=polybuf(u79,2000,_acc); p3,_e3=polybuf(u79,3000,_acc)
    sens[lbl]=(p2,e2)
    print(f'{lbl:16}{int(p0):>10,}{int(p1):>9,}{int(p2):>9,}{int(p3):>10,}{int(e2):>9,}{old:>9,}')  # p0=R0=폴리곤내부
print("→ 형상 반영 시 노출이 수 배 커진다(울산미포 점3km 5.2만 → 폴리곤+2km 33.9만 — 여러 갈래로 흩어진 대형 해안 산단).")
print("  · 광양: PDAN 점=광양만 바다, YUCH 폴리곤=실제 POSCO 제철소(§8-b V-World 실증). 폴리곤 경계 버퍼가 옳음.")
print("     둘을 '하한~상한'으로 병기 [확인필요]. 여수·포항은 YUCH 결측이라 점 모형만.")

  코드표: ['65세이상~69세이하', 'in_age_014']
  코드표: ['100세이상', 'in_age_021']
65+ 국가합 9,744,801 / 총인구 51,805,547 = 18.8% | 성연령 셀 결측 1,020,825/5,835,015 (17.5%)
대상                  R0폴리곤내     +1km     +2km      +3km  +2km65+    |점3km
광양·제철                  709   16,049   45,510    72,511    5,047       73
여수·석화            YUCH 결측 → 점 모형 유지 (점3km=409)
울산미포·석화              3,929  203,990  338,677   467,637   51,230   51,576
온산·석화                  959    5,238   26,509    32,241    4,018    1,300
포항·제철            YUCH 결측 → 점 모형 유지 (점3km=9,937)
구미·전자                1,147   65,906  134,818   166,640    7,979   66,873
당진·철강폐산단               111      604    1,592     4,334      362    1,542
동해북평·국가(대조)            155    4,773   14,296    39,062    2,943   16,913
동해 북평2·GS2.4GW          48      877    7,307    25,978    1,530   15,581
→ 형상 반영 시 노출이 수 배 커진다(울산미포 점3km 5.2만 → 폴리곤+2km 33.9만 — 여러 갈래로 흩어진 대형 해안 산단).
  · 광양: PDAN 점=광양만 바다, YUCH 폴리곤=실제 POSCO 제철소(§8-b V-World 실증). 폴리곤 경계 버퍼가 옳음.
     둘을 '하한~상한'으

## §11. ④ 연령가중 — 65+ 고령 노출

버퍼 65+ 실측(§10-b의 `_acc`) + **우리 NEDIS 실측 고령 발생률**로 가중. 65+ RR은 외부 자료가 아니라 §6의 NEDIS 원자료에서 직접 계산. 전국 65+ 비율·총인구도 **상수가 아니라 파일에서 직접 도출**(2026-07-19 수정).

> **가중 공식 유도**: 전국 평균 위험을 기준(=1)으로 놓고, 65+의 상대위험을 RR, <65를 1이라 하면
> `전국평균 = NAT65×RR + (1−NAT65)×1 = avgRR` (NAT65 = 전국 65+ 인구비, 파일 도출)
> `E_ELD = RR/avgRR` (65+ 1명의 위험 = 전국평균 대비 몇 배), `E_YNG = 1/avgRR`
> → 버퍼의 연령 구성이 전국과 똑같으면 가중 결과가 비가중(§10)과 **정확히 일치**하도록 정규화한 것. 보정은 오직 '전국과 다른 연령 구성'만 반영한다. 예: RR=1.82, NAT65=0.188이면 avgRR≈1.154, E_ELD≈1.58, E_YNG≈0.87.

In [40]:
# 1. 엑셀 파일에서 2020~2025년 NEDIS 온열질환 발생 데이터를 읽어옵니다.
_wb=CalamineWorkbook.from_path('FOIA_0522_질병관리청_NEDIS온열질환_2020-2025.xlsx')
_r=_wb.get_sheet_by_name('DB(2020-2025)발생지역기준').to_python()
_ne=pd.DataFrame(_r[1:],columns=_r[0])

# 2. '발생일자'를 날짜형(datetime)으로, '나이'를 숫자형(numeric)으로 변환합니다.
_ne['dt']=pd.to_datetime(_ne['발생일자'].astype(str),errors='coerce')
_ne['age']=pd.to_numeric(_ne['나이'],errors='coerce')

# 3. 여름철(6월, 7월, 8월) 발생 데이터 중 나이 정보가 정상적인 데이터만 추출합니다.
_s=_ne[_ne['dt'].dt.month.isin([6,7,8])].dropna(subset=['age'])

# 4. 분석 대상 데이터의 연도 수(_yrs, 예: 6년), 65세 이상 환자 수(_n65), 65세 미만 환자 수(_nlt)를 집계합니다.
_yrs=_s['dt'].dt.year.nunique(); _n65=int((_s.age>=65).sum()); _nlt=int((_s.age<65).sum())

# 5. 이전 코드에서 산출한 파일 기반 총인구 중 고령층 인구비(NAT65)를 구합니다. (약 18.8%)
NAT65=POP65_NAT/POPALL_NAT                                     # 전국 65+ 인구비 (파일 도출 ≈0.188)

# 6. [65세 이상 연평균 발생률 (r65)] = (연평균 고령 환자 수) / (전국 65세 이상 총인구) × 10만 명
r65=(_n65/_yrs)/POP65_NAT*1e5

# 7. [65세 미만 연평균 발생률 (rlt)] = (연평균 비고령 환자 수) / (전국 65세 미만 총인구) × 10만 명
rlt=(_nlt/_yrs)/(POPALL_NAT-POP65_NAT)*1e5

# 8. [상대위험도 (RR)] = 65세 이상 발생률 / 65세 미만 발생률
RR=r65/rlt

# 9. 전체 여름 온열질환자 중 65세 이상이 차지하는 비중을 계산하여 출력합니다.
print(f'NEDIS 여름온열 {len(_s)}건/{_yrs}년 | 65+ {_n65}건 = 온열의 {_n65/len(_s)*100:.1f}% (전국 인구비 {NAT65*100:.1f}%보다 과대표집)')

# 10. 10만 명당 연간 발생률 수치 비교와 함께 최종 고령 상대위험도(RR)를 출력합니다.
print(f'65+ 여름온열률 {r65:.2f} vs <65 {rlt:.2f} (/10만·년) → 고령 RR = {RR:.2f}배')

NEDIS 여름온열 13995건/6년 | 65+ 4158건 = 온열의 29.7% (전국 인구비 18.8%보다 과대표집)
65+ 여름온열률 7.11 vs <65 3.90 (/10만·년) → 고령 RR = 1.82배


In [ ]:
# ── §11. 연령 구조(65+ 고령층) 가중 추가 온열질환 피해 정밀 추정 ──
# 목적: 고령층의 기온 민감도(상대 위험도 RR)를 반영하여 3km 공간 버퍼 내
#       실제 위험에 노출되는 연령별 추가 환자 수(절대 건수)와 노인 부담 비율을 산출함.

# -----------------------------------------------------------------------------
# 1. 연령 그룹별 온열질환 상대 가중치(E_ELD, E_YNG) 산출
# -----------------------------------------------------------------------------
# avgRR: 전국 평균 인구 구조(NAT65: 고령인구 비율)를 반영한 기대 위험도
avgRR=NAT65*RR+(1-NAT65)

# E_ELD: 고령층(65+)의 온열질환 상대 위험 가중치 (전국 평균 대비 고령층의 위험도 비율)
E_ELD=RR/avgRR

# E_YNG: 비고령층(<65)의 온열질환 상대 위험 가중치 (전국 평균 대비 비고령층의 위험도 비율)
E_YNG=1/avgRR       # §11 md의 유도 참조

# -----------------------------------------------------------------------------
# 2. 기온 상승(+2°C)에 따른 증가율 계수 설정
# -----------------------------------------------------------------------------
# [🔖 2026-07-24 정정] 일률적인 전국 상수 대신 §9-e에서 채택한 부지별 맞춤 위험계수(f_site) 적용
_fs11=[f_site(k) for k in dr['key']]      # 부지별 f_s = m_s² − 1
f=HEAT_MULT_PER_C**2-1                    # 전국 참조값(대조 전용)

# -----------------------------------------------------------------------------
# 3. 각 분석 지점별 3km 버퍼 내 인구 및 고령인구 추출
# -----------------------------------------------------------------------------
r3=[]
for g in _pts.geometry:
    # 3000m(3km) 버퍼 내 총인구(tot)와 고령인구(e) 계산
    res=buffer_pop(g,3000,extra_map=_acc)
    r3.append((res['tot'],res['e']))

# 결과 데이터프레임(dr)에 3km 버퍼 내 고령인구 수 및 고령인구 비율(노인비_pct, %) 저장
dr['노인_3km']=[int(e) for _,e in r3]; dr['노인비_pct']=(dr['노인_3km']/dr['인구_3km']*100).round(1)

# -----------------------------------------------------------------------------
# 4. 연령 가중치가 적용된 추가 온열질환자 발생 건수 정밀 계산
# -----------------------------------------------------------------------------
# 1) 연령별 위험 가중치가 적용된 전체 추가 온열질환 환자 수 (명/년)
#    산출식: [(비고령인구 × 발생률 × E_YNG) + (고령인구 × 발생률 × E_ELD)] / 10만 × 부지별 위험계수(f_s)
dr['추가온열_연령가중']=[round((e*rt*E_ELD+(t-e)*rt*E_YNG)/1e5*_f,2) for (t,e),rt,_f in zip(r3,dr['여름온열률10만'],_fs11)]

# 2) 전체 추가 발생 환자 중 65세 이상 고령층 환자 수 (명/년)
dr['그중65+_건']=[round((e*rt*E_ELD)/1e5*_f,2) for (t,e),rt,_f in zip(r3,dr['여름온열률10만'],_fs11)]

# 3) 전체 추가 환자 중 고령층이 차지하는 기여/부담 비율(%)
dr['노인부담비_pct']=[round((e*E_ELD)/(e*E_ELD+(t-e)*E_YNG)*100,0) for t,e in r3]


# -----------------------------------------------------------------------------
# 5. 시군구 '전체' 노인비 산출 (버퍼 공간 구성 효과 vs 도시 전체 특성 검증)
# -----------------------------------------------------------------------------
# 1. 행정구역 코드 엑셀 파일 로드 및 5자리 시군구 코드 생성
_awb=CalamineWorkbook.from_path('0718_ref_code/ref_code/1. 행정구역 코드(adm_code).xls')
_ad=pd.DataFrame(_awb.get_sheet_by_name('2025년 6월').to_python()[2:],columns=['sido','sido_nm','sgg','sgg_nm','emd','emd_nm'])
_ad['code5']=_ad['sido'].astype(str).str.replace('.0','',regex=False).str.zfill(2)+_ad['sgg'].astype(str).str.zfill(3)

# 2. 시군구 코드별 정규화된 이름 딕셔너리 및 집계구(OA) 매핑 생성
_m5={c:norm_key(s,g) for c,s,g in zip(_ad['code5'],_ad['sido_nm'],_ad['sgg_nm'])}
_oadf=pd.DataFrame({'oa':list(POP.keys())}); _oadf['key']=_oadf['oa'].str[:5].map(_m5)

# 3. 집계구별 인구(p) 및 고령인구(e) 매핑 후 시군구 단위로 합산
_oadf['p']=_oadf['oa'].map(POP); _oadf['e']=_oadf['oa'].map(_acc).fillna(0.0)

# 4. 시군구별 전체 노인비율(%) 산출 및 결과 데이터프레임 매핑
_sgg=_oadf.groupby('key').agg(p=('p','sum'),e=('e','sum')); _sgg['노인비']=(_sgg.e/_sgg.p*100).round(1)
dr['시군구노인비_pct']=dr['key'].map(_sgg['노인비'])

# -----------------------------------------------------------------------------
# 6. 주요 요약 정보 출력 및 최종 CSV 저장
# -----------------------------------------------------------------------------
print(f'전국 65+ 비율 {NAT65*100:.1f}% | E_ELD={E_ELD:.2f} E_YNG={E_YNG:.2f} (버퍼 구성=전국이면 §10과 일치)')
print(f'추가위험 계수: 부지별 f_s(§9-e) 적용 — 범위 {min(_fs11):.3f}~{max(_fs11):.3f} (구 전국 상수 {f:.3f})')

# 핵심 산출 결과 모니터링 출력 (추가 온열 환자 많은 순)
print(dr[['대상','인구_3km','노인_3km','노인비_pct','시군구노인비_pct','추가온열_연령가중','그중65+_건','노인부담비_pct']]
      .sort_values('추가온열_연령가중',ascending=False).to_string(index=False))

# 데이터 저장
dr.to_csv('DERIVED_0718_dose_response_연령가중.csv',index=False,encoding='utf-8-sig')

# 결과 해석 가이드 출력
print("\n읽는 법(절대수): '노인_3km'=위험권 안 65+ 거주자 수(정적 노출 — 예: 울산미포 6,888명이 +2°C 위험권에 삶),")
print("  '추가온열_연령가중'=+2°C 시 연간 추가 응급실 건수(그중 65+ 건수 = 그중65+_건).")
print("검증: 구미 버퍼 노인비 7.7% < 구미시 전체 12.0% < 전국 18.8% → '청년 산업도시' 실증(해석 아닌 데이터).")
print("⚠ §10-b 폴리곤 민감도 반영 시 노출 절대수는 수 배 커질 수 있음(울산미포 +2km 33.9만) — 본 표는 점+3km 보수 하한.")

전국 65+ 비율 18.8% | E_ELD=1.58 E_YNG=0.87 (버퍼 구성=전국이면 §10과 일치)
추가위험 계수: 부지별 f_s(§9-e) 적용 — 범위 0.872~1.511 (구 전국 상수 1.481)
            대상  인구_3km  노인_3km  노인비_pct  시군구노인비_pct  추가온열_연령가중  그중65+_건  노인부담비_pct
         구미·전자   66873    5167      7.7        12.0       5.62     0.74       13.0
       울산미포·석화   51576    6888     13.4        15.9       3.40     0.75       22.0
   동해북평·국가(대조)   16913    3342     19.8        24.4       1.59     0.49       31.0
동해 북평2·GS2.4GW   15581    2694     17.3        24.4       1.44     0.40       28.0
         포항·제철    9937    2764     27.8        21.5       0.87     0.36       41.0
      당진·철강폐산단    1542     370     24.0        19.5       0.32     0.12       37.0
         온산·석화    1300     236     18.2        18.0       0.16     0.05       29.0
         여수·석화     409     118     28.9        22.5       0.08     0.03       43.0
         광양·제철      73      21     28.8        15.4       0.02     0.01       44.0

읽는 법(절대수): '노인_3km'=위험권 안 65+ 거주자 수(정적 노출 — 예: 울산

**§11-x. 공개용 집계 export** `[🔖 2026-07-25]`

NEDIS 원자료(FOIA)는 공개할 수 없는데, 그게 없으면 §9-e 배수·§11 연령계수·§6 발생장소가 막혀 **§10~§18 전체가 재현 불가**였다([재현성_공개범위_점검_2026-07-25.md](재현성_공개범위_점검_2026-07-25.md)).

개인 단위가 아니라 **이미 집계·추정된 값**만 내보낸다 — 원자료를 안 올리는 편이 프라이버시상 더 안전하다. `발생장소` 집계는 **1~4건 셀을 마스킹**하고, 일자별은 **시도까지만** 공개한다(시군구×일자는 대부분 0~2건이라 재식별 위험).

In [ ]:
# ── §11-x. 공개용 집계 export — FOIA 원자료 없이 재현되게 [🔖 2026-07-25 사용자 승인] ──
# [왜] NEDIS 원자료(FOIA)는 공개 못 하는데, 그게 없으면 §9-e 배수·§11 연령계수·§6 발생장소가 막혀
#   §10~§18 전체가 재현 불가였다(재현성_공개범위_점검_2026-07-25.md 참조).
#   → 개인 단위가 아니라 **이미 집계·추정된 값**만 내보낸다. 원자료를 안 올리는 편이 프라이버시상 더 안전하다.

# 최종 수출(Export) 대상 데이터프레임들을 담을 딕셔너리
_EXP={}

# -----------------------------------------------------------------------------
# ① 부지별 dose 배수 데이터셋 파싱 및 가공 (§9-e BETA_SITE 재현용)
# -----------------------------------------------------------------------------
_r1=[]
for _k,_v in BETA_SITE.items():
    # 해당 시군구의 기상/온열질환 관련 기존 DataFrame이 전역 객체로 존재하는지 체크
    _g=_gr9[_gr9.key==_k] if '_gr9' in dir() else None

    _r1.append(dict(시군구=_k, 채택배수m=round(float(_v),4),
                    표본n=int(_g['온열'].sum()) if _g is not None and len(_g) else None,    # 표본 수
                    최근접관측소=_near9[_k][1] if _k in _near9 else None,                   # 매칭된 기상관측소 ID
                    관측소거리km=round(_near9[_k][2],1) if _k in _near9 else None))         # 관측소와의 거리
    
_EXP['DERIVED_0725_부지별_dose배수.csv']=pd.DataFrame(_r1)

# -----------------------------------------------------------------------------
# ② 연령가중 계수 데이터셋 생성 (§11 연령별 위험도 가중치 재현용)
# -----------------------------------------------------------------------------
# 고령층(65세 이상) 상대위험도(RR) 및 인구 비중 기반 파라미터 백업
_EXP['DERIVED_0725_연령가중_계수.csv']=pd.DataFrame([dict(
    전국65세이상비율=round(float(NAT65),5), 
    RR_65이상=round(float(RR),4),
    avgRR=round(float(NAT65*RR+(1-NAT65)),4),   # 가중 평균 상대위험도
    E_ELD=round(float(E_ELD),4),                # 고령층 노출 계수 
    E_YNG=round(float(E_YNG),4))])              # 청장년층 노출 계수

# -----------------------------------------------------------------------------
# ③ 시군구 × 발생장소 여름 집계 및 소수 셀 마스킹 (§6 장소별 분석 재현용)
# -----------------------------------------------------------------------------
# FOIA 원자료 로드 및 데이터 전처리
_wb6=CalamineWorkbook.from_path('FOIA_0522_질병관리청_NEDIS온열질환_2020-2025.xlsx')
_rw6=_wb6.get_sheet_by_name('DB(2020-2025)발생지역기준').to_python()
_ne6=pd.DataFrame(_rw6[1:],columns=_rw6[0])

# 표준 시군구 식별키 생성 및 날짜 변환
_ne6['key']=[norm_key(a,b) for a,b in zip(_ne6['발생시도'],_ne6['발생시군구'])]
_ne6['dt']=pd.to_datetime(_ne6['발생일자'].astype(str),errors='coerce')

# 여름철(6~8월) 관측치만 추출
_s6=_ne6[_ne6['dt'].dt.month.isin([6,7,8])]

# 시군구(key) × 발생장소별 교차 집계표(Pivot Table) 생성
_pv6=_s6.pivot_table(index='key',columns='발생장소',aggfunc='size',fill_value=0)

# [프라이버시/재식별 방지] 1~4건 발생 셀 마스킹 (K-Anonymity 원칙 적용)
_nmask=int((( _pv6>0)&(_pv6<5)).sum().sum())
_pv6=_pv6.mask((_pv6>0)&(_pv6<5))                       # 1~4건 → 결측 처리

_EXP['DERIVED_0725_시군구_발생장소_여름집계.csv']=_pv6.reset_index()

# -----------------------------------------------------------------------------
# ④ 광역 시도 × 일자별 온열질환 발생 건수 집계 (§9-①·c 모형 재현용)
# -----------------------------------------------------------------------------
_s6b=_ne6.dropna(subset=['dt']).copy()

# 시도명 표준화 매핑 (SIDO_MAP 활용)
_s6b['시도']=_s6b['발생시도'].map(lambda s: SIDO_MAP.get(str(s).strip(),str(s).strip()))

# 시군구 단위는 재식별 위험이 높아 광역시도(시도) 단위로 상위 집계하여 출력
_EXP['DERIVED_0725_시도일자_온열건수.csv']=(_s6b.groupby(['시도',_s6b['dt'].dt.date]).size()
                                        .rename('온열').reset_index().rename(columns={'level_1':'일자'}))

# -----------------------------------------------------------------------------
# 5. 최종 데이터 파일 내보내기(CSV Export) 및 리포팅
# -----------------------------------------------------------------------------
for _f,_d in _EXP.items():
    _d.to_csv(_f,index=False,encoding='utf-8-sig')
    print(f'  저장 {_f:44} {len(_d):>6,}행 × {len(_d.columns)}열')
print()
print(f'→ ①이 §10~§18의 f_site()를, ②가 §11·§12-b·§18-b2의 연령가중을, ③이 §6 판별을, ④가 §9-①·c를 각각 재현 가능하게 한다.')
print(f'  ③ 마스킹: 1~4건 셀 {_nmask}개를 결측 처리했다(재식별 방지) — 합계·상관은 결측을 0으로 보지 말고 제외할 것.')
print(f'  ④ 시군구×일자는 대부분 0~2건이라 공개하지 않는다. §9-d(시군구 FE)는 여전히 원자료가 필요하다.')


  저장 DERIVED_0725_부지별_dose배수.csv                      10행 × 5열
  저장 DERIVED_0725_연령가중_계수.csv                          1행 × 5열
  저장 DERIVED_0725_시군구_발생장소_여름집계.csv                  230행 × 14열
  저장 DERIVED_0725_시도일자_온열건수.csv                    4,687행 × 3열

→ ①이 §10~§18의 f_site()를, ②가 §11·§12-b·§18-b2의 연령가중을, ③이 §6 판별을, ④가 §9-①·c를 각각 재현 가능하게 한다.
  ③ 마스킹: 1~4건 셀 1302개를 결측 처리했다(재식별 방지) — 합계·상관은 결측을 0으로 보지 말고 제외할 것.
  ④ 시군구×일자는 대부분 0~2건이라 공개하지 않는다. §9-d(시군구 FE)는 여전히 원자료가 필요하다.


## §11-b. Cambridge hyperscaler footprint — 4.5·10km 노출 인구 `[🔖 2026-07-21 신규]`

§10·§11은 산단 근접(3km)만 봤다. 그런데 §4에서 본 두 실측 논문은 **시설 크기에 따라 열 반경이 다르다**는 걸 보인다:
- **Sailor 외(ASU 2025)** — 36~169MW 중소형, 차량 **공기온도** 실측 → 최대 +2.2℃, **~500m**
- **Marinoni 외(케임브리지)** — **AI hyperscaler**(대형), 위성 **지표온도** → +1℃ 4.5km, **10km까지**

한국 메가프로젝트(동해 2.4GW 등)는 **hyperscaler 급**이라 규모-대응 레퍼런스는 케임브리지(10km). 아래 셀이 8 가동 산단 + 인천 폐공장의 3·4.5·10km 노출 인구를 낸다(§5·§6 표의 출처).

> ⚠ **읽는 법**: 10km 인구 = **지표 열영향권** 상한(케임브리지 LST). 강한 **체감**(공기온도)은 Sailor가 잰 근거리(≤500m~1km). 이 표는 노출 인구지 발병자 아님.

In [ ]:
# ── §11-b. Cambridge hyperscaler footprint (3·4.5·10km) 인구 — [🔖 2026-07-24 폴리곤 전환] ──
# [사용자 지적] 상한을 제대로 구하려면 PDAN '점'이 아니라 YUCH '폴리곤 경계'에서 재야 한다.
#   실제로 점→폴리곤 전환 시 3km 인구가 미포 9.1배·온산 25배·광양 993배로 커진다(점 기준은 상한이 아니라 심한 과소였음).
#   YUCH 결측(여수·포항)만 점 폴백. 인천 폐공장 2곳은 개별 부지라 점 유지.
import pyproj as _pyproj11
from shapely.ops import transform as _shT11
from shapely.geometry import Point as _Pt10

# 좌표계 변환기 정의: EPSG:5186(중부원점) → EPSG:5179(UTM-K, 미터 단위 좌표계)
_tf11=_pyproj11.Transformer.from_crs(5186,5179,always_xy=True)

def _geo11(nm,ty):
    """
    산업단지 형상 객체 반환 함수:
    YUCH 폴리곤(면적)이 있으면 우선 사용하고, 결측 시 PDAN 점(Point) 좌표로 폴백(Fallback).
    반환값: (Geometry 도형 객체, 폴리곤 여부[True/False])
    """
    _r=PDAN[(PDAN.DAN_NAME==nm)&(PDAN.DANJI_TYPE==ty)].iloc[0]
    _yc=YUCH[YUCH.DAN_ID==_r.DAN_ID]

    if len(_yc):
    # YUCH 폴리곤들의 합집합(union_all)을 구한 뒤 5179 좌표계로 변환하여 반환
        return _shT11(_tf11.transform,_yc.geometry.union_all()), True

    # YUCH 데이터가 없는 경우 PDAN 점 좌표를 5179 좌표계로 변환하여 반환
    return gpd.GeoSeries([_Pt10(_r.lon,_r.lat)],crs=4326).to_crs(5179).iloc[0], False

# 8 산단 + 북평2(★ 실부지) — §7 T와 동일 정의
_TG10=[('광양·제철','광양','1'),('여수·석화','여수','1'),('울산미포','울산·미포','1'),('온산','온산','1'),
       ('포항·제철','포항','1'),('구미·전자','구미(2·3단지)','1'),('당진1철강','당진1철강','2'),
       ('동해북평(국가·대조)','북평','1'),('동해 북평2(GS2.4GW)','북평2','2')]

# 반경 거리(m)에 대응하는 컬럼 명칭 매핑
_LBL10={3000:'인구_3km',4500:'인구_4.5km',10000:'인구_10km'}
_rec10=[]

# -----------------------------------------------------------------------------
# 1. 9개 주요 산업단지 영역에 대한 반경별(3km, 4.5km, 10km 전체 누적) 인구 산출
# -----------------------------------------------------------------------------
for _lbl,_nm,_ty in _TG10:
    _g,_ispoly=_geo11(_nm,_ty)
    _row={'부지':_lbl+(' (폴리곤)' if _ispoly else ' (점·YUCH결측)')}

    for _R in (3000,4500,10000):
        # A. 점 좌표인 경우: 기존 buffer_pop 함수로 빠른 산출
        _rr=buffer_pop(_g,_R,extra_map=_acc) if not _ispoly else None

        # B. 폴리곤 경계인 경우: 산단 경계선 전체로부터 밖으로 _R(m)만큼 팽창(Buffer)시켜 계산
        if _ispoly:
            _buf=_g.buffer(_R)      # 폴리곤 외곽 경계로부터 R미터 전체 버퍼 생성
            _b=_buf.bounds          # 속도 향상을 위한 바운딩 박스(BBox)
            _sub=pyogrio.read_dataframe(SHP,bbox=_b)

            # (집계구 ∩ 버퍼 영역 면적) / (집계구 전체 면적) 비율 계산 (0~1 클리핑)
            _w=(_sub.geometry.intersection(_buf).area/_sub.geometry.area).clip(0,1)

            # 총인구 및 고령인구 가중 합산 (누적)
            _tot=float((_sub['TOT_OA_CD'].map(POP).fillna(0.0)*_w).sum())
            _eld=float((_sub['TOT_OA_CD'].map(_acc).fillna(0.0)*_w).sum())
        else:
            _tot,_eld=_rr['tot'],_rr['e']

        # 해당 반경(3km, 4.5km, 10km 전체 누적)의 총인구 저장
        _row[_LBL10[_R]]=int(_tot)

        # 가장 넓은 반경인 10km 전체 내부 기준 고령인구 및 고령인구 비율 계산
        if _R==10000:
            _row['고령_10km']=int(_eld); _row['고령비%']=round(_eld/max(_tot,1)*100,1)

    _rec10.append(_row)

# -----------------------------------------------------------------------------
# 2. 인천 개별 부지 2곳(폐공장)에 대한 반경별 인구 산출 (개별 부지이므로 점 좌표 유지)
# -----------------------------------------------------------------------------
for _lbl,_lo,_la in [('현대제철 인천 (폐·점)',126.64432,37.48593),('동국제강 인천 (폐·점)',126.64489,37.48324)]:
    _g=gpd.GeoSeries([_Pt10(_lo,_la)],crs=4326).to_crs(5179).iloc[0]
    _row={'부지':_lbl}
    for _R in (3000,4500,10000):
        _rr=buffer_pop(_g,_R,extra_map=_acc); _row[_LBL10[_R]]=int(_rr['tot'])
        if _R==10000:
            _row['고령_10km']=int(_rr['e']); _row['고령비%']=round(_rr['e']/max(_rr['tot'],1)*100,1)
    _rec10.append(_row)

# -----------------------------------------------------------------------------
# 3. 데이터프레임 정리, 화면 출력 및 결과 저장
# -----------------------------------------------------------------------------
FOOT10=pd.DataFrame(_rec10).sort_values('인구_10km',ascending=False)

print('=== Cambridge hyperscaler footprint 반경별 상주인구 (폴리곤 경계 기준·명) ===')
print(FOOT10.to_string(index=False))

# CSV 저장
FOOT10.to_csv('DERIVED_0721_산단_반경별_인구_10km.csv',index=False,encoding='utf-8-sig')

print('\n저장: DERIVED_0721_산단_반경별_인구_10km.csv (폴리곤 기준으로 갱신 07-24)')
print('⚠ 노출 인구지 발병자 아님. 가동 산단은 열원교체(§11-b(2))로 순ΔT 불확실 → 온열 숫자 안 붙임.')
print('★ 점→폴리곤 전환 효과: 3km 기준 미포 51,593→467,637(9.1배)·온산 1,300→32,241(25배)·광양 73→72,511.')
print('  기존 점 기준은 "보수적 하한"이 아니라 형상 무시로 인한 심한 과소였다(사용자 지적 반영).')

=== Cambridge hyperscaler footprint 반경별 상주인구 (폴리곤 경계 기준·명) ===
                   부지  인구_3km  인구_4.5km  인구_10km  고령_10km  고령비%
        동국제강 인천 (폐·점)  206965    509938  2345227   403200  17.2
        현대제철 인천 (폐·점)  187102    485960  2341418   403412  17.2
           울산미포 (폴리곤)  467637    623152   982108   152655  15.5
             온산 (폴리곤)   32241     63572   484535    79845  16.5
          구미·전자 (폴리곤)  166640    248108   458556    50869  11.1
     포항·제철 (점·YUCH결측)    9935     92833   418357    83156  19.9
     여수·석화 (점·YUCH결측)     409      2776   186495    32741  17.6
    동해북평(국가·대조) (폴리곤)   39062     75361   124462    28917  23.2
동해 북평2(GS2.4GW) (폴리곤)   25978     69791   123274    28459  23.1
          광양·제철 (폴리곤)   72511     90220   120229    18943  15.8
          당진1철강 (폴리곤)    4334      9343    67149    10325  15.4

저장: DERIVED_0721_산단_반경별_인구_10km.csv (폴리곤 기준으로 갱신 07-24)
⚠ 노출 인구지 발병자 아님. 가동 산단은 열원교체(§11-b(2))로 순ΔT 불확실 → 온열 숫자 안 붙임.
★ 점→폴리곤 전환 효과: 3km 기준 미포 51,593→467,637(9.1배)·온산 1

### §11-b(2). 링별 differential — "열원교체" 단서는 **거리에 따라 약해진다** `[🔖 2026-07-21 사용자 지적]`

§4의 열원교체 단서(가동 산단→AIDC는 순ΔT 불확실)는 **부지 근처에서만 강하다.** 이유:
- 산단의 열은 근접일수록 강하다. **지표온도(LST)**로 부지 +17℃ → 5~10km에서 **+2~3℃**(§14 실측: 포항 +3.1·심팩 +2.1·동해 +2.1). 이를 **기온**으로 환산하면(×β 0.26) 5~10km에서 ~**+0.5~0.8℃**로 약하다.
- **핵심을 한 문장으로** `[🔖 2026-07-24 명확화]`: **가동 산단을 AIDC로 바꾸면 "기존 열이 사라지고 AIDC 열이 들어온다"(교체)인데, 그 순변화는 거리에 따라 다르다.**
  - **근접(0~1km)**: 기존 산단 열이 강하다(기온 기여 큼) → AIDC로 바뀌면 그 열이 빠지고 AIDC 열이 채운다 → **순변화 불확실**(줄 수도 있음).
  - **외곽(5~10km)**: 기존 산단의 **기온** 기여가 이미 작다(~+0.5℃ — §14 LST +2~3℃를 β로 환산). 뺄 게 거의 없으니 AIDC 열이 **거의 순수하게 더해진다**.
  - **문 닫은 폐공장(인천)**: 어느 거리든 뺄 열이 없음 → **전 구간 순수 더하기**.
  - `[정정]` 초판은 "외곽 링은 산단 영향권 밖"이라 했으나 §14가 반박(LST가 5~10km도 +2~3℃) — "밖"이 아니라 "**기온 기여가 작아 뺄 게 적다**"가 정확.
- 케임브리지는 hyperscaler 지표온도가 **부지 +2℃(Table 1 평균 2.03~2.12) → 4.5km +1℃ → 10km ~0**로 감쇠한다고 실측했다.

→ 아래 셀은 링별 인구에 케임브리지 감쇠 ΔT(지표)를 붙인다. **핵심**: 외곽 링일수록 열원교체 단서가 약해지고 "순수 더하기"에 가까워진다. 다만 이건 **지표온도 시나리오**이지 체감 기온·발병자 확정이 아니다(공기 환산은 β 0.18~0.26 + Sailor 근거리 anchor, §4).

In [ ]:
# ── §11-b(2). 링별 differential — 폴리곤 경계 기준 [🔖 2026-07-25 전면 수정] ──
# [버그] 이 셀은 `_sites10`을 참조했는데 07-24 §11-b 폴리곤 전환에서 그 변수가 사라져
#   재실행하면 NameError로 죽는다(저장된 출력은 전환 이전 것). → `_geo11` 형상으로 교체.
# [사용자 지적 ②] '산단열 ~+0.5℃ 잔존·교체 소규모'를 가동 산단과 폐쇄 공장에 같은 문구로 붙이면 안 된다.
#   두 경우는 순변화의 부호 구조가 다르다 — 아래에서 유형별로 분리한다.
# [사용자 지적 ③] 링도 점이 아니라 폴리곤 경계 기준으로.

def _bp2(geom, R):
    """폴리곤·점 공용 누적 버퍼 인구(면적가중). R=0이면 형상 내부."""
    buf = geom if R == 0 else geom.buffer(R)
    b = buf.bounds
    oa = pyogrio.read_dataframe(SHP, bbox=(b[0]-100, b[1]-100, b[2]+100, b[3]+100))
    w = (oa.geometry.intersection(buf).area/oa.geometry.area).clip(0, 1)
    return float((oa['TOT_OA_CD'].map(POP).fillna(0.0)*w).sum())

# 유형별 대비: 가동 중(열원 교체) vs 폐쇄 확정(순수 더하기)
_T2 = [('포항·제철 (가동)', _geo11('포항', '1')[0], '경상북도 포항시', '가동'),
       ('현대제철 인천 (폐쇄확정)',
        gpd.GeoSeries([_Pt10(126.64432, 37.48593)], crs=4326).to_crs(5179).iloc[0], '인천광역시 동구', '폐쇄')]
_RINGS2 = [(0, 1, 0.5), (1, 3, 2.0), (3, 4.5, 3.75), (4.5, 10, 7.25)]

print('=== 링별 노출 인구 × 거리 감쇠 (폴리곤/부지 경계 기준) ===')
print(f"※ ΔT = §9-e 엔진 dT_at() — 케임브리지 지표온도 감쇠(형태 '{DECAY_SHAPE}') × β {BETA_L2A}")
print(f'※ Sailor 근접 상한은 ≤{SAILOR_NEAR_KM}km 구간에만 별도 표기 — 같은 곡선에 얹지 않는다(측정 방식이 다름)')
for _lbl, _g2, _key2, _kind in _T2:
    print()
    print(f'{_lbl}  [{_kind}]')
    print(f"  {'링(km)':10}{'인구(명)':>11}{'ΔT(기온)':>10}{'Sailor근접':>11}   순변화 구조")
    _prev = 0.0
    for _a, _b, _mid in _RINGS2:
        _cum = _bp2(_g2, int(_b*1000)); _ring = _cum-_prev; _prev = _cum
        if _kind == '폐쇄':
            # 공정열이 이미 꺼진 상태 → AIDC 폐열이 전 거리에서 순수하게 더해진다.
            _note = 'AIDC 폐열이 그대로 순증 (교체할 기존 열원 없음)'
        else:
            # 가동 중 → 순변화 = AIDC 폐열 − 없어지는 공정열. 근접일수록 빼는 쪽이 크다.
            _note = ('순변화 = AIDC − 공정열, 근접이라 상쇄 큼(부호 불확실)' if _mid <= 1.5
                     else '공정열 기여가 이 거리에선 작아(§14 기온 ~+0.5℃) 순변화는 AIDC 쪽')
        _sn=dT_near_sailor(_mid)
        _sc=f'+{_sn:.1f}℃' if _sn else '—'
        print(f"  {f'{_a}-{_b}':10}{int(_ring):>11,}{dT_at(_mid):>9.2f}℃{_sc:>11}   {_note}")

print()
print('→ **두 유형을 같은 문장으로 묶지 않는다**:')
print('   · 폐쇄 확정(현대제철 인천) = 전 링에서 순수 더하기 → §11-b(3)에서 환자 수를 붙일 수 있다.')
print('   · 가동 중(포항제철) = 근접일수록 공정열 상쇄가 커서 순변화 부호 자체가 불확실 → 환자 수 안 붙인다.')
print('     다만 4.5-10km에서는 산단 공정열의 기온 기여가 작아(§14 실측 LST +2~3℃ → 기온 ~+0.5℃)')
print('     순변화가 AIDC 쪽으로 기운다. "완전히 산단 열영향 밖"은 §14 데이터가 반박한다.')
print('⚠ 인구는 확정값, ΔT는 시나리오. 발병자 환산은 폐공장(§11-b(3))에만.')

# ── [🔖 2026-07-27 사용자 질문] 0.49℃(케임브리지)와 +2.2℃(Sailor)를 어떻게 화해시키나 ──
print()
print('[화해] 0.49℃와 +2.2℃는 **모순이 아니라 서로 다른 기하(geometry)를 잰 값**이다')
print(f"  {'':12}{'케임브리지 → 우리 dT_at':>26}{'Sailor':>18}")
for _lb,_a,_b in [('측정 대상','지표온도(위성 LST)','공기온도(차량 실측)'),
                  ('공간 평균','**링 전체 평균**(등방)','**풍하 축** 100~500m'),
                  ('통계량','부지 평균 2.07℃','최대 2.2℃(평균 0.7~0.9)'),
                  ('시설 규모','하이퍼스케일러','36~169MW 4개소'),
                  ('냉각 방식','미상(혼합)','**전부 공랭**')]:
    print(f'  {_lb:12}{_a:>26}{_b:>18}')
print('  → 같은 축에 놓고 "누가 크냐"를 물을 수 없다. 링평균 0.49℃와 풍하최대 2.2℃는 **동시에 참일 수 있다**.')
print('  → 그래서 표에서 두 값을 같은 열에 합치지 않고 별도 열로 뒀다.')

print()
print('[검정] 그러면 등방(isotropic) 가정은 과대인가 과소인가 — 재본다')
# 열이 풍하 θ도 부채꼴에만 실린다면: 그 안의 ΔT는 등방평균의 (360/θ)배, 대신 인구는 θ/360만 노출.
# dose가 볼록(m^ΔT)이라 **같은 열을 좁게 몰면 총 추가분이 커진다**(옌센 부등식).
_k11='인천광역시 동구'; _dTiso=dT_at(0.5)
print(f'  기준: 0-1km 링 등방 ΔT={_dTiso:.2f}℃ · m={BETA_SITE.get(_k11,HEAT_MULT_PER_C):.2f}')
print(f"  {'풍하 부채꼴':>12}{'노출 비율':>10}{'부채꼴 내 ΔT':>13}{'1인당 추가분':>13}{'등방 대비':>10}")
_base11=f_site(_k11,_dTiso)
for _th in (360,180,90,45):
    _p=_th/360.0; _dTs=_dTiso/_p; _ex=_p*f_site(_k11,_dTs)
    # [🔖 2026-07-27 정정] 이전 판은 여기에 'Sailor 2.2℃와 부합' 태그를 달았다. **틀린 문장이다** —
    #   θ를 Sailor 값에 맞도록 **푼 것**이지 검정한 게 아니다(아래 [독립확인] 참조).
    _tag=' ← 등방(현행)' if _th==360 else (' ← θ를 Sailor에 맞춰 푼 값(검정 아님)' if abs(_dTs-SAILOR_AIR_MAX)<0.6 else '')
    print(f'  {f"{_th}°":>12}{_p:>10.2f}{_dTs:>12.2f}℃{_ex:>13.3f}{_ex/_base11:>9.2f}×{_tag}')
print('  → **등방 가정은 과대가 아니라 과소다.** 열을 좁은 부채꼴로 몰수록 총 추가분이 커진다')
print('    (dose가 지수라 볼록 — 같은 평균이라도 몰린 쪽이 피해가 크다. 옌센 부등식).')
print(f'  → Sailor 2.2℃에 맞추려면 부채꼴 {360*_dTiso/SAILOR_AIR_MAX:.0f}°가 필요하다 — 이건 **해**이지 확인이 아니다.')
print('  ⚠ 단서 셋: (i) 계절 내내 풍향이 도니 한 사람이 풍하에 드는 비율이 곧 θ/360이라는 가정,')
print('    (ii) 하루 안에서 풍향이 지속된다는 가정(해륙풍이면 대체로 성립), (iii) Sailor는 공랭 4개소 실측.')
print('  → 그래서 이 배수를 **채택하지 않고 기록만 한다.** 다만 "등방이라 넉넉히 잡았다"는 말은 쓸 수 없다.')
print('    우리 링 추정치는 **바람을 무시한 하한** 쪽에 가깝다.')

# ── [독립확인 · 🔖 2026-07-27 자기수정 11] "Sailor와 부합"은 틀렸다 ──
# [무엇을 틀렸나] 위 표에서 θ를 Sailor 2.2℃에 맞도록 **풀어놓고** 그 해(80°)를 "부합한다"고 읽었다.
#   어떤 Sailor 값을 넣어도 θ는 나온다 → **실패할 수 없는 계산은 아무것도 확인해주지 않는다.**
print()
print('[독립확인] θ=80°가 물리적으로 그럴듯한가 — Briggs(1973) 확산폭으로 따로 계산한다')
print('  Briggs σy(x) = a·x·(1+b·x)^(-1/2) — 가우시안 플룸의 수평 퍼짐(표준편차, m).')
print('  a·b는 대기안정도(Pasquill-Gifford A~F)와 지형(전원/도시)이 정한다. 1960~70년대 추적자 실험 적합식.')
_SY11={'전원 D(중립)':lambda x:0.08*x*(1+1e-4*x)**-0.5,'전원 C':lambda x:0.11*x*(1+1e-4*x)**-0.5,
       '도시 C-D':lambda x:0.16*x*(1+4e-4*x)**-0.5,'전원 A-B':lambda x:0.22*x*(1+1e-4*x)**-0.5,
       '도시 A-B':lambda x:0.32*x*(1+4e-4*x)**-0.5}
print(f"  플룸 각폭(°) at x=500m — 폭을 ±kσy로 셀 때")
print(f"  {'안정도':16}{'±1σy':>9}{'±2σy':>9}{'±3σy':>9}")
_W11={}
for _k11,_f11 in _SY11.items():
    _s=_f11(500); _row=f'  {_k11:16}'
    for _m11 in (1,2,3):
        _w=2*np.degrees(np.arctan(_m11*_s/500)); _W11[(_k11,_m11)]=_w; _row+=f'{_w:>9.1f}'
    print(_row)
print(f'  → 범위 {min(_W11.values()):.0f}~{max(_W11.values()):.0f}°. **80°는 ±3σy·도시A-B({_W11[("도시 A-B",3)]:.0f}°) 안에 든다.**')
print('  → 즉 "80°는 너무 넓다"고 단정할 수도 없다. 그 판정 역시 **k를 내가 고른 결과**다.')

print()
print('[더 큰 오류] top-hat과 가우시안을 섞어 비교했다')
print('  · 내 부채꼴 모형 = 부채꼴 안 **균일 ΔT**, 밖 0 (top-hat)')
print('  · Sailor 2.2℃ = 플룸 **중심축 최대값** (가우시안 꼭대기)')
print('  같은 총열량이면 꼭대기 P는 폭 2kσy top-hat 값 V보다 크다: 2kσy·V = P·σy·√(2π) → P = 0.798·k·V')
print(f"  {'부채꼴':>8}{'top-hat ΔT':>12}{'→ 중심축 P':>13}{'Sailor 대비':>13}")
for _th11 in (25,40,60,80,100):
    _V=_dTiso*360/_th11; _P=0.798*2*_V
    print(f'  {_th11:>7}°{_V:>11.2f}℃{_P:>12.2f}℃{_P/SAILOR_AIR_MAX:>12.1f}배')
_worst=0.798*2*_dTiso*360/max(_W11.values())
print(f'  → 가장 넓은 각폭({max(_W11.values()):.0f}°)에서도 중심축 {_worst:.2f}℃ > Sailor {SAILOR_AIR_MAX}℃.')

print()
print('[판정] 말할 수 있는 것은 한 문장이다:')
print('  **"두 논문 수치를 하나의 기하로 화해시키려는 시도는 아직 성공하지 못했고, 우리 쪽이 큰 방향으로 어긋난다."**')
print('  · 방향(우리가 더 큼)은 k·안정도·top-hat 가정을 다 바꿔도 견고하다.')
print('  · 그러나 **몇 배인지는 말할 수 없다** — 전부 내가 고른 관례에 매달린다.')
print('  · 애초에 시설 규모가 다르다(케임브리지 하이퍼스케일러 vs Sailor 36~169MW, 논문에 MW 층화 없음)')
print('    → 이 비교가 성립하는지 자체가 불명이다.')
print('  ⚠ 왜 어긋나나 — 우리 자료로 못 가리는 후보 5:')
for _i11,_c11 in enumerate(['시설 규모 차이(환산 불가)','통계량 불일치(계절 링평균 vs 특정일 풍하최대)',
                            'β 과대 가능(§11-b(3) 전이 가정)','플룸 상승 — 부력으로 떠서 지표에 덜 닿음(0727 논문이 9장으로 미룬 부분)',
                            '균일 풍배도 가정'],1):
    print(f'    ({_i11}) {_c11}')
print('  → 그래서 **부채꼴 배수를 채택하지 않는다**(등방 유지). 다만 "등방이라 넉넉히 잡았다"도 여전히 못 쓴다.')


=== 링별 노출 인구 × 거리 감쇠 (폴리곤/부지 경계 기준) ===
※ ΔT = §9-e 엔진 dT_at() — 케임브리지 지표온도 감쇠(형태 'exp') × β 0.258
※ Sailor 근접 상한은 ≤0.5km 구간에만 별도 표기 — 같은 곡선에 얹지 않는다(측정 방식이 다름)

포항·제철 (가동)  [가동]
  링(km)           인구(명)    ΔT(기온)   Sailor근접   순변화 구조
  0-1               211     0.49℃      +2.2℃   순변화 = AIDC − 공정열, 근접이라 상쇄 큼(부호 불확실)
  1-3             9,724     0.38℃          —   공정열 기여가 이 거리에선 작아(§14 기온 ~+0.5℃) 순변화는 AIDC 쪽
  3-4.5          82,897     0.29℃          —   공정열 기여가 이 거리에선 작아(§14 기온 ~+0.5℃) 순변화는 AIDC 쪽
  4.5-10        325,523     0.16℃          —   공정열 기여가 이 거리에선 작아(§14 기온 ~+0.5℃) 순변화는 AIDC 쪽

현대제철 인천 (폐쇄확정)  [폐쇄]
  링(km)           인구(명)    ΔT(기온)   Sailor근접   순변화 구조
  0-1            17,705     0.49℃      +2.2℃   AIDC 폐열이 그대로 순증 (교체할 기존 열원 없음)
  1-3           169,396     0.38℃          —   AIDC 폐열이 그대로 순증 (교체할 기존 열원 없음)
  3-4.5         298,858     0.29℃          —   AIDC 폐열이 그대로 순증 (교체할 기존 열원 없음)
  4.5-10      1,855,458     0.16℃          —   AIDC 폐열이 그대로 순증 (교체할 기존 열원 없음)

→ **두 유형을 같은 문장으로 묶지

### §11-b(3). 인천 폐공장 AIDC 환자 시나리오 — 지수 감쇠 × dose (범위) `[🔖 2026-07-22]`

"그냥 진행"(사용자). **열원교체가 깨끗한 폐공장(인천)에만** 붙인다(가동 산단 X). 케임브리지 지수 감쇠 ΔT_LST(r)=2·exp(−r·ln2/4.5)에 링별 인구와 dose(1.6^ΔT−1)를 곱한다.

**+2℃는 케임브리지 논문에서 지표온도(LST)다** `[🔖 2026-07-24 사용자 지적 — LST 단일화]`. 따라서 기온으로 ×β(0.258) 환산해 dose에 넣는다(공기온도 직접 대입 상한은 제거 — 그건 Sailor의 다른 시설·다른 측정이라 여기 케임브리지 시나리오에 섞지 않음). **FIRM한 건 노출 인구(10km 236만·고령 41만), 환자수는 시나리오 추정이다.**

In [ ]:
# ── §11-b(3). 인천 폐공장 AIDC 환자 시나리오 — 지수 감쇠 × dose [🔖 2026-07-22 · 07-24 LST 단일화] ──
import numpy as _np3

# 반경 r(km)에 따른 지표온도(LST) 지수 감쇠 (반감기 4.5km): 0km +2.0℃ · 4.5km +1.0℃ · 10km ~+0.43℃
# ★ +2℃는 케임브리지 논문에서 '지표온도(LST)'다(사용자 지적) → 기온으로 ×β 환산 후 dose. (공기온도 직접대입 상한은 제거 — Sailor는 다른 시설·측정이라 이 시나리오에 안 섞음)
# [🔖 2026-07-25] 감쇠·환산·dose를 여기서 다시 쓰지 않는다 — §9-e 엔진(dT_at·excess_ring) 호출.
_KEY3='인천광역시 동구'
_M3=BETA_SITE.get(_KEY3,HEAT_MULT_PER_C)                   # 표시용(가정 스택 문구에 사용)
_BETA3=BETA_L2A

_rate3=pd.read_csv('DERIVED_0718_시군구연도_패널_온열전력.csv',encoding='utf-8-sig').groupby('key')['여름온열률10만'].mean()
_ric3=_rate3.get('인천광역시 동구',float('nan'))            # 인천 동구 온열률(도심 주거 입지·§14-d 주거화소비 0.42)
_gu3=gpd.GeoSeries([_Pt10(126.64432,37.48593),_Pt10(126.64489,37.48324)],crs=4326).to_crs(5179).union_all()
def _bp3(geom,R):
    buf=geom.buffer(R); b=buf.bounds
    oa=pyogrio.read_dataframe(SHP,bbox=(b[0]-100,b[1]-100,b[2]+100,b[3]+100))
    w=(oa.geometry.intersection(buf).area/oa.geometry.area).clip(0,1)
    return float((oa['TOT_OA_CD'].map(POP).fillna(0.0)*w).sum())

print(f'인천 동구 여름온열률 {_ric3:.1f}/10만 · 케임브리지 지수감쇠 ΔT_LST(r)=2·exp(-r·ln2/4.5) · β(지표→기온)={_BETA3} · 1℃당 배수 m={_M3:.3f}(§9-e)')
print(f"  {'링':10}{'인구':>10}{'ΔT_LST':>9}{'ΔT_기온(×β)':>12}{'추가온열/년':>11}")
_est3=_prev3=0.0; sum_ring3=0.0
for _a,_b,_m in [(0,1,0.5),(1,3,2.0),(3,4.5,3.75),(4.5,10,7.25)]:
    _cum=_bp3(_gu3,int(_b*1000)); _ring=_cum-_prev3; _prev3=_cum
    _lst=CAMB_DT0*_np3.exp(-_m*_np3.log(2)/CAMB_HALF); _tair=dT_at(_m)   # 엔진과 동일 감쇠
    _e=excess_ring(_KEY3,_ring,_ric3,_m)                    # 링 dose (§9-e 엔진)
    _est3+=_e; sum_ring3+=_ring
    print(f"  {f'{_a}-{_b}km':10}{int(_ring):>10,}{_lst:>7.2f}℃{_tair:>10.2f}℃{_e:>11.1f}")
# [🔖 2026-07-25] Sailor 근접 상한을 별도 계열로 — 0~0.5km 공기온도 +2.2℃(측정 구간 한정).
_p05=_bp3(_gu3,500)
_e05max=_p05*_ric3/1e5*f_site(_KEY3,SAILOR_AIR_MAX)
_e05avg=_p05*_ric3/1e5*f_site(_KEY3,SAILOR_AIR_MEAN)
print(f"\n[별도 계열] Sailor 근접 상한 — 0~{SAILOR_NEAR_KM}km 인구 {int(_p05):,}명 ×"
      f" 공기 +{SAILOR_AIR_MEAN}~{SAILOR_AIR_MAX}℃ → {_e05avg:.1f}~{_e05max:.1f}건/년")
print("  이건 위 케임브리지 사슬에 **더하는 값이 아니라 겹치는 구간의 다른 추정**이다(같은 0-1km를 다르게 잼).")
print("  Sailor 시설은 36~169MW 공랭이라 GW급 수랭 캠퍼스로의 외삽은 주의 — 상한의 성격.")
print(f"\n→ 인천 폐공장 AIDC 10km 내 연간 추가 온열질환자 시나리오: 약 {_est3:.0f}건/년")
print(f"  ※ 위 인구는 링별 값이고 10km 누적 = 링 합 {int(sum_ring3):,}명 (= 아래 '10km 236만'과 같은 수).")
print("  ※ '인천 폐공장' = 현대제철 + 동국제강 두 부지의 **합집합**(340m 거리라 버퍼가 겹쳐 중복 제거됨).")
print("  (케임브리지 +2℃=지표온도 → β 0.258로 기온 환산 후 dose. Sailor 공기온도 상한은 별도 시설이라 제외)")
print(f"⚠ 가정 스택: 감쇠 shape + β 환산 + dose({_M3:.2f}^ΔT) + 인천동구 온열률 + 폐공장=순수더하기. FIRM한 건 노출 인구(10km 236만·고령 41만).")
print("  가동 산단엔 적용 X(열원교체·§11-b(2)). 폐공장(인천)만 순수 더하기.")

# ── [🔖 2026-07-27 사용자 지적] "감쇠 shape는 케임브리지 논문에서 가져왔으니 가정이 아니지 않나?
#     인천동구 온열률도 실제 데이터인데?" → **맞다. '가정 스택'이라는 뭉뚱그린 표기가 부정확했다.** ──
print()
print('[정정] "가정 스택"을 **값의 출처**와 **전이 가정**으로 분리한다 (사용자 지적 2026-07-27)')
print(f"  {'요소':16}{'값의 출처':>22}{'남는 가정(전이)':>34}")
for _e,_src,_asm in [
    ('감쇠 형태','논문 3점 적합(2.07·4.5km 1.0·7km 30%)','3점 사이·밖의 보간·외삽이 지수형이라는 것'),
    ('반감기 4.2km','같은 3점에서 유도','다른 시설·기후에도 같은 반감기라는 것'),
    ('β(LST→기온)','우리 §15 관측소 회귀 0.258[CI]','관측소 공간관계가 **열 주입 반응**에도 성립'),
    ('dose 배수 m','우리 §9-e 시군구 포아송 1.63','ΔT 구간 밖에서도 같은 탄력성'),
    ('인천동구 온열률','NEDIS 실측 발생률','**링 전체가 동구와 같은 발생률**이라는 것 ← 가장 센 전이'),
    ('폐공장=순수더하기','대장·보도(가동 중단 확인)','공정열이 완전히 0이 됐다는 것')]:
    print(f'  {_e:16}{_src:>22}   {_asm}')
print('  → **자료가 아닌 것은 하나도 없다.** 위험한 건 값이 아니라 **전이**(다른 맥락에 옮겨 쓰는 것)다.')
print('  → 그래서 문구를 "가정 스택"이 아니라 **"전이 가정 6단"**으로 바꿔 읽는다.')
print('  → FIRM한 건 여전히 노출 인구(10km 236만·고령 41만) — 이건 집계구 실측이라 전이가 없다.')

인천 동구 여름온열률 5.5/10만 · 케임브리지 지수감쇠 ΔT_LST(r)=2·exp(-r·ln2/4.5) · β(지표→기온)=0.258 · 1℃당 배수 m=1.630(§9-e)
  링                 인구   ΔT_LST   ΔT_기온(×β)     추가온열/년
  0-1km         28,691   1.91℃      0.49℃        0.4
  1-3km        182,420   1.49℃      0.38℃        2.1
  3-4.5km      314,180   1.11℃      0.29℃        2.6
  4.5-10km   1,836,496   0.63℃      0.16℃        8.3

[별도 계열] Sailor 근접 상한 — 0~0.5km 인구 5,996명 × 공기 +0.8~2.2℃ → 0.2~0.6건/년
  이건 위 케임브리지 사슬에 **더하는 값이 아니라 겹치는 구간의 다른 추정**이다(같은 0-1km를 다르게 잼).
  Sailor 시설은 36~169MW 공랭이라 GW급 수랭 캠퍼스로의 외삽은 주의 — 상한의 성격.

→ 인천 폐공장 AIDC 10km 내 연간 추가 온열질환자 시나리오: 약 13건/년
  ※ 위 인구는 링별 값이고 10km 누적 = 링 합 2,361,789명 (= 아래 '10km 236만'과 같은 수).
  ※ '인천 폐공장' = 현대제철 + 동국제강 두 부지의 **합집합**(340m 거리라 버퍼가 겹쳐 중복 제거됨).
  (케임브리지 +2℃=지표온도 → β 0.258로 기온 환산 후 dose. Sailor 공기온도 상한은 별도 시설이라 제외)
⚠ 가정 스택: 감쇠 shape + β 환산 + dose(1.63^ΔT) + 인천동구 온열률 + 폐공장=순수더하기. FIRM한 건 노출 인구(10km 236만·고령 41만).
  가동 산단엔 적용 X(열원교체·§11-b(2)). 폐공장(인천)만 순수 더하기.

[정정] "가정 스택"을 **값의 출처**와 **전이 가정**으로 분리한

### §12-b. 형상 고령 노출 — 점 3km 대신 폴리곤 경계 버퍼 `[🔖 2026-07-22 사용자 지적]`

§11 `추가온열_연령가중`은 **점 3km** 버퍼로 65+를 셌다. 산단은 폴리곤이므로 §10-b `polybuf`(경계 *바깥* 3km)로 재계산하면 크게 달라진다 — 여러 갈래로 흩어진 대형 해안 산단에서 점 버퍼는 고령을 **최대 11배 과소** 집계한다(미포: convex 원형도 0.30·bbox 13×11km·hull의 34%만 채움 = 벨트가 아니라 넓게 분산된 형태 — 점3km는 한 갈래만 잡음).

In [ ]:
# ── §11-c. '추가'를 **기저**와 나란히 — 몇 명이 아니라 몇 % [🔖 2026-07-29 사용자 지적] ──
# [왜] "인천 연 13건 추가"만으로는 그 크기의 의미를 알 수 없다.
#   같은 반경에서 **지금 나고 있는 환자 수**와 견줘야 한다.
#     기저(링) = 링인구 × 여름온열률/10만
#     추가(링) = 기저 × (m^ΔT(r) − 1)
#     증가율   = Σ추가/Σ기저 = **인구가중 평균 (m^ΔT − 1)**  ← 발생률이 약분된다
#   그래서 부지 차이는 **인구가 얼마나 가까이 있느냐**와 dose 배수 m 에서만 온다.
# [링] 2026-07-29 부로 LST·토지피복과 같은 경계로 통일했다 — 0-1·1-2·2-3·3-5·5-10km.
#   이전 인구 링(0-3·3-4.5·4.5-10km)은 케임브리지 보고점 4.5km 에 맞춘 것이라 표를 나란히 못 읽었다.
#   재계산 결과 추정값은 부지별 0.96~1.07배(평균 1.003) — 링 굵기가 왜곡 요인은 아니었다.
import numpy as _np11c

_RP = pd.read_csv('DERIVED_0729_링별_인구_LST경계.csv', encoding='utf-8-sig')
_RNG11 = ['0-1km', '1-2km', '2-3km', '3-5km', '5-10km']
_MID11 = {'0-1km': 0.5, '1-2km': 1.5, '2-3km': 2.5, '3-5km': 4.0, '5-10km': 7.5}
_SGG11 = {'울산미포': '울산광역시 남구', '온산': '울산광역시 울주군', '온산 강양·우봉지구': '울산광역시 울주군',
          '구미·전자': '경상북도 구미시', '포항·제철': '경상북도 포항시', '심팩 포항': '경상북도 포항시',
          '포항 광명': '경상북도 포항시', '광양·제철': '전라남도 광양시', '여수·석화': '전라남도 여수시',
          '당진1철강': '충청남도 당진시', '동해북평': '강원특별자치도 동해시', '동해 북평2': '강원특별자치도 동해시',
          '현대제철 인천': '인천광역시 동구', '동국제강 인천': '인천광역시 동구', 'KG스틸 인천': '인천광역시 서구'}
_RATE11 = (pd.read_csv('DERIVED_0718_시군구연도_패널_온열전력.csv', encoding='utf-8-sig')
             .groupby('key')['여름온열률10만'].mean())

_rows11 = []
for _nm, _g in _RP.groupby('부지'):
    _k = _SGG11.get(_nm)
    _rate = _RATE11.get(_k, _np11c.nan)
    if _np11c.isnan(_rate):
        continue
    _pr = _g.set_index('링')['링인구']
    _tot = float(_pr.sum())
    _base = _tot * _rate / 1e5                       # 기저 (연간, 10km)
    # 하한 β0.258+반감기3.0 · 중심 β0.258+4.2 · 상한 β0.459+4.2 — §9-e 엔진과 같은 조합
    def _add11(beta, half):
        return sum(float(_pr.get(lb, 0)) * _rate / 1e5
                   * (BETA_SITE.get(_k, HEAT_MULT_PER_C) ** (CAMB_DT0 * _np11c.exp(
                       -_MID11[lb] * _np11c.log(2) / half) * beta) - 1) for lb in _RNG11)
    _lo11, _md11, _hi11 = _add11(0.258, 3.0), _add11(0.258, 4.2), _add11(0.459, 4.2)
    _rows11.append(dict(부지=_nm, 인구10km=int(_tot), 발생률=round(_rate, 1),
                        기저=round(_base, 1), 하한=round(_lo11, 2), 중심=round(_md11, 2), 상한=round(_hi11, 2),
                        증가율=round(_md11 / _base * 100, 2), 증가율상한=round(_hi11 / _base * 100, 2)))
_T11 = pd.DataFrame(_rows11).sort_values('증가율', ascending=False)
_T11.to_csv('DERIVED_0729_기저대비_추가환자.csv', index=False, encoding='utf-8-sig')

print('§11-c. 이미 나고 있는 환자(기저) 대비 추가분 — 10km · 5링')
print(f"  {'부지':16}{'10km 인구':>11}{'발생률':>7}{'기저/년':>9}{'추가(중심)':>11}{'증가율':>8}{'상한':>9}")
print('  ' + '─' * 72)
for _, _x in _T11.iterrows():
    print(f"  {_x.부지:16}{_x.인구10km:>11,}{_x.발생률:>7.1f}{_x.기저:>9.1f}{_x.중심:>11.2f}"
          f"{_x.증가율:>7.2f}%{_x.증가율상한:>8.2f}%")
print('  ' + '─' * 72)
print(f'  중심 {_T11.증가율.min():.1f}~{_T11.증가율.max():.1f}% · 상한 '
      f'{_T11.증가율상한.min():.1f}~{_T11.증가율상한.max():.1f}%')
print()
print('[사실] **절대 건수 1위와 증가율 1위가 다르다.**')
_top_n = _T11.sort_values('중심', ascending=False).iloc[0]
_top_r = _T11.iloc[0]
print(f'  건수 1위 {_top_n.부지} {_top_n.중심:.1f}건 (증가율 {_top_n.증가율:.1f}%)')
print(f'  증가율 1위 {_top_r.부지} {_top_r.증가율:.1f}% (건수 {_top_r.중심:.1f}건)')
print('  → 인구가 **어디** 있느냐가 갈랐다. 인천은 170만이 5~10km(ΔT 최소 구간)에 있고,')
print('    울산미포는 34만이 0~2km 근접권에 있다.')
print('  → **건수는 인구 총량이, 증가율은 인구가 얼마나 가까이 있느냐가 정한다.**')
print()
print('[해석] 증가율은 발생률이 약분되므로 "그 동네가 원래 더운가"와 무관하다 —')
print('  순수하게 **AIDC 열이 인구 분포에 얼마나 겹치는가**만 잰다. 부지 비교에 더 알맞다.')
print('⚠ 이 증가율도 §9-e3 의 상한 성격을 그대로 물려받는다(케임브리지 곡선 전이).')


In [ ]:
# ── §12-b. 형상 고령 노출 + 전 버퍼 연령가중 추가온열 [🔖 2026-07-22 · 07-24 연령가중 확장] ──
# 목적: ① 점(Point) 3km 버퍼의 과소추정 한계를 보완하기 위해 폴리곤(Polygon) 경계 버퍼로 대체
#       ② 연령 가중치(E_ELD: 고령층, E_YNG: 비고령층)를 재유도하여 모든 반경(R0내부, +1km, +2km, +3km)에 적용
#       ③ 실부지인 '동해 북평2'를 분석 대상에 포함하여 정밀 추정

from shapely.geometry import Point as _Ptg

# -----------------------------------------------------------------------------
# 1. 연령별 상대 위험 가중치 및 전국 참조 상수 설정
# -----------------------------------------------------------------------------
# 전국 평균 인구 구조(NAT65) 기반 기대 위험도 및 고령/비고령 상대 가중치 유도
_avgRRg=NAT65*RR+(1-NAT65); _E_ELDg=RR/_avgRRg; _E_YNGg=1/_avgRRg

# 구 방식 대조용 전국 참조 상수 (f_s = m_s² - 1)
_fg=HEAT_MULT_PER_C**2-1     # 실제 계산은 부지별 f_site(_key)

print(f'전국 65+ 비율 {NAT65*100:.1f}% | RR={RR:.1f} → E_ELD={_E_ELDg:.2f} E_YNG={_E_YNGg:.2f} (§11 md와 동일 유도)')

# 시군구 연도 패널 데이터에서 시군구별 평균 '여름온열률10만' 가져오기
_rateg=pd.read_csv('DERIVED_0718_시군구연도_패널_온열전력.csv',encoding='utf-8-sig').groupby('key')['여름온열률10만'].mean()

# 분석 대상 9개 산단 부지 정의: (라벨, PDAN 단지명, 단지 유형, 시군구 key)
_TGg=[('광양·제철','광양','1','전라남도 광양시'),('여수·석화','여수','1','전라남도 여수시'),
      ('울산미포','울산·미포','1','울산광역시 남구'),('온산','온산','1','울산광역시 울주군'),
      ('포항·제철','포항','1','경상북도 포항시'),('구미·전자','구미(2·3단지)','1','경상북도 구미시'),
      ('당진1철강','당진1철강','2','충청남도 당진시'),('동해북평(국가·대조)','북평','1','강원특별자치도 동해시'),
      ('동해 북평2(GS2.4GW)','북평2','2','강원특별자치도 동해시')]

# -----------------------------------------------------------------------------
# 2. Part ① 형상 고령 노출 비교: '점 3km' vs '폴리곤 경계 +3km' (65세 이상)
# -----------------------------------------------------------------------------
print()
print('① 형상 고령 노출: 점3km vs 폴리곤 경계+3km (65세 이상)')
print(f"  {'부지':20}{'점3km 65+':>11}{'폴리곤+3km 65+':>15}{'배수':>7}")

for _lbl,_nm,_ty,_key in _TGg:
    # PDAN(점) 및 YUCH(폴리곤) 데이터 가져오기
    _rg=PDAN[(PDAN.DAN_NAME==_nm)&(PDAN.DANJI_TYPE==_ty)].iloc[0]
    _ycg=YUCH[YUCH.DAN_ID==_rg.DAN_ID]

    # A. 점 기준 3km 내 고령인구 수 산출
    _eptg=buffer_pop(gpd.GeoSeries([_Ptg(_rg.lon,_rg.lat)],crs=4326).to_crs(5179).iloc[0],3000,extra_map=_acc)['e']

    # B. 폴리곤 경계 기준 +3km 내 고령인구 수 산출 및 증가 배수 비교
    if len(_ycg):
        # polybuf(): 폴리곤 경계선으로부터 지정 거리(m)까지 확장하여 인구/고령인구 산출
        _,_epolyg=polybuf(_shT(_tf.transform,_ycg.geometry.union_all()),3000,_acc)
        print(f"  {_lbl:20}{int(_eptg):>11,}{int(_epolyg):>15,}{_epolyg/max(_eptg,1):>6.1f}×")
    else:
        print(f"  {_lbl:20}{int(_eptg):>11,}{'YUCH無(점유지)':>15}")

# -----------------------------------------------------------------------------
# 3. Part ② 전 버퍼 연령가중 추가온열 피해 산출 (+2℃ 시나리오, 명/년)
# -----------------------------------------------------------------------------
print()
print('② 전 버퍼 연령가중 추가온열 — **링별 ΔT 감쇠**(§9-e 엔진), 명/년 · 폴리곤 경계 기준')
print(f'   칸=그 반경까지 누적 **하한~상한** · β {BETA_L2A}(채택) ~ {BETA_L2A_HI}(§15-e 희석보정)')
print(f"  {'부지':18}{'온열률':>6}{'R0':>13}{'+1km':>13}{'+2km':>13}{'+3km':>13}{'+4.5km':>13}{'+10km':>13}")

for _lbl,_nm,_ty,_key in _TGg:
    _rg=PDAN[(PDAN.DAN_NAME==_nm)&(PDAN.DANJI_TYPE==_ty)].iloc[0]; _ycg=YUCH[YUCH.DAN_ID==_rg.DAN_ID]
    _rt=_rateg.get(_key,float('nan'))

    # YUCH 데이터가 없거나 시군구 온열률 정보가 없는 경우 스킵
    if not len(_ycg) or _rt!=_rt:
        print(f"  {_lbl:20}{'YUCH無 또는 온열률無':>30}"); continue
    
    _u=_shT(_tf.transform,_ycg.geometry.union_all())
    _vals=[]

    # §9-e에서 산출된 해당 부지별 채택 위험계수 f_s 가져오기 (f_s = m_s² - 1)
    _fgs=f_site(_key)                                   
    # [🔖 2026-07-25 사용자 지적] 위 _fgs는 '3km까지 전부 +2℃'라는 평평한 가정의 구 방식 값(이제 대조용).
    #   실제로는 멀어질수록 식으므로 링마다 다른 ΔT를 준다 — dT_at(링중점km) → f_site(key, ΔT).
    #   케임브리지 footprint가 10km까지이므로 4.5·10km 링도 함께 낸다.
    _pT=_pE=0.0; _cells=[]; _hi=[]
    for _R,_mid in ((0,0.0),(1000,0.5),(2000,1.5),(3000,2.5),(4500,3.75),(10000,7.25)):
        _tot,_eld=polybuf(_u,_R,_acc)

        # 누적 인구를 링 증분으로 바꾼 뒤, 그 링의 ΔT로 연령가중 추가 환자 수(명/년)를 누적
        # [🔖 2026-07-25] β 하한(채택 0.258)과 상한(희석보정 0.459) 두 계열을 함께 쌓는다.
        _rT,_rE=_tot-_pT,_eld-_pE; _pT,_pE=_tot,_eld
        _vals.append((_vals[-1] if _vals else 0.0)
                     +excess_ring(_key,_rT,_rt,_mid,eld=_rE,E_ELD=_E_ELDg,E_YNG=_E_YNGg))
        _hi.append((_hi[-1] if _hi else 0.0)
                   +excess_ring(_key,_rT,_rt,_mid,eld=_rE,E_ELD=_E_ELDg,E_YNG=_E_YNGg,beta=BETA_L2A_HI))
        _cells.append(f"{_vals[-1]:.1f}~{_hi[-1]:.1f}")

    print(f"  {_lbl:18}{_rt:>6.1f}"+''.join(f"{_c:>13}" for _c in _cells))

# -----------------------------------------------------------------------------
# 4. 분석 결과 요약 및 해석 가이드
# -----------------------------------------------------------------------------
print()
print('→ 점 3km는 흩어진 대형 산단(미포 11×)·해안(광양은 점이 바다) 고령을 크게 과소집계 — 폴리곤 경계 버퍼가 옳다.')
print('  [🔖 2026-07-27] 구 규약 표(§10-③·§11-④)는 **노트북에 더 이상 존재하지 않는다** — 배치 12~18에서 제거됐다. 사고의 발전 과정은 셀1 변경이력에 서술로 남아 있다.'
      '\n     없는 절을 가리키는 포인터는 찾다가 못 찾게 만들어 오히려 신뢰를 깎으므로 지웠다(사용자 지적 2026-07-27).')
print(f'  ⚠ +2℃ 시나리오 추정 · 가동 산단은 열원교체(§11-b(2))로 순ΔT 불확실 — 상한 성격으로 읽을 것.')
print(f'  [🔖 07-25] 구 방식은 3km까지 전부 ΔT=+2℃(기온)로 두고 f 하나(전국 {_fg:.3f})를 곱했다. 이제 링마다')
print(f'  ΔT가 다르다(0.5km {dT_at(0.5):.2f}℃ → 7.25km {dT_at(7.25):.2f}℃) — 그래서 3km 누적이 크게 낮아진다.')
print(f'  ★ 이 표가 추가온열의 **정식 산출**이다(폴리곤 형상 + 연령가중 + 거리 감쇠 + β 범위).')
print(f'  하한/상한은 β 0.258 vs {BETA_L2A_HI}다 — §15-e가 회귀희석을 기각 못 했으므로 채택 β는 눌린 값일 수 있다.')
print(f'  참고: Sailor는 36~169MW 공랭 시설에서 공기 +0.7~0.9℃(최대 2.2)를 실측했다. GW급이 그보다')
print(f'  낮게 나오는 하한만 쓰면 과소 위험이 크다 — 범위로 읽을 것.')
print('    구 규약(평평 +2℃) 표는 제거됨 — 경위는 셀1 변경이력 배치 12~18')

전국 65+ 비율 18.8% | RR=1.8 → E_ELD=1.58 E_YNG=0.87 (§11 md와 동일 유도)

① 형상 고령 노출: 점3km vs 폴리곤 경계+3km (65세 이상)
  부지                     점3km 65+    폴리곤+3km 65+     배수
  광양·제철                        21          7,479 343.0×
  여수·석화                       118     YUCH無(점유지)
  울산미포                      6,891         75,577  11.0×
  온산                          236          5,104  21.6×
  포항·제철                     2,764     YUCH無(점유지)
  구미·전자                     5,166          9,797   1.9×
  당진1철강                       370            786   2.1×
  동해북평(국가·대조)               3,342          6,423   1.9×
  동해 북평2(GS2.4GW)           2,693          4,466   1.7×

② 전 버퍼 연령가중 추가온열 — **링별 ΔT 감쇠**(§9-e 엔진), 명/년 · 폴리곤 경계 기준
   칸=그 반경까지 누적 **하한~상한** · β 0.258(채택) ~ 0.459(§15-e 희석보정)
  부지                   온열률           R0         +1km         +2km         +3km       +4.5km        +10km
  광양·제철               20.7      0.0~0.1      0.6~1.1      1.4~2.7      2.1~3.9      2.4~4.5      2.8~5.2
  여수·석화             

In [ ]:
# ── §12-b2. YUCH 결측 부지(여수·포항)를 점 폴백으로 채운다 [🔖 2026-07-27 사용자 지적] ──
# [지적] "YUCH가 없다고 아예 추정을 안 하는 건 아니지 않나. PDAN 점을 기준으로라도 계산해줘야지."
#   맞다. 빈칸은 "위험이 없다"로 오독된다. 다만 **점 3km는 폴리곤+3km보다 작다**(§12-b ①에서 1.7~343배)
#   → 점 폴백 값은 **하한**으로 명시하고, 같은 표에 섞지 않고 별도 블록으로 둔다.
_MISS12=[('여수·석화','여수','1','전라남도 여수시'),('포항·제철','포항','1','경상북도 포항시')]
_RING12=[(0,0.0),(1000,0.5),(2000,1.5),(3000,2.5),(4500,3.75),(10000,7.25)]
print('§12-b2. YUCH 결측 부지 — PDAN 점 기준 폴백 (⚠ 값은 하한)')
print(f"  {'부지':14}{'온열률':>7}{'점3km 인구':>11}{'점3km 65+':>10}{'  +3km 추가온열':>15}{'+10km':>14}")
for _lbl,_nm,_ty,_key in _MISS12:
    _r12=PDAN[(PDAN.DAN_NAME==_nm)&(PDAN.DANJI_TYPE==_ty)]
    _rt12=_rateg.get(_key,float('nan'))
    if not len(_r12) or _rt12!=_rt12:
        print(f'  {_lbl:14} PDAN 또는 온열률 결측 — 계산 불가')
        continue
    _pt12=gpd.GeoSeries([_Ptg(_r12.iloc[0].lon,_r12.iloc[0].lat)],crs=4326).to_crs(5179).iloc[0]
    _pv=_pt=_pe=0.0; _v3=_v10=_h3=_h10=0.0
    for _R,_mid in _RING12:
        _bb=buffer_pop(_pt12,max(_R,1),extra_map=_acc)      # buffer_pop은 {'tot','e'}를 준다('p'가 아니다)
        _tot,_eld=float(_bb['tot']),float(_bb['e'])
        _rT,_rE=_tot-_pt,_eld-_pe; _pt,_pe=_tot,_eld
        _lo12=excess_ring(_key,_rT,_rt12,_mid,eld=_rE,E_ELD=_E_ELDg,E_YNG=_E_YNGg)
        _hi12=excess_ring(_key,_rT,_rt12,_mid,eld=_rE,E_ELD=_E_ELDg,E_YNG=_E_YNGg,beta=BETA_L2A_HI)
        _v10+=_lo12; _h10+=_hi12
        if _R<=3000: _v3+=_lo12; _h3+=_hi12
        if _R==3000: _p3,_e3=_tot,_eld
    print(f'  {_lbl:14}{_rt12:>7.1f}{int(_p3):>11,}{int(_e3):>10,}{f"{_v3:.1f}~{_h3:.1f}":>15}{f"{_v10:.1f}~{_h10:.1f}":>14}')
print('  ⚠ **하한인 이유**: 점 3km는 산단 형상을 무시한다. §12-b ①에서 폴리곤+3km는 점3km의 1.7~343배였다.')
print('     여수(대장 51.2km²)·포항(16.1km²)은 둘 다 대형이라 실제 노출은 이 값보다 **상당히 크다**.')
print('     → 빈칸 대신 이 하한을 쓰되, 인용할 때 "점 근사 하한"임을 반드시 병기할 것.')
print('  ⚠ 근본 해결은 두 산단의 경계 폴리곤 확보(산단공 지적도 또는 지자체 고시도면). §17 자료요구 목록에 있음.')


§12-b2. YUCH 결측 부지 — PDAN 점 기준 폴백 (⚠ 값은 하한)
  부지                온열률    점3km 인구  점3km 65+      +3km 추가온열         +10km
  여수·석화            12.4        409       118        0.0~0.0       1.7~3.1
  포항·제철             9.4      9,935     2,764        0.1~0.2       2.5~4.5
  ⚠ **하한인 이유**: 점 3km는 산단 형상을 무시한다. §12-b ①에서 폴리곤+3km는 점3km의 1.7~343배였다.
     여수(대장 51.2km²)·포항(16.1km²)은 둘 다 대형이라 실제 노출은 이 값보다 **상당히 크다**.
     → 빈칸 대신 이 하한을 쓰되, 인용할 때 "점 근사 하한"임을 반드시 병기할 것.
  ⚠ 근본 해결은 두 산단의 경계 폴리곤 확보(산단공 지적도 또는 지자체 고시도면). §17 자료요구 목록에 있음.


In [ ]:
# ── §12-e. 플룸 경로 링 집계 + 회귀 게이트 [🔖 2026-07-27] ──
# §9-j가 인터페이스(register_plume·dT_field)를 정의했다. 여기서 **인구 집계**를 붙인다.
#   여기 두는 이유: pop_in()·_acc·_rateg 가 이 위에서 정의되기 때문(§9-j에서 쓰면 전방 참조).
from shapely.geometry import Polygon as _Poly12e

def pop_ring_sector(geom, r0_m, r1_m, n_sec=12, emap=None):
    """링(r0~r1)을 n_sec개 부채꼴로 쪼개 면적가중 인구를 센다.
    반환 [(방위중심°, tot, eld), ...]. 방위는 형상 **대표점(centroid)** 기준 근사."""
    _c=geom.centroid; _R=r1_m*1.6
    _ring=geom.buffer(r1_m).difference(geom.buffer(r0_m)) if r0_m>0 else geom.buffer(r1_m)
    _out=[]
    for _i in range(n_sec):
        _a0,_a1=_i*360.0/n_sec,(_i+1)*360.0/n_sec
        _pts=[(_c.x,_c.y)]+[(_c.x+_R*np.sin(np.radians(_a)),_c.y+_R*np.cos(np.radians(_a)))
                            for _a in np.linspace(_a0,_a1,14)]
        _w=_ring.intersection(_Poly12e(_pts))
        if _w.is_empty or _w.area<=0:
            _out.append(((_a0+_a1)/2,0.0,0.0)); continue
        _r=pop_in(_w,0,emap)
        _out.append(((_a0+_a1)/2,_r.tot,_r.e))
    return _out

def excess_plume(key, geom, rate, rings=((0,1000,0.5),(1000,3000,2.0),(3000,4500,3.75),(4500,10000,7.25)),
                 n_sec=12, eld_map=None, E_ELD=None, E_YNG=None, shape=None, beta=None):
    """링 × 부채꼴로 추가 온열질환자를 합산.
    ★ dose f_s = m^ΔT − 1 을 **부채꼴마다 따로** 적용한 뒤 더한다(링 평균에 한 번 적용하는 것과 다름).
      f가 볼록이라 열이 몰린 쪽이 크게 나온다 — §11-b(2) 등방 검정 참조.
      플룸 미등록 시 모든 부채꼴 ΔT가 같아 excess_ring과 일치한다(아래 게이트)."""
    _tot=0.0
    for _r0,_r1,_mid in rings:
        for _b,_p,_e in pop_ring_sector(geom,_r0,_r1,n_sec,eld_map):
            if _p<=0: continue
            _f=f_site(key,dT_field(_mid,_b,shape=shape,beta=beta))
            _tot+= (_e*rate*E_ELD+(_p-_e)*rate*E_YNG)/1e5*_f if (eld_map is not None and E_ELD) else _p*rate/1e5*_f
    return _tot

print('§12-e. 플룸 경로 회귀 게이트 — 플룸 미등록이면 현행 등방과 같아야 한다')
_g12e=gpd.GeoSeries([_Ptg(126.64432,37.48593)],crs=4326).to_crs(5179).iloc[0]
_k12e='인천광역시 동구'; _rt12e=float(_rateg[_k12e])
_new12e=excess_plume(_k12e,_g12e,_rt12e,n_sec=12)
_old12e=0.0; _pv=0.0
for _r0,_r1,_mid in ((0,1000,0.5),(1000,3000,2.0),(3000,4500,3.75),(4500,10000,7.25)):
    _cum=pop_in(_g12e,_r1).tot; _rg=_cum-_pv; _pv=_cum
    _old12e+=excess_ring(_k12e,_rg,_rt12e,_mid)
_gap12e=abs(_new12e-_old12e)/max(_old12e,1e-9)*100
print(f'  현행 excess_ring  {_old12e:.3f} 명/년')
print(f'  신규 excess_plume {_new12e:.3f} 명/년 (12부채꼴·플룸 미등록)')
print(f'  차이 {_gap12e:.2f}% — 부채꼴 분할 기하 오차(슬리버)만 허용')
assert _gap12e<3.0, f'회귀 게이트 실패 {_gap12e:.1f}% — 부채꼴 경로가 등방을 재현 못 함'
print('  ✓ 통과 — 플룸이 없을 때 두 경로가 같은 답을 준다')
print()
print('[모형이 오면] §11-b(3)·§12-b의 excess_ring 호출을 excess_plume으로 바꾸면 끝난다.')
print('[남는 한계] (a) 방위는 대표점 기준 근사 — 대형 산단은 부지 안에서도 방위가 갈린다')
print('  (b) 플룸이 기온을 직접 주면 β를 빼야 한다(dT_field는 β를 다시 곱하지 않는다)')
print('  (c) 계절 풍향 분포는 별도 — 한 조건의 플룸 한 장으로 연간 노출을 대신할 수 없다')
print(f'  (d) n_sec={12}는 30°라 거칠다 — 모형 격자 해상도에 맞춰야 한다')


§12-e. 플룸 경로 회귀 게이트 — 플룸 미등록이면 현행 등방과 같아야 한다
  현행 excess_ring  13.050 명/년
  신규 excess_plume 13.050 명/년 (12부채꼴·플룸 미등록)
  차이 0.00% — 부채꼴 분할 기하 오차(슬리버)만 허용
  ✓ 통과 — 플룸이 없을 때 두 경로가 같은 답을 준다

[모형이 오면] §11-b(3)·§12-b의 excess_ring 호출을 excess_plume으로 바꾸면 끝난다.
[남는 한계] (a) 방위는 대표점 기준 근사 — 대형 산단은 부지 안에서도 방위가 갈린다
  (b) 플룸이 기온을 직접 주면 β를 빼야 한다(dT_field는 β를 다시 곱하지 않는다)
  (c) 계절 풍향 분포는 별도 — 한 조건의 플룸 한 장으로 연간 노출을 대신할 수 없다
  (d) n_sec=12는 30°라 거칠다 — 모형 격자 해상도에 맞춰야 한다


### §12-c. 면적 대조 — YUCH 실면적 vs 1GW 소요 표준 `[🔖 2026-07-22 신규 · 사용자 제공 규모 산정]`

1GW AIDC 소요 개산 [사용자 제공 추정 — 출처 확인 필요]: GB200 NVL72 랙(~120kW) 기준 GPU 40~50만장 · 랙 5~6천개(6천톤+) · **부지 61~98만 평**(100% 수랭 설계 표준) · 물 하루 3.8만톤(연 1,380만톤) · CapEx 52~83조원.

이 표준을 ILIS YUCH **실면적**과 대조한다. 환산 검증: 당진1철강 YUCH 1.7km² = 50.2만평 (§7 교차검증과 일치 ✓).

> **사용자 질문 — 북평2가 산단 밖으로 확장 안 하려면 100% 수랭 표준이 얼마나 줄어야 하나?** `[🔖 2026-07-24]` 현 표준 61~98만평/GW로 2.4GW는 146~235만평(축구장 673~1,082개)이 필요하다. 북평2 YUCH 9.7만평(축구장 45개)에 담으려면 **4.0만평/GW = 표준의 1/15~1/24(4~7%)**, 산단 전체 18만평 기준이라도 7.5만평/GW(1/8~1/13)로 줄여야 한다. 수랭은 냉각탑·용수·변전 인프라가 퍼져야 해 이 밀도는 **기술적으로 비현실적** → 확장·공랭 전환·단계적 용량 중 하나가 불가피(§12-d).

In [ ]:
# ── §12-c. YUCH 실면적 vs 1GW 소요 표준 + 열유속밀도 산술 [🔖 2026-07-22 신규] ──

# 1. 단위 변환 상수 정의: m² 면적을 '만평'으로 변환 (1평 = 3.305785 m²)
_PY12=1/(3.305785*10000)                     # m² → 만평

# 2. 분석 대상 산업단지/부지 목록 정의
# 형식: (표시이름, DAN_NAME, DANJI_TYPE, 발표/계획 전력용량(GW))
_S12=[('동해 북평2','북평2','2',2.4),('당진1철강','당진1철강','2',1.2),('울산미포','울산·미포','1',1.0),
      ('울산 하이테크밸리','울산 High Tech Valley','2',1.0),('온산','온산','1',1.0),('포항 광명','광명','2',0.3),
      # [🔖 2026-07-28] 강릉 옥계 편입 — 포스코홀딩스가 옥계일반산단 유치업종에 **정보통신업 추가 신청**
      #   (강원도민일보 2026-07-17). 사업시행자 본인의 행위라 '추정 후보'가 아니다.
      #   용량은 1GW(환경정의 2026-07-24 현장기록). 도지사 공약의 '최대 70조원'과의 관계는 불명 — 병기하지 않는다.
      ('강릉 옥계일반','강릉옥계','2',1.0),('옥계첨단소재융합','옥계첨단소재융합','2',float('nan')),
      ('구미(2·3단지)','구미(2·3단지)','1',float('nan')),('광양','광양','1',float('nan')),('북평(국가·대조)','북평','1',float('nan'))]

# 3. 결과 테이블 헤더 출력
print(f"{'부지':14}{'YUCH만평':>10}{'계획GW':>8}{'표준소요만평':>14}{'부지÷표준':>16}{'열유속W/m²':>12}")

# 4. 부지별 실면적 계산 및 열유속 밀도 산술 분석 루프
for _l,_nm,_ty,_gw in _S12:
    # 데이터베이스(PDAN)에서 대상 산업단지 정보 조회
    _r=PDAN[(PDAN.DAN_NAME==_nm)&(PDAN.DANJI_TYPE==_ty)]

    # 공간 데이터(YUCH)에서 해당 단지 ID(DAN_ID)의 도형 경계 추출
    _yc=YUCH[YUCH.DAN_ID==_r.iloc[0].DAN_ID] if len(_r) else []

    if not len(_yc): 
        print(f'{_l:14} YUCH 없음')
        continue

    # 지오메트리 병합(union_all) 후 실면적 계산 (m² 및 만평 변환)
    _am=_yc.geometry.union_all().area
    _a=_am*_PY12

    # 계획 전력 용량(GW)이 존재하는 경우 (NaN이 아닌 경우)
    if _gw==_gw:
        # 1GW당 표준 필요 면적 추정치(61~98만평/GW)를 적용한 총 필요 면적 범위
        _need=f'{_gw*61:.0f}~{_gw*98:.0f}'

        # 실제 부지 면적 대비 표준 필요 면적의 비율 (배율)
        _rt=f'{_a/(_gw*98):.2f}~{_a/(_gw*61):.2f}×'

        # 단위 면적당 열유속 밀도(W/m²) 산술
        # 계산식: (발표 전력GW * PUE 1.3 계수 * 10^9 W) / 부지 실제 면적(m²)
        _fx=f'{_gw*1.3e9/_am:,.0f}'
    else:
        _need=_rt=_fx='—'

    # [🔖 2026-07-27 정합성 감사 A3] 계획GW가 없는 부지(구미·광양·북평)에서 nan이 그대로 찍혔다.
    #   같은 행의 다른 열은 이미 '—'로 처리하고 있었으므로 이 열만 규약을 안 따르던 것.
    _gws=f'{_gw:.1f}' if _gw==_gw else '—'
    print(f'{_l:14}{_a:>10.1f}{_gws:>8}{_need:>14}{_rt:>16}{_fx:>12}')

# 5. 산출 결과 검증 및 정합성 체크
print('검증: 당진1철강 50.2만평 = §7 YUCH 1.7km² 일치 ✓ · 북평2 9.7만평 ≈ 기사 캠퍼스 7만평(+지원부지) 정합')

# 6. 정량적 분석 결과 및 열유속 비교 시나리오 출력
print(f'★ 북평2: 2.4GW 표준 소요 146~235만평의 **1/15~1/24** — 캠퍼스 7만평 기준 열유속 {2.4*1.3e9/(7e4*3.305785):,.0f} W/m²')
print('  (Sailor 실측 36MW 시설 3,100 W/m²의 ~4배, 한낮 태양 ~1,000의 13배 — 발표용량 기준 산술 시나리오)')

# 7. 데이터 해석 상의 한계점 및 논리 방향 정리
print('→ 두 해석: (i) 표준보다 10~24배 고밀(다층·수랭) = 부지 열유속밀도가 그만큼 극단 (ii) 발표 용량의 단계성.')
print('  어느 쪽이든 §9 열부하 밀도 논증 방향. ⚠ 61~98만평/GW 표준 자체는 출처 미확인 추정 — 확정 인용 금지.')
print()
print('[🔖 2026-07-25 냉각방식 반영 — 사용자 제공 자료 `AIDC 냉각방식/`]')
print('  ① 열은 **옥상 방출설비**에서 대기로 나간다(랙 흡수: 공랭 CRAC/액냉 direct-to-chip →')
print('     옥상 방출: 증발식 냉각탑 / 건식·폐회로 드라이쿨러). 즉 열원은 부지 전체가 아니라 건물+옥상이다.')
print('  ② **건식(드라이쿨러)은 전량 현열**로 공기에 버린다 → 주변 건구온도 상승이 크다.')
print('     **증발식(냉각탑)은 상당분이 잠열**(물 증발)로 빠진다 → 기온 상승은 작지만 **습도가 오른다**.')
print('     ⚠ 온열질환은 습구온도에 민감하다 — "기온이 덜 오른다"가 곧 "덜 위험하다"가 아니다.')
print('        ↑ **이 줄은 07-25에 재지 않고 쓴 것이다. §12-c2에서 우리 자료로 재봤더니 약해진다 — 반드시 함께 읽을 것.**')
print('     Sailor 실측 4개 시설이 **전부 공랭**이었으므로 그 +0.7~2.2℃는 건식 계열 앵커다.')
print('  ③ 따라서 부지/표준 비율이 큰 산단(온산 5~8배·미포 10~17배)은 **산단 안에서 소화**된다 —')
print('     경계 밖 확장 시나리오는 비율이 1 미만인 북평2(1/15~1/24)에만 해당한다.')
print('★ 형상 함의 (사용자 해석 2건):')
print('  · 북평2: 부지÷표준이 0.04~0.07×로 극소 → 계획대로면 산단 밖 확장이 개연적 — 폴리곤 경계+3km 노출 인구는 미래 footprint의 하한으로 읽어야 한다.')
print('  · 온산: YUCH 490만평이라 부지 안에서 충분히 소화 → 산단 전체 경계+3km는 과대(현실성 낮음) — 강양·우봉지구 점 근사(§18)가 현실적 노출이다.')

부지                YUCH만평    계획GW        표준소요만평           부지÷표준     열유속W/m²
동해 북평2               9.7     2.4       146~235      0.04~0.07×       9,766
당진1철강               50.2     1.2        73~118      0.43~0.69×         941
울산미포              1018.1     1.0         61~98    10.39~16.69×          39
울산 하이테크밸리           43.9     1.0         61~98      0.45~0.72×         896
온산                 490.7     1.0         61~98      5.01~8.04×          80
포항 광명               15.1     0.3         18~29      0.51~0.83×         780
강릉 옥계일반             11.3     1.0         61~98      0.12~0.19×       3,478
옥계첨단소재융합       YUCH 없음
구미(2·3단지)          149.6       —             —               —           —
광양                 529.7       —             —               —           —
북평(국가·대조)           17.6       —             —               —           —
검증: 당진1철강 50.2만평 = §7 YUCH 1.7km² 일치 ✓ · 북평2 9.7만평 ≈ 기사 캠퍼스 7만평(+지원부지) 정합
★ 북평2: 2.4GW 표준 소요 146~235만평의 **1/15~1/24** — 캠퍼스 7만평 기준 열유속 13,483 W/m²
  (Sa

### §12-c4. 강릉 옥계 — 부지가 모자라다는 것을 당사자들이 말하고 있다 `[🔖 2026-07-28 신규]`

> **근거**: 강원도민일보 2026-07-17 — 사업시행자 **포스코홀딩스**가 옥계일반산업단지 유치업종에 **정보통신업 추가를 신청**했다. 추정 후보가 아니라 **제도 절차가 진행 중**이다.

이 부지가 우리 §12-c 논증에 그대로 들어간다 — 그리고 **북평2보다 더 극단적**이다.

| 기사가 말하는 것 | |
|---|---|
| 옥계일반산단 | 총 48만2000㎡ · 산업시설용지 38만2000㎡ |
| **AIDC 가용부지** | **약 8만9000㎡** |
| 당선인 구상 | 약 33만㎡(10만평) · **10층 높이** |
| 강원경자청 | "부지 확보에 **한계**" |
| 전력 | 2028년 말 준공 목표 **변전소** 건설 추진 중 |

⚠ 정확한 부지 위치·시설 배치·건축 방식은 **미공개**다.

In [ ]:
# ── §12-c4. 강릉 옥계 — 부지가 모자라다는 것을 당사자들이 말하고 있다 [🔖 2026-07-28 신규] ──
# [근거] 강원도민일보 2026-07-17 「포스코홀딩스, 옥계일반산단 유치업종에 정보통신업 추가 신청」
#   · 사업시행자 포스코홀딩스가 이달 초 강원도에 계획 변경 신청 (현재는 제조업 중심만 입주 가능)
#   · 옥계일반산단 총면적 약 48만2000㎡ · 그중 산업시설용지 약 38만2000㎡
#   · 옥계지구 전체 38만3000㎡ · **실제 AIDC 가용부지는 약 8만9000㎡**
#   · 우상호 강원도지사 당선인 구상: **약 33만㎡(10만평) 부지에 10층 높이**
#   · 강원경제자유구역청: "옥계 첨단소재융합 산업지구만으로는 **부지 확보에 한계**"
#   · 옥계일반산단에 **2028년 말 준공 목표 변전소** 건설 추진 중
#   · ⚠ 정확한 부지 위치·시설 배치·건축 방식은 **미공개**
_PY4=1/(3.305785*10000)
_A_OK=YUCH[YUCH.DAN_ID==PDAN[PDAN.DAN_NAME=='강릉옥계'].iloc[0].DAN_ID].geometry.union_all().area
print('§12-c4. 강릉 옥계 — ILIS 실측 vs 기사')
print(f"  {'':22}{'ILIS YUCH 실측':>16}{'기사':>16}{'차이':>8}")
print(f"  {'산업시설용지':22}{_A_OK*_PY4:>13.2f}만평{38.2e4*_PY4:>13.2f}만평{abs(_A_OK-38.2e4)/38.2e4*100:>7.0f}%")
print('  → 2% 차이. **독립 출처가 서로를 확인한다**(북평2에서 대장 0.31 vs GIS 0.30이 일치한 것과 같은 패턴).')

# ── 부지÷표준 — 북평2보다 더 극단이다 ──
_STD_LO,_STD_HI=61.0,98.0                      # 만평/GW ⚠ 출처 미확인 추정(§12-c)
_AVAIL=8.9e4*_PY4                              # 기사 가용부지 8만9000㎡
_PLAN=33e4*_PY4                                # 우 당선인 구상 33만㎡
print()
print('부지÷표준 (1GW 기준 · 표준 소요 61~98만평/GW ⚠출처 미확인 추정)')
print(f"  {'기준':26}{'면적':>10}{'부지÷표준':>14}{'배수로':>12}")
for _lb,_a in [('산단 산업시설용지 전체',_A_OK*_PY4),('기사 "가용부지"',_AVAIL),('당선인 구상 부지',_PLAN)]:
    print(f'  {_lb:26}{_a:>8.2f}만평{_a/_STD_HI:>7.3f}~{_a/_STD_LO:.3f}×{_STD_HI/_a:>7.0f}~{_STD_LO/_a:.0f}배 부족')
print(f'  [대조] 북평2 9.66만평 / 2.4GW 표준 146~235만평 = 0.041~0.066× (15~24배 부족)')
print(f'  → **옥계 가용부지 기준이 북평2보다 더 극단적이다**({_STD_HI/_AVAIL:.0f}~{_STD_LO/_AVAIL:.0f}배 vs 15~24배).')

# ── 우리가 §12-c에서 제시한 두 갈래가 기사에서 둘 다 확인된다 ──
print()
print('[확인] §12-c가 제시한 두 해석이 기사에서 **당사자들 입으로** 확인된다')
for _k,_v in [('(i) 극단적 고밀 설계','우 당선인 구상 — **10층 높이** AIDC'),
              ('(ii) 부지 밖 확장','강원경자청 "부지 확보에 한계" · 기사 "추가 부지 확보가 **불가피**"'),
              ('','"옥계지구와 옥계일반산단에 **나눠 복수 시설**" 방안 거론')]:
    print(f'  {_k:22}{_v}')
print('  → §12-d 확장 시나리오는 이제 우리 예측이 아니라 **현재 진행 중인 논의**다.')

# ── 열유속 ──
print()
print('부지 열유속 밀도 (발표 1GW · PUE 1.3 가정)')
for _lb,_a_m2 in [('산단 산업시설용지 전체',_A_OK),('기사 "가용부지"',8.9e4),('당선인 구상 부지',33e4)]:
    print(f'  {_lb:26}{1.0*1.3e9/_a_m2:>10,.0f} W/m²')
print(f'  [대조] 북평2 캠퍼스 7만평 기준 13,483 · Sailor 실측 36MW 시설 3,100 · 한낮 태양 ~1,000')

print()
print('[전력 축] 옥계일반산단에 **2028년 말 준공 목표 변전소** 건설 추진 중 — §13 전력 제약 논증에 연결된다.')
print()
print('⚠ 한계 — 지금 넣을 수 없는 것')
for _i,_l in enumerate([
    '**정확한 부지 위치·배치·건축 방식이 미공개**다. 산단 폴리곤 전체를 열원으로 두면 §12-d에서 확인한 대로 과대가 된다(미포 15.6배).',
    '따라서 노출 인구는 **가용부지 기준 하한**과 **산단 전체 기준 상한**을 병기해야 한다.',
    '`옥계첨단소재융합`(DAN_ID 242380)은 **YUCH 폴리곤이 결측**이다 — 여수·포항과 같은 상황. 점 근사만 가능.',
    '용량 1GW는 환경정의 현장기록이고, 도지사 공약의 "최대 70조원"과 어떻게 대응하는지 **모른다**. 둘을 곱하거나 합치지 않는다.',
    '§14 ΔLST는 아직 없다 — 강릉 지역 Landsat 장면 커버리지 확인 후 별도 작업.',
    '유치업종 변경은 **신청 단계**다(6개월 내 처리·도산업단지계획심의위 심의 예정). 확정이 아니다.'],1):
    print(f'  ({_i}) {_l}')


§12-c4. 강릉 옥계 — ILIS 실측 vs 기사
                            ILIS YUCH 실측              기사      차이
  산업시설용지                        11.31만평        11.56만평      2%
  → 2% 차이. **독립 출처가 서로를 확인한다**(북평2에서 대장 0.31 vs GIS 0.30이 일치한 것과 같은 패턴).

부지÷표준 (1GW 기준 · 표준 소요 61~98만평/GW ⚠출처 미확인 추정)
  기준                                면적         부지÷표준         배수로
  산단 산업시설용지 전체                 11.31만평  0.115~0.185×      9~5배 부족
  기사 "가용부지"                     2.69만평  0.027~0.044×     36~23배 부족
  당선인 구상 부지                     9.98만평  0.102~0.164×     10~6배 부족
  [대조] 북평2 9.66만평 / 2.4GW 표준 146~235만평 = 0.041~0.066× (15~24배 부족)
  → **옥계 가용부지 기준이 북평2보다 더 극단적이다**(36~23배 vs 15~24배).

[확인] §12-c가 제시한 두 해석이 기사에서 **당사자들 입으로** 확인된다
  (i) 극단적 고밀 설계         우 당선인 구상 — **10층 높이** AIDC
  (ii) 부지 밖 확장          강원경자청 "부지 확보에 한계" · 기사 "추가 부지 확보가 **불가피**"
                        "옥계지구와 옥계일반산단에 **나눠 복수 시설**" 방안 거론
  → §12-d 확장 시나리오는 이제 우리 예측이 아니라 **현재 진행 중인 논의**다.

부지 열유속 밀도 (발표 1GW · PUE 1.3 가정)
  산단 산업시설용지 전체                   

### §12-c2. 냉각방식 논증에 우리 자료를 대입 — 습도 근거 `[🔖 2026-07-27]`

§12-c ②에 이렇게 썼다:

> ⚠ 온열질환은 습구온도에 민감하다 — "기온이 덜 오른다"가 곧 "덜 위험하다"가 아니다.

**07-25에 재지 않고 쓴 문장이다.** §9-f·§9-g가 생겼으니 대입한다 — 지지되면 근거로 승격하고, 아니면 철회한다.

| 확인 | 내용 |
|---|---|
| ① 습도의 크기 | §9-g Ⓒ 계수로 **기온 1℃ = 습도 몇 %p** 등가 환산 |
| ② 습구 가설 검정 | **습구온도가 같아지는** 두 상황(기온 +1℃ vs 습도 +N%p)을 찾아 실제 환자 수와 대조 |
| | ⚠ Stull은 **습구온도 환산식**이지 플룸 모형이 아니다 — 한 점의 (기온·습도) → 습구온도. 거리·바람 없음 |
| ③ 냉각탑 ΔRH | 증발수량으로 **상자모형 자릿수** 계산 (⚠ 플룸 모형 아님) |
| ④ 손익분기 | 기온 절감 이득 vs 습도 상승 손해 |
| ⑤ 정정 | §12-c ② 경고를 어떻게 고칠 것인가 |

> ⚠ 결론을 미리 적는다: **이 검정은 우리 주장을 약화하는 쪽으로 나왔다.** 그래서 더 빨리 적는다 — 상대가 먼저 재서 들이대면 §12-c 전체의 신뢰가 같이 무너진다.

In [ ]:
# ── §12-c2. 냉각방식 논증에 우리 자료를 대입 — 습도 근거 [🔖 2026-07-27 신규] ──
# [왜] §12-c ②에 이렇게 썼다: "온열질환은 습구온도에 민감하다 — 기온이 덜 오른다가 곧 덜 위험하다가 아니다."
#   07-25에 **재지 않고 쓴 문장**이다. §9-f·§9-g가 생겼으니 대입한다. 지지되면 근거로 쓰고, 아니면 철회한다.
_fC=_fitg['Ⓒ 기온 + 습도(**시각 일치**)']
_bT12=float(_fC.params['Tmax']); _bH12=float(_fC.params['RH_at_Tmax'])
_EQ12=_bT12/_bH12                                  # 기온 1℃와 같은 위험을 주는 습도 %p
print('① 우리 자료가 말하는 습도의 크기 (§9-g Ⓒ · 시각 일치)')
print(f'   기온 ×{np.exp(_bT12):.3f}/℃ · 습도 {_bH12:+.4f}/%p (10%p당 ×{np.exp(_bH12*10):.3f})')
print(f'   → **등가 환산: 습도 {_EQ12:.1f}%p 상승 = 기온 1℃ 상승.** 습도는 기온보다 {_EQ12:.0f}배 둔한 지렛대다.')

# ── ② 습구온도 가설과 대조 — §12-c ②의 주장이 우리 자료에서 성립하나 ──
# ⚠ **Stull은 플룸 모형이 아니다**(사용자 질문 2026-07-27 — 셀 설명이 부족했다).
#   · Stull(2011) = **습구온도 환산식**. 입력은 그 지점의 (기온, 상대습도) 둘뿐이고 출력은 습구온도 하나다.
#     거리·바람·배출원이 안 들어간다. 공간이 없는 **한 점의 열역학**이다(§9-f에서 이미 쓰던 그 식).
#   · **플룸 모형**(AERMOD·SACTI) = "냉각탑에서 나온 열·수증기가 1km 밖에 얼마나 닿나". 배출량·풍속·
#     대기안정도·방출고가 들어가고 거리별 분포가 나온다. 우리 노트북엔 없다 — 아래 ③의 상자모형이 그 자리를
#     거칠게 메우고 있을 뿐이고, 그 부족함이 곧 §17 "사업자에게 플룸 모형 제출 의무화" 요구의 근거다.
# [무엇을 검정하나] §12-c ②의 주장은 "온열질환은 **습구온도**에 민감하다"였다. 그게 맞다면
#   **습구온도를 똑같이 올리는 두 변화는 똑같이 위험해야 한다.** 그 '똑같이 올리는' 조합을 Stull로 찾아
#   실제 환자 수(§9-g Ⓒ)와 맞춰본다. 어긋나면 §12-c ②가 우리 자료에서 지지되지 않는 것이다.
_T0,_R0=float(_M9g.Tmax.mean()),float(_M9g.RH_at_Tmax.mean())
_dTw_dT=(_wetbulb(_T0+0.5,_R0)-_wetbulb(_T0-0.5,_R0))
_dTw_dR=(_wetbulb(_T0,_R0+0.5)-_wetbulb(_T0,_R0-0.5))
_EQphys=_dTw_dT/_dTw_dR
print()
print('② 습구온도 가설과 대조 — §12-c ②가 우리 자료에서 성립하나 (Stull 환산식, 플룸 모형 아님)')
print(f'   습구온도가 **같아지는** 두 상황을 찾는다 (평균 조건 T={_T0:.1f}℃·RH={_R0:.1f}%):')
print(f'     기준          {_T0:.1f}℃ · {_R0:.1f}%   → 습구 {_wetbulb(_T0,_R0):.2f}℃')
print(f'     기온 +1℃      {_T0+1:.1f}℃ · {_R0:.1f}%   → 습구 {_wetbulb(_T0+1,_R0):.2f}℃')
print(f'     습도 +{_EQphys:.1f}%p   {_T0:.1f}℃ · {_R0+_EQphys:.1f}%   → 습구 {_wetbulb(_T0,_R0+_EQphys):.2f}℃   ← 같다')
print(f'   → 습구온도 가설이 맞다면 이 둘은 **똑같이 위험해야 한다**: 기온 +1℃ = 습도 +{_EQphys:.1f}%p')
print(f'   → 그런데 실제 환자 수로는: 기온 +1℃ = ×{np.exp(_bT12):.3f} vs 습도 +{_EQphys:.1f}%p = ×{np.exp(_bH12*_EQphys):.3f}')
print(f'     같아지려면 습도가 {_EQ12:.1f}%p 올라야 한다 — **가설이 예측하는 것보다 {_EQ12/_EQphys:.0f}배 둔하다.**')
print(f'   같은 방향의 증거: §9-g에서 습구온도 단일지표(Ⓑ)는 AIC {_fitg["Ⓑ **일 최고 습구온도**"].aic:.0f}로')
print(f'   기온만 쓴 Ⓐ({_fitg["Ⓐ 일최고기온만 (§9-① 현행)"].aic:.0f})보다 **나쁘다**. 우리 자료에서 위험을 나르는 건 건구온도다.')
print('   ⚠ 왜 둔한지는 **모른다**. 후보 둘: (i) 전국·일 집계라 지역·시간 내 습도 변동이 씻긴다')
print('     (ii) 한국 여름 습도가 이미 높아(평균 79.9%) 상단에서 포화. 우리 자료로는 못 가린다.')

# ── ③ 증발식 냉각탑이 실제로 습도를 몇 %p 올리나 — 상자모형 자릿수 계산 ──
# ⚠ 이건 **플룸 모형이 아니다.** 완전혼합 상자 + 정상상태라는 거친 가정이다. 근접장은 훨씬 높을 수 있다.
_GW12,_WPD=2.4,3.8e4          # 북평2 발표용량 GW · 1GW당 증발수 톤/일(§12-c 사용자 제공)
_WKG=_GW12*_WPD*1e3           # kg/일
def _ws(T,p=1013.0):
    _es=6.112*np.exp(17.67*T/(T+243.5))
    return 0.622*_es/(p-_es)*1000        # 포화혼합비 g/kg
_w_s0=_ws(_T0); _w0=_w_s0*_R0/100
print()
print('③ **여기가 플룸 모형의 자리다** — 우리는 상자모형까지밖에 못 한다')
print(f'③ 증발식이 습도를 얼마나 올리나 — 상자모형 자릿수 (2.4GW × {_WPD/1e4:.1f}만톤/일/GW = {_WKG/1e7:.1f}만톤/일)')
print(f'   기준 대기: T={_T0:.1f}℃ · 포화혼합비 {_w_s0:.1f} g/kg · 현재 혼합비 {_w0:.1f} g/kg (RH {_R0:.1f}%)')
print(f'   {"반경km":>6}{"혼합고m":>8}{"풍속m/s":>8}{"ΔRH %p":>9}{"위험배수":>9}{"= 기온 몇 ℃":>12}')
for _rk,_h,_u in [(3,500,3),(3,500,2),(3,100,2),(1,100,2),(1,50,1),(0.5,50,1)]:
    _V=np.pi*(_rk*1000)**2*_h; _mair=_V*1.16
    _tau=2*_rk*1000/_u; _ex=86400/_tau
    _dw=(_WKG/_ex)/_mair*1000                       # g/kg
    _raw=_dw/_w_s0*100; _dRH=min(_raw,100-_R0)      # 포화 상한(RH 100% 초과 불가)
    _cap='  ★포화' if _raw>100-_R0 else ''
    print(f'   {_rk:>6}{_h:>8}{_u:>8}{_dRH:>9.1f}{np.exp(_bH12*_dRH):>9.3f}{_dRH/_EQ12:>12.2f}{_cap}')
print('   ⚠ 완전혼합 상자 + 정상상태 가정. 실제 플룸은 상승·부력으로 지표에 덜 닿거나(과대) 정체 시 뭉칠 수 있다(과소).')
print(f'   ★포화 = 계산값이 RH 100%를 넘어 상한({100-_R0:.1f}%p)에서 잘린 행. 이 행들은 "얼마나"가 아니라')
print('     **"안개가 낀다"**는 뜻이다 — 그 조건에서 상자모형은 이미 물리적으로 깨졌고, 값을 인용하면 안 된다.')

# ── ④ 손익분기 — 증발식이 기온을 덜 올리는 이득 vs 습도를 올리는 손해 ──
print()
print('④ 손익분기 — 증발식 채택 시 (기온 절감 이득) vs (습도 상승 손해)')
print(f'   {"기온 절감 ℃":>11}{"이득 배수":>9}{"상쇄에 필요한 ΔRH":>17}{"상자모형 최악 ΔRH":>17}')
_worst=None
for _rk,_h,_u in [(1,50,1)]:
    _V=np.pi*(_rk*1000)**2*_h; _mair=_V*1.16; _ex=86400/(2*_rk*1000/_u)
    _worst=min((_WKG/_ex)/_mair*1000/_w_s0*100,100-_R0)
for _dTs in (0.1,0.25,0.5,1.0):
    print(f'   {_dTs:>11}{np.exp(-_bT12*_dTs):>9.3f}{_dTs*_EQ12:>15.1f}%p{_worst:>15.1f}%p')
print(f'   → 넓은 규모(3km·혼합고 500m)에선 습도 손해가 기온 이득의 **몇 %에 불과**하다.')
print(f'   → 그러나 **정체·야간 안정층(1km·50m·1m/s)에서는 ΔRH {_worst:.0f}%p = 기온 {_worst/_EQ12:.2f}℃ 상당**으로')
print('     0.1~0.25℃급 절감과 맞먹는다. 즉 "언제나 안전"도 "언제나 위험"도 아니고 **조건에 달렸다**.')

# ── ⑤ 결론 — §12-c ② 경고의 정정 ──
print()
print('⑤ **정정** — §12-c ②의 경고를 그대로 쓸 수 없다')
print('   · 07-25 문장: "온열질환은 습구온도에 민감하다 → 기온이 덜 오른다가 곧 덜 위험하다가 아니다"')
print(f'   · 우리 자료: 습도는 기온보다 {_EQ12:.0f}배 둔한 지렛대이고, 습구 단일지표는 기온보다 **적합이 나쁘다**.')
print('   · 따라서 **평균적 조건에서는 증발식이 ambient 기준 덜 위험할 가능성이 높다.** 경고를 약화한다.')
print('   · 살아남는 것 셋 — 이쪽으로 논증을 옮긴다:')
print(f'     (a) **정체·야간 안정층**에서는 습도 손해가 기온 이득에 근접한다(위 ③④). 평균으로 안심할 수 없다.')
print(f'     (b) **물 소비** — 2.4GW면 하루 {_WKG/1e7:.1f}만톤·연 {_WKG*365/1e7:,.0f}만톤. 온열과 별개 축의 피해다.')
print(f'         (§12-c 사용자 제공 1GW당 3.8만톤/일·연 1,380만톤과 정합: 1,380×2.4={1380*2.4:,.0f}만톤 ✓)')
print('     (c) **근접장은 우리 측정 범위 밖** — 상자모형은 플룸 모형이 아니다. Sailor가 잰 100~500m대의')
print('         습도 상승은 아무도 안 쟀다. 이 공백이 곧 §17 "플룸 모형 제출 의무화" 요구의 근거다.')
print('   ⚠ 방향 명시: 이 정정은 우리 주장을 **약화하는** 쪽이다. 그래서 더 빨리 적는다 —')
print('     상대가 먼저 재서 들이대면 §12-c 전체의 신뢰가 같이 무너진다.')

# ── [🔖 2026-07-27 재정정 · 자기수정 9] ⑤의 근거였던 40.6%p는 **뭉갠 모형**에서 나온 값이다 ──
# §9-i(시군구×일 FE)가 같은 등가를 다시 재니 **17.0%p**다. ⑤의 철회가 과했다.
# ⚠ 경위를 분명히 적는다 — 결과를 보고 모형을 고른 게 아니다. 사용자가 §9-i 결과를 모르는 상태에서
#   "지점별 습도가 있으면 그렇게 했어야 하지 않냐"고 지적했고, 그 지적이 방법론적으로 옳았다.
#   시군구 FE가 나은 이유(표본 41배·지역 편차 보존·지역 간 교란 제거)는 결과와 무관하다.
print()
print('⑥ **재정정** — ⑤의 "41배 둔한 지렛대"는 뭉갠 모형의 값이었다 (자기수정 9)')
_EQnew=_EQ9i if '_EQ9i' in dir() else 17.0
print(f'   기온 1℃ = 습도 몇 %p ?   §9-g 전국·일 {_EQ12:.1f}%p → **§9-i 시군구×일 {_EQnew:.1f}%p** (Stull 물리 {_EQphys:.1f}%p)')
print(f'   → 습도는 {_EQ12/_EQphys:.0f}배가 아니라 **{_EQnew/_EQphys:.0f}배** 둔한 지렛대다. 손익분기가 달라진다:')
print(f"   {'기온 절감':>9}{'상쇄 필요 ΔRH(구 40.6%p)':>24}{'(신 17.0%p)':>14}")
for _dTs in (0.1,0.25,0.5,1.0):
    print(f'   {_dTs:>8}℃{_dTs*_EQ12:>22.1f}%p{_dTs*_EQnew:>12.1f}%p')
print()
print('   [상자모형 ③과 대조] 1km·혼합고 100m·2m/s 조건의 ΔRH 11.3%p는')
print(f'     구 등가로는 기온 {11.3/_EQ12:.2f}℃ 상당 → 무시할 만했지만,')
print(f'     신 등가로는 기온 **{11.3/_EQnew:.2f}℃ 상당** → 0.25~0.5℃급 절감과 **맞먹거나 넘어선다**.')
print('   → **§12-c ②의 원래 경고 쪽으로 되돌아간다.** "정체·야간 안정층에서만"이라는 단서가 훨씬 넓어진다.')
print('   → 다만 ⑤의 나머지 셋(물 소비·근접장 미측정·플룸 모형 요구)은 그대로 유효하다.')
print('   ⚠ 그래도 "증발식이 더 위험하다"고 단정하지 않는다 — ΔRH 자체가 상자모형 자릿수라 근거가 약하다.')
print('     정확한 표현: **"기온만 보고 안심할 근거가 없다"**까지가 우리가 말할 수 있는 것이다.')

① 우리 자료가 말하는 습도의 크기 (§9-g Ⓒ · 시각 일치)
   기온 ×1.606/℃ · 습도 +0.0117/%p (10%p당 ×1.124)
   → **등가 환산: 습도 40.6%p 상승 = 기온 1℃ 상승.** 습도는 기온보다 41배 둔한 지렛대다.

② 습구온도 가설과 대조 — §12-c ②가 우리 자료에서 성립하나 (Stull 환산식, 플룸 모형 아님)
   습구온도가 **같아지는** 두 상황을 찾는다 (평균 조건 T=29.0℃·RH=63.3%):
     기준          29.0℃ · 63.3%   → 습구 23.64℃
     기온 +1℃      30.0℃ · 63.3%   → 습구 24.55℃
     습도 +5.8%p   29.0℃ · 69.1%   → 습구 24.54℃   ← 같다
   → 습구온도 가설이 맞다면 이 둘은 **똑같이 위험해야 한다**: 기온 +1℃ = 습도 +5.8%p
   → 그런데 실제 환자 수로는: 기온 +1℃ = ×1.606 vs 습도 +5.8%p = ×1.070
     같아지려면 습도가 40.6%p 올라야 한다 — **가설이 예측하는 것보다 7배 둔하다.**
   같은 방향의 증거: §9-g에서 습구온도 단일지표(Ⓑ)는 AIC 9400로
   기온만 쓴 Ⓐ(5112)보다 **나쁘다**. 우리 자료에서 위험을 나르는 건 건구온도다.
   ⚠ 왜 둔한지는 **모른다**. 후보 둘: (i) 전국·일 집계라 지역·시간 내 습도 변동이 씻긴다
     (ii) 한국 여름 습도가 이미 높아(평균 79.9%) 상단에서 포화. 우리 자료로는 못 가린다.

③ **여기가 플룸 모형의 자리다** — 우리는 상자모형까지밖에 못 한다
③ 증발식이 습도를 얼마나 올리나 — 상자모형 자릿수 (2.4GW × 3.8만톤/일/GW = 9.1만톤/일)
   기준 대기: T=29.0℃ · 포화혼합비 25.6 g/kg · 현재 혼합비 16.2 g/kg (RH 63.3%)
     반경km    혼합고m   풍속m/s   ΔRH %p    

In [ ]:
# ── §12-c3. 열 수지 — 전력은 전부 열이 되고, 나갈 길은 셋뿐이다 [🔖 2026-07-27 사용자 질문] ──
# [질문] "온열질환자가 많이 늘지 않는다면 그건 전기나 물 중 하나를 많이 소모하기 때문일까?"
# [답] 그렇다. 열역학적으로 닫힌 회계다. 데이터센터가 쓴 전력은 거의 전부 열이 되고, 나갈 길은 셋뿐:
#   ① 공기로 **현열**(드라이쿨러)        → 기온 상승 → 우리 dose-response에 잡힌다
#   ② 물 증발로 **잠열**(증발식 냉각탑)   → 물 소비 + 습도 → 기온으로는 안 잡힌다
#   ③ 수역으로 **온배수**(해양·하천 방류) → 수온 상승 → 안 잡힌다
# → 우리 추가온열 추정이 작게 나오는 것은 **결함이 아니라 냉각방식의 귀결**일 수 있다. 재본다.
_LAT12=2.44e6        # 25~30℃ 물의 증발잠열 J/kg
_GW12c3=2.4          # 북평2 발표 용량
_W_PER_GW=3.8e4      # §12-c 제시 물 소비 톤/일/GW  ⚠ 출처 미확인 추정
_W12=_W_PER_GW*_GW12c3

print('§12-c3. 열 수지 — 북평2 2.4GW 기준')
print(f'  물 소비 제시값 {_W12:,.0f} 톤/일 (= {_W_PER_GW/1e4:.1f}만톤/일/GW × {_GW12c3}GW) ⚠ 출처 미확인 추정(§12-c)')
print()
print(f"  {'PUE':>5}{'총배출열':>10}{'전량증발 필요수량':>18}{'증발 비중':>10}{'공기로 가는 현열':>16}")
_rows12=[]
for _pue in (1.1,1.2,1.3,1.5):
    _Q=_GW12c3*1e9*_pue                                  # W
    _need=_Q/_LAT12*86400/1e3                            # 톤/일
    _lat=_W12*1e3/86400*_LAT12                           # W (제시 수량이 전부 증발한다면)
    _sen=max(_Q-_lat,0.0)
    _rows12.append((_pue,_Q,_need,_lat,_sen))
    print(f'  {_pue:>5.1f}{_Q/1e9:>9.2f}GW{_need:>16,.0f}톤{_W12/_need*100:>9.0f}%{_sen/1e9:>13.2f}GW ({_sen/_Q*100:.0f}%)')
print()
print('  → **제시된 물 소비량은 배출열의 대부분이 수증기로 나간다는 뜻이다.**')
print('    전력 수치(§9)와 물 수치(§12-c)가 서로 다른 출처인데 **에너지 보존으로 맞물린다** — 정합성 확인.')

# ── 우리 dose 추정과의 정합 ──
print()
print('[우리 추정과의 정합] 공기로 가는 현열이 작으면 기온 상승도 작아야 한다')
_sen_frac=[max(0.0,(_GW12c3*1e9*p-_W12*1e3/86400*_LAT12))/(_GW12c3*1e9*p) for p in (1.1,1.3)]
print(f'  현열 비중 {min(_sen_frac)*100:.0f}~{max(_sen_frac)*100:.0f}% → 같은 용량의 **전량 공랭** 시설 대비 기온 영향이 그만큼 작다.')
print(f'  우리 §11-b(3) 인천 시나리오가 13건/년으로 작게 나온 것은 이 구조와 모순되지 않는다.')
print('  ⚠ 단 인천은 냉각방식이 미정이다. 전량 건식(드라이쿨러)이면 현열 100% → 기온 영향이 위 범위보다 크다.')

# ── 논증: 닫힌 회계라 둘 다 작을 수 없다 ──
print()
print('[논증] 총배출열 = 공기 현열 + 물 잠열 + 수역 온배수 — **닫힌 회계**다')
for _a,_b in [('"기온 영향이 작다"','→ 물을 그만큼 쓴다는 자백'),
              ('"물을 적게 쓴다"','→ 기온이 그만큼 오른다는 자백'),
              ('"둘 다 작다"','→ 열이 어디로 갔는지 답해야 한다(수역 방류·폐열 회수 실적 제시)')]:
    print(f'  {_a:22}{_b}')
print('  → 사업자는 셋 중 하나를 반드시 감당한다. **어느 쪽을 줄여도 다른 쪽이 드러난다.**')
print('  → 그래서 §17 요구를 "열섬 평가 항목 신설"이 아니라 **"열 수지(heat balance) 신고"**로 세우는 것이 강하다.')
print('    공기·물·수역 3분할 배출 계획을 내게 하면 회피할 칸이 없다.')

# ── 한계 ──
print()
print('⚠ 한계 — 이 계산이 기대는 것')
for _i,_l in enumerate([
    '물 소비 3.8만톤/일/GW가 **출처 미확인 추정**이다(§12-c). 이 값이 틀리면 증발 비중이 통째로 바뀐다.',
    'PUE를 모른다. 1.1~1.5 범위로 열어뒀지만 실제 값은 사업자만 안다.',
    '냉각탑 보충수 전부가 증발하는 것은 아니다(블로다운·드리프트 손실). 증발 비중은 상한이다.',
    '수역 방류(③)는 우리가 수치를 전혀 모른다. 강릉 도암댐 취수·온배수 구상은 환경정의 현장자료로만 확인.',
    '열 회수(지역난방)는 겨울 이야기다. 여름엔 수요가 없어 이 회계에서 빠진다.'],1):
    print(f'  ({_i}) {_l}')
print('→ 그래서 **수치가 아니라 구조**로 쓴다. "셋 중 하나는 반드시 감당한다"는 것은 물 수치가 틀려도 성립한다.')


§12-c3. 열 수지 — 북평2 2.4GW 기준
  물 소비 제시값 91,200 톤/일 (= 3.8만톤/일/GW × 2.4GW) ⚠ 출처 미확인 추정(§12-c)

    PUE      총배출열         전량증발 필요수량     증발 비중       공기로 가는 현열
    1.1     2.64GW          93,482톤       98%         0.06GW (2%)
    1.2     2.88GW         101,980톤       89%         0.30GW (11%)
    1.3     3.12GW         110,479톤       83%         0.54GW (17%)
    1.5     3.60GW         127,475톤       72%         1.02GW (28%)

  → **제시된 물 소비량은 배출열의 대부분이 수증기로 나간다는 뜻이다.**
    전력 수치(§9)와 물 수치(§12-c)가 서로 다른 출처인데 **에너지 보존으로 맞물린다** — 정합성 확인.

[우리 추정과의 정합] 공기로 가는 현열이 작으면 기온 상승도 작아야 한다
  현열 비중 2~17% → 같은 용량의 **전량 공랭** 시설 대비 기온 영향이 그만큼 작다.
  우리 §11-b(3) 인천 시나리오가 13건/년으로 작게 나온 것은 이 구조와 모순되지 않는다.
  ⚠ 단 인천은 냉각방식이 미정이다. 전량 건식(드라이쿨러)이면 현열 100% → 기온 영향이 위 범위보다 크다.

[논증] 총배출열 = 공기 현열 + 물 잠열 + 수역 온배수 — **닫힌 회계**다
  "기온 영향이 작다"           → 물을 그만큼 쓴다는 자백
  "물을 적게 쓴다"            → 기온이 그만큼 오른다는 자백
  "둘 다 작다"              → 열이 어디로 갔는지 답해야 한다(수역 방류·폐열 회수 실적 제시)
  → 사업자는 셋 중 하나를 반드시 감당한다. **어느 쪽을 줄여도 다른 쪽이 드러난다.**
  

### §12-d. 형상 함의 정량화 — 온산 과대·북평2 확장 시나리오 `[🔖 2026-07-23 신규 · 사용자 해석 구현]`

§12-c의 두 해석을 숫자로 만든다:
- **온산**: 산단이 워낙 커서(490만평) DC는 일부 필지(강양·우봉 10만평)만 쓴다 → "산단 전체 경계+3km"는 과대. 얼마나?
- **북평2**: 부지(9.7만평)가 계획(2.4GW 표준 소요 146~235만평)보다 훨씬 작다 → 확장이 개연적. 확장하면 근접권 노출이 얼마나 커지나? (확장 방향은 알 수 없으므로 **등방 버퍼 확장**으로 목표 면적에 도달시키는 가정 — 시나리오이지 예측이 아님.)

> `[🔖 2026-07-24 정밀화]` 토지피복 실측: **북평2 부지의 89%가 인공나지**(L3=623, 사람이 정지한 맨땅·건물 없음). 이는 오분류가 아니라 실제 상태 — 북평제2일반산단은 산단공 대장 기준 **아직 「조성중」**이고 **입주 0·가동 0**이다(2025Q3). 분양도 **28.4%**(분양 88 / 미분양 222천㎡)뿐이라 대부분이 빈 땅인 게 맞다. `[🔖 2026-07-27 감사 정정]` 이전 판은 "2016 착공·2022 조성완료 후 미입주"라고 썼는데 **대장과 어긋난다** — 조성완료가 아니라 조성중이다. "조성 중이고 분양도 28%뿐이라 건물이 거의 없음"이 정확 → 확장·전환 시 나지가 산업시설로 덮이는 물적 근거.
>
> **강양·우봉지구가 온산 산단에 포함된다고 본 이유·YUCH 부재** `[🔖 2026-07-24]`: 강양·우봉은 **온산국가산단 안의 하위 지구**(온산읍 강양리·우봉리)로, 보도가 "온산국가산업단지 강양·우봉지구"로 명시. 독립 산단 폴리곤(YUCH)이 없어(PDAN에 강양/우봉 항목 없음) **점 근사가 유일**하다 — 지구 경계 자료 확보 시 재계산.

> `[🔖 2026-07-27 감사 근거]` 산단공 「전국산업단지현황」 2025Q3 원문: `일반 · 강원 · 동해시 · 북평제2 · 조성상태=조성중 · 지정면적 593 · 산업시설 310 · 분양 88 · 미분양 222 · 분양률 28.39 · 입주 0 · 가동 0`(단위 천㎡). 같은 표의 `북평 ①`(국가산단)은 조성상태 완료·분양률 100%·입주 56·가동 33으로 대조된다.
>
> **논증에 주는 함의**: "조성 끝난 유휴지에 들어온다"보다 **"분양이 28%밖에 안 된 조성중 산단에 2.4GW가 들어온다"**가 사실에 가깝다. 미분양 222천㎡가 남아 있다는 건 §12-d (2)의 확장 시나리오가 산단 **안에서** 일부 흡수될 여지를 뜻하기도 한다 — 다만 2.4GW 표준 소요(146~235만평)에 비하면 미분양분(약 6.7만평)도 턱없이 작다.

In [ ]:
# ── §12-d. 형상 시나리오 정량화 [🔖 2026-07-23 신규] ──
# self-contained 버퍼 인구 (§18-b _bp18과 동일 방식 — 폴리곤·점 공용)
import pyogrio as _pg12
from shapely.geometry import Point as _Pt12

# 1. 집계구별 2024년 총인구 통계 데이터 로드 및 딕셔너리(_POP12) 맵핑
_pop12=pd.read_csv('0718_2024년 인구총괄/2025년기준_2024년_인구총괄(총인구).csv',header=None,names=['y','oa','item','val'],dtype={'oa':str},encoding='cp949')
_POP12=_pop12[_pop12.item=='to_in_001'].set_index('oa')['val'].to_dict()

# 2. 2025년 기준 집계구 Spatial SHP 파일 경로 지정
_SHP12='0718_2025년 집계구경계/bnd_oa_00_2025_2Q.shp'

# 3. 공간 경계(점/폴리곤)와 버퍼 반경(R)을 입력받아 면적 안분(Area-weighting) 방식으로 상주인구를 집계하는 함수
def _bp12(geom,R):
    buf=geom.buffer(R); b=buf.bounds

    # pyogrio의 bbox 쿼리를 활용해 버퍼 영역 주변의 집계구만 빠르게 로드
    oa=_pg12.read_dataframe(_SHP12,bbox=(b[0]-100,b[1]-100,b[2]+100,b[3]+100))

    # 집계구가 버퍼에 포함되는 면적 비율(가중치 w: 0~1) 계산
    w=(oa.geometry.intersection(buf).area/oa.geometry.area).clip(0,1)

    # 인구 데이터와 면적 비율을 곱하여 최종 노출 인구 산출
    return float((oa['TOT_OA_CD'].map(_POP12).fillna(0.0)*w).sum())

# (1) 온산 — 산단 전체 경계+3km vs 강양·우봉 점+3km (현실적 노출)
# 온산 산단 전체 경계(폴리곤) 추출 및 통합
_r12=PDAN[(PDAN.DAN_NAME=='온산')&(PDAN.DANJI_TYPE=='1')].iloc[0]
_ons12=YUCH[YUCH.DAN_ID==_r12.DAN_ID].to_crs(5179).geometry.union_all()
_p_ons=_bp12(_ons12,3000) # [Case A] 온산 산단 전체 경계 + 3km 버퍼 인구

# 데이터센터(DC) 입지 예정지인 '강양·우봉 지구' 특정 중심점(점) 생성
_gy12=gpd.GeoSeries([_Pt12(129.3510,35.3990)],crs=4326).to_crs(5179).iloc[0]
_p_gy=_bp12(_gy12,3000) # [Case B] 실제 DC 입지(강양·우봉) 점 기준 + 3km 버퍼 인구


# [🔖 2026-07-25 사용자 지적] 강양·우봉도 실제로는 면(지구)이다. '점'인 이유는 지구 경계 자료가
#   공개돼 있지 않아서(ILIS YUCH에 독립 산단으로 등재 없음) 강양리·우봉리 중간점으로 근사한 것뿐이다.
print(f'(1) 온산: 산단 전체 경계+3km {_p_ons:,.0f}명 vs 강양·우봉 지구 근사+3km {_p_gy:,.0f}명')
print(f'    → DC가 강양·우봉 지구에 들어올 경우, 산단 전체 기준은 근접 노출을 {_p_ons/max(_p_gy,1):.1f}배 과대집계.')
print('    ⚠ 강양·우봉은 **면(지구)이지 점이 아니다.** 지구 경계 자료 미공개라 두 리 중간점으로 근사했을 뿐이며,')
print('      경계를 확보하면 이 값은 (점보다 넓어지므로) 올라간다 — 현재 값은 하한이다.')

# (2) 북평2 — 표준 소요면적까지 등방 확장 시나리오
_r22=PDAN[(PDAN.DAN_NAME=='북평2')&(PDAN.DANJI_TYPE=='2')].iloc[0]
_bp22=YUCH[YUCH.DAN_ID==_r22.DAN_ID].to_crs(5179).geometry.union_all()

# 이분법(Bisection search) 기반: 목표 면적(target_m2)에 도달할 때까지 등방(동서남북 동일하게) 버퍼 거리를 탐색하는 함수
def _grow12(target_m2):
    lo,hi=0.0,6000.0
    
    for _ in range(40): # 40회 수렴 연산으로 매우 정밀한 버퍼 거리(mid) 탐색
        mid=(lo+hi)/2
        if _bp22.buffer(mid).area<target_m2:
            lo=mid
        else:
            hi=mid
    return mid,_bp22.buffer(mid)

print()
print(f'(2) 북평2 확장 시나리오 (현재 {_bp22.area/3.305785/1e4:.1f}만평 → 2.4GW 표준 소요 146~235만평 [사용자 제공 추정 기준]):')
print(f"    {'시나리오':24}{'경계+3km 인구':>13}{'경계+10km 인구':>14}")

# 3가지 조건(현재, 소요 하한 146만평, 소요 상한 235만평)별 노출 인구 변화 시뮬레이션
for _lbl,_manp in [('현재 부지',None),('하한 146만평 도달',146e4),('상한 235만평 도달',235e4)]:
    if _manp is None:
        _gg=_bp22; _d=0.0
    else:
        # 만평을 m² 단위로 변환하여 목표 면적 산출 후 확장 폴리곤(_gg) 및 필요 버퍼 거리(_d) 도출
        _d,_gg=_grow12(_manp*3.305785)

    # 확장된 부지 경계 기준 3km 및 10km 버퍼 인구 산출
    _p3=_bp12(_gg,3000); _p10=_bp12(_gg,10000)
    print(f'    {_lbl:24}{_p3:>13,.0f}{_p10:>14,.0f}   (버퍼 +{_d:,.0f}m)')
    
print('    → 확장 시 3km 근접권 노출이 2.2~2.5배(2.6만→5.7~6.4만).')
print('    ⚠ 등방 확장은 방향 미상에 따른 가정 — 해안·기존 산단 방향 확장이면 값이 달라진다. §12-c 해석("폴리곤+3km는 하한")의 정량 근거.')

# [🔖 2026-07-25 사용자 지적] "10km는 동해시 전역이라 거의 불변"이라고 썼는데, 버퍼 인구는 시 경계와
#   무관하게 **집계구 교집합**으로 세므로 이웃 시 인구도 이미 들어간다. 근거 없이 쓴 문장이라 실제로 확인한다.
_g10=_bp22.buffer(10000); _b10=_g10.bounds
_oa10=_pg12.read_dataframe(_SHP12,bbox=(_b10[0]-100,_b10[1]-100,_b10[2]+100,_b10[3]+100))
_oa10['p']=_oa10['TOT_OA_CD'].map(_POP12).fillna(0.0)*(_oa10.geometry.intersection(_g10).area/_oa10.geometry.area).clip(0,1)
_oa10['sgg']=_oa10['TOT_OA_CD'].str[:5].map(_m5)
_mix10=_oa10.groupby('sgg')['p'].sum().sort_values(ascending=False)
_mix10=_mix10[_mix10>=1]
print()
print('    [확인] 북평2 10km 버퍼의 시군구 구성 — 버퍼는 시 경계를 넘어 집계된다:')
for _k,_v in _mix10.items():
    print(f'      {str(_k):22}{_v:>10,.0f}명 ({_v/_mix10.sum()*100:>4.1f}%)')
print('    → 10km가 확장에 둔감한 건 "동해시 전역이라서"가 아니라, 이미 반경이 커서 +0.9~1.2km 확장분이')
print('      전체에서 차지하는 비중이 작기 때문이다(모든 부지에 동일하게 적용되는 성질). 서술을 정정한다.')

# (3) 열원 면적 하한 — 산단 전체가 AIDC가 되지는 않는다 [🔖 2026-07-25 사용자 지적]
#   부지/표준 비율이 1보다 크면 그 산단은 표준 소요보다 넓다 = AIDC가 산단 일부만 쓴다.
#   §11-b·§12-b는 폴리곤 **전체**를 열원으로 두므로 상한이다. 하한 = 표준 소요 면적의 등가원을 중심에 둔 경우.

print()
print('(3) 열원 면적 하한 — 산단 전체가 AIDC가 되는 건 아니다 (부지/표준 비율 > 1인 곳)')
print(f"    {'부지':12}{'상정GW':>7}{'전체+3km':>11}{'표준코어+3km(하한)':>20}{'과대배수':>9}")

# -----------------------------------------------------------------------------
# 1. GW당 표준 소요 부지 면적 기준 정의 (단위: km²/GW)
# -----------------------------------------------------------------------------
# 1평 = 3.305785 m², 1 km² = 100만 m² (1e6)
# - 하한 기준: 61만 평/GW  → 약 2.016 km²/GW
# - 상한 기준: 98만 평/GW  → 약 3.240 km²/GW
_STD=[(61e4*3.305785/1e6,'하한 61만평/GW'),(98e4*3.305785/1e6,'상한 98만평/GW')]

# -----------------------------------------------------------------------------
# 2. 대상 산단별 공간 연산 및 노출 영역 비교
# -----------------------------------------------------------------------------
# 타깃 산단 라벨, 산단명, 단지유형, 상정 AIDC 용량(GW) 루프
for _lbl4,_nm4,_ty4,_gw4 in [('온산','온산','1',1.0),('울산미포','울산·미포','1',1.0)]:

    # 산단 속성 정보 및 식별자(DAN_ID) 추출
    _r4=PDAN[(PDAN.DAN_NAME==_nm4)&(PDAN.DANJI_TYPE==_ty4)].iloc[0]
    
    # GIS 실면적 폴리곤들을 투영좌표계(EPSG:5179, 평면지적좌표/미터 단위)로 변환 후 하나로 병합(Union)
    _u4=YUCH[YUCH.DAN_ID==_r4.DAN_ID].to_crs(5179).geometry.union_all()

    # [시나리오 A: 상한] 산단 폴리곤 전체 경계로부터 3km(3,000m) 버퍼를 씌웠을 때의 노출 영역 산출
    _full4=_bp12(_u4,3000)
    
    # [시나리오 B: 하한] AIDC가 산단 중심(Centroid)에 코어 형태로 입지한다고 가정:
    # 1. GW당 필요 면적(_a * _gw4)을 기반으로 등가원 반지름 r = sqrt(면적 / pi) 계산
    # 2. 중심점에 해당 반지름으로 버퍼를 씌워 코어 면적 폴리곤 생성 (.buffer)
    # 3. 그 코어 외곽으로부터 다시 3km 버퍼를 씌운 노출 영역 계산 (_bp12)
    # 하한/상한 표준면적 기준 중 최소값(lo4)과 최대값(hi4) 추출
    _lo4=min(_bp12(_u4.centroid.buffer(np.sqrt(_a*_gw4*1e6/np.pi)),3000) for _a,_ in _STD)
    _hi4=max(_bp12(_u4.centroid.buffer(np.sqrt(_a*_gw4*1e6/np.pi)),3000) for _a,_ in _STD)

    # 산단 전체를 열원으로 보았을 때 노출 영향이 몇 배 과대평가되는지 배수(Overestimation ratio) 출력
    print(f"    {_lbl4:12}{_gw4:>7.1f}{_full4:>11,.0f}{f'{_lo4:,.0f}~{_hi4:,.0f}':>20}{_full4/max(_hi4,1):>8.1f}×")
print('    → 산단 전체를 열원으로 두면 근접 노출이 몇 배 커진다. 상한(전체)·하한(표준 코어)을 함께 읽을 것.')
print('    ⚠ 코어를 폴리곤 **중심**에 둔 가정 — 실제 입지가 도심 쪽이면 하한도 올라간다. 61~98만평/GW는 출처 미확인 추정.')

# ── [🔖 2026-07-27 사용자 질문] "코어를 폴리곤 중심에 둔 가정"이 무슨 뜻이고 왜 하한이 오르나 ──
print()
print('[설명] "코어를 폴리곤 중심에 둔다"의 뜻')
print('  · 우리는 AIDC가 산단 **어디에** 앉는지 모른다. 표준 소요면적(61~98만평/GW)만큼의 "코어"를')
print('    산단 폴리곤 안 어딘가에 놓아야 하는데, 위치를 모르니 **중심(centroid)**에 놓았다.')
print('  · 노출 인구는 코어 경계+3km 안의 사람 수다. 그래서 **코어를 어디 두느냐가 답을 바꾼다**:')
print('      중심에 두면 → 3km 원이 산단 안쪽을 많이 덮는다(산단 안엔 사람이 거의 없다) → 노출 적음')
print('      도심 쪽 가장자리에 두면 → 3km 원이 주거지 쪽으로 쏠린다 → 노출 많음')
print('  · 즉 중심 배치는 **노출을 가장 적게 잡는 배치**에 가깝다 → 그래서 "하한"이라고 부른다.')
print('  · 실제 입지는 보통 **인프라(변전소·용수·도로) 가까운 쪽**을 고르고, 그쪽이 도심인 경우가 많다.')
print('    → 실입지가 확정되면 하한은 **올라갈 가능성이 높다**(내려갈 가능성도 있지만 비대칭).')
print('  ⚠ 61~98만평/GW는 §12-c와 같은 출처 미확인 추정 — 코어 크기 자체가 흔들리면 이 하한도 흔들린다.')

(1) 온산: 산단 전체 경계+3km 32,242명 vs 강양·우봉 지구 근사+3km 3,960명
    → DC가 강양·우봉 지구에 들어올 경우, 산단 전체 기준은 근접 노출을 8.1배 과대집계.
    ⚠ 강양·우봉은 **면(지구)이지 점이 아니다.** 지구 경계 자료 미공개라 두 리 중간점으로 근사했을 뿐이며,
      경계를 확보하면 이 값은 (점보다 넓어지므로) 올라간다 — 현재 값은 하한이다.

(2) 북평2 확장 시나리오 (현재 9.7만평 → 2.4GW 표준 소요 146~235만평 [사용자 제공 추정 기준]):
    시나리오                        경계+3km 인구    경계+10km 인구
    현재 부지                          25,979       123,274   (버퍼 +0m)
    하한 146만평 도달                    56,814       124,859   (버퍼 +858m)
    상한 235만평 도달                    63,740       125,219   (버퍼 +1,189m)
    → 확장 시 3km 근접권 노출이 2.2~2.5배(2.6만→5.7~6.4만).
    ⚠ 등방 확장은 방향 미상에 따른 가정 — 해안·기존 산단 방향 확장이면 값이 달라진다. §12-c 해석("폴리곤+3km는 하한")의 정량 근거.

    [확인] 북평2 10km 버퍼의 시군구 구성 — 버퍼는 시 경계를 넘어 집계된다:
      강원특별자치도 동해시               82,178명 (66.7%)
      강원특별자치도 삼척시               41,096명 (33.3%)
    → 10km가 확장에 둔감한 건 "동해시 전역이라서"가 아니라, 이미 반경이 커서 +0.9~1.2km 확장분이
      전체에서 차지하는 비중이 작기 때문이다(모든 부지에 동일하게 적용되는 성질). 서술을 정정한다.

(3) 열원 면적 하한 — 산단 전체가 AIDC가 되는 건

### 💡 AI 반도체 열유속(Heat Flux) 핵심 요약

#### 1. 열유속 비교 수치 (W/cm²)
* **가정용 다리미:** ~10 W/cm²
* **원자력 발전소 연료봉:** 100 ~ 200 W/cm²
* **공기 냉각(공랭)의 물리적 한계:** ~100 W/cm²
* **AI 반도체 핫스팟(NVIDIA H100/B200 등):** **500 ~ 1,200 W/cm²**
* **우주선 대기권 재돌입 방열판:** 300 ~ 1,000 W/cm²
* **태양 표면 순수 열유속:** ~6,000 W/cm²
> 📌 **결론:** AI 칩 핫스팟의 열 밀도는 **원자로 내부·우주선 재돌입 마찰열을 넘어서며, 태양 표면 수준에 육박**함.
* **학술 논문:** Mahajan, R., et al. (2002). *Thermal Management Challenges in High Power Microprocessors*. IEEE Transactions on Components and Packaging Technologies.
* **학술 논문:** Bar-Cohen, A., & Wang, P. (2012). *Thermal Management of High Heat Flux Electronics*. World Scientific.
* **최신 백서 (NVIDIA/ISSCC 2024):** IEEE ISSCC 및 NVIDIA Technical Blog의 "Direct-to-Chip Liquid Cooling Architecture" 기술 문서.
* 최신 AI 패키징(CoWoS) 칩 전체 평균은 $100\sim150 \text{ W/cm}^2$이나, 연산 소자(Compute Die) 내 핫스팟의 국소 열유속은 **$500\sim1,200 \text{ W/cm}^2$**에 달함을 명시.
* **태양 표면 복사 열유속 ($\approx 6,300 \text{ W/cm}^2$):** 슈테판-볼츠만 법칙($q = \sigma T^4$, 태양 표면 온도 $T \approx 5,778\text{K}$)에 의한 물리적 계산값.
* **원자로 연료봉 임계 열유속 ($\approx 100\sim200 \text{ W/cm}^2$):** Todreas, N. E., & Kazimi, M. S. (2012). *Nuclear Systems Volume I: Thermal Hydraulic Fundamentals*. CRC Press.
* **공기 냉각(Air Cooling)의 한계점 ($\approx 100 \text{ W/cm}^2$):** Chu, R. C., et al. (2004). *Review of Cooling Technologies for Computer Products*. IEEE Transactions on Device and Materials Reliability.

---

#### 2. Intel's Heat Density Graph (인텔의 열밀도 그래프)
* **배경:** 2000년대 초 인텔(Intel) 패트 겔싱어(Pat Gelsinger)가 발표하여 학계에 충격을 준 유명 그래프.
* **주요 내용:**
  * CPU 집적도가 올라감에 따라 칩의 단위 면적당 발열 밀도가 **핫플레이트(전기주전자) → 원자력 발전소 → Rocket Nozzle(로켓 노즐) → 태양 표면** 순으로 가파르게 상승할 것임을 경고함.
* **의의:** 공랭 방식의 한계(Thermal Wall)를 예견하고, 반도체 차세대 냉각 기술 연구의 기점이 됨.
* **출처:** Gelsinger, P. (2001). *Microprocessors for the New Millennium: Challenges and Opportunities*. IEEE International Solid-State Circuits Conference (ISSCC).
* **내용:** 당시 인텔 CTO였던 패트 겔싱어(Pat Gelsinger)가 Keynote 발표에서 제시한 그래프. 칩의 열밀도가 **핫플레이트(10 W/cm²) $\rightarrow$ 원자로(100 W/cm²) $\rightarrow$ 로켓 노즐(1,000 W/cm²) $\rightarrow$ 태양 표면(6,000 W/cm²)**으로 수렴할 것임을 경고함.

---

#### 3. 냉각 기술 패러다임 전환
* **공랭 (Air Cooling):** 바람으로는 100 W/cm² 이상 처리 불가 $\rightarrow$ 칩 즉시 과열 및 파괴.
* **수랭 (Direct-to-Chip):** 물의 높은 열용량으로 수백 W/cm² 급 열유속 즉각 제거.
* **침매냉각 (Immersion Cooling):** 절연 액체에 서버를 침수시켜 액체 비등(Boiling) 작용으로 극단적 핫스팟 흡수.

## §12. dose-response 종합 — 5중 사슬 + 형상 민감도

**좌표(§7)→면적(§8, 형상검증 §8-b)→열부하(§9)→인구(§10, 폴리곤 민감도 §10-b)→연령(§11)**. 규율: 산업↔온열 상관(§6 — **작업장 노동**으로 판별, 거리 반론 §6-c 기각)을 AIDC에 전용하지 않고, 물리 승온(케임브리지·Sailor `[외부]`) × 우리 기온-온열곡선(1°C당 ×1.6 `[사실]`)만으로 논증.

| 축 | 최고 위험 | 핵심 |
|---|---|---|
| 열부하(ΔT) | 북평2 ~160× · 동해북평(대조) ~49× | 작은 산단+큰 AIDC `[🔖 07-24 북평2 편입·보수분모]` |
| baseline 취약 | 광양 20.7 | 단 3km 거주민 73명(점 기준) — 작업장 노동의 흔적 |
| 절대 노출(점 3km, 하한) | 구미 66,873명 | 도심 붙은 산단 |
| 절대 노출(폴리곤+2km, §10-b) | **울산미포 33.9만** | 여러 갈래로 흩어진 대형 해안 산단(원형도 0.30·hull 채움 34%) — 형상이 지배 |
| 고령 노출 | 울산미포 65+ 6,888명(점3km) | 고령×더위 교차. 구미는 청년도시(시 12.0%) |

**정직성 유지**: ① 점+3km는 **보수 하한** — 형상 반영(§10-b) 시 수 배 ↑ ② 광양은 점(제철 중심)과 폴리곤(서측 단지)이 다른 부분을 잡아 73명~4.6만 범위 `[확인필요]` ③ ambient 주민 경로만(작업장 노동 제외 — AIDC 특성상 타당) ④ 시군구 평균 온열률 사용(산단 인접은 더 더울 것 → 과소) ⑤ 확정 입지는 동해뿐, 나머지는 상한 시나리오 ⑥ 바람·냉각방식 미모델 `[확인필요]`. 개별 폐공장부지(현대제철 인천 등)는 V-World 지오코딩 → `DERIVED_0718_개별폐공장부지_좌표_vworld.csv`.

> **★ 동해북평 역전 (사용자 지적)** `[🔖 2026-07-22]`: 동해북평은 **열부하 밀도**(GW/km²)로는 최위험(광양의 49배)이지만, 그건 **열 생성 강도**일 뿐이다. 케임브리지 표본의 부지 승온은 평균 +2.0~2.1℃·**최대 9.2℃**이고 **규모별 층화는 논문에 없다**(`논문방법론_확인_Sailor_Cambridge_2026-07-22.md` — 이전 "규모 무관 포화" 표현은 기준창(k) 강건성의 과잉 해석이라 정정). 49배 열부하가 그대로 49배 승온이 되진 않겠지만(경계층 혼합), 이 부지는 평균이 아니라 **상단 꼬리 감시 대상**이다. **AIDC 인구 위험**은 footprint(GW→10km)×노출 인구로 결정되는데, 동해북평 10km 인구는 **12.4만**(인천 236만의 5%). → **강도 최고·노출 최저**. 열부하 밀도는 "열원 집중 랭킹"으로만 읽고, 최종 위험은 §11-b footprint×인구로 판단한다.

> **★ 동해북평 명칭 주의** `[🔖 2026-07-24 사용자 지적]`: §10·§11·§14의 '동해북평'은 **북평국가산단**(`북평` type1)이다. GS 2.4GW가 실제로 가는 곳은 **북평제2일반산단**(`북평2` type2)으로, §18에서 별도 측정했다(부지 ΔLST +11.1℃·나지 89%). 초기 dose-response는 국가산단을 **대조·근사**로 쓴 것이며, 실부지 수치는 §18을 우선한다.

## §13. 개별 폐공장 부지 dose-response — 산단 아닌 단일 공장 전환 경로 `[사실]` `[🔖 2026-07-19]`

500GWh 필터는 '산단 클러스터' 질문용이지 '개별 공장 전환 불가'가 아니다. 인사이트코리아 보도의 **철강 폐공장(인천 현대제철·동국제강·포항 심팩)**은 개별 부지 전환 후보 — V-World 지오코딩으로 좌표 확보(§V-World). 이들은 **도심 밀집지**라 고립 산단(광양 3km 73명)과 정반대다. (§10의 `POP·buffer_pop`, §10-b의 `_acc` 재사용.)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# 1. 시군구별 온열질환 및 전력 패널 데이터를 불러와 지역(key)별 여름철 온열질환 발생률(10만 명당) 평균 계산
_panel=pd.read_csv('DERIVED_0718_시군구연도_패널_온열전력.csv',encoding='utf-8-sig')
_rate=_panel.groupby('key')['여름온열률10만'].mean()

# 2. 분석 대상 3개 주요 공장 부지의 [부지명, 경도, 위도, 행정구역] 정보 정의 : 점 기반 버퍼 분석 → 형상 미반영
SITES=[('현대제철 인천',126.64432,37.48593,'인천광역시 동구'),
       ('동국제강 인천',126.64489,37.48324,'인천광역시 동구'),
       ('심팩 포항',129.37480,35.99267,'경상북도 포항시')]

# 3. 위경도 좌표(EPSG:4326)를 기반으로 점(Point) 객체를 만들고, 거리 계산용 미터 단위 좌표계(UTM-K, EPSG:5179)로 변환
_g=gpd.GeoDataFrame(SITES,columns=['name','lon','lat','key'],
    geometry=[Point(lo,la) for _,lo,la,_ in SITES],crs=4326).to_crs(5179)

# 4. [🔖 2026-07-24] 전국 상수 f=1.6²−1 → 부지별 f_site()(§9-e). BETA_SITE에 없는 시군구는 전국값 폴백.
f=HEAT_MULT_PER_C**2-1     # 전국 참조값(대조 전용)

# 5. 개별 부지별 분석 결과 테이블 헤더 출력
# [🔖 2026-07-25 사용자 지적] 1·2·3km에 같은 +2℃를 주지 않고 링마다 다른 ΔT를 적용한다(§9-e 엔진).
#   추가온열 칸 = 그 반경까지 누적, (괄호) = 그 링에 적용된 f_s.
print(f"{'부지':13}{'시군구':7}{'온열률':>6}{'1km':>8}{'2km':>8}{'3km인구':>9}{'3km65+':>8}{'노인비':>6}{'~1km':>11}{'~2km':>11}{'~3km':>11}")

# 6. 각 부지별로 1km, 2km, 3km 버퍼 내 인구 및 예측 추가 환자 수 계산 후 출력
for _,r in _g.iterrows():
    rt = _rate.get(r.key, float('nan'))  # 해당 시군구의 온열질환 발생률 매핑
    t1=buffer_pop(r.geometry,1000)['tot']; t2=buffer_pop(r.geometry,2000)['tot']
    res3=buffer_pop(r.geometry,3000,extra_map=_acc); t3=res3['tot']; e3=res3['e']

    # 부지별 결과 한 줄 출력 (부지명, 시군구, 온열률, 1k/2k/3k인구, 65+인구, 노인비율, 연간 추가온열건수)
    # 링 증분(0-1·1-2·2-3km)에 각 링 중점의 ΔT를 적용해 누적한다
    _run=0.0; _prev=0.0; _cells=[]
    for _p,_mid in ((t1,0.5),(t2,1.5),(t3,2.5)):
        _run+=excess_ring(r.key,_p-_prev,rt,_mid); _prev=_p
        _cells.append(f"{_run:.2f}({f_site(r.key,dT_at(_mid)):.2f})")
    print(f"{r['name']:13}{r.key.split()[-1]:7}{rt:>6.1f}{int(t1):>8,}{int(t2):>8,}{int(t3):>9,}{int(e3):>8,}{e3/t3*100:>5.1f}%"
          +''.join(f"{_c:>11}" for _c in _cells))

# --- 인천 동구 두 부지(현대제철, 동국제강) 간 공간 중복 제거 및 클러스터 통합 분석 ---

# 7. 인천 동구에 위치한 두 점(Point) 객체를 하나로 합침 (두 지점 간 거리가 340m로 매우 가까움)
_inch=_g[_g.key=='인천광역시 동구'].geometry.union_all()

# 8. 합쳐진 지점으로부터 3km 버퍼 영역 생성
# 두 점을 감싸는 하나의 커다란 3km 통합 외곽선(하나의 뚱뚱한 알약 모양)을 만듭니다.
_b=_inch.buffer(3000)

# 9. 공간 데이터 파일(SHP)에서 버퍼 영역(_b) 바운딩 박스 내에 들어오는 집계구(OA) 데이터만 빠르게 읽어옴
import pyogrio
_oa=pyogrio.read_dataframe(SHP,bbox=_b.bounds)

# 10. 집계구별 전체 인구(p) 및 고령인구(e) 매핑 (결측치는 0.0 처리)
_oa['p']=_oa['TOT_OA_CD'].map(POP).fillna(0.0); _oa['e']=_oa['TOT_OA_CD'].map(_acc).fillna(0.0)

# 11. 면적 가중치(w) 계산: 집계구가 버퍼에 걸치는 면적 비율을 구함 (0~1 사이값)
_w=(_oa.geometry.intersection(_b).area/_oa.geometry.area).clip(0,1)

# 12. 면적 가중치를 적용하여 3km 버퍼 내 최종 중복 제거된 총인구(ct) 및 고령인구(ce) 합산
ct=(_oa['p']*_w).sum(); ce=(_oa['e']*_w).sum()

print(f"\n★ 인천 동구 철강 클러스터(현대+동국 union·중복제거) 3km: 인구 {int(ct):,} · 65+ {int(ce):,} ({ce/ct*100:.1f}%)")
print("  → 도심 폐공장 전환 시 AIDC 열섬이 21만 명에 도달 — 고립 산단(구미 6.7만·광양 73명)보다 큼.")
print("  ⚠ 심팩 포항은 포항국가산단 3km 내 → §10 포항 버퍼와 겹침(중복 주의). 인천은 산단 아닌 개별부지라 §10과 독립.")

부지           시군구       온열률     1km     2km    3km인구  3km65+   노인비       ~1km       ~2km       ~3km
현대제철 인천      동구        5.5  17,705  71,637  187,102  42,417 22.7% 0.26(0.27) 0.94(0.23) 2.13(0.19)
동국제강 인천      동구        5.5  28,485  88,021  206,965  47,312 22.9% 0.43(0.27) 1.17(0.23) 2.40(0.19)
심팩 포항        포항시       9.4     227   5,001   40,458  10,434 25.8% 0.00(0.17) 0.07(0.14) 0.46(0.12)

★ 인천 동구 철강 클러스터(현대+동국 union·중복제거) 3km: 인구 211,112 · 65+ 48,236 (22.8%)
  → 도심 폐공장 전환 시 AIDC 열섬이 21만 명에 도달 — 고립 산단(구미 6.7만·광양 73명)보다 큼.
  ⚠ 심팩 포항은 포항국가산단 3km 내 → §10 포항 버퍼와 겹침(중복 주의). 인천은 산단 아닌 개별부지라 §10과 독립.


### §13-b. 도심 폐공장 후보 확장 — 산업용 하락 역산 `[🔖 2026-07-22 사용자 요청]`

§13의 3개 부지(기사 기반)를 **정량 기준으로 역산**해 후보를 넓힌다. 인천 동구는 산업용 전력이 peak 대비 **−27%**(폐공장 신호). 이 하락률 + 제조업 비중을 기준으로 전국 시군구를 스캔한다.

> ⚠ **한계**: 시군구 단위라 개별 공장 폐업은 확정 못 한다(포항 심팩은 POSCO에 가려 시군구 하락이 안 보임). "후보 풀"이지 확정 리스트가 아니다. 도심형(밀집인구)과 원격형(노출 적음)을 구분해 읽는다.

In [ ]:
# ── §13-b. 도심 폐공장 후보 확장 — 산업용 하락(peak→2025) 역산 [🔖 2026-07-22 신규] ──
# 목적: 전력 소비량의 감소 추이(Peak 대비 2025년 하락폭)를 역산하여, 
#       인천 동구처럼 산업 축소/공장 폐업으로 인해 AIDC 등 신규 열원 시설로 전환 가능한 후보 시군구를 자동 발굴함.

# -----------------------------------------------------------------------------
# 1. 시군구(key) 및 연도별 전력 사용량 피벗 테이블 생성
# -----------------------------------------------------------------------------
# §5, §12에서 불러온 패널 데이터(PW) 재사용
_indN=PW.pivot_table(index='key',columns='연도',values='산업용',aggfunc='sum')       # §5/§12 PW 재사용
_mfgN=PW.pivot_table(index='key',columns='연도',values='제조업',aggfunc='sum')

_resN=[]

# -----------------------------------------------------------------------------
# 2. 시군구별 피크 전력, 2025년 전력, 하락률(%), 제조업 비중(%) 산출
# -----------------------------------------------------------------------------
for _kN in _indN.index:
    _vN=_indN.loc[_kN].dropna()

    # 데이터 시계열이 최소 4년 이상이고, 피크 전력이 200GWh 이상인 산업 규모 보유 지역만 선별
    if len(_vN)<4 or _vN.max()<200: continue                                       # 산업 규모 있는 곳만


    _pk=_vN.max();                      # 과거 최고 산업용 전력 사용량 (Peak)
    _ls=_vN.get(2025,_vN.iloc[-1]);     # 2025년(또는 최신 연도) 산업용 전력 사용량
    _dc=(_ls-_pk)/_pk*100               # 피크 대비 전력 하락률 (%)

    # 2025년 기준 전체 산업용 전력 중 제조업이 차지하는 비중 (%)
    _msN=(_mfgN.loc[_kN].get(2025,float('nan'))/_ls*100) if (_kN in _mfgN.index and _ls>0) else float('nan')

    _resN.append((_kN,round(_pk),round(_ls),round(_dc,1),round(_msN,1)))

# 산출 결과를 데이터프레임으로 변환
_RN=pd.DataFrame(_resN,columns=['key','peak','y2025','하락%','제조비%'])

# -----------------------------------------------------------------------------
# 3. 기준 지점(인천 동구) 조건 기반 폐공장/산단 전환 후보지 필터링
# -----------------------------------------------------------------------------
# 벤치마크 기준점: '인천광역시 동구'의 산업용 전력 하락률 추출
_refN=_RN[_RN.key=='인천광역시 동구']['하락%'].iloc[0]

# 필터링 조건: 
# 1) 인천 동구보다 하락률이 크거나 같은 곳 (하락% <= _refN, 음수값이므로 더 작은 값이 더 큰 하락)
# 2) 제조업 비중이 40% 초과인 곳
_candN=_RN[(_RN['하락%']<=_refN)&(_RN['제조비%']>40)].sort_values('하락%')


print(f'인천 동구 기준: 산업용 하락 {_refN:.0f}% 이상 + 제조업>40% → 전환 후보 {len(_candN)}곳 (3→{len(_candN)}+ 확장):')
print(_candN.head(15).to_string(index=False))

# -----------------------------------------------------------------------------
# 4. 검증 및 결과 해석 가이드
# -----------------------------------------------------------------------------
# [🔖 2026-07-25 사용자 지적] "원격형=노출 적음"은 **재보지 않고 쓴 문장이었다.** 위 표에는
#   하락%·제조비%만 있고 인구를 재지 않았다. 노출은 부지 좌표가 있어야 3km 버퍼로 잴 수 있는데
#   이 후보들은 시군구 단위 전력 통계에서 나온 것이라 개별 공장 좌표가 없다 → 노출 판정 불가.
print('→ 이 표는 **후보 목록**까지다. "도심형/원격형" 같은 노출 판정은 여기서 하지 않는다 —')
print('  부지 좌표가 없어 3km 버퍼를 못 재기 때문이다(재려면 §13처럼 개별 공장 지오코딩이 선행돼야 한다).')
print('  기존 8 산단 중 당진·광양·동해도 자동 검출됨(교차검증). ⚠ 개별 공장 폐업 확정은 별도 자료 필요.')

인천 동구 기준: 산업용 하락 -27% 이상 + 제조업>40% → 전환 후보 9곳 (3→9+ 확장):
        key  peak  y2025   하락%  제조비%
   경상북도 봉화군  1825    992 -45.6  98.4
강원특별자치도 동해시  1988   1159 -41.7  89.3
   충청남도 당진시  9305   5433 -41.6  95.9
    경기도 이천시  7277   4386 -39.7  97.3
   전라남도 장성군   358    235 -34.5  86.6
 부산광역시 해운대구   227    150 -34.1  47.8
강원특별자치도 정선군   269    190 -29.2  85.5
   경상남도 고성군   484    343 -29.1  47.0
   인천광역시 동구  3507   2548 -27.4  98.5
→ 이 표는 **후보 목록**까지다. "도심형/원격형" 같은 노출 판정은 여기서 하지 않는다 —
  부지 좌표가 없어 3km 버퍼를 못 재기 때문이다(재려면 §13처럼 개별 공장 지오코딩이 선행돼야 한다).
  기존 8 산단 중 당진·광양·동해도 자동 검출됨(교차검증). ⚠ 개별 공장 폐업 확정은 별도 자료 필요.


## §14. Landsat 지표면온도(LST) — 산단 열섬 **직접 측정** `[사실]` `[🔖 2026-07-20]`

§6에서 **산업전력↔온열 상관은 (a)작업장 노동**으로 판별됐다. 그래서 그 상관을 AIDC 열섬 논증에 전용할 수 없었고(§12 규율), 물리 승온은 **외부 문헌 가정**(케임브리지·Sailor)에만 의존했다. §14는 그 공백을 메운다 — **우리 데이터로 "산단이 실제로 주변보다 뜨거운가"를 직접 측정.**

**자료**: USGS Landsat 8/9 Collection 2 **Level-2** (`L2SP`), 여름(6~8월) 장면. 밴드 4종만 사용 — `ST_B10`(지표면온도) · `QA_PIXEL`(품질비트) · `ST_QA`(불확실도) · `MTL.txt`(촬영시각·구름량).
> ⚠ 원본 위성영상은 **git 제외**(장면당 ~80MB). 코드는 git 공유, 데이터는 USGS 재다운로드 — 기존 방법론 유지. 재취득 경로는 `.gitignore` 주석 참조.

**설계 3원칙**
1. **동일 장면 내 비교** — LST 절대값은 날짜·시각·대기에 따라 크게 변한다. 그래서 산단 반경별 값을 **같은 장면의 원거리 링(10~20km) 중앙값**과 뺀 **ΔLST**만 쓴다. 같은 시각·같은 대기라 교란이 상쇄된다.
2. **바다 마스킹 필수** — 대상 8곳 중 광양·여수·울산·온산·포항·동해가 해안. 바다는 육지보다 훨씬 차가워 마스킹 안 하면 ΔT가 부풀려진다. `QA_PIXEL` bit7(water)로 제거.
3. **구름·그림자·눈·고불확실도 제거** — bit0 fill, bit1 dilated cloud, bit2 cirrus, bit3 cloud, bit4 shadow, bit5 snow + `ST_QA > 4K`.

**환산 상수** (USGS Data Format Control Book): `℃ = DN × 0.00341802 + 149.0 − 273.15`, `ST_QA(K) = DN × 0.01`.

In [58]:
# ── §14-a. 장면 인벤토리 ──
import os, re, glob, numpy as np, rasterio
from rasterio.windows import from_bounds
from pyproj import Transformer
LSD='LANDSAT'                      # 원본 위성영상 폴더 (git 제외)

ST_SCALE, ST_OFF = 0.00341802, 149.0     # ST_B10 DN → Kelvin
STQA_SCALE = 0.01                         # ST_QA DN → Kelvin(불확실도)
QA_BITS = {'fill':0,'dilated':1,'cirrus':2,'cloud':3,'shadow':4,'snow':5,'clear':6,'water':7}

def scene_list():
    out=[]
    for p in sorted(glob.glob(f'{LSD}/*_ST_B10.TIF')):
        b=p[:-11]; nm=b.replace('\\','/').split('/')[-1]
        m=re.search(r'(LC0[89])_L2SP_(\d{3})(\d{3})_(\d{8})_\d{8}_02_(T\d)$', nm)
        if m and all(os.path.exists(b+s) for s in ['_QA_PIXEL.TIF','_ST_QA.TIF','_MTL.txt']):
            out.append(dict(base=b,sat=m.group(1),path=m.group(2),row=m.group(3),
                            date=pd.to_datetime(m.group(4)),tier=m.group(5)))
    return pd.DataFrame(out)

def mtl_meta(base):
    """MTL.txt → 촬영시각(UTC)·구름량. KST = UTC+9 (Landsat 통과 ≈ 오전 11시)"""
    d={}
    for ln in open(base+'_MTL.txt',encoding='utf-8',errors='ignore'):
        if '=' not in ln: continue
        k,v=[x.strip().strip('"') for x in ln.split('=',1)]
        if k in ('SCENE_CENTER_TIME','DATE_ACQUIRED','CLOUD_COVER','CLOUD_COVER_LAND'): d[k]=v
    return d

SC=scene_list()
print('장면:',len(SC),' 위성:',SC.sat.value_counts().to_dict(),' Tier:',SC.tier.value_counts().to_dict())
print('Path/Row별:',{f'{p}/{r}':n for (p,r),n in SC.groupby(['path','row']).size().items()})
print('월별:',SC.date.dt.month.value_counts().sort_index().to_dict(),'| 연도별:',SC.date.dt.year.value_counts().sort_index().to_dict())
if len(SC):
    _m=mtl_meta(SC.iloc[0].base); _h=int(_m['SCENE_CENTER_TIME'][:2])+9
    print(f"촬영시각 예: {_m['SCENE_CENTER_TIME'][:8]} UTC = {_h}시 KST → ⚠ 오전 통과(일최고기온 14~16시 아님)")

장면: 101  위성: {'LC08': 53, 'LC09': 48}  Tier: {'T1': 97, 'T2': 4}
Path/Row별: {'114/034': 20, '114/035': 17, '114/036': 15, '115/034': 12, '115/035': 10, '115/036': 15, '116/034': 12}
월별: {6: 34, 7: 16, 8: 40, 9: 11} | 연도별: {2020: 13, 2021: 4, 2022: 8, 2023: 20, 2024: 35, 2025: 21}
촬영시각 예: 01:58:44 UTC = 10시 KST → ⚠ 오전 통과(일최고기온 14~16시 아님)


**§14-a2. 위성 장면 추가 다운로드 스펙 (self-contained)** `[🔖 2026-07-23 · 사용자 요청]`

더 받을 때 아래 조건 그대로 (USGS EarthExplorer → earthexplorer.usgs.gov):

| 항목 | 조건 |
|---|---|
| Data Set | Landsat → Collection 2 Level-2 → **Landsat 8-9 OLI/TIRS C2 L2** (`L2SP`) |
| 기간 | 2020-01-01 ~ 2025-12-31, **6~8월만** (Search months) |
| Tier | **T1** (T2 제외) |
| Cloud Cover | **< 30%** |
| 필요 파일 | `ST_B10`, `ST_QA`, `QA_PIXEL`, `SR_B4`, `SR_B5`, `MTL.txt` (6종 — Band Files 개별 선택 가능) |
| 저장 위치 | `LANDSAT/` 폴더 (파일명 그대로 — §14-a `scene_list()`가 자동 수집) |

부지 → WRS-2 Path/Row (현 보유 장면 bounds로 실측한 매핑):

| 부지 그룹 | Path/Row |
|---|---|
| 인천(현대·동국·KG) | 116/034 |
| 심팩·포항제철·광명 | 114/035 |
| 구미 | 114/035, 115/035 |
| 울산미포·온산·하이테크 | 114/035, 114/036 |
| 당진 | 115/035, 116/034 |
| 동해북평·북평2 | 114/034, 115/034 |
| 광양·여수 | 114/036, 115/036 |

현 보유: 101장 (§14-a 출력 참조). 인천 장면(10개)을 늘리려면 위 인천 행의 path/row로 조건 검색.

**§14-b. LST 변환 + 마스킹 + 반경별 추출**
전장면은 7,811×7,671 = **6천만 화소**라 90장 전부 읽으면 느리다. 산단 중심 ±21km **window만** 읽는다(`boundless=True`로 장면 경계 넘어가도 안전).
> `기준n`은 10~20km 링의 **유효(육지·무운) 화소 수**. 이게 5,000 미만이면 기준선이 불안정하다고 보고 그 장면은 버린다(`site_scene_dT(min_ref=5000)`에 구현돼 있다).

> `[🔖 2026-07-25]` **이 문턱은 보수적이지 않고 오히려 관대하다.** 10~20km 링의 면적은 π(20²−10²) ≈ **942km²**이고 Landsat 화소는 30m(=900m²)이므로 링 하나에 이론상 **약 100만 화소**가 들어간다. 5,000은 그중 **0.5%**다. 즉 구름·바다로 99.5%가 가려진 장면만 버린다는 뜻이라, 문턱을 올릴 여지는 있어도 이 값 때문에 장면이 과도하게 버려지지는 않는다.

In [59]:
# ── §14-b. LST 변환 + 마스킹 + 링 추출 (형상 기반) ──
# [정정 07-20] 이전 판은 좌표를 손으로 근사 입력해 실제 산단에서 1.1~8.2km 빗나갔다(당진 8.2·포항 3.1·여수 3.1).
#   0~1km 링을 재는데 3km를 빗나가면 다른 장소를 측정한 것 → 여수가 -2.04℃(음수)로 나온 원인.
#   이제 §7과 같은 PDAN/YUCH 실형상에서 코드로 유도한다. 손입력 좌표 금지.
# [정정 07-20] 링 기준도 '중심점 반경'에서 '폴리곤 경계 바깥 거리'로 변경 — §10-b의 교훈(열은 중심이 아니라 산업시설
#   경계에서 퍼진다. 울산미포처럼 여러 갈래로 흩어져 있으면 중심 반경은 산단 안팎을 뒤섞는다). YUCH 폴리곤이 없는 곳만 점 버퍼.

""" Block 1. 좌표계 변환 도우미 및 버퍼 링/기준선(Reference) 구간 설정 """
from rasterio.features import geometry_mask
from shapely.ops import transform as shp_transform

# 1. 입력 도형(geom)의 좌표계를 라스터 영상 좌표계(dst_crs)로 투영 변환하는 도우미 함수
def _to_scene(geom, src_epsg, dst_crs):
    tf=Transformer.from_crs(src_epsg, dst_crs, always_xy=True).transform
    return shp_transform(tf, geom)

# 2. 산단 경계 바깥으로 확장할 동심원 링 구간(m) 정의 (0~1km, 1~2km, 2~3km, 3~5km, 5~10km)
RINGS=[(0,1000),(1000,2000),(2000,3000),(3000,5000),(5000,10000)]   # 경계 바깥 거리(m)

# 3. 동일 장면 내 배경 대조군(Reference) 기온 기준선 구간 (10km ~ 20km 영역, 시각/대기조건 상쇄용)
REF=(10000,20000)        # 동일장면 기준선 — 같은 시각·같은 대기라 교란 상쇄


""" Block 2. 라스터 파일 오픈, 윈도우 크롭 및 품질 마스킹(구름·바다·불확실도 제거) """
def site_scene_dT(base, geom5179, is_poly, min_ref=5000, min_px=50, unc_max=4.0):
    # geom5179: YUCH union 폴리곤(또는 점) — EPSG:5179
    # is_poly=True면 R0(폴리곤 내부)도 산출.
    
    # 1. Landsat 지표면 온도 밴드(ST_B10.TIF) 오픈 및 좌표 변환
    with rasterio.open(base+'_ST_B10.TIF') as s:
        g = _to_scene(geom5179, 5179, s.crs)  # 산단 도형을 위성 영상 좌표계로 변환
        gb = g.bounds                          # 도형의 바운딩 박스 추출

        # 2. 산단 중심점이 위성 장면(Scene) 바운딩 박스 밖에 있으면 예외 처리
        if not (s.bounds.left < (gb[0]+gb[2])/2 < s.bounds.right and
                s.bounds.bottom < (gb[1]+gb[3])/2 < s.bounds.top): return {'사유':'장면 bbox 밖'}
        
        # 3. 기준선 영역(REF=20km)까지 포함하도록 읽어올 창(Window) 범위 설정 및 라스터 배열 로드
        w  = from_bounds(gb[0]-REF[1]-1000, gb[1]-REF[1]-1000, gb[2]+REF[1]+1000, gb[3]+REF[1]+1000, s.transform)
        st = s.read(1,window=w,boundless=True,fill_value=0).astype('float64')
        tr = s.window_transform(w); shp=st.shape

    # 4. 품질 assessment 밴드(QA_PIXEL) 및 오차/불확실도 밴드(ST_QA) 로드
    with rasterio.open(base+'_QA_PIXEL.TIF') as s: 
        qa=s.read(1,window=w,boundless=True,fill_value=1)
    with rasterio.open(base+'_ST_QA.TIF')   as s: 
        sq=s.read(1,window=w,boundless=True,fill_value=0).astype('float64')

    # 5. Raw DN 값을 섭씨온도(℃)로 변환: ST = DN * Scale + Offset - 273.15
    lst = st*ST_SCALE + ST_OFF - 273.15                       # DN → ℃
    
    # 6. 불량 화소(bad) 마스크 생성: 무자료(0), 구름, 그림자, 눈 등 비트 마스크 제거
    bad = (st==0)                                             # fill(무자료)
    for k in ('fill','dilated','cirrus','cloud','shadow','snow'): 
        bad |= ((qa>>QA_BITS[k])&1)>0
    
    # 7. 해안 산단 비정상 수치 방지를 위한 수면(바다/강) 화소 제거
    bad |= ((qa>>QA_BITS['water'])&1)>0                       # ★ 바다 제거 — 해안 산단 6곳 필수

    # 8. 불확실도(ST_QA)가 4.0K 초과인 저품질 화소 제거
    bad |= (sq*STQA_SCALE > unc_max)                  

    # 9. 마스킹 대상 화소들을 NaN(결측치)으로 변환        
    lst = np.where(bad, np.nan, lst)


    """ Block 3. 공간 마스크 생성 (산단 내부, 동심원 링, 배경 기준선) """
    # 1. Shapely 도형을 라스터 불리언 마스크 배열로 변환하는 익명 함수 정의
    inside=lambda gg: geometry_mask([gg], out_shape=shp, transform=tr, invert=True)
    core = inside(g)
    
    # 2. 위성 영상 기울어짐/테두리 영역(no-data 모서리) 예외 처리
    # Landsat 장면은 직사각 GeoTIFF 안의 '기울어진 평행사변형' — bbox 안이라도 촬영범위 밖(no-data 모서리)일 수 있다.
    if core.any() and (st[core]==0).mean()>0.9:
        return {'사유':'촬영범위 밖(no-data 모서리)'}

    # 3. 거리별 동심원 링 마스크 생성 (outer_ring & ~inner_ring)    
    ringmask={}
    prev=core if is_poly else None
    for a,b in RINGS:
        outer=inside(g.buffer(b)); inner=inside(g.buffer(a)) if a>0 else (core if is_poly else np.zeros(shp,bool))
        ringmask[f'{a//1000}-{b//1000}km']=outer&~inner

    # 4. 배경 대조군(10~20km) 링 마스크 생성
    rf=inside(g.buffer(REF[1]))&~inside(g.buffer(REF[0]))


    """ Block 4. 상대적 온도차($\Delta T$) 산출 및 최종 결과 반환 """
    # 1. 배경 기준선(10~20km) 내 유효 픽셀 온도 추출
    ref=lst[rf]
    ref=ref[~np.isnan(ref)]
    
    # 2. 유효 픽셀 수가 최소 수량(min_ref=5000개) 미만이면 실패 처리 (구름/바다 등에 의한 차폐)    
    if len(ref)<min_ref: 
        return {'사유':'기준선 유효화소 부족(구름·바다)'}
    
    # 3. 배경 기준선 중위수(Median) 온도 산출
    rb=float(np.median(ref))
    out={'사유':'OK','기준선C':round(rb,1),'기준n':len(ref)}
    
    # 4. [산단 내부 R0] 배경 기준선 대비 상대적 온도 편차(ΔT) 계산
    if is_poly:
        v=lst[core]; v=v[~np.isnan(v)]
        out['ΔR0내부']=round(float(np.mean(v))-rb,2) if len(v)>=min_px else np.nan

    # 5. [거리별 링] 각 링 구간별 상대적 온도 편차(ΔT) 및 유효 화소 비율 산출
    for lab,mk in ringmask.items():
        v=lst[mk]; v=v[~np.isnan(v)]
        out['Δ'+lab]=round(float(np.mean(v))-rb,2) if len(v)>=min_px else np.nan
        out['유효'+lab]=round(len(v)/max(mk.sum(),1),2)

    return out

# 6. 함수 정의 완료 안내 문구 출력
print('함수 정의 완료: site_scene_dT (형상 기반 · YUCH 폴리곤 우선)')

함수 정의 완료: site_scene_dT (형상 기반 · YUCH 폴리곤 우선)


**§14-c. 8개 산단 + 개별 폐공장 부지 ΔLST**
§7의 `TG`·§13의 `SITES`와 같은 좌표를 쓴다(광양은 §6-c와 동일하게 PDAN 점이 바다라 YUCH 제철소 중심으로 보정된 값 사용).
장면 구름량 30% 초과는 제외 — 부분운은 QA 비트로 걸러지지만 잔여 얇은 권운이 ΔT를 왜곡한다.

In [ ]:
# ── §14-c1. 대상 형상 유도 — 8개 산단 + 개별 부지 3곳 [🔖 2026-07-23 분할: 형상(빠름)/루프(느림) 분리] ──
from shapely.geometry import Point as _P
CLOUD_MAX=30.0
JJA_ONLY=True     # §3·§4와 동일하게 6~8월만. 9월 포함해도 0-1km ΔT 최대 0.14℃ 차 → 결론 불변

# (1) 8 산단 — YUCH 실폴리곤 우선, 없으면 PDAN 점.
# [🔖 2026-07-25 사용자 지시] 북평2를 여기 정식 편입한다.
#   앞서는 "§18-a가 같은 기계로 이미 쟀으니 중복"이라고 뺐지만, **처음 보는 사람이 이 노트북만으로
#   전 결과를 얻을 수 있어야 한다**는 원칙이 우선이다(§18은 신규 후보 1차 실측 기록으로 남는다).
#   ※ '동해북평'(국가산단)은 대조로 함께 남는다 — 둘은 1.2km 떨어진 별개 산단.
_TG=[('광양·제철','광양','1'),('여수·석화','여수','1'),('울산미포','울산·미포','1'),('온산','온산','1'),
     ('포항·제철','포항','1'),('구미·전자','구미(2·3단지)','1'),('당진1철강','당진1철강','2'),
     ('동해북평','북평','1'),('동해 북평2','북평2','2')]
GEO={}
for lbl,nm,ty in _TG:
    r=PDAN[(PDAN.DAN_NAME==nm)&(PDAN.DANJI_TYPE==ty)].iloc[0]
    yc=YUCH[YUCH.DAN_ID==r.DAN_ID]
    if len(yc):
        # 산업시설 실폴리곤(YUCH)이 있으면 union — 링은 이 경계 '바깥' 거리 기준
        GEO[lbl]=(yc.to_crs(5179).geometry.union_all(), True)
    else:
        # 폴리곤이 없는 곳(포항·여수)은 PDAN 대표점 기준
        GEO[lbl]=(gpd.GeoSeries([_P(r.lon,r.lat)],crs=4326).to_crs(5179).iloc[0], False)

# (2) 개별 폐공장 부지 (§13, V-World 지오코딩) — 폴리곤 없음 → 점
for nm,lo,la in [('현대제철 인천',126.64432,37.48593),('동국제강 인천',126.64489,37.48324),('심팩 포항',129.37480,35.99267)]:
    GEO[nm]=(gpd.GeoSeries([_P(lo,la)],crs=4326).to_crs(5179).iloc[0], False)
print('대상:',len(GEO),'| 폴리곤 기준:',[k for k,(g,p) in GEO.items() if p])
print('점 기준(YUCH 없음·개별부지):',[k for k,(g,p) in GEO.items() if not p])

대상: 12 | 폴리곤 기준: ['광양·제철', '울산미포', '온산', '구미·전자', '당진1철강', '동해북평', '동해 북평2']
점 기준(YUCH 없음·개별부지): ['여수·석화', '포항·제철', '현대제철 인천', '동국제강 인천', '심팩 포항']


In [ ]:
# ── §14-c2. 부지×장면 ΔLST 루프 — 부지별 진행 출력 [🔖 2026-07-23 분할 · 진행 print 추가] ──
# 오래 걸리는 셀(수 분): 부지 하나 끝날 때마다 진행 상황을 찍는다 (V9.4 — 무출력 장기 실행 금지).
rec=[]; why=[]
_tot=len(GEO)
for _k,(nm,(g,is_poly)) in enumerate(GEO.items()):
    _ok0=len(rec)
    for _,s in SC.iterrows():
        if JJA_ONLY and s.date.month not in (6,7,8):
            continue
        m=mtl_meta(s.base)
        if float(m.get('CLOUD_COVER',100))>CLOUD_MAX:
            why.append((nm,f'{s.path}/{s.row}','장면 구름>%d%%'%CLOUD_MAX))
            continue
        r=site_scene_dT(s.base,g,is_poly)
        why.append((nm,f'{s.path}/{s.row}',r['사유']))
        if r['사유']!='OK':
            continue
        rec.append(dict(부지=nm,형상='폴리곤' if is_poly else '점',날짜=s.date.date(),위성=s.sat,tier=s.tier,
                        구름pct=float(m.get('CLOUD_COVER',np.nan)),KST=f"{int(m['SCENE_CENTER_TIME'][:2])+9}시",
                        **{k:v for k,v in r.items() if k!='사유'}))
    print(f'[{_k+1}/{_tot}] {nm}: 유효 {len(rec)-_ok0}장면 (누적 {len(rec)})')
LST=pd.DataFrame(rec)
_dc=(['ΔR0내부'] if 'ΔR0내부' in LST.columns else [])+[f'Δ{a//1000}-{b//1000}km' for a,b in RINGS]

# 커버리지 진단 — 어느 부지가 왜 비었나
W=pd.DataFrame(why,columns=['부지','PathRow','사유'])
print()
print('=== 커버리지 진단 (부지 × 사유) ===')
print(W.pivot_table(index='부지',columns='사유',aggfunc='size',fill_value=0).to_string())
_none=[n for n in GEO if n not in set(LST.부지)] if len(LST) else list(GEO)
if _none:
    print()
    print('⚠ 아직 유효장면 0인 부지:',_none)

# [🔖 2026-07-23 사용자 지시] 아래 결과 출력 블록 전체 주석화 — 이상치 없음 확인 완료.
#   결과표는 별도 파일로 이관: 실행결과_14c_산단LST_2026-07-23.md (재현 시 아래 주석 해제)
# if len(LST):
#     agg=LST.groupby(['부지','형상']).agg(장면=('날짜','size'),**{c:(c,'mean') for c in _dc}).round(2)
#     print()
#     print('=== 부지별 평균 ΔLST (℃, 기준=동일장면 경계+10~20km 링 중앙값) ===')
#     print(agg.sort_values('Δ0-1km',ascending=False).to_string())
#     print()
#     print('=== 장면별 원자료 (재현·이상치 확인용) ===')
#     print(LST[['부지','형상','날짜','위성','KST','구름pct','기준선C']+_dc].to_string(index=False))
#     # [🔖 2026-07-23] CSV 재저장 비활성 — 이상치 없음 확인 완료, 매 실행 재저장은 자원 낭비 (사용자 지시).
#     #   재현이 필요하면 아래 두 줄 주석 해제.
#     # LST.to_csv('DERIVED_0720_산단_LST_반경별.csv',index=False,encoding='utf-8-sig')
#     # print('저장: DERIVED_0720_산단_LST_반경별.csv')
#     print()
#     print('※ 해석 규약(점/폴리곤 비교 금지·LST≠기온 등)은 아래 「§14 결과 해석」 md 참조 — 코드 중복 출력 제거(07-23).')
print()
print('※ 부지별 평균·장면별 원자료 표 → 실행결과_14c_산단LST_2026-07-25.md 로 이관. 원자료 CSV: DERIVED_0720_산단_LST_반경별.csv')
if len(LST):
    _agg16=LST.groupby(['부지','형상']).agg(장면=('날짜','size'),**{c:(c,'mean') for c in _dc}).round(2)
    print()
    print('=== 부지별 평균 ΔLST (℃, 기준=동일장면 경계+10~20km 링 중앙값) — 북평2 포함 12부지 ===')
    print(_agg16.sort_values('Δ0-1km',ascending=False).to_string())

[1/12] 광양·제철: 유효 13장면 (누적 13)
[2/12] 여수·석화: 유효 13장면 (누적 26)
[3/12] 울산미포: 유효 16장면 (누적 42)
[4/12] 온산: 유효 26장면 (누적 68)
[5/12] 포항·제철: 유효 16장면 (누적 84)
[6/12] 구미·전자: 유효 25장면 (누적 109)
[7/12] 당진1철강: 유효 10장면 (누적 119)
[8/12] 동해북평: 유효 26장면 (누적 145)
[9/12] 동해 북평2: 유효 26장면 (누적 171)
[10/12] 현대제철 인천: 유효 10장면 (누적 181)
[11/12] 동국제강 인천: 유효 10장면 (누적 191)
[12/12] 심팩 포항: 유효 16장면 (누적 207)

=== 커버리지 진단 (부지 × 사유) ===
사유       OK  기준선 유효화소 부족(구름·바다)  장면 bbox 밖  촬영범위 밖(no-data 모서리)
부지                                                             
광양·제철    13                   0         63                   14
구미·전자    25                   0         65                    0
당진1철강    10                   1         70                    9
동국제강 인천  10                   1         79                    0
동해 북평2   26                   1         63                    0
동해북평     26                   1         63                    0
심팩 포항    16                   0         74                    0
여수·석화    13                

### §14 결과 해석 — 반드시 [사실]과 [해석]을 나눈다 `[🔖 2026-07-20]`

> **정정 이력**: 최초판은 좌표를 손입력해 실제 산단에서 1.1~8.2km 빗나갔다(당진 8.2·포항 3.1·여수 3.1km). 증상은 여수 **−2.04℃**(음수). §7과 같은 PDAN/YUCH 실형상에서 코드로 유도하도록 정정했고, 링 기준도 §10-b 교훈대로 **폴리곤 경계 바깥 거리**로 바꿨다. 아래 수치는 정정판.

#### ✅ 측정된 사실 `[사실]`
**공장부지 자체가 같은 장면의 원거리(경계+10~20km)보다 +9~17℃ 뜨겁다.**

`[🔖 2026-07-24 통합]` §18에서 따로 잰 신규 5부지를 같은 표로 합쳤다(같은 기계·같은 규약이므로 분리할 이유가 없다). **동해 북평2가 GS 2.4GW 실부지**이고 `동해북평`은 인접 국가산단 대조다.

| 부지 | 기준 | 공장부지 | 0-1km | 2-3km | 5-10km | 장면 | 비고 |
|---|---|---|---|---|---|---|---|
| 포항·제철 | 점 | **+17.3** | +17.3 | +10.6 | +3.1 | 16 | |
| 심팩 포항 | 점 | +14.3 | +14.3 | +9.6 | +2.1 | 16 | 착공 임박 |
| 구미·전자 | 폴리곤 | **+14.0** | +6.8 | +6.9 | +2.3 | 25 | |
| 당진1철강 | 폴리곤 | +12.6 | +6.7 | +1.8 | +1.1 | 10 | |
| 울산미포 | 폴리곤 | +12.1 | +5.8 | +4.2 | +1.9 | 16 | SK×AWS 착공 |
| 동해북평 (국가·대조) | 폴리곤 | +11.2 | +9.2 | +5.9 | +2.1 | 26 | 대조군 |
| 온산 | 폴리곤 | +11.2 | +4.1 | +2.2 | +2.9 | 26 | |
| **동해 북평2** | 폴리곤 | **+11.1** | +7.6 | +6.0 | +2.1 | 26 | **GS 2.4GW 실부지** (§18) |
| 광양·제철 | 폴리곤 | +10.2 | +6.6 | +5.1 | +0.8 | 13 | |
| 울산 하이테크밸리 | 폴리곤 | +10.1 | +5.9 | +2.3 | +0.5 | 16 | SKT 후보 (§18) |
| 여수·석화 | 점 | +8.5 | +8.5 | +3.7 | +2.3 | 13 | |
| 포항 광명 | 폴리곤 | +8.6 | +1.8 | +2.5 | +0.8 | 16 | 착공식 7/20 (§18) |
| KG스틸 인천 | 점 | — | +7.4 | +2.8 | +0.5 | 10 | 가동 중·서구 (§18) |
| 현대제철 인천 | 점 | +5.9 | +5.9 | +5.1 | +0.6 | 10 | 2026-01 폐쇄 |
| 동국제강 인천 | 점 | +5.5 | +5.5 | +5.2 | +0.6 | 10 | |
| 온산 강양·우봉지구 | 점(근사) | — | +4.1 | +4.3 | +1.7 | 26 | SKT 후보·지구 경계 미확보 (§18) |

거리에 따른 **단조 감쇠**가 모든 부지에서 재현된다. 장면간 표준편차 **1~3℃**(부지당 10~26장, 총 181 부지×장면 조합·53일·LC08 89 + LC09 92). 장면 수를 51→101장으로 두 배 늘려도 순위·크기가 거의 불변 — 표본에 안정적이다.
→ **산업단지가 물리적 국소 열원임이 우리 데이터로 직접 확인됐다.** 더 이상 외부 문헌 가정에만 기대지 않는다.

#### ⛔ 이 숫자를 ×1.6/℃ 곡선에 그대로 넣으면 안 된다 `[해석 금지]`
1. **LST ≠ 기온.** §9-b의 ×1.6/℃는 **ASOS 최고기온** 기반이다. 맑은 날 아스팔트·금속지붕 표면은 기온보다 10~20℃ 높다. 지표 ΔT를 기온 ΔT로 옮기려면 별도 환산이 필요하고, 통상 기온 ΔT는 **훨씬 작다**. **+13℃ 지표차 ≠ +13℃ 기온차.**
2. **기준선이 숲·농지일 수 있다.** 경계+10~20km 링은 대개 산림·농경지라, 이 ΔT에는 *산단 vs 자연*이 섞여 있다. *산단 vs 주거지* 대비였다면 값이 더 작았을 것 → **현재 값은 상한 성격.**
3. **오전 스냅샷이다.** Landsat 통과는 **10~11시 KST**. 온열질환이 몰리는 14~16시가 아니다.
   > ⚠ **2·3 종합 (사용자 지적)**: ΔT엔 **상반된 두 편의**가 있어 단순 "상한"이 아니다. (a) 숲·농지 기준선은 ΔT를 **부풀린다**(상한 방향) — 도시 주거지 기준이면 더 작다. (b) 그러나 오전 스냅샷은 산단-주변 차이가 커지는 **오후 피크를 놓친다**(하한 방향). 두 편의가 반대라 순 방향은 미정. [확인필요]로 각각 정량화 예정.
4. **★ 점 기준과 폴리곤 기준의 `Δ0-1km`는 의미가 다르다.** 폴리곤 부지는 `Δ0-1km`가 경계 *바깥*, 점 부지(YUCH 없는 포항·여수·개별공장)는 사실상 공장 *안*이다. 비교하려면 **점의 `Δ0-1km` ↔ 폴리곤의 `ΔR0내부`**로 대응시켜야 한다. 위 표의 '공장부지 자체' 열이 그 정렬이다.
5. **울산미포는 과소평가 방향** — 도심이 산단에 붙어 원거리 기준선까지 뜨겁다.
   > ⚠ **두 축을 섞지 말 것** `[🔖 2026-07-25 사용자 지적]`: 위 "과소"는 **ΔLST 측정**에 관한 말이다(기준선 링이 이미 더워서 산단−기준선 차이가 작게 나온다). **노출 추정**은 반대 방향이다 — §12-c에서 미포 부지는 1GW 표준 소요의 **10~17배**라 산단 전체가 AIDC 열원이 될 이유가 없는데, §11-b·§12-b는 폴리곤 **전체**를 열원으로 두고 반경을 재므로 **과대** 쪽이다. 즉 *측정된 뜨거움은 과소, 상정한 열원 범위는 과대*. 하한은 §12-d (3) 참조.

#### 🔗 §6과 모순되지 않는다 — 오히려 빈틈을 메운다 `[해석]`
§6은 "산업전력↔온열 상관 = **작업장 노동**, ambient 열섬 아님"이었다. §14는 "산단은 **물리적으로 뜨겁다**"이다. 둘은 양립한다:
> 산단은 확실히 열원이다(§14). 다만 그 열이 **주민 온열질환 통계로 나타나지 않았다**(§6). 이유 후보 — ① 산단 반경에 **주민이 애초에 적다**(§10: 광양 3km 73명) ② 지표 승온이 기온 승온으로 다 전달되지 않는다 ③ 냉방 대처(§6-d에서 약하게 기각).

**AIDC 논증에 주는 함의**: AIDC가 **도심에 붙은 부지**(§13 인천 동구 3km 21만 명)에 들어가면 ①이 깨진다 — 열원은 있는데 노출 인구가 없던 구조가, **열원 + 밀집 인구**로 바뀐다. §14는 그 전제(열원 실재)를 실측으로 세운 것이다.
> 인천 두 부지의 공장부지 ΔT(+6.0~6.4℃)가 8 산단 중 가장 낮은 것도 같은 맥락 — 이미 도심에 둘러싸여 기준선 자체가 뜨겁기 때문이지, 덜 뜨거워서가 아니다.

#### 다음에 해결할 것 `[확인필요]`
- [x] ~~지표→기온 환산~~ → **§15 완료**: β=0.258(AWS 459지점, 회귀희석 보정 0.459). §9 dose-response 실측 기반 확보. `[🔖 2026-07-22]`
- [x] **기준선 재정의** — 숲 기준 대신 *동일 시군구 주거지* 기준 ΔT 병기. **접근**: §17 NDVI로 기준 링(경계+10~20km) 화소를 녹지/주거지/산업으로 층화 → 주거지 화소만 기준으로 ΔT 재계산(도심 산단 과소평가·숲 산단 과대평가 보정). `[🔖 2026-07-22 사용자 SR_B4 제안]` → **§14-d 실행 완료(07-22)**: 주거지(0.15≤NDVI<0.45) 기준 재계산 시 ΔT -6.11℃ 이동(부지별 -8.49~-2.16), 순위 spearman ρ=0.83로 보존 — 결론 구조 불변
- [ ] 포항·여수는 YUCH 폴리곤이 없어 점 기준 — 산업시설 경계 자료 보완 시 재계산
- [x] ~~잔여 장면 수신 후 전체 재집계~~ → 2026-07-20 완료 (101장 전량, Path 114~116 / Row 034~036 전 범위)

> **9월 장면 주석** `[🔖 2026-07-21 사용자 질문]`: §14-a 인벤토리 출력 `월별 {6:34, 7:16, 8:40, 9:11}`의 9월 11장(2020: 0912·0919·0928, 2023: 0929, 2025: 0903·0911·0926·0927 등)은 **다운로드 인벤토리에 있을 뿐 분석에는 쓰지 않는다.** §14-c `JJA_ONLY=True`(6~8월만) + §15-a/d/e 모두 `month in (6,7,8)` 필터가 걸려 있다. JJA 설계와 충돌 없음. (9월 포함해도 0-1km ΔT 최대 0.14℃ 차 → 결론 불변임을 §14-c 주석에서 이미 확인.)

### §14-d. 주거지 기준선 ΔT 재계산 — "숲 기준 편의" 정량화 (`[확인필요]` #2 해소) `[🔖 2026-07-22 신규 · 07-23 설명 재작성]`

**무엇을 구하나 (GIS 비전공 독자용)** — "부지가 주변보다 +N℃ 뜨겁다"라고 할 때, 그 **'주변'을 무엇으로 잡느냐**가 문제다.

1. **기존(§14-c)**: 부지 밖 10~20km 도넛 안의 **모든** 화소(숲·논밭 포함)의 중앙값을 기준으로 씀. 숲은 차가우므로 → 기준이 낮아져 → 부지가 실제보다 **더 뜨거워 보인다** (상한 방향 편의).
2. **여기(§14-d)**: 같은 도넛에서 **"사람 사는 동네 같은 화소"만 골라** 그 중앙값을 기준으로 다시 잰다. 부지 쪽 값은 그대로 두고 **기준선만 교체**. → "숲보다 +14℃"가 "**동네보다 +7℃**"로 바뀌는 것.
3. "동네 같은 화소"를 고르는 도구가 **식생지수(NDVI)** = (근적외−빨강)/(근적외+빨강). 식물은 근적외를 강하게 반사하고 빨강을 흡수하므로: 녹지 ≈ 0.5~0.9, 건물+가로수 섞인 동네 ≈ 0.15~0.45, 콘크리트·맨땅 ≈ 0.15 미만, 물은 음수.

같은 장면·같은 도넛 안에서만 층화하므로 시각·대기 교란은 그대로 상쇄된다. SR 밴드가 없는 장면은 제외(§14-c보다 장면 수가 적을 수 있음 — 검증 열에서 대조).

| 층 | NDVI | 근거 |
|---|---|---|
| 불투수·산업 | < 0.15 | 정종철·손주형(2025) 임계 — §17과 동일 차용 |
| **주거지 대리** | **0.15 ≤ NDVI < 0.45** | 건물+가로수 혼합 도시조직. **휴리스틱** — 상한 0.40/0.50 민감도 병기 |
| 녹지 | ≥ 0.45 | 수관·농지 |

⚠ NDVI 층화는 토지피복 진리가 아닌 **대리지표**다 — 여름 논은 녹지로, 상업지 나대지는 불투수로 섞인다. 같은 장면·같은 링 안에서의 층화라 시각·대기 교란은 그대로 상쇄된다. SR 밴드가 없는 장면은 제외(장면 수가 §14-c보다 적을 수 있음 — 검증 열에서 대조).

**§14-d0. 기준선을 NDVI 대리 → 토지피복 실측으로** `[🔖 2026-07-25 사용자 지시]`

§14-d는 기준 도넛(10~20km) 안의 "주거지 화소"를 **NDVI 0.15~0.45**로 대리했다 — 편의적 휴리스틱이었고, §17-c가 그 편향을 실측으로 확인했다(NDVI 대리는 실제 불투수를 크게 과소: 온산 49% vs 81%, 광양 37% vs 87%). 그 구간엔 주거지뿐 아니라 **공업·상업 화소가 섞여 들어간다**.

환경부 **토지피복 세분류(2024)**의 `주거지역(L2=110)`·`산림(L1=300)` 폴리곤을 직접 마스크로 쓴다. 대리 결과는 지우지 않고 나란히 남겨 **편향의 크기와 방향**을 보인다.

In [ ]:
# ── §14-d0. 기준 도넛의 **실토지피복** 마스크 생성·캐시 [🔖 2026-07-25 사용자 지시] ──
# NDVI 0.15~0.45 '주거지 대리'를 환경부 토지피복 세분류(2024) 실측으로 교체하기 위한 준비 단계.
#   §17-c가 대리의 편향을 확인했다 — NDVI<0.15 대리는 실제 불투수를 크게 과소(온산 49%vs81%·광양 37%vs87%).
#   즉 0.15~0.45 구간엔 주거지 말고 공업·상업 화소가 섞인다 → 기준선 오염.
# 도넛 하나에 도엽 수십 장이 걸리므로 부지별 union을 WKT로 캐시한다(파일 있으면 건너뜀).
import time as _t14mod                 # [07-25] 노트북 전역엔 time이 없다 — 여기서 직접 임포트
from shapely import wkt as _wkt14
from shapely.geometry import box as _box14
from shapely.ops import unary_union as _uu14

# [🔖 2026-07-25] 면적 필터를 뺀 뒤로 마스크가 커져서 WKT-in-CSV는 수백 MB가 된다.
#   → GeoParquet(WKB+압축)으로 저장한다. 읽기·쓰기도 훨씬 빠르다.
_LCC14='DERIVED_0725_기준도넛_피복마스크.parquet'
LCMASK={}
if os.path.exists(_LCC14):
    _c14=gpd.read_parquet(_LCC14)
    for _,_r in _c14.iterrows():
        _g=_r.geometry
        LCMASK.setdefault(_r['부지'],{})[_r['종류']]=None if (_g is None or _g.is_empty) else _g
    print(f'{_LCC14} 로드 — {len(LCMASK)}부지 (다시 만들려면 파일 삭제)')
else:
    _idx14=pd.read_csv('DERIVED_0723_토지피복_도엽인덱스.csv',encoding='utf-8-sig').dropna(subset=['w'])
    _idxg14=gpd.GeoDataFrame(_idx14,geometry=[_box14(w,s,e,n) for w,s,e,n in zip(_idx14.w,_idx14.s,_idx14.e,_idx14.n)],
                             crs=4326).to_crs(5179)
    _sh14={}
    def _sheets14(target):
        parts=[]
        for _,r in _idxg14[_idxg14.intersects(target)].iterrows():
            if r.도엽 not in _sh14:
                try:
                    _sh14[r.도엽]=gpd.read_file(f"zip://{r.zip}!{r.도엽}.shp")[['L1_CODE','L2_CODE','geometry']].to_crs(5179)
                except Exception:
                    _sh14[r.도엽]=None
            if _sh14[r.도엽] is not None:
                parts.append(_sh14[r.도엽])
        return pd.concat(parts,ignore_index=True) if parts else None

    _rows14=[]; _t14=_t14mod.time()
    for _k14,(_nm14,_gv14) in enumerate(GEO.items()):
        _g14=_gv14[0] if isinstance(_gv14,tuple) else _gv14
        _don=_g14.buffer(REF[1]).difference(_g14.buffer(REF[0]))      # 기준 도넛 10~20km
        _lc=_sheets14(_don)
        if _lc is None or not len(_lc):
            print(f'  [{_k14+1}/{len(GEO)}] {_nm14}: 도엽 없음 — 대리 유지',flush=True)
            _rows14+=[dict(부지=_nm14,종류='res',wkt=''),dict(부지=_nm14,종류='grn',wkt='')]
            continue
        _cl=gpd.clip(gpd.GeoDataFrame(_lc,geometry='geometry'),_don)
        _l1=_cl['L1_CODE'].astype(str).str.zfill(3); _l2=_cl['L2_CODE'].astype(str).str.zfill(3)
        _o={}
        for _kind,_sel in (('res',_l2=='110'),('grn',_l1=='300')):
            _sub=_cl[_sel]
            if not len(_sub):
                _o[_kind]=''
                continue
            # [🔖 2026-07-25 2차 정정 — 실측으로 원인 재확인]
            #   1차엔 simplify(15)를 범인으로 봤지만 실제 주범은 **900m² 면적 필터**였다.
            #   세분류는 필지 단위라 시가화 폴리곤이 원래 작다 — 실측 중위면적(도엽 37909016):
            #     주거 40m² · 공업 41m² · 인공초지 224 · 인공나지 268
            #     (반면 침엽수림 4,753 · 활엽수림 4,039 · 밭 926)
            #   한 화소(30m×30m=900m²)보다 작은 걸 버리면 주거·공업이 **100% 사라지고 산림만 남는다**.
            #   → 면적 필터와 단순화를 **둘 다 제거**한다. 화소화(geometry_mask)는 작은 폴리곤을
            #     알아서 처리한다(화소 중심을 덮으면 포함). 인접 필지가 뭉치면 화소를 채우므로 손실 없다.
            _o[_kind]=_uu14(list(_sub.geometry)).wkt
        _rows14+=[dict(부지=_nm14,종류=k,wkt=v) for k,v in _o.items()]
        _ar=_wkt14.loads(_o['res']).area/1e6 if _o['res'] else 0.0
        _ag=_wkt14.loads(_o['grn']).area/1e6 if _o['grn'] else 0.0
        print(f'  [{_k14+1}/{len(GEO)}] {_nm14}: 주거 {_ar:.1f}km² · 산림 {_ag:.1f}km² / 도넛 {_don.area/1e6:.0f}km² ({int(_t14mod.time()-_t14)}s)',flush=True)
    _gdf14=gpd.GeoDataFrame(
        {'부지':[r['부지'] for r in _rows14],'종류':[r['종류'] for r in _rows14]},
        geometry=[_wkt14.loads(r['wkt']) if r['wkt'] else None for r in _rows14], crs=5179)
    _gdf14.to_parquet(_LCC14)
    for _,_r in _gdf14.iterrows():
        _g=_r.geometry
        LCMASK.setdefault(_r['부지'],{})[_r['종류']]=None if (_g is None or _g.is_empty) else _g
    print(f'저장 {_LCC14} ({os.path.getsize(_LCC14)/1e6:.1f}MB)')
# [🔖 2026-07-25] 검증 게이트 — 마스크가 비면 아래 '실측'이 전부 허수가 된다. 조용히 넘어가지 않게 막는다.
_nres=sum(1 for v in LCMASK.values() if v.get('res') is not None and not v['res'].is_empty)
_ares={k:(v['res'].area/1e6 if v.get('res') is not None else 0.0) for k,v in LCMASK.items()}
print('실측 주거 마스크 보유 부지:',_nres,'/',len(LCMASK))
print('  부지별 주거 면적(km²):',{k:round(v,1) for k,v in sorted(_ares.items(),key=lambda x:-x[1])})
assert min(_ares.values())>0.05, (
    '주거 마스크 면적이 0에 가까운 부지: '+str({k:round(v,3) for k,v in _ares.items() if v<=0.05})
    +' — 세분류 필지는 수십 m²라 면적 필터를 걸면 안 된다(2026-07-25 사건). '
    '이 게이트가 울리면 아래 "실측" 값은 전부 허수다.')
assert _nres>=int(len(LCMASK)*0.7), (
    f'주거 마스크가 {_nres}/{len(LCMASK)}뿐 — L2 코드(110=주거지역)나 단순화/면적필터를 점검하라. '
    f'비어 있으면 §14-d1의 base_res_lc가 전부 NaN이 되어 "실측"이 허수가 된다.')


  [1/12] 광양·제철: 주거 5.5km² · 산림 244.3km² / 도넛 1173km² (128s)
  [2/12] 여수·석화: 주거 4.1km² · 산림 137.3km² / 도넛 941km² (212s)
  [3/12] 울산미포: 주거 2.3km² · 산림 255.7km² / 도넛 1318km² (308s)
  [4/12] 온산: 주거 3.4km² · 산림 195.4km² / 도넛 1127km² (373s)
  [5/12] 포항·제철: 주거 2.0km² · 산림 300.5km² / 도넛 941km² (467s)
  [6/12] 구미·전자: 주거 1.4km² · 산림 192.0km² / 도넛 1079km² (530s)
  [7/12] 당진1철강: 주거 3.7km² · 산림 101.0km² / 도넛 1002km² (642s)
  [8/12] 동해북평: 주거 0.1km² · 산림 103.0km² / 도넛 983km² (658s)
  [9/12] 동해 북평2: 주거 0.2km² · 산림 119.4km² / 도넛 966km² (669s)
  [10/12] 현대제철 인천: 주거 3.5km² · 산림 23.2km² / 도넛 941km² (708s)
  [11/12] 동국제강 인천: 주거 3.5km² · 산림 23.7km² / 도넛 941km² (730s)
  [12/12] 심팩 포항: 주거 2.2km² · 산림 299.0km² / 도넛 941km² (786s)
저장 DERIVED_0725_기준도넛_피복마스크.parquet (131.1MB)
실측 주거 마스크 보유 부지: 12 / 12
  부지별 주거 면적(km²): {'광양·제철': 5.5, '여수·석화': 4.1, '당진1철강': 3.7, '동국제강 인천': 3.5, '현대제철 인천': 3.5, '온산': 3.4, '울산미포': 2.3, '심팩 포항': 2.2, '포항·제철': 2.0, '구미·전자': 1.4, '동해 북평2': 0.2, '동해북평': 0.1}


In [ ]:
# ── §14-d1. 층화 기준선 함수 정의 [🔖 2026-07-23 분할·주석 보강] ──
# ⚠ NDVI 계산 로직은 §17과 동일 (self-contained 원칙으로 의도적 중복) — 수정 시 양쪽 동기화할 것.
import numpy as _n14, os as _os14
SR_SC14,SR_OF14=0.0000275,-0.2      # USGS 공식: 반사율 = DN×0.0000275 − 0.2 (SR_B4·B5 공통)

def site_scene_dT_res(base,geom5179,is_poly,unc_max=4.0,min_px=50,min_ref=2000,lcm=None):
    """한 장면에서: 부지 평균 LST + 기준 도넛(10~20km)의 [전체/주거지/녹지] 층별 기준선 LST를 구한다."""
    # (0) NDVI용 SR 밴드가 없는 장면은 측정 불가
    for _b in ('_SR_B4.TIF','_SR_B5.TIF'):
        if not _os14.path.exists(base+_b) or _os14.path.getsize(base+_b)==0:
            return {'사유':'SR밴드 없음/0바이트'}
    # (1) 온도(ST_B10)·품질(QA)·불확실도(ST_QA)·반사율(SR_B4 빨강, SR_B5 근적외) — 같은 창으로 읽기
    with rasterio.open(base+'_ST_B10.TIF') as s:
        g=_to_scene(geom5179,5179,s.crs); gb=g.bounds; _scrs=s.crs   # [07-25] 실측 마스크 투영용
        if not (s.bounds.left<(gb[0]+gb[2])/2<s.bounds.right and s.bounds.bottom<(gb[1]+gb[3])/2<s.bounds.top):
            return {'사유':'장면 bbox 밖'}
        w=from_bounds(gb[0]-REF[1]-1000,gb[1]-REF[1]-1000,gb[2]+REF[1]+1000,gb[3]+REF[1]+1000,s.transform)
        st=s.read(1,window=w,boundless=True,fill_value=0).astype('float64'); tr=s.window_transform(w); shp=st.shape
    with rasterio.open(base+'_QA_PIXEL.TIF') as s:
        qa=s.read(1,window=w,boundless=True,fill_value=1)
    with rasterio.open(base+'_ST_QA.TIF') as s:
        sq=s.read(1,window=w,boundless=True,fill_value=0).astype('float64')
    with rasterio.open(base+'_SR_B4.TIF') as s:
        b4=s.read(1,window=w,boundless=True,fill_value=0).astype('float64')
    with rasterio.open(base+'_SR_B5.TIF') as s:
        b5=s.read(1,window=w,boundless=True,fill_value=0).astype('float64')
    # (2) DN → 물리량: 지표온도(℃), 반사율(0~1)
    lst=st*ST_SCALE+ST_OFF-273.15
    bad=(st==0)                                    # 무자료
    for k in ('fill','dilated','cirrus','cloud','shadow','snow'):
        bad|=((qa>>QA_BITS[k])&1)>0                # 구름·그림자·눈 비트
    bad|=((qa>>QA_BITS['water'])&1)>0              # 물 (해안 산단 필수)
    bad|=(sq*STQA_SCALE>unc_max)                   # 온도 불확실도 큰 화소
    lst=_n14.where(bad,_n14.nan,lst)
    r4=b4*SR_SC14+SR_OF14                          # 빨강 반사율 (0~1이어야 물리적으로 유효)
    r5=b5*SR_SC14+SR_OF14                          # 근적외 반사율
    # 유효 조건: 품질 통과 + 두 반사율 모두 물리 범위(0~1) 안 + 분모가 0 근처 아님(0나눗셈 방지)
    okr=(~bad)&(r4>0)&(r4<1)&(r5>0)&(r5<1)&((r4+r5)>1e-6)
    ndvi=_n14.where(okr,(r5-r4)/(r5+r4),_n14.nan)  # 식생지수: 식물↑일수록 +1에 가까움
    # (3) 공간 마스크: 부지(core)와 기준 도넛(10~20km)
    inside=lambda gg: geometry_mask([gg],out_shape=shp,transform=tr,invert=True)
    core=inside(g)
    if core.any() and (st[core]==0).mean()>0.9:
        return {'사유':'촬영범위 밖'}
    if is_poly:
        sm=core                                    # 폴리곤 부지: 내부(R0)
    else:
        sm=inside(g.buffer(1000))                  # 점 부지: 0-1km (§14-c 규약 동일)
    sv=lst[sm]; sv=sv[~_n14.isnan(sv)]
    if len(sv)<min_px:
        return {'사유':'부지 유효화소 부족'}
    rf=inside(g.buffer(REF[1]))&~inside(g.buffer(REF[0]))
    rl=lst[rf]; rn=ndvi[rf]
    ok=~_n14.isnan(rl); rl=rl[ok]; rn=rn[ok]
    if len(rl)<min_ref:
        return {'사유':'기준선 유효화소 부족'}
    # (4) 기준선 층화: 전체 / 주거지(0.15≤NDVI<0.40·0.45·0.50 민감도) / 녹지(≥0.45)
    both=~_n14.isnan(rn)
    out={'사유':'OK','site':float(_n14.mean(sv)),'base_all':float(_n14.median(rl)),'n_all':int(len(rl))}
    for ub,lab in [(0.40,'r40'),(0.45,'r45'),(0.50,'r50')]:
        m=both&(rn>=0.15)&(rn<ub)
        if m.sum()>=min_px:
            out['base_'+lab]=float(_n14.median(rl[m]))
        else:
            out['base_'+lab]=_n14.nan
        if lab=='r45':
            out['frac_res']=float(m.sum()/max(both.sum(),1))
    mg=both&(rn>=0.45)
    if mg.sum()>=min_px:
        out['base_grn']=float(_n14.median(rl[mg]))
    else:
        out['base_grn']=_n14.nan
    # (5) [🔖 2026-07-25 사용자 지시] **실측 토지피복 층** — NDVI 대리 대신 주거지역(L2=110)·산림(L1=300).
    #     lcm은 §14-d0이 만든 {'res':geom(5179), 'grn':geom(5179)}. 장면 CRS로 옮겨 화소 마스크를 만든다.
    #     대리(위 base_r45/base_grn)는 **지우지 않고 나란히 남긴다** — 편향의 크기·방향을 보이기 위해서다.
    out['base_res_lc']=_n14.nan; out['base_grn_lc']=_n14.nan; out['frac_res_lc']=_n14.nan
    if lcm:
        for _k,_col in (('res','base_res_lc'),('grn','base_grn_lc')):
            _gg=lcm.get(_k)
            if _gg is None or getattr(_gg,'is_empty',True):
                continue
            _mk=inside(_to_scene(_gg,5179,_scrs))&rf          # 기준 도넛 ∩ 실측 피복
            _vv=lst[_mk]; _vv=_vv[~_n14.isnan(_vv)]
            if len(_vv)>=min_px:
                out[_col]=float(_n14.median(_vv))
            if _k=='res':
                out['frac_res_lc']=float(_mk.sum()/max(rf.sum(),1))
    return out
print('함수 정의 완료: site_scene_dT_res (기준선 NDVI 층화판)')

함수 정의 완료: site_scene_dT_res (기준선 NDVI 층화판)


In [ ]:
# ── §14-d2. 부지×장면 루프 — 부지별 진행 출력 [🔖 2026-07-23 분할] ──
# [🔖 2026-07-25 중복 제거] 여기서 대상 목록을 다시 만들지 않는다 — §14-c1의 `GEO`를 그대로 쓴다.
#   기존엔 같은 목록을 두 벌 갖고 있어 §14-c1에 북평2를 넣어도 §14-d는 따라오지 않았다(정의 표류의 원인).
GEO14=GEO

_rec14=[]; _sk14={}
for _j,(_nm,(_g,_ip)) in enumerate(GEO14.items()):
    _ok0=len(_rec14)
    for _,_s in SC.iterrows():
        if _s.date.month not in (6,7,8):
            continue
        _m=mtl_meta(_s.base)
        if float(_m.get('CLOUD_COVER',100))>30.0:
            continue
        _r=site_scene_dT_res(_s.base,_g,_ip,lcm=LCMASK.get(_nm))   # [07-25] 실측 피복 마스크 전달
        _sk14[_r['사유']]=_sk14.get(_r['사유'],0)+1
        if _r['사유']!='OK':
            continue
        _rec14.append(dict(부지=_nm,폴리곤=_ip,날짜=_s.date.date(),**{k:v for k,v in _r.items() if k!='사유'}))
    print(f'[{_j+1}/{len(GEO14)}] {_nm}: 유효 {len(_rec14)-_ok0}장면 (누적 {len(_rec14)})')
D14=pd.DataFrame(_rec14)
print()
print('장면-부지 쌍 처리 사유:',_sk14)

[1/12] 광양·제철: 유효 11장면 (누적 11)
[2/12] 여수·석화: 유효 7장면 (누적 18)
[3/12] 울산미포: 유효 16장면 (누적 34)
[4/12] 온산: 유효 26장면 (누적 60)
[5/12] 포항·제철: 유효 13장면 (누적 73)
[6/12] 구미·전자: 유효 20장면 (누적 93)
[7/12] 당진1철강: 유효 9장면 (누적 102)
[8/12] 동해북평: 유효 22장면 (누적 124)
[9/12] 동해 북평2: 유효 21장면 (누적 145)
[10/12] 현대제철 인천: 유효 9장면 (누적 154)
[11/12] 동국제강 인천: 유효 9장면 (누적 163)
[12/12] 심팩 포항: 유효 14장면 (누적 177)

장면-부지 쌍 처리 사유: {'장면 bbox 밖': 813, '촬영범위 밖': 51, 'OK': 177, '부지 유효화소 부족': 39}


In [ ]:
# ── §14-d3. 집계 — 현행 vs 주거지 기준 + §14-c 대조 검증 [🔖 2026-07-23 분할] ──
# 목적: 1) 기존 기준선(숲·녹지가 포함되어 온도가 낮게 잡힘) 대비 주거지 전용 기준선을 적용했을 때의 온열 격차(ΔT) 재계산
#       2) 주거지 화소 비율 조건(r40, r45, r50)에 따른 민감도 분석 수행
#       3) 이전 단계(§14-c) 결과 파일과 대조하여 데이터 정합성 검증

# -----------------------------------------------------------------------------
# 1. 현행 기준선 대비 및 주거지 민감도 기준선별 ΔT(온도차) 산출
# -----------------------------------------------------------------------------
# ΔT현행: 부지 온도(site) - 숲/녹지를 포함한 주변 전체 평균 온도(base_all)
# [🔖 2026-07-25] §14-d2를 다시 돌리지 않고 이 절만 손볼 수 있게 폴백을 둔다(장면 원자료는 CSV에 있다).
if 'D14' not in dir():
    D14=pd.read_csv('DERIVED_0722_주거지기준선_ΔT.csv',encoding='utf-8-sig')
    print(f'§14-d2 미실행 → DERIVED_0722_주거지기준선_ΔT.csv 로드 ({len(D14)}행)')

D14['ΔT현행']=D14['site']-D14['base_all']

# 주거지 화소 비율 조건(r40: 40%, r45: 45%, r50: 50% 이상 주거지)별 기준선 대비 ΔT 산출
for _lab in ['r40','r45','r50']:
    D14['ΔT'+_lab]=D14['site']-D14['base_'+_lab]

# 부지별로 위성 장면(Scene) 수, 평균 ΔT 현행/주거지, 민감도 조건, 주거 화소 비율 집계
# [🔖 2026-07-25] 실측 토지피복 기준선 ΔT도 함께 — 대리(ΔTr45)와 나란히 놓아 편향을 본다.
D14['ΔT주거_실측']=D14['site']-D14['base_res_lc']
D14['ΔT산림_실측']=D14['site']-D14['base_grn_lc']
A14=D14.groupby('부지').agg(장면=('날짜','size'),ΔT현행=('ΔT현행','mean'),ΔT주거_대리=('ΔTr45','mean'),
    ΔT주거_실측=('ΔT주거_실측','mean'),ΔT산림_실측=('ΔT산림_실측','mean'),
    주거비_대리=('frac_res','mean'),주거비_실측=('frac_res_lc','mean')).round(4)
# [🔖 2026-07-25 표시 정정] 전체를 round(2)로 뭉개니 실측 주거비 0.005가 0.00으로 찍혀
#   "실측 0%"라는 잘못된 진술이 나왔다. ΔT는 2자리, 비율은 4자리로 나눠 반올림한다.
A14[[c for c in A14.columns if c.startswith('ΔT')]]=A14[[c for c in A14.columns if c.startswith('ΔT')]].round(2)
A14['ΔT주거지']=A14['ΔT주거_실측'].fillna(A14['ΔT주거_대리'])     # 실측 우선, 없으면 대리

# -----------------------------------------------------------------------------
# 2. §14-c 저장 CSV 파일(전체 위성 장면)과의 데이터 정합성 교차검증
# -----------------------------------------------------------------------------
# 기존 분석 CSV 파일 로드
_c14=pd.read_csv('DERIVED_0720_산단_LST_반경별.csv',encoding='utf-8-sig')

# 산단 형상이 폴리곤인 경우 ΔR0내부, 점인 경우 Δ0-1km를 현행 CSV 참조값으로 선택
_c14['현행CSV']=_c14.apply(lambda r: r['ΔR0내부'] if r['형상']=='폴리곤' and pd.notna(r.get('ΔR0내부')) else r['Δ0-1km'],axis=1)

# 기존 CSV와의 대조값 결합 및 '기준선 이동'에 따른 ΔT 변화량(주거지 - 현행) 계산
A14['§14-c대조']=_c14.groupby('부지')['현행CSV'].mean().round(2)
A14['이동']=(A14['ΔT주거지']-A14['ΔT현행']).round(2)     # 음수 = 주거지 기준으로 ΔT가 줄어듦

# -----------------------------------------------------------------------------
# 3. 분석 결과 출력 및 스피어만 순위 상관계수(Spearman ρ) 검증
# -----------------------------------------------------------------------------
print('=== 부지별 ΔT: 현행(숲 포함) vs 주거지 기준 — 대리(NDVI) vs 실측(토지피복 2024) (℃) ===')
print(A14.sort_values('ΔT현행',ascending=False).to_string())
_dl=(A14['ΔT주거_실측']-A14['ΔT주거_대리']).dropna()
if len(_dl):
    print()
    print(f'[사실] 대리→실측 교체 효과: 평균 {_dl.mean():+.2f}℃ (부지별 {_dl.min():+.2f}~{_dl.max():+.2f}) · n={len(_dl)}부지')
    print(f'  기준 도넛에서 주거가 차지하는 비율: 대리(NDVI) {A14["주거비_대리"].mean()*100:.1f}% vs 실측(토지피복) {A14["주거비_실측"].mean()*100:.2f}% — '
          '대리가 잡던 화소 중 상당수는 실제 주거지가 아니었다(§17-c 편향 방향과 일치).')

# 평균 기준선 이동값 및 현행 vs 주거지 순위 보존도(Spearman 상관계수) 계산
_sh14=A14['이동'].mean()
_rk14=A14[['ΔT현행','ΔT주거지']].corr(method='spearman').iloc[0,1]


print()
print(f'[사실] 기준선 이동 평균 {_sh14:+.2f}℃ (부지별 {A14["이동"].min():+.2f}~{A14["이동"].max():+.2f}) · 순위 보존 spearman ρ={_rk14:.3f}')

# 기준선 이동 효과 해석 도출
if _sh14<0:
    print(f'[해석] 주거지 기준 ΔT가 평균 {-_sh14:.1f}℃ 작다 — "숲 기준이 ΔT를 부풀린다"(상한 방향 편의) 실측 확인. 순위 ρ={_rk14:.2f} → 부지 간 비교·결론 구조 불변.')
else:
    print(f'[해석] 주거지 기준 ΔT가 평균 {_sh14:.1f}℃ 크다 — 기존 기준이 오히려 보수적이었음.')

print('[해석] 두 편의 종합: 숲 기준(상한 방향, 여기서 정량화) vs 오전 스냅샷(하한 방향, §15-f) — 반대 방향이라 범위로 읽는다.')

# -----------------------------------------------------------------------------
# 4. 장면 단위 원자료 CSV 저장
# -----------------------------------------------------------------------------
D14.to_csv('DERIVED_0722_주거지기준선_ΔT.csv',index=False,encoding='utf-8-sig')
print()
# [🔖 2026-07-25] 실측 표본이 얼마나 얇은지, 그게 실제로 불안정을 낳는지 **재서** 적는다.
#   (처음엔 "얇으면 불안정할 것"이라고 추정했는데 데이터가 그걸 지지하지 않았다 — 아래 SEM 참조.)
print()
_px=(D14.groupby('부지').apply(lambda g:(g['n_all']*g['frac_res_lc']).mean())).rename('실측화소')
_dr=(D14['site']-D14['base_r45']); _ds=(D14['site']-D14['base_res_lc'])
_st=pd.DataFrame({'부지':D14['부지'],'대리':_dr,'실측':_ds}).groupby('부지').agg(
    SEM대리=('대리',lambda x:x.dropna().std()/max(np.sqrt(x.notna().sum()),1)),
    SEM실측=('실측',lambda x:x.dropna().std()/max(np.sqrt(x.notna().sum()),1)))
_st['실측화소']=_px.round(0); _st['SEM배수']=(_st.SEM실측/_st.SEM대리).round(2)
print(f'⚠ 실측 주거 표본은 얇다 — 유효 도넛 화소의 {A14["주거비_실측"].mean()*100:.2f}%뿐(대리의 약 1/70).')
print(f'  도넛(10~20km)이 대부분 산림·농지·바다라서다. 부지별 편차가 크다:')
print('  ' + _st.sort_values('실측화소')[['실측화소','SEM대리','SEM실측','SEM배수']].round(2).to_string().replace(chr(10), chr(10) + '  '))
print()
print('  → **화소 수와 불안정이 대응하지 않는다.** 48화소인 동해북평(SEM배수 1.30)보다')
print('    729화소인 온산(1.99)이 더 흔들리고, 여수·포항은 실측이 오히려 더 안정적이다(0.64·0.71).')
print('    장면 간 변동(구름·계절)이 화소 수보다 크게 작용한다 → 화소가 적다는 이유만으로 버리지 않는다.')
print('  → 다만 동해북평 48·북평2 52는 min_px=50 문턱에 걸쳐 일부 장면이 결측됐다(각 2·3장면).')
print('    그 두 부지는 실측값에 SEM을 반드시 병기해 읽을 것.')
print('  → 대리와의 차이가 평균 +0.49℃로 작다 — 대리(NDVI 0.15~0.45)가 이 용도로는 쓸 만했다는 뜻이다.')
# [🔖 2026-07-26 감사] "왜 비슷한가"에 두 번 설명을 붙였다가 두 번 다 데이터에 기각당했다(audit3).
#   ① "삼킨 화소가 공업·상업·나지라서" → 실제 구성은 **교통시설(도로) 33%** 지배, 공업+상업은 8%뿐.
#   ② "뜨거운 것과 시원한 것이 상쇄돼서" → 부지별 편의가 도로비(r=−0.01)·논밭비(r=−0.28)와 무관.
#   → 세 번째 설명을 지어내지 않는다. **왜 비슷한지는 모른다.** 경험적으로 작다는 것만 확인됐다.
print('    ⚠ 다만 **왜** 비슷한지는 설명하지 못한다. 대리가 삼킨 화소의 실제 구성은 도로 33%·논밭 19%·')
print('      공업+상업 8%로 "도시 화소"라 부르기 어렵고, 부지별 편의(−0.8~+1.4℃)도 그 구성비로 예측되지')
print('      않는다(도로비와 r=−0.01). 기전 없이 경험적으로만 작다 → SD 0.66℃를 불확실성으로 안고 간다.')
print('    ⚠ 대리는 실측 주거의 **32~71%만 포착**한다(superset이 아니라 부분집합에 가깝다).')
print('저장: DERIVED_0722_주거지기준선_ΔT.csv (장면 단위 원자료)')

§14-d2 미실행 → DERIVED_0722_주거지기준선_ΔT.csv 로드 (177행)
=== 부지별 ΔT: 현행(숲 포함) vs 주거지 기준 — 대리(NDVI) vs 실측(토지피복 2024) (℃) ===
         장면   ΔT현행  ΔT주거_대리  ΔT주거_실측  ΔT산림_실측  주거비_대리  주거비_실측  ΔT주거지  §14-c대조    이동
부지                                                                                 
포항·제철    13  17.26    10.46    10.53    18.34  0.0726  0.0022  10.53    17.25 -6.73
심팩 포항    14  14.34     6.82     7.15    15.41  0.0797  0.0024   7.15    14.34 -7.19
구미·전자    20  14.02     6.45     6.92    15.25  0.0988  0.0013   6.92    13.10 -7.10
당진1철강     9  12.58     9.55    10.33    15.87  0.2931  0.0037  10.33    12.58 -2.25
울산미포     16  12.10     4.14     6.09    13.06  0.0817  0.0017   6.09    12.10 -6.01
동해북평     22  11.17     4.04     4.51    12.66  0.0331  0.0001   4.51    10.93 -6.66
동해 북평2   21  11.06     3.96     4.46    12.06  0.0331  0.0001   4.46      NaN -6.60
온산       26  10.94     2.45     2.61    12.67  0.1238  0.0029   2.61    11.16 -8.33
광양·제철    11  10.21     3.03     2.88    12.

## §15. 지표면온도 → 기온 환산계수 **자체 추정** `[방법]` `[🔖 2026-07-20]`

§14는 산단 지표가 +5~17℃ 뜨겁다는 걸 실측했지만, **§9-b의 ×1.6/℃ 곡선은 기온(ASOS 최고기온) 기반**이라 지표 ΔT를 그대로 넣을 수 없다(§14 해석 ⛔2). §15는 그 환산계수 β를 **외부 상수 차용 없이 우리 데이터로** 추정한다.

**설계 — 같은 장면 안, 관측소들 사이의 기울기**
$$T_{air,is} = \alpha_s + \beta \cdot LST_{is} + \varepsilon_{is}$$
- $i$ = ASOS 관측소, $s$ = Landsat 장면(날짜×Path/Row)
- $\alpha_s$ = **장면 고정효과** — 그날 그 지역이 통째로 덥고 추운 것을 흡수. 남는 건 **같은 시각·같은 대기 안에서 지표가 뜨거운 지점이 공기도 더 뜨거운가**라는 공간 기울기뿐이다.
- 표준오차는 **관측소 클러스터**(같은 관측소가 여러 장면에 반복 등장)
- → $\beta$ = 지표 1℃ 상승당 기온 상승분. **산단 ΔLST × β = 기온 ΔT**

**왜 이게 중요한가**: §9는 AIDC 폐열로 **+2℃**를 가정했다. β가 나오면 "실제 제철소가 만드는 기온 상승"과 대조할 수 있다 — §9의 가정이 과대/과소인지 우리 숫자로 판정 가능해진다.

---

### ⚠ 먼저 받아야 할 파일 — ASOS 관측지점 좌표

ASOS 일자료(`OPEN_0522_기상청_ASOS일자료_*_summer.csv`)에는 **지점번호·지점명만 있고 위경도가 없다.** 좌표 없이는 위성 화소를 못 고른다.

**받는 법** — [기상자료개방포털](https://data.kma.go.kr) → **[데이터] → [메타데이터] → [지점정보]** → 관측종류 `종관기상관측(ASOS)` → CSV 내려받기
필요 컬럼: **지점번호(지점)·위도·경도**. 파일명을 `OPEN_0720_기상청_ASOS지점정보.csv`로 해서 노트북과 같은 폴더에 두면 아래 셀이 자동 인식한다.

> ⚠ 좌표를 손으로 입력하지 말 것 — §14에서 그렇게 했다가 최대 8.2km 이탈했다(§14-b 주석). 반드시 공식 파일에서 읽는다.

> `[🔖 2026-07-23 표기 정합]` 여기서 $i$ = **관측소**(같은 형태의 모형을 ASOS·AWS **각각**에 적합), $s$ = 위성 장면. 최종 채택 β는 **AWS 459지점**(§15-e), ASOS는 대조·강건성 확인용 — "$i$=ASOS"로만 읽히지 않도록 명시.

In [62]:
# ── §15-a. 관측망 로더 (ASOS·AWS 공통) ──
# [🔖 2026-07-21] AWS 로더 261→459지점(ASOS+AWS/OBS_AWS_DD_*.csv)으로 갱신
import glob, os
# 두 관측망을 같은 함수로 처리한다. 파일 형식이 동일(지점·시작일·종료일·지점명·위도·경도 + 일자료 3기온).
NETS={'ASOS':dict(stn='OPEN_0720_기상청_ASOS지점정보.csv',
                  day=sorted(glob.glob('OPEN_0522_기상청_ASOS일자료_*_summer.csv'))),
      'AWS' :dict(stn='OPEN_0720_기상청_AWS지점정보.csv',
                  day=sorted(glob.glob('ASOS+AWS/OBS_AWS_DD_*.csv')) or ['OPEN_0720_기상청_AWS일자료_summer.csv'])}

def _rd(p):
    for e in ('cp949','utf-8-sig','utf-8'):
        try:
            d=pd.read_csv(p,encoding=e); d.columns=[str(c).strip() for c in d.columns]; return d
        except Exception: continue
    raise IOError(p)

def load_net(name):
    cfg=NETS[name]
    if not os.path.exists(cfg['stn']) or not all(os.path.exists(f) for f in cfg['day']):
        print(f'⏸ {name}: 파일 없음 — 건너뜀'); return None,None
    s=_rd(cfg['stn'])
    s=s.rename(columns={c:'lat' for c in s.columns if '위도' in c}|{c:'lon' for c in s.columns if '경도' in c})
    for c in ['지점','lat','lon']: s[c]=pd.to_numeric(s[c],errors='coerce')
    # [중요] 지점정보는 이전(relocation)·번호 재할당 이력이 여러 행으로 들어있다.
    #   ASOS 천안 15.4km · AWS 최대 417km(= 이동이 아니라 지점번호 재사용).
    #   최신 행을 그냥 쓰면 과거 날짜의 위성 화소를 엉뚱한 곳에서 뽑는다(§14 좌표 이탈과 같은 실수).
    s['시작일']=pd.to_datetime(s.get('시작일'),errors='coerce').fillna(pd.Timestamp('1900-01-01'))
    s['종료일']=pd.to_datetime(s.get('종료일'),errors='coerce')
    s=s.dropna(subset=['지점','lat','lon'])[['지점','lat','lon','시작일','종료일']].reset_index(drop=True)

    d=pd.concat([_rd(f) for f in cfg['day']],ignore_index=True)
    d=d.rename(columns={'평균기온(°C)':'Tavg','최고기온(°C)':'Tmax','최저기온(°C)':'Tmin'})
    d['date']=pd.to_datetime(d['일시'],errors='coerce').dt.date
    d['지점']=pd.to_numeric(d['지점'],errors='coerce')
    d=d[['지점','지점명','date','Tavg','Tmax','Tmin']].dropna(subset=['date','지점'])
    d=d[pd.to_datetime(d.date).dt.month.isin([6,7,8])]          # JJA만 (§3·§4와 동일 규율)
    nmulti=(s.groupby('지점').size()>1).sum()
    print(f'{name}: 지점 {s.지점.nunique()}개(정보 {len(s)}행, 이전이력 {nmulti}개) · 일자료 {len(d):,}행(JJA) · 관측지점 {d.지점.nunique()}개')
    return s,d

def stn_on(stn, day):
    # day에 유효한 좌표만 — 시작일 ≤ day ≤ 종료일(없으면 현재까지)
    t=pd.Timestamp(day)
    m=stn[(stn.시작일<=t)&(stn.종료일.isna()|(stn.종료일>=t))]
    return m.sort_values('시작일').drop_duplicates('지점',keep='last')[['지점','lat','lon']].reset_index(drop=True)

NET={}
for _n in NETS:
    _s,_d=load_net(_n)
    if _s is not None: NET[_n]=(_s,_d)
print('\n사용 가능 관측망:',list(NET))

ASOS: 지점 105개(정보 148행, 이전이력 37개) · 일자료 52,985행(JJA) · 관측지점 97개
AWS: 지점 576개(정보 2518행, 이전이력 538개) · 일자료 241,009행(JJA) · 관측지점 459개

사용 가능 관측망: ['ASOS', 'AWS']


**§15-b. 관측소 화소의 LST 추출 — 버퍼 크기도 함께 추정**

기온계는 점이지만 **공기는 상류 수백 m의 지표 영향을 통합**한다. 그래서 관측소 주변 반경 R의 LST 평균을 쓰고, **R을 30m·100m·500m·1km로 바꿔가며** 어느 반경이 기온을 가장 잘 설명하는지(=유효 footprint) 함께 본다.

관측소 화소가 구름·물로 가려지면 그 (관측소, 장면) 쌍은 버린다. 유효 화소가 반경 내 30% 미만이어도 버린다.

In [63]:
# ── §15-b. 관측소 화소 LST 추출 (지점당 소형 window) ──
from shapely.geometry import Point as _P

# 관측소 주변 평균 LST 계산을 위한 반경 설정 목록 (단위: 미터) - 유효 footprint 탐색용
RADII=[30,100,500,1000]        

# [성능 정정 07-20] 초판은 '전 관측소를 감싸는 하나의 거대 window'(장면 전역 ~7000×7000)를 읽고
#   지점마다 그 위에서 거리 배열을 다시 계산했다 → 사실상 끝나지 않음. 지점당 작은 window만 읽도록 교체.

def lst_at_points(base, lats, lons, radii=RADII, unc_max=4.0, min_frac=0.3):
    """
    관측소 좌표(위경도) 주변의 래스터(Landsat 등) 화소 데이터를 읽어 지정된 반경별 평균 LST(지표면 온도)를 산출하는 함수
    
    :param base: 래스터 파일의 기본 경로 (파일명 접두사)
    :param lats: 관측 지점들의 위도 리스트/배열
    :param lons: 관측 지점들의 경도 리스트/배열
    :param radii: 공간 집계를 수행할 반경 리스트(m)
    :param unc_max: 허용 가능한 최대 불확실성(Uncertainty) 임계값 (℃)
    :param min_frac: 평균 계산에 필요한 최소 유효 화소 비율 (0.0 ~ 1.0)
    :return: (반경별 평균 온도 및 유효 비율 결과 딕셔너리, 경계 내 처리된 유효 지점 개수)
    """
    n=len(lats)
    
    # 반경(R)별 결과 저장을 위한 구조 생성: {R: (평균 LST 배열, 유효 화소 비율 배열)}
    out={R:(np.full(n,np.nan),np.zeros(n)) for R in radii}

    # Window 크롭 시 여유를 두기 위한 최대 반경 마진 설정 (최대 반경 + 60m)
    rmax=max(radii)+60

    # 1. 지표온도(ST) 래스터 파일 오픈
    with rasterio.open(base+'_ST_B10.TIF') as s0:

        # WGS84(4326) 경위도 좌표를 래스터 데이터의 좌표계(CRS)로 일괄 변환
        xs,ys=Transformer.from_crs(4326,s0.crs,always_xy=True).transform(list(lons),list(lats))
        xs,ys=np.asarray(xs),np.asarray(ys)
        
        # 래스터의 전체 공간 경계(Bounds) 확인
        b=s0.bounds

        # 래스터 내부 영역(마진 rmax 적용)에 포함되는 유효 관측점 인덱스 필터링
        ok=np.where((xs>b.left+rmax)&(xs<b.right-rmax)&(ys>b.bottom+rmax)&(ys<b.top-rmax))[0]

        # 영역 내에 들어오는 지점이 없으면 빈 결과 반환
        if len(ok)==0: return out,0

        tr0=s0.transform # 기본 아핀 변환 행렬
        
        # 2. QA(화질/구름) 및 ST_QA(불확실성) 래스터 함께 오픈
        with rasterio.open(base+'_QA_PIXEL.TIF') as s1, rasterio.open(base+'_ST_QA.TIF') as s2:

            # 유효 지점들에 대해 순회하며 소형 Window 단위 처리
            for j in ok:

                # 관측점 중심으로 rmax 마진 범위만큼의 소형 영역(Window) 정의
                w=from_bounds(xs[j]-rmax,ys[j]-rmax,xs[j]+rmax,ys[j]+rmax,tr0)

                # ST(지표온도 Raw 값) 읽기
                st=s0.read(1,window=w,boundless=True,fill_value=0).astype('float64')
                if st.size==0 or (st==0).all(): continue

                # QA 데이터 읽기
                qa=s1.read(1,window=w,boundless=True,fill_value=1)
                sq=s2.read(1,window=w,boundless=True,fill_value=0).astype('float64')

                # Raw DN 값을 실제 섭씨 온도(℃)로 변환: (DN * Scale + Offset) - 273.15(K->℃)
                lst=st*ST_SCALE+ST_OFF-273.15

                # --- 품질 마스킹(QA Masking) 처리 ---
                bad=(st==0) # # 데이터 없음(NoData)

                # QA 비트 플래그 검사 (구름, 그림자, 눈, Cirrus 등 결함 화소 제거)
                for k in ('fill','dilated','cirrus','cloud','shadow','snow'): 
                    bad|=((qa>>QA_BITS[k])&1)>0
                
                # 수계(Water) 영역 제외
                bad|=((qa>>QA_BITS['water'])&1)>0

                # 불확실성(Uncertainty) 임계값 초과 화소 제외
                bad|=(sq*STQA_SCALE>unc_max)

                # 결함 화소는 NaN으로 마스킹
                lst=np.where(bad,np.nan,lst)

                # --- 소형 Window 내 화소별 중심 좌표 및 거리 계산 ---
                tr=s0.window_transform(w); ny,nx=lst.shape
                cc,rr=np.meshgrid(np.arange(nx)+0.5,np.arange(ny)+0.5)
                px,py=tr*(cc,rr)
                
                d2=(px-xs[j])**2+(py-ys[j])**2
                for R in radii:
                    m=d2<=R*R
                    if not m.any(): continue
                    g=lst[m]; g=g[~np.isnan(g)]
                    fr=len(g)/m.sum(); out[R][1][j]=fr
                    if fr>=min_frac and len(g): out[R][0][j]=g.mean()
    return out,len(ok)

print('함수 정의 완료: lst_at_points (지점당 소형 window)')

함수 정의 완료: lst_at_points (지점당 소형 window)


**§15-c. (관측소 × 위성장면) 패널 구축** `[🔖 2026-07-25 설명 추가]`

§15-d가 돌리는 회귀 `Tair ~ LST + C(scene)`에는 **한 행이 (기온, LST) 한 쌍**이어야 한다. 그런데 두 자료는 출처가 다르다 — 기온은 기상관측소 일자료(지점×날짜), LST는 Landsat 장면(경로/행×촬영일)이다.

§15-c가 하는 일은 그 **둘을 촬영일 기준으로 맞물려 패널을 만드는 것**이다: 장면을 훑어 그날 그 장면 안에 있는 관측소를 찾고, §15-b가 뽑아둔 반경별 LST를 붙이고, 구름·물·품질(ST_QA) 불량 쌍을 버린다.

> **왜 §15-d만으론 안 되나** — §15-d는 이미 만들어진 패널에 회귀를 돌리는 절이다. 입력이 없으면 돌릴 대상이 없다. 실제로 §15-d 코드는 `if not len(SP): print('⏸ §15-c 선행 필요')`로 시작한다. 둘은 **자료 생성(§15-c) → 추정(§15-d)** 관계이지 중복이 아니다.

In [ ]:
# ── §15-c. (관측소 × 위성장면) 패널 데이터 구축 — 관측망별 ──
# 목적: 기상관측소(AWS/ASOS) 위치와 Landsat 위성 LST(지표면온도) 데이터를 
#       날짜/시점 기준으로 매칭하여 패널(Panel) 데이터셋을 생성하고 변이성(Leverage)을 모니터링함.

def build_panel(name, stn, day):
    """
    관측망별 (관측소 × 위성장면) 패널 생성 함수
    :param name: 관측망 이름 (예: AWS, ASOS 등)
    :param stn: 해당 관측망의 관측소 위치/메타 정보
    :param day: 해당 관측망의 일별 기온 데이터 (Tavg, Tmax, Tmin 등)
    """
    rows=[]

    # -------------------------------------------------------------------------
    # 1. 위성 장면(Scene) 스캔 및 필터링
    # -------------------------------------------------------------------------
    for _,s in SC.iterrows():
        # [조건 1] 여름철(JJA: 6, 7, 8월) 한정 분석 옵션 적용 시 해당 월만 필터링
        if JJA_ONLY and s.date.month not in (6,7,8): continue

        # [조건 2] 구름량이 기준치(CLOUD_MAX) 초과 시 해당 장면 제외
        if float(mtl_meta(s.base).get('CLOUD_COVER',100))>CLOUD_MAX: continue

        # [조건 3] 해당 날짜/시점에 정상 작동 중인 관측소(cs) 추출
        cs=stn_on(stn,s.date.date())
        if not len(cs): continue

        # 관측소 위경도 위치에 해당하는 위성 LST 추출 (반경 RADII별)
        r,nin=lst_at_points(s.base,cs.lat.values,cs.lon.values)
        if nin==0: continue

        # 기본 메타 정보 작성
        base=dict(scene=os.path.basename(s.base),date=s.date.date(),path=s.path,row=s.row)

        # ---------------------------------------------------------------------
        # 2. 관측소별 반경(RADII) LST 추출 및 행 레코드 구축
        # ---------------------------------------------------------------------
        for j in range(len(cs)):
            # 모든 반경에서 LST가 NaN(결측)인 관측소는 스킵
            if all(np.isnan(r[R][0][j]) for R in RADII): continue

            # 레코드 추가: 관측소 ID 및 반경별 LST 매핑
            rows.append({**base,'지점':int(cs.지점[j]),**{f'LST_{R}m':r[R][0][j] for R in RADII}})

    # 데이터프레임 변환
    P=pd.DataFrame(rows)
    if not len(P): print(f'{name}: 유효 쌍 0'); return P

    # -------------------------------------------------------------------------
    # 3. 기상 관측 데이터(지상 기온) 결합 및 변이성 분석
    # -------------------------------------------------------------------------
    # 지상관측 기온 데이터(Tavg, Tmax, Tmin)와 Inner Join 결합
    P=P.merge(day[['지점','지점명','date','Tavg','Tmax','Tmin']],on=['지점','date'],how='inner')
    P['net']=name; P['scene_id']=P['scene']

    # 장면당 결합된 관측소 수 집계
    per=P.groupby('scene').size()

    # 요약 정보 출력    
    print(f"\n[{name}] (관측소×장면) 쌍 {len(P):,} · 관측소 {P.지점.nunique()}개 · 장면 {P.scene.nunique()}개")
    print(f"  장면당 관측소: 중앙값 {per.median():.0f} (최소 {per.min()} · 최대 {per.max()})")

    # ★ 핵심 검증: 한 장면 내 관측소 간 LST의 표준편차(SD) 측정
    # LST의 넓은 변이폭(=회귀 Leverage)이 확보되어야 향후 LST-기온 회귀 모델의 추정력/신뢰도가 높아짐
    print(f"  ★ 장면 내 관측소간 LST 표준편차(회귀 leverage) 중앙값:")
    for R in RADII:
        print(f"     반경 {R:>4}m: {P.groupby('scene')[f'LST_{R}m'].std().median():5.2f}℃  (유효 {P[f'LST_{R}m'].notna().mean()*100:3.0f}%)")

    # 종속변수인 지상 기온(Tmax, Tavg)의 장면 내 표준편차(SD) 중앙값 출력
    print(f"  기온 SD 중앙값: Tmax {P.groupby('scene')['Tmax'].std().median():.2f}℃ · Tavg {P.groupby('scene')['Tavg'].std().median():.2f}℃")
    return P

# -----------------------------------------------------------------------------
# 4. 전체 관측망(NET) 패널 데이터 구축 및 병합
# -----------------------------------------------------------------------------
PANEL={n:build_panel(n,*NET[n]) for n in NET}

# 생성된 관측망별 패널 데이터들을 단일 데이터프레임(SP)으로 통합
SP=pd.concat([p for p in PANEL.values() if len(p)],ignore_index=True) if PANEL else pd.DataFrame()


[ASOS] (관측소×장면) 쌍 905 · 관측소 83개 · 장면 85개
  장면당 관측소: 중앙값 11 (최소 1 · 최대 29)
  ★ 장면 내 관측소간 LST 표준편차(회귀 leverage) 중앙값:
     반경   30m:  3.00℃  (유효  90%)
     반경  100m:  2.88℃  (유효  92%)
     반경  500m:  2.68℃  (유효  93%)
     반경 1000m:  2.82℃  (유효  97%)
  기온 SD 중앙값: Tmax 1.38℃ · Tavg 0.96℃

[AWS] (관측소×장면) 쌍 3,405 · 관측소 370개 · 장면 88개
  장면당 관측소: 중앙값 26 (최소 1 · 최대 104)
  ★ 장면 내 관측소간 LST 표준편차(회귀 leverage) 중앙값:
     반경   30m:  3.45℃  (유효  86%)
     반경  100m:  3.38℃  (유효  87%)
     반경  500m:  3.37℃  (유효  92%)
     반경 1000m:  3.25℃  (유효  98%)
  기온 SD 중앙값: Tmax 1.66℃ · Tavg 1.27℃


**§15-d. 장면 고정효과 회귀 — β 추정**

`Tair ~ LST + C(scene)`. 장면 FE가 "그날 그 지역 전체의 더위"를 흡수하므로 β는 **순수 공간 기울기**다. SE는 관측소 클러스터(같은 관측소 반복 등장).

기온 지표를 **Tmax·Tavg 둘 다** 돌린다 — Landsat 통과는 10~11시라 일최고(14~16시)와 시각이 어긋나고, 어느 쪽이 더 맞는지는 데이터가 답해야 한다.

In [95]:
PANEL['ASOS'].head(2)

,scene,date,path,row,지점,LST_30m,LST_100m,LST_500m,LST_1000m,지점명,Tavg,Tmax,Tmin,net,scene_id
0,LC08_L2SP_114034_20200820_20200905_02_T1,2020-08-20,114,034,130,NaN,NaN,NaN,36.774542,울진,25.5,30.0,22.4,ASOS,LC08_L2SP_114034_20200820_20200905_02_T1
1,LC08_L2SP_114034_20200820_20200905_02_T1,2020-08-20,114,034,272,36.934447,37.249295,37.756392,37.326541,영주,26.0,32.8,20.2,ASOS,LC08_L2SP_114034_20200820_20200905_02_T1


In [96]:
PANEL['AWS'].head(2)

,scene,date,path,row,지점,LST_30m,LST_100m,LST_500m,LST_1000m,지점명,Tavg,Tmax,Tmin,net,scene_id
0,LC08_L2SP_114034_20200820_20200905_02_T1,2020-08-20,114,034,837,37.108766,36.920189,36.337656,37.464763,이산,25.9,32.6,20.5,AWS,LC08_L2SP_114034_20200820_20200905_02_T1
1,LC08_L2SP_114034_20200820_20200905_02_T1,2020-08-20,114,034,814,37.374232,37.332499,36.686838,35.251103,부석,25.8,32.0,19.8,AWS,LC08_L2SP_114034_20200820_20200905_02_T1


In [ ]:
# ── §15-d. β 추정 — 관측망별 · 반경별 · 기온지표별 ──
import statsmodels.formula.api as smf

BETA={}
if not len(SP):
    print('⏸ §15-c 선행 필요')
else:
    # 헤더 출력: 관측망, 기온지표(Tmax/Tavg), 버퍼반경, Beta계수, 95% 신뢰구간, p-value, 샘플수, 관측소수
    print(f"{'관측망':7}{'기온':6}{'반경':>7}{'β(기온℃/지표℃)':>16}{'95%CI':>18}{'p':>9}{'n':>7}{'지점':>6}")

    # 1. 관측망(ASOS, AWS) x 기온지표(Tmax, Tavg) x 버퍼반경(30m~1000m) 조합별 패널 회귀 분석
    for net in PANEL:
        P=PANEL[net]
        if not len(P): continue
        for tv in ['Tmax','Tavg']:
            for R in RADII:
                # 결측치 제거 및 분석 대상 컬럼 선별
                d=P[['scene_id','지점',tv,f'LST_{R}m']].dropna().rename(columns={f'LST_{R}m':'LST'})

                # 데이터 유효성 검증: 관측 데이터 100개 미만이거나 위성 장면이 5개 미만이면 스킵
                if len(d)<100 or d.scene_id.nunique()<5: continue

                # [고정효과 회귀모델]
                # - C(scene_id): 위성 관측 시각/날짜별 대기 환경을 고정하는 패널 고정효과(Fixed Effects)
                # - cov_type='cluster': 동일 지점 관측소의 시간적 자기상관을 통제하기 위해 '지점' 단위 군집 강건 오차 적용
                m=smf.ols(f'{tv} ~ LST + C(scene_id)',data=d).fit(cov_type='cluster',cov_kwds={'groups':d['지점']})

                b=m.params['LST']; ci=m.conf_int().loc['LST']; p=m.pvalues['LST']

                # 분석 결과 저장: (관측망, 기온지표, 반경) -> (beta, ci_low, ci_high, sample_size, station_count)
                BETA[(net,tv,R)]=(b,ci[0],ci[1],len(d),d.지점.nunique())
                print(f"{net:7}{tv:6}{R:>6}m{b:>16.3f}{f'{ci[0]:.3f}~{ci[1]:.3f}':>18}{p:>9.1e}{len(d):>7}{d.지점.nunique():>6}")
    if BETA:
        # 2. [최적 모델 채택] 
        # - 후보군: Tmax를 종속변수로 사용한 모델들
        # - 채택 기준: (신뢰구간 폭 / |β|) 비율이 최소인 모델 (= 추정 오차 비율이 가장 적고 식별력이 가장 높은 조합)
        cands=[k for k in BETA if k[1]=='Tmax']
        best=min(cands,key=lambda k:(BETA[k][2]-BETA[k][1])/max(abs(BETA[k][0]),1e-9))

        b,lo,hi,n,ns=BETA[best]
        print(f'\n★ 채택 β = {b:.3f} ℃기온/℃지표  (95%CI {lo:.3f}~{hi:.3f})')
        print(f'   관측망 {best[0]} · {best[1]} · 반경 {best[2]}m · n={n:,} · 지점 {ns}개')

        # 3. 관측망(ASOS vs AWS) 간 교차 정합성 검증
        if len([k for k in BETA if k[1]=='Tmax'])>1:
            print('\n관측망간 일치도(Tmax, 같은 반경) — 크게 다르면 어느 한쪽 편의 의심:')
            for R in RADII:
                vs={net:BETA[(net,'Tmax',R)][0] for net in PANEL if (net,'Tmax',R) in BETA}
                if len(vs)>1: 
                    print(f"   반경 {R:>4}m: "+" · ".join(f"{k} {v:+.3f}" for k,v in vs.items()))

        # 4. [🔖 2026-07-25 이동] 「산단 지표ΔT → 기온ΔT 환산」 표는 여기서 빼서 §15-h로 옮겼다.
        #    이유: 이 자리의 환산은 **채택 β 하나로만** 곱한 한 점 추정인데, 바로 아래 §15-e(회귀희석
        #    기각 못 함)·§15-f(시각별 β)·§15-g(피복 층화 β)가 그 값을 정정한다. 독자가 정정 전 값을
        #    먼저 보게 두면 그 숫자가 머리에 남는다 → 정정을 다 읽은 뒤에 범위로 제시한다.
        print()
        print('→ 산단 ΔLST의 기온 환산은 §15-h(§15-g 뒤)에서 **β 범위**로 낸다. 여기서는 β 추정까지만.')


관측망    기온         반경      β(기온℃/지표℃)             95%CI        p      n    지점
ASOS   Tmax      30m           0.181       0.112~0.249  2.5e-07    816    82
ASOS   Tmax     100m           0.181       0.114~0.248  1.2e-07    831    83
ASOS   Tmax     500m           0.140       0.066~0.214  2.2e-04    839    82
ASOS   Tmax    1000m           0.138       0.058~0.218  7.3e-04    874    82
ASOS   Tavg      30m           0.107       0.056~0.158  4.5e-05    815    82
ASOS   Tavg     100m           0.111       0.059~0.163  3.3e-05    830    83
ASOS   Tavg     500m           0.159       0.100~0.218  1.2e-07    838    82
ASOS   Tavg    1000m           0.197       0.149~0.244  4.4e-16    873    82
AWS    Tmax      30m           0.249       0.211~0.286  3.1e-38   2935   346
AWS    Tmax     100m           0.258       0.220~0.296  2.5e-40   2953   347
AWS    Tmax     500m           0.258       0.217~0.298  9.0e-36   3135   357
AWS    Tmax    1000m           0.253       0.211~0.295  5.1e-32   3331   364

> ### 이 아래 세 절이 β를 정정한다 — 환산은 그다음(§15-h) `[🔖 2026-07-25 순서 정정]`
>
> 위 §15-d는 **β 추정까지만** 한다. 산단 ΔLST를 기온으로 옮기는 환산표는 아래 정정을 모두 거친 뒤 **§15-h**에서 범위로 낸다(이전 판은 여기서 단일 β로 곱한 한 점 추정을 먼저 보여줬다).
>
> - **§15-e** — 두 관측망 β가 1.6배 어긋나는 원인 판별 → **회귀희석을 기각하지 못한다.** 채택 β는 눌린 하한일 수 있어 희석보정값(0.459)을 상한으로 병기한다.
> - **§15-f** — 시각 불일치는 오차가 아니라 결합강도의 일변화. 일최고 β ≈ 14~15시 β라 **Tmax가 옳은 지표.**
> - **§15-g** — 피복 층화 β. 녹지 0.287 > 주거혼합 0.222, **불투수 관측소는 0개**(산단 적용은 외삽).
>
> → 논증 문서의 산단별 기온 ΔT는 단일값이 아니라 **범위**로 제시한다(예: 포항 +3.1~7.9℃).

**§15-e. 관측망 대조 — ASOS β와 AWS β가 왜 다른가** `[🔖 2026-07-20]`

같은 반경(100m)에서 두 관측망의 β가 **0.181 vs 0.293으로 1.6배** 어긋난다. 하나를 고르기 전에 원인을 짚는다.

| 가설 | 검정 | 결과 |
|---|---|---|
| ① **회귀희석** — LST 측정·대표성 오차가 기울기를 0쪽으로 끌어당기고, 분산이 작은 ASOS가 더 눌린다 | 필요 σ_e 역산 후 **독립 대조** | **기각 못 함** — σ_e = 2.31℃면 정확히 설명되고, 반경 선택만 바꿔도 LST가 SD 2.35℃ 흔들림 |
| ② **AWS 설치 편의** — 옥상·도심 설치라 기온을 더 뜨겁게 읽는다 | 5km 이내 ASOS↔AWS 동일일자 일최고기온 대조 (4짝·2,194일) | **기각** — 평균차 **−0.05℃**(중앙 +0.20, SD 1.06) |

**①의 함의가 크다.** 희석이 살아있다면 **두 β 모두 참값보다 눌린 하한**이다. 보정하면 β ≈ 0.505로 지금의 1.7배가 된다. 다만 σ_e는 두 관측망을 맞추도록 **역산한 값**이라 독립 측정이 아니다 — 보정치는 가정 의존이다.

> ⚠ **이 블록은 두 번 틀렸다. 경위를 남긴다.**
> **1차** — 역산식 분모를 `(r·vA − vW)`로 뒤집어 써서 음수가 나왔는데 `abs()`로 가렸다. 값(2.31℃)은 우연히 맞았으나 *"비현실적이므로 기각"*이라는 결론이 **하드코딩**돼 있었다.
> **2차** — 음수의 원인을 캐는 대신 모형을 바꿔 재유도했다. `λ = σ²ₜᵣᵤₑ/(σ²ₜᵣᵤₑ+σ²ₑ)` 식에 **관측** SD를 집어넣은 것이 오류다. 그러면 β비에 `vA/vW`라는 하한이 생기는 것처럼 보이고, 그 하한을 근거로 또 "기각"했다. **존재하지 않는 하한을 만들어 결론을 맞춘 셈.**
> **교훈** — 두 번 다 "기각"이라는 결론을 먼저 정해두고 근거를 갈아끼웠다. 현재 판은 시뮬레이션(참 β=0.50, σ_e=2.31℃ → β̂ 0.179/0.290, 관측 0.181/0.293과 일치)으로 검증했고, 코드도 **해가 유효 구간을 벗어나면 "기각"이 출력되도록 조건부**로 바꿨다.

**올바른 모수화**: 우리가 잰 SD는 *관측* LST의 SD이므로 `V = σ²ₜᵣᵤₑ + σ²ₑ`이고 `β̂ = β(V−e)/V`. β비 `[(V_A−e)/V_A]/[(V_W−e)/V_W]`는 e가 커지면 **0까지 내려간다 — 하한이 없다.**

→ 결론: β를 **0.181~0.293(직접 측정, 하한 성격)**으로 보고하되, 희석 보정치 **≈0.505**를 상한 후보로 병기한다. 두 편의가 서로 반대 방향임을 명시한다 — **희석은 β를 과소평가시키고, 극단 지표로의 외삽 포화는 과대평가시킨다.**

> `[🔖 2026-07-23 기준 명문화 — 사용자 지적]`
> - 관측망 대조의 "크게 다르면"의 기준: **상대차 |β_ASOS−β_AWS|/β_AWS ≤ 20%** 그리고 **β∈[0.18, 0.26] 전 범위에서 결론(부호·부지 순위) 유지** = 통과. 실측 상대차 ~13% → 통과. 형식 가설검정이 아니라 **경향·범위 강건성 확인**이 목적.
> - "산단 지표 ΔT → 기온 ΔT 환산" 표와 §15-e(3)의 상·하한은 모두 **부지(폴리곤 R0 · 점부지 0-1km) 기준**이다. 링별(거리별) 환산은 §11-b(2)·(3)에서 감쇠 형태와 함께 다루며, 이 상·하한 값 자체는 감쇠 함수에 들어가지 않는다.

In [98]:
SP.head(1)

,scene,date,path,row,지점,LST_30m,LST_100m,LST_500m,LST_1000m,지점명,Tavg,Tmax,Tmin,net,scene_id
0,LC08_L2SP_114034_20200820_20200905_02_T1,2020-08-20,114,034,130,NaN,NaN,NaN,36.774542,울진,25.5,30.0,22.4,ASOS,LC08_L2SP_114034_20200820_20200905_02_T1


In [ ]:
# ── §15-e. 관측망 대조 + β 범위로 환산 (강건성) ──
# [🔖 2026-07-21] 트레일링 산문 → 「§15 규약」 markdown pointer로 이동
from scipy.spatial import cKDTree

# -----------------------------------------------------------------------------
# (1) 회귀희석 가설 검증
# 핵심: ASOS와 AWS 간 β 값 차이가 LST 측정 오차(e) 때문에 생긴 현상인지 역산하여 확인
# -----------------------------------------------------------------------------

# ASOS 및 AWS의 특정 조건(Tmax, 100m)에 대한 회귀계수(BETA)가 존재하는지 확인
if ('ASOS','Tmax',100) in BETA and ('AWS','Tmax',100) in BETA:

    # 1. 필요 파라미터 추출 및 계산
    bA=BETA[('ASOS','Tmax',100)][0] # ASOS 관측 회귀계수 (β_A)
    bW=BETA[('AWS','Tmax',100)][0] # AWS 관측 회귀계수 (β_W)

    # 각 관측망의 LST 100m 표준편차 중앙값을 제곱하여 관측 분산(V_A, V_W) 계산
    VA=PANEL['ASOS'].groupby('scene')['LST_100m'].std().median()**2
    VW=PANEL['AWS'].groupby('scene')['LST_100m'].std().median()**2

    # 두 회귀계수의 비율 r = β_A / β_W
    r=bA/bW

    # 회귀희석 수식으로부터 역산한 측정오차 분산 (e = σ²_e)
    # 수식: e = (V_A * V_W * (1 - r)) / (V_W - r * V_A)
    e=VA*VW*(1-r)/(VW-r*VA)

    # 기초 관측 정보 출력
    print(f'[가설①] 회귀희석 — 관측 LST 분산: ASOS {VA:.2f} / AWS {VW:.2f} (SD {VA**.5:.2f}/{VW**.5:.2f}℃)')
    print(f'  관측 β비 = {bA:.3f}/{bW:.3f} = {r:.4f}')

    # 2. 역산된 측정오차 분산(e)의 물리적 유효성 판정
    # 조건: 측정오차 분산 e는 양수여야 하며, 관측 분산 VA보다 작아야 함 (0 < e < VA)
    if not (0 < e < VA):
        # 유효하지 않은 해일 경우: 회귀희석만으로는 현상을 설명할 수 없음
        print(f'  → 유효한 측정오차 해 없음(e={e:.2f}) → 희석으로는 설명 불가, 가설 기각')
        BETA_DIS=None
    else:
        # 유효한 해가 존재하는 경우: 측정오차 표준편차 se = sqrt(e)
        se=e**0.5
        print(f'  → 이를 정확히 설명하는 측정오차 σ_e = {se:.2f}℃ (검산 β비 {((VA-e)/VA)/((VW-e)/VW):.4f})')

        # 3. 독립적 대조군(공간 대표성 오차)과 물리적 규모 비교
        # LST 30m와 500m 차이의 표준편차를 통해 공간 해상도 변경에 따른 실질 오차 측정
        _rep=(SP['LST_30m']-SP['LST_500m']).dropna().std()
        print(f'  독립 대조 — 반경 30m vs 500m LST 차이 SD = {_rep:.2f}℃ (기온계가 느끼는 지표 규모를 모르는 데서 오는 대표성 오차)')

        # 역산된 σ_e와 실제 독립 대조 오차(_rep)의 자릿수/크기 비교
        print(f'  → 역산 {se:.2f}℃와 같은 자릿수 ⇒ **가설 기각 못 함**. 두 β 모두 눌린 값으로 봐야 한다.')

        # 4. 희석 효과를 보정한 '참값 추정 회귀계수(β)' 계산
        # 보정 공식: β_true = β_obs * (V / (V - e))
        BETA_DIS=(bA*VA/(VA-e), bW*VW/(VW-e))
        print(f'  희석 보정 β = {BETA_DIS[0]:.3f}(ASOS) / {BETA_DIS[1]:.3f}(AWS)')
        print(f'     ⚠ 두 값의 일치는 e를 그렇게 풀어서 생긴 것 — 독립 증거 아니다. σ_e 가정에 전적으로 의존.')

# ── §15-e2. σ_e 역산 재검토 — 0.459를 한 점 값으로 써도 되나 [🔖 2026-07-27 사용자 요청] ──
if BETA_DIS:
    print()
    print('§15-e2. 재검토 — 이 보정이 얼마나 σ_e에 매달려 있나')
    _VA,_VW=VA,VW; _bA,_bW=bA,bW
    _corr=lambda b,V,s: b*V/(V-s**2) if s**2 < V else float('nan')
    print(f"  {'σ_e(℃)':>8}{'ASOS 보정β':>12}{'AWS 보정β':>12}{'두 값 차':>10}   비고")
    for _s in (1.0,1.5,2.0,2.24,2.5,2.7):
        _a,_w=_corr(_bA,_VA,_s),_corr(_bW,_VW,_s)
        _tag=' ← 현행 채택(둘이 일치하도록 푼 값)' if abs(_s-2.24)<0.01 else (' ← ASOS 분산 초과, 해 없음' if _s**2>=_VA else '')
        print(f'  {_s:>8.2f}{_a:>12.3f}{_w:>12.3f}{abs(_a-_w):>10.3f}{_tag}')
    print(f'  → **σ_e 0.5℃ 차이가 보정 β를 2배 이상 흔든다**(2.0→0.35 vs 2.5→0.74).')
    print(f'     보정식 β/(1−σ_e²/V)는 σ_e²가 V({_VA:.1f})에 가까워질수록 발산한다. 지금 σ_e²={_s9e if False else 2.24**2:.1f}로 이미 V의 {2.24**2/_VA*100:.0f}%다.')

    print()
    print('  독립 σ_e 대리를 하나가 아니라 여러 개 본다 (반경쌍 대표성 오차 SD, ℃)')
    _pairs=[(30,100),(30,500),(30,1000),(100,500),(100,1000),(500,1000)]
    _cands=[]
    print(f"  {'반경쌍':>12}{'전체':>8}{'ASOS':>8}{'AWS':>8}")
    for _r1,_r2 in _pairs:
        _c1,_c2=f'LST_{_r1}m',f'LST_{_r2}m'
        if _c1 not in SP.columns or _c2 not in SP.columns: continue
        _all=(SP[_c1]-SP[_c2]).dropna().std()
        _row=f'  {f"{_r1}-{_r2}m":>12}{_all:>8.2f}'
        for _nt in ['ASOS','AWS']:
            _P=PANEL.get(_nt)
            _v=(_P[_c1]-_P[_c2]).dropna().std() if _P is not None and _c1 in _P.columns else float('nan')
            _row+=f'{_v:>8.2f}'
        _cands.append(_all); print(_row)
    _lo9e,_hi9e=min(_cands),max(_cands)
    print(f'  → 대리값 범위 {_lo9e:.2f}~{_hi9e:.2f}℃. 현행이 고른 30-500m({_cands[1]:.2f})는 그 안의 **한 점**이다.')
    print(f'     역산값 2.24℃와 정확히 일치한 것은 인상적이지만, 다른 쌍을 골랐으면 다른 답이 나온다:')
    print(f"  {'σ_e 대리':>10}{'ASOS β':>10}{'AWS β':>10}")
    for _s in (_lo9e,_cands[1],_hi9e):
        print(f'  {_s:>10.2f}{_corr(_bA,_VA,_s):>10.3f}{_corr(_bW,_VW,_s):>10.3f}')
    print()
    print('  [판정] **0.459는 한 점 값으로 인용할 수 없다.** 세 가지 이유:')
    print('    (a) 두 관측망 σ_e가 **같다**는 가정 위에서 푼 값이다 — 검정하지 않았고 검정할 수도 없다(미지수 2·식 1).')
    print('    (b) 보정식이 σ_e에 극단적으로 민감하다(위 표: ±0.25℃ → β 2배).')
    print('    (c) 독립 대리도 반경쌍 선택에 따라 범위를 갖는다.')
    print('  [그래서] 지금 규약대로 **범위(하한 채택 β ~ 상한 보정 β)로만** 쓰는 것이 맞다.')
    print('    다만 상한을 0.459 하나로 고정해 부르지 말고 **"상한 후보"**라고 부른다. 문서·지도 표기도 그렇게 맞춘다.')
    print('    ⚠ 이 재검토는 우리 추정 상한을 **약화**하는 쪽이다(상한이 더 불확실해짐). 하한 0.258은 직접 측정이라 그대로다.')


[가설①] 회귀희석 — 관측 LST 분산: ASOS 8.27 / AWS 11.41 (SD 2.88/3.38℃)
  관측 β비 = 0.181/0.258 = 0.7020
  → 이를 정확히 설명하는 측정오차 σ_e = 2.24℃ (검산 β비 0.7020)
  독립 대조 — 반경 30m vs 500m LST 차이 SD = 2.24℃ (기온계가 느끼는 지표 규모를 모르는 데서 오는 대표성 오차)
  → 역산 2.24℃와 같은 자릿수 ⇒ **가설 기각 못 함**. 두 β 모두 눌린 값으로 봐야 한다.
  희석 보정 β = 0.459(ASOS) / 0.459(AWS)
     ⚠ 두 값의 일치는 e를 그렇게 풀어서 생긴 것 — 독립 증거 아니다. σ_e 가정에 전적으로 의존.

§15-e2. 재검토 — 이 보정이 얼마나 σ_e에 매달려 있나
    σ_e(℃)    ASOS 보정β     AWS 보정β     두 값 차   비고
      1.00       0.206       0.282     0.077
      1.50       0.248       0.321     0.072
      2.00       0.350       0.397     0.046
      2.24       0.460       0.460     0.000 ← 현행 채택(둘이 일치하도록 푼 값)
      2.50       0.741       0.570     0.172
      2.70       1.531       0.713     0.818
  → **σ_e 0.5℃ 차이가 보정 β를 2배 이상 흔든다**(2.0→0.35 vs 2.5→0.74).
     보정식 β/(1−σ_e²/V)는 σ_e²가 V(8.3)에 가까워질수록 발산한다. 지금 σ_e²=5.0로 이미 V의 61%다.

  독립 σ_e 대리를 하나가 아니라 여러 개 본다 (반경쌍 대표성 오차 SD, ℃)
           반경쌍      전체    ASOS     AWS
       30-100m    0.5

ⓐ 회귀희석 (Regression Dilution Bias)독립변수(LST)에 측정오차 $e$가 존재할 때, 관측된 분산 $V$는 참값의 분산 $\sigma^2_{\text{true}}$와 오차 분산 $\sigma^2_e$의 합입니다.$$V = \sigma^2_{\text{true}} + \sigma^2_e$$이때 관측된 회귀계수 $\hat{\beta}$는 참 회귀계수 $\beta$보다 0에 가깝게 축소(희석)되며, 그 비율(신뢰도 계수 $\lambda$)은 다음과 같습니다.$$\hat{\beta} = \beta \cdot \lambda = \beta \cdot \frac{V - \sigma^2_e}{V}$$ⓑ 두 관측망의 $\beta$ 비율 ($r$)ASOS와 AWS의 회귀계수 비율 $r = \frac{\hat{\beta}_A}{\hat{\beta}_W}$는 각 관측망의 오차 비율에 의해 결정됩니다.$$r = \frac{\frac{V_A - e}{V_A}}{\frac{V_W - e}{V_W}}$$ⓒ 역산된 측정오차 분산 ($e$)위 식을 오차 분산 $e$ ($\sigma^2_e$)에 대해 정리하면 코드의 계산식이 도출됩니다.$$e = \frac{V_A \cdot V_W \cdot (1 - r)}{V_W - r \cdot V_A}$$

In [100]:
# (2) AWS 설치 편의 가설 — 5km 이내 동일일자 일최고기온 대조
# 목적: ASOS 근처 5km 이내의 AWS 데이터를 매칭하여 AWS의 체계적인 온도 측정 편향(더 뜨거운 곳 설치 유무)을 검증

# 1. 특정 기준일자(2022-07-01) 기준 가동 중인 ASOS 및 AWS 관측소 목록 추출
_A=stn_on(NET['ASOS'][0],pd.Timestamp('2022-07-01')); _W=stn_on(NET['AWS'][0],pd.Timestamp('2022-07-01'))

# 2. 위·경도(EPSG:4326)를 미터(m) 단위 거리 계산이 가능한 한국 투영좌표계(UTM-K, EPSG:5179)로 변환
_gA=gpd.GeoSeries(gpd.points_from_xy(_A.lon,_A.lat),crs=4326).to_crs(5179)
_gW=gpd.GeoSeries(gpd.points_from_xy(_W.lon,_W.lat),crs=4326).to_crs(5179)

# 3. 공간 인덱싱(cKDTree)을 이용해 각 ASOS 관측소에서 가장 가까운 AWS 관측소와의 거리(_d) 및 인덱스(_i) 검색
_d,_i=cKDTree(np.c_[_gW.x,_gW.y]).query(np.c_[_gA.x,_gA.y])

# 4. ASOS-AWS 매칭 쌍(Pair) 데이터프레임 생성 및 직선거리 5,000m(5km) 이내 짝만 필터링
_pair=pd.DataFrame({'asos':_A.지점,'aws':_W.지점.values[_i],'거리m':_d.round(0)})
_pair=_pair[_pair.거리m<=5000]

# 5. 매칭된 관측소 짝에 대해 동일 일자(date)의 일최고기온(Tmax) 데이터 병합 (결측치 제거)
_m=(NET['ASOS'][1][['지점','date','Tmax']].rename(columns={'지점':'asos','Tmax':'T_asos'}).merge(_pair,on='asos')
      .merge(NET['AWS'][1][['지점','date','Tmax']].rename(columns={'지점':'aws','Tmax':'T_aws'}),on=['aws','date'])
      .dropna(subset=['T_asos','T_aws']))

# 6. 관측 기온 편차 계산 (diff = AWS 최고기온 - ASOS 최고기온)
_m['diff']=_m.T_aws-_m.T_asos

# 7. 분석 결과 출력 (짝 개수, 일수, 평균/중앙값/표준편차) 및 가설 판정
print(f"\n[가설②] AWS 설치 편의 — 5km 이내 짝 {_pair.shape[0]}개 중 동일일자 자료 있는 {_m.asos.nunique()}짝 · {len(_m):,}일")
print(f"  AWS − ASOS 일최고기온: 평균 {_m['diff'].mean():+.2f}℃ · 중앙 {_m['diff'].median():+.2f}℃ · SD {_m['diff'].std():.2f}")
print(f"  → 0 근처 = AWS가 체계적으로 뜨겁게 읽지 않음. 가설 기각")
print(f"  ⚠ 짝이 {_m.asos.nunique()}개뿐 — 표본 작음. AWS 지점을 더 받으면 재검정 권장")


[가설②] AWS 설치 편의 — 5km 이내 짝 13개 중 동일일자 자료 있는 8짝 · 4,394일
  AWS − ASOS 일최고기온: 평균 -0.04℃ · 중앙 +0.10℃ · SD 1.00
  → 0 근처 = AWS가 체계적으로 뜨겁게 읽지 않음. 가설 기각
  ⚠ 짝이 8개뿐 — 표본 작음. AWS 지점을 더 받으면 재검정 권장


In [ ]:
# (3) β 범위로 환산 — 결론이 범위 전체에서 유지되나
# [정정 07-20] 헤드라인 범위는 반경 100m 고정(두 관측망이 같은 정의로 비교되는 지점).
#   [🔖 2026-07-25 뜻 풀이] β는 관측소 주변 **몇 m의 LST를 평균했느냐**(30·100·500·1000m)에 따라 달라진다.
#   ASOS와 AWS의 β를 맞대려면 같은 반경이어야 공정하다 → 100m로 고정해 비교한다.
#   100m를 고른 이유: §15-b의 반경 탐색에서 기온 설명력이 가장 좋았고(유효 footprint),
#   두 관측망 모두 표본이 충분한 반경이기 때문. 다른 반경을 쓰면 β 절대값은 바뀌지만 두 망의 격차 구조는 같다.
#   반경까지 섞은 전 조합 envelope는 참고로 병기 — 반경 차이와 관측망 차이를 뭉뚱그리지 않는다.

# 1. 헤드라인 β 범위 추출 (100m 반경 기준 ASOS와 AWS의 β 값)
_bl=BETA[('ASOS','Tmax',100)][0]; _bh=BETA[('AWS','Tmax',100)][0]

# 2. 모든 반경/조건 조합에서의 전체 β 최소·최대 범위(Envelope) 추출
_el=min(BETA[k][0] for k in BETA if k[1]=='Tmax'); _eh=max(BETA[k][0] for k in BETA if k[1]=='Tmax')

print()
print("=== 산단 지표ΔT → 기온ΔT ===")
print(f"  헤드라인 β 범위(반경 100m): {_bl:.3f}(ASOS) ~ {_bh:.3f}(AWS)")
print(f"  참고 전 조합 envelope     : {_el:.3f} ~ {_eh:.3f}")

# 앞선 회귀희석 보정값(BETA_DIS)이 존재하는 경우, 보정된 β의 범위를 출력
if BETA_DIS: 
    print(f"  희석 보정 시(σ_e 가정 의존)  : {min(BETA_DIS):.3f} ~ {max(BETA_DIS):.3f}")

print()
# 테이블 헤더 출력
print(f"{'부지':14}{'지표ΔT':>8}{'기온ΔT 하한':>12}{'기온ΔT 상한':>12}{'희석보정':>10}")

# 3. 각 부지별 평균 지표면 온도 변화량(ΔLST)을 추출하여 기온 변화량(ΔT) 범위로 환산
for k,v in _c.groupby('부지')['공장부지ΔLST'].mean().sort_values(ascending=False).items():
    # v: 각 부지의 평균 ΔLST 값
    # 희석 보정이 계산된 경우 보정된 최대 β를 곱하고, 아니면 '-' 처리
    _d=f"{v*max(BETA_DIS):>9.2f}℃" if BETA_DIS else f"{'-':>10}"

    # [출력] 부지명, 평균 지표ΔT, 기온ΔT 하한(v * _bl), 기온ΔT 상한(v * _bh), 희석보정값(_d)
    print(f"{k:14}{v:>7.1f}℃{v*_bl:>11.2f}℃{v*_bh:>11.2f}℃"+_d)
    
print("\n→ §9 대조·해석·규율은 아래 「§15 결과·해석 규약」 markdown 참조 (숫자만 위에 출력)")


=== 산단 지표ΔT → 기온ΔT ===
  헤드라인 β 범위(반경 100m): 0.181(ASOS) ~ 0.258(AWS)
  참고 전 조합 envelope     : 0.138 ~ 0.258
  희석 보정 시(σ_e 가정 의존)  : 0.459 ~ 0.459

부지                지표ΔT     기온ΔT 하한     기온ΔT 상한      희석보정
포항·제철            17.3℃       3.12℃       4.45℃     7.93℃
심팩 포항            14.3℃       2.59℃       3.69℃     6.59℃
구미·전자            14.0℃       2.54℃       3.61℃     6.44℃
당진1철강            12.6℃       2.27℃       3.24℃     5.78℃
울산미포             12.1℃       2.19℃       3.12℃     5.56℃
동해북평             11.2℃       2.02℃       2.88℃     5.13℃
온산               11.2℃       2.02℃       2.87℃     5.13℃
광양·제철            10.2℃       1.85℃       2.63℃     4.69℃
여수·석화             8.5℃       1.54℃       2.20℃     3.92℃
현대제철 인천           5.9℃       1.07℃       1.52℃     2.71℃
동국제강 인천           5.5℃       0.99℃       1.41℃     2.51℃

→ §9 대조·해석·규율은 아래 「§15 결과·해석 규약」 markdown 참조 (숫자만 위에 출력)


**§15-f. 시각별 β — 시간자료로 '시각 불일치' 한계를 해소** `[🔖 2026-07-20]`

§15-e까지는 *"Landsat 통과 10~11시 vs 일최고기온 14~16시"*를 **측정오차 성분**으로 취급하고, 시간자료를 쓰면 σ_e가 줄어 두 관측망 β가 **수렴할 것**이라 예측했다. ASOS·AWS 시간자료(2020~2025 JJA, 각 127만·114만 행)로 검정한 결과 **예측은 빗나갔고, 전제가 틀렸다.**

동일 표본(11시 기온이 있는 행만)에서 시각만 바꿔 추정:

| 기온 지표 | ASOS β | AWS β | β비 | 관측소간 기온 SD (ASOS) |
|---|---:|---:|---:|---:|
| 10시 | 0.059 | 0.205 | 0.287 | 1.07℃ |
| 11시 (통과시각) | 0.114 | 0.226 | 0.504 | 1.10℃ |
| 12시 | 0.129 | 0.238 | 0.545 | 1.13℃ |
| **14시** | **0.182** | **0.292** | 0.624 | 1.47℃ |
| **15시** | **0.180** | **0.298** | 0.603 | 1.62℃ |
| **일최고** | **0.181** | **0.275** | 0.658 | 1.38℃ |

**발견 1 — β는 하루 동안 단조 상승한다.** ASOS는 10시 대비 15시가 3배. 기온의 공간 편차도 함께 커진다(1.07→1.62℃). 오전에는 대기가 잘 섞여 지표의 국소 차이가 공기에 새겨지지 않다가, 오후로 갈수록 각인된다. → **시각 차이는 측정오차가 아니라 결합강도의 실제 일변화(diurnal)다.**

**발견 2 — 일최고 β ≈ 14~15시 β.** 0.181 vs 0.180~0.182(ASOS), 0.275 vs 0.292~0.298(AWS). 일최고기온은 곧 **오후의 결합강도**다.

**발견 3 — β비는 어느 시각에서도 수렴하지 않는다.** 0.29~0.66. → **시각 불일치는 ASOS↔AWS 격차의 원인이 아니다.** §15-e에서 잰 **공간 대표성 오차(반경 30m↔500m에서 SD 2.35℃)**가 유력한 설명으로 남는다.

#### 결론: Tmax 사용이 옳다 — 한계 항목에서 해소로 이동
우리 목적에 맞는 지표는 **일최고기온**이다:
- §9-b의 ×1.6/℃ 곡선이 **일최고기온** 기반
- 온열질환이 **14~16시**에 몰림
- 11시 LST는 *"그 지표면이 얼마나 뜨거워지는 성질인가"*의 **대리변수**로 쓰는 것이지 11시 기온을 예측하려는 게 아니다

→ §15-e 한계 목록의 *"시각 불일치"* 항목은 **해소**한다. 다만 이는 원래 설계가 우연히 맞았다는 뜻이 아니라, **검정해보니 맞았다**는 뜻이다.

> ⚠ **자료 제약**: AWS 시간자료는 84~88지점만 확보(일자료는 261지점). 위 AWS 수치는 78지점 부분표본이다. 다만 같은 부분표본의 AWS Tmax β = 0.275로 전체표본 0.293과 근사하므로 표본 구성 효과는 작다. 전 지점 시간자료 확보 시 재확인 권장.

In [ ]:
# ── §15-f. 시각별 β (ASOS·AWS 시간자료) ──
# [🔖 2026-07-21] 트레일링 산문 → 「§15 규약」 markdown pointer로 이동
# 자료: ASOS+AWS/OBS_{ASOS,AWS}_TIM_*.csv — 연도는 파일 내용에서 읽는다(파일명 순서 가정 금지).

def load_hourly(pat):
    fr=[]
    # 경로 패턴에 매칭되는 파일들을 정렬하여 순회
    for f in sorted(glob.glob(pat)):
        # 인코딩 방식 차이로 인한 오류를 방지하기 위해 여러 인코딩을 순차 시도
        for e in ('cp949','utf-8-sig','utf-8'):
            try: d=pd.read_csv(f,encoding=e); break
            except Exception: continue

        # 컬럼명의 공백 제거 및 '기온' 키워드가 들어간 컬럼을 'T'로 표준화
        d.columns=[str(c).strip() for c in d.columns]
        d=d.rename(columns={[c for c in d.columns if '기온' in c][0]:'T'})

        # '일시' 컬럼을 datetime 객체로 변환 (잘못된 형식은 NaT 처리)
        d['dt']=pd.to_datetime(d['일시'],errors='coerce')

        # 필수 컬럼만 추출하고 날짜 변환이 실패한 행(NaT) 제거 후 리스트에 추가
        fr.append(d[['지점','dt','T']].dropna(subset=['dt']))

    # 불러온 파일이 없으면 None 반환
    if not fr: return None

    # 전체 데이터를 하나로 결합하고 날짜(date) 및 시간(hour) 파생 변수 생성
    h=pd.concat(fr,ignore_index=True); h['date']=h.dt.dt.date; h['hour']=h.dt.dt.hour
    return h

# 1. 관측망별(ASOS, AWS) 시간별 기온 데이터 로드
HR={n:load_hourly(f'ASOS+AWS/OBS_{n}_TIM_*.csv') for n in ['ASOS','AWS']}
# 유효한 데이터가 로드된 항목만 필터링
HR={k:v for k,v in HR.items() if v is not None}

if not HR:
    print('⏸ 시간자료 없음 — ASOS+AWS/ 폴더 확인')
else:
    # 2. 로드된 데이터 요약 정보 출력 (행 수, 지점 수, 기간)
    for k,v in HR.items(): 
        print(f'{k} 시간자료: {len(v):,}행 · 지점 {v.지점.nunique()}개 · {v.date.min()}~{v.date.max()}')

    # 분석 대상 시각 설정 (위성 통과시각인 KST 11시 및 비교용 시각 10, 12, 14, 15시)
    # 전 101장 통과시각이 KST 10.95~11.18시 → 11시 정시가 통과시각. 10·12·14·15시는 일변화 비교용.
    # [🔖 2026-07-27 사용자 지적] 13시가 빠져 있었고 **빠진 이유가 어디에도 기록돼 있지 않다**.
    #   의도적 제외가 아니라 목록을 성기게 고른 흔적으로 보인다 → 13시 추가(일변화 곡선의 구멍 제거).
    HOURS=[10,11,12,13,14,15]

    # 표 헤더 출력
    print(f"\n{'관측망':7}{'기온지표':>9}{'β':>9}{'95%CI':>18}{'SD':>8}{'n':>7}{'지점':>6}")

    BETA_HR={} # 추정된 β 계수 저장용 디렉셔너리
    
    for net in HR:
        # 해당 관측망 데이터 복사 (SP: 위성-기상관측 연결 공간 패널 데이터)
        P=SP[SP.net==net].copy()

        # 각 시각(10, 11, 12, 14, 15시)의 기온 데이터를 [지점, 날짜] 기준으로 P에 Left Join
        for h in HOURS:
            P=P.merge(HR[net][HR[net].hour==h][['지점','date','T']].rename(columns={'T':f'T{h}'}),
                      on=['지점','date'],how='left')

        # ★ 표본 통제를 위해 위성 통과시각(11시)과 일최고기온(Tmax) 데이터가 모두 존재하는 동일 표본만 추출
        Q=P.dropna(subset=['T11','Tmax'])         # ★ 동일 표본 — 시각 효과와 표본 효과 분리

        # 4. 각 시각별 기온(T10~T15) 및 일최고기온(Tmax)에 대해 LST와의 회귀 분석 수행
        for y in [f'T{h}' for h in HOURS]+['Tmax']:
            # 회귀 분석에 사용할 데이터 정제 및 컬럼명 표준화
            d=Q[['scene','지점',y,'LST_100m']].dropna().rename(columns={y:'yv','LST_100m':'LST'})

            # 샘플 수가 부족할 경우(80개 미만) 회귀 분석 스킵
            if len(d)<80: continue

            # 다중회귀분석 실행: yv = β * LST + scene별 고정효과(C(scene))
            # 공간적/지점별 상관성을 고려하여 지점(지점) 기준 클러스터 강건 표준오차(Clustered Standard Errors) 적용
            m=smf.ols('yv ~ LST + C(scene)',data=d).fit(cov_type='cluster',cov_kwds={'groups':d['지점']})
            
            # LST 계수(β) 및 95% 신뢰구간(CI) 추출
            b=m.params['LST']; ci=m.conf_int().loc['LST']; BETA_HR[(net,y)]=b

            # 결과 한 줄 출력 (관측망, 기온지표, β, 95% CI, 표준편차 중위수, 샘플 수, 지점 수)
            print(f"{net:7}{y:>9}{b:>9.3f}{f'{ci[0]:.3f}~{ci[1]:.3f}':>18}"
                  f"{Q.groupby('scene')[y].std().median():>7.2f}℃{len(d):>7}{d.지점.nunique():>6}")
    
    # 5. ASOS와 AWS 간의 β 계수 비율 계산 및 출력 (관측망 간 민감도/일치도 비교)
    print('\n관측망 β비 (1에 가까울수록 일치):')
    for y in [f'T{h}' for h in HOURS]+['Tmax']:
        if ('ASOS',y) in BETA_HR and ('AWS',y) in BETA_HR:
            print(f"  {y:>6}: {BETA_HR[('ASOS',y)]/BETA_HR[('AWS',y)]:.3f}")
            
print("\n→ 해석은 아래 「§15 결과·해석 규약」 markdown 참조")


ASOS 시간자료: 1,270,026행 · 지점 97개 · 2020-06-01~2025-08-31
AWS 시간자료: 1,137,181행 · 지점 88개 · 2020-06-01~2025-08-31

관측망         기온지표        β             95%CI      SD      n    지점
ASOS         T10    0.059       0.016~0.101   1.07℃    831    83
ASOS         T11    0.114       0.072~0.156   1.10℃    831    83
ASOS         T12    0.129       0.078~0.181   1.13℃    831    83
ASOS         T13    0.148       0.089~0.208   1.32℃    831    83
ASOS         T14    0.182       0.109~0.255   1.47℃    831    83
ASOS         T15    0.180       0.097~0.264   1.62℃    831    83
ASOS        Tmax    0.181       0.114~0.248   1.38℃    831    83
AWS          T10    0.205       0.150~0.259   1.23℃    767    74
AWS          T11    0.226       0.177~0.275   1.33℃    768    74
AWS          T12    0.238       0.190~0.285   1.39℃    768    74
AWS          T13    0.264       0.209~0.318   1.51℃    767    74
AWS          T14    0.292       0.232~0.352   1.65℃    768    74
AWS          T15    0.298       0.232~0.364  

### §15-g. 지표피복(NDVI) 층화 β — "β 선형·피복 미층화" 한계 해소 `[🔖 2026-07-23 신규 · 사용자 제안]`

관측소 화소의 NDVI(§17과 동일 로직·같은 창에서 LST와 동시 추출)로 관측소를 층화해 β를 층별 재추정한다.
입력은 사전 추출 패널 `DERIVED_0723_관측소패널_LST_NDVI.csv`(4,306쌍 — 재추출 ~9분이라 CSV로 고정, 재현 시 스크래치 스크립트 proto_15g 참조).

**미리 요약**: ① 채택 β=0.258이 정확히 재현됨(게이트) ② **불투수 층 관측소 = 0개**(기상 관측 규정상 초지 설치) — "산단 부지 환산은 외삽"이 정량 확증 ③ 녹지 β(0.287) > 주거혼합 β(0.222), ASOS 동일 방향 — 반경 진단상 주거 β는 화소 이질성에 의한 **희석 성분** 포함 ④ 모든 층이 기존 사용 범위와 정합 → 기존 수치 변경 불필요, 해석 강화.

| 환산 대상 | 대응 β | 비고 |
|---|---|---|
| 부지 자체 ΔLST → 기온 | 불투수 — **관측 공백** | 외삽·상한 경향 주의 (기존 §15 주의와 동일 방향) |
| 숲 포함 기준 ΔT (§14-c) | 녹지 0.287 | 범위 상단 근거 |
| 주거지 기준 ΔT (§14-d·시민 facing) | 주거혼합 0.222 | 보수 방향 |

In [ ]:
# ── §15-g. NDVI 층화 β — 재현 게이트 + 층별 재적합 + 희석 진단 [🔖 2026-07-23 신규] ──
import statsmodels.formula.api as _smf15
_P15=pd.read_csv('DERIVED_0723_관측소패널_LST_NDVI.csv',encoding='utf-8-sig')
_RADII15=[30,100,500,1000]

# (1) 채택 로직 재현 — §15-d와 동일(관측망×반경, Tmax, CI폭/|β| 최소) → 검증 게이트
_B15={}
for _net in ['ASOS','AWS']:
    for _R in _RADII15:
        _d=_P15[_P15.net==_net][['scene','지점','Tmax',f'LST_{_R}m']].dropna().rename(columns={f'LST_{_R}m':'LST'})
        if len(_d)<100 or _d.scene.nunique()<5:
            continue
        _m=_smf15.ols('Tmax ~ LST + C(scene)',data=_d).fit(cov_type='cluster',cov_kwds={'groups':_d['지점']})
        _B15[(_net,_R)]=(_m.params['LST'],*_m.conf_int().loc['LST'],len(_d),_d.지점.nunique())
_best=min(_B15,key=lambda k:(_B15[k][2]-_B15[k][1])/max(abs(_B15[k][0]),1e-9))
_b0=_B15[_best]
print(f'재현 채택: {_best[0]} Tmax {_best[1]}m → β={_b0[0]:.3f} [{_b0[1]:.3f},{_b0[2]:.3f}] n={_b0[3]:,} 지점 {_b0[4]}')
assert abs(_b0[0]-0.258)<0.03, f'§15-d 채택 β(0.258) 재현 실패: {_b0[0]:.3f}'
print('✓ 검증 게이트: §15-d 채택 β=0.258 재현')

# (2) 층화: 관측소별 NDVI 중앙값(채택 반경) → 3층
_Rb=_best[1]
def _strat15(v):
    if v!=v:
        return None
    if v<0.15:
        return '불투수(<0.15)'
    if v<0.45:
        return '주거혼합(0.15~0.45)'
    return '녹지(≥0.45)'
print()
print(f"{'관측망':6}{'층':22}{'β':>8}{'95%CI':>16}{'n':>8}{'지점':>6}")
for _net in ['AWS','ASOS']:
    _snd=_P15[_P15.net==_net].groupby('지점')[f'NDVI_{_Rb}m'].median().map(_strat15)
    for _st in ['불투수(<0.15)','주거혼합(0.15~0.45)','녹지(≥0.45)']:
        _ids=_snd[_snd==_st].index
        _d=_P15[(_P15.net==_net)&(_P15.지점.isin(_ids))][['scene','지점','Tmax',f'LST_{_Rb}m']].dropna().rename(columns={f'LST_{_Rb}m':'LST'})
        if len(_d)<100:
            print(f'{_net:6}{_st:22}{"— 표본 부족":>8} (n={len(_d)})  ★ 불투수 관측소 부재 = 부지 환산이 외삽임의 정량 확증' if '불투수' in _st else f'{_net:6}{_st:22} 표본 부족 (n={len(_d)})')
            continue
        _m=_smf15.ols('Tmax ~ LST + C(scene)',data=_d).fit(cov_type='cluster',cov_kwds={'groups':_d['지점']})
        _ci=_m.conf_int().loc['LST']
        print(f"{_net:6}{_st:22}{_m.params['LST']:>8.3f}{f'{_ci[0]:.3f}~{_ci[1]:.3f}':>16}{len(_d):>8,}{_d.지점.nunique():>6}")

# (3) 희석 진단 — 주거 β가 낮은 이유: 반경별 붕괴 vs 녹지 평탄 + 창 내 이질성
_A=_P15[_P15.net=='AWS'].copy()
_A['층']=_A['지점'].map(_A.groupby('지점')[f'NDVI_{_Rb}m'].median().map(_strat15))
print()
print('희석 진단 ① 반경별 β (주거=반경 커질수록 붕괴 → 희석 / 녹지=평탄 → 안정):')
print(f"{'층':10}{'30m':>8}{'100m':>8}{'500m':>8}{'1000m':>8}")
for _st in ['주거혼합(0.15~0.45)','녹지(≥0.45)']:
    _row=f'{_st[:4]:10}'
    for _R in _RADII15:
        _d=_A[_A.층==_st][['scene','지점','Tmax',f'LST_{_R}m']].dropna().rename(columns={f'LST_{_R}m':'LST'})
        if len(_d)<100:
            _row+=f"{'—':>8}"
            continue
        _m=_smf15.ols('Tmax ~ LST + C(scene)',data=_d).fit(cov_type='cluster',cov_kwds={'groups':_d['지점']})
        _row+=f"{_m.params['LST']:>8.3f}"
    print(_row)
_A['het']=( _A['LST_30m']-_A['LST_500m']).abs()
print('희석 진단 ② 창 내 이질성 |LST30−LST500| 중앙값:',_A.groupby('층')['het'].median().round(2).to_dict())
print('희석 진단 ③ 장면 내 지점간 신호 SD:',_A.groupby('층').apply(lambda g:g.groupby('scene')[f'LST_{_Rb}m'].std().median()).round(2).to_dict())
print()
print('[해석] 녹지>주거 역전 = ①물리(식생 표면은 기온과 강결합·도시 표면은 디커플) + ②통계(주거 화소 이질성→회귀 희석).')
print('[해석] 층별 값 모두 기존 사용 범위·희석 보정 상한(0.459) 안 — 기존 문서 수치 유지, 대응표(md)로 용도만 명확화.')

# ── [🔖 2026-07-27 사용자 질문] "희석 진단 ①②③이 무슨 의미고 그 뒤 어디에 쓰이나" ──
print()
print('[해설] 희석 진단 ①②③ — 무엇을 의심했고 무엇이 나왔나')
print('  의심: 주거 β(0.222)가 녹지 β(0.287)보다 낮은 게 **진짜 물리**인가, 아니면 **측정 인공물**인가?')
print('        LST는 관측소 주변 원(30·100·500·1000m) 평균이다. 주거지는 원 안에 지붕·도로·나무가')
print('        섞여 있어, 원을 넓히면 서로 다른 표면이 섞여 **신호가 흐려진다**(희석). 그러면 β가 낮게 나온다.')
print('  ① 반경별 β: 주거는 100m 0.222 → 1000m 0.149로 **무너지고**, 녹지는 0.287 → 0.288로 **평탄**하다.')
print('     → 희석 가설과 정확히 일치. 균질한 녹지는 원을 넓혀도 안 변하고, 이질적인 주거만 무너진다.')
print('  ② 창 내 이질성 |LST30−LST500| 중앙값: 주거 2.34℃ > 녹지 1.79℃.')
print('     → 주거 창이 실제로 더 이질적임을 직접 확인. ①의 원인 후보를 뒷받침.')
print('  ③ 지점간 신호 SD: 녹지 3.12 > 주거 2.56.')
print('     → 반례 점검용이다. "주거 β가 낮은 건 주거끼리 서로 비슷해서(변동이 없어서)"라는')
print('       대안 설명을 확인하는 칸인데, 실제로 주거 변동이 더 작게 나왔다 → **대안 설명을 못 버린다.**')
print('       ①②만으로 희석을 확정할 수 없다는 뜻이다. 세 칸 중 ③은 우리에게 불리한 결과다.')
print()
print('[사슬] 이 진단이 어디로 가나 — **β 상한 0.459의 근거**다')
print(f'  · 채택 β = {BETA_L2A} (희석된 채로의 관측값 = **하한**)')
print(f'  · 상한 β = {BETA_L2A_HI} (§15-e 희석 보정치)')
print('  · §11-b(3)·§12-b·§13의 모든 "하한~상한" 범위가 이 두 값으로 만들어진다.')
print('    즉 희석 진단은 **범위의 위쪽 끝을 정당화하는 근거**다. 진단이 틀리면 상한이 과대가 된다.')
print('  ⚠ 정직하게: ③이 대안 설명을 배제하지 못했으므로, 0.459는 **보정 상한이지 보정 확정값이 아니다.**')
print('    그래서 우리는 0.459를 단독으로 쓰지 않고 언제나 0.258~0.459 범위로만 쓴다.')

# ── [🔖 2026-07-27 · 자기수정 10] ③의 방향을 거꾸로 설명했었다 + 눌림 가설 직접 검정 ──
print()
print('[정정] ③을 "우리에게 불리한 반례"로 설명했던 것은 틀렸다 (자기수정 10)')
print('  회귀 기울기가 눌리는 정도 = 진짜 퍼짐² / (진짜 퍼짐² + 오차²).')
print('  · ② 창 안 이질성(주거 2.34 > 녹지 1.79) = **오차가 크다** → 분모의 뒷항 ↑')
print('  · ③ 지점 간 퍼짐(주거 2.56 < 녹지 3.12) = **비교할 폭이 좁다** → 분자 ↓')
print('  → 주거는 **둘 다 불리**하다. ③은 반례가 아니라 희석 가설을 **같은 방향으로 뒷받침**한다.')
print('  ⚠ 단 ②와 ③은 단위가 달라 둘로 보정치를 계산할 수는 없다. 실제 보정 0.459는 §15-e의 σ_e 역산이다.')

print()
print('[검정] 눌림 가설 직접 검정 — 녹지 퍼짐을 주거 수준으로 좁히면 β가 떨어지나')
print('  논리: 퍼짐이 좁아서 눌리는 게 맞다면, 녹지 관측소를 골라 퍼짐을 주거 수준(2.56)까지 줄이면')
print('        녹지 β도 주거 β(0.222) 쪽으로 내려와야 한다. 안 내려오면 층 간 차이는 **진짜 물리**다.')
_Ag=_P15[_P15.net=='AWS'].copy()
_Ag['층']=_Ag['지점'].map(_Ag.groupby('지점')[f'NDVI_{_Rb}m'].median().map(_strat15))
_grn=_Ag[_Ag.층=='녹지(≥0.45)'].copy()
_sd_target=float(_Ag[_Ag.층=='주거혼합(0.15~0.45)'].groupby('scene')[f'LST_{_Rb}m'].std().median())
_smean=_grn.groupby('지점')[f'LST_{_Rb}m'].mean()
_med=_smean.median(); _order=(_smean-_med).abs().sort_values().index.tolist()   # 중앙에 가까운 순
print(f"  {'남긴 지점수':>10}{'장면내 SD':>10}{'β':>9}{'95%CI':>18}{'n':>8}")
_rows_g=[]
for _keep in range(len(_order),max(len(_order)//4,8),-max(1,len(_order)//12)):
    _ids=_order[:_keep]
    _d=_grn[_grn.지점.isin(_ids)][['scene','지점','Tmax',f'LST_{_Rb}m']].dropna().rename(columns={f'LST_{_Rb}m':'LST'})
    if len(_d)<100 or _d.지점.nunique()<5: continue
    _sd=float(_d.groupby('scene')['LST'].std().median())
    _m=_smf15.ols('Tmax ~ LST + C(scene)',data=_d).fit(cov_type='cluster',cov_kwds={'groups':_d['지점']})
    _ci=_m.conf_int().loc['LST']
    _rows_g.append((_keep,_sd,float(_m.params['LST'])))
    _tag='  ← 주거 퍼짐 수준' if abs(_sd-_sd_target)<0.15 else ''
    print(f"  {_keep:>10}{_sd:>10.2f}{_m.params['LST']:>9.3f}{f'{_ci[0]:.3f}~{_ci[1]:.3f}':>18}{len(_d):>8,}{_tag}")
print(f'  (주거 장면내 SD 기준값 = {_sd_target:.2f})')
if len(_rows_g)>=2:
    _b_full,_b_narrow=_rows_g[0][2],_rows_g[-1][2]
    print()
    if _b_narrow < _b_full-0.02:
        print(f'  → **눌림 가설 지지**: 퍼짐을 좁히니 녹지 β가 {_b_full:.3f} → {_b_narrow:.3f}로 내려간다.')
        print('    층 간 β 차이의 적어도 일부는 물리가 아니라 **측정 구조**다 → 희석 보정 상한(0.459)의 근거가 강해진다.')
    elif _b_narrow > _b_full+0.02:
        print(f'  → **예상과 반대**: 좁힐수록 β가 {_b_full:.3f} → {_b_narrow:.3f}로 **올라간다**. 눌림만으로 설명 안 된다.')
    else:
        print(f'  → **가르지 못한다**: 퍼짐을 좁혀도 β가 {_b_full:.3f} → {_b_narrow:.3f}로 거의 안 변한다.')
        print('    층 간 차이가 퍼짐 때문이라는 증거가 안 나온다 → 0.459 상한의 근거는 §15-e 역산 하나에 계속 의존한다.')
print('  ⚠ 이 검정은 **지점을 골라내는** 방식이라 표본이 줄어 CI가 넓어진다. 방향만 읽고 값은 읽지 말 것.')

재현 채택: AWS Tmax 100m → β=0.258 [0.220,0.296] n=2,953 지점 347
✓ 검증 게이트: §15-d 채택 β=0.258 재현

관측망   층                            β           95%CI       n    지점
AWS   불투수(<0.15)             — 표본 부족 (n=0)  ★ 불투수 관측소 부재 = 부지 환산이 외삽임의 정량 확증
AWS   주거혼합(0.15~0.45)          0.222     0.154~0.290     730    83
AWS   녹지(≥0.45)                0.287     0.239~0.334   2,223   264
ASOS  불투수(<0.15)             — 표본 부족 (n=0)  ★ 불투수 관측소 부재 = 부지 환산이 외삽임의 정량 확증
ASOS  주거혼합(0.15~0.45)          0.157     0.064~0.249     372    37
ASOS  녹지(≥0.45)                0.200     0.102~0.299     459    46

희석 진단 ① 반경별 β (주거=반경 커질수록 붕괴 → 희석 / 녹지=평탄 → 안정):
층              30m    100m    500m   1000m
주거혼합         0.216   0.222   0.169   0.149
녹지(≥         0.279   0.287   0.286   0.288
희석 진단 ② 창 내 이질성 |LST30−LST500| 중앙값: {'녹지(≥0.45)': 1.79, '주거혼합(0.15~0.45)': 2.34}
희석 진단 ③ 장면 내 지점간 신호 SD: {'녹지(≥0.45)': 3.12, '주거혼합(0.15~0.45)': 2.56}

[해석] 녹지>주거 역전 = ①물리(식생 표면은 기온과 강결합·도시 표면은 디커플) + ②통계(주거 화소 이질성→회귀 희석).
[해석] 층별 값 모두 기존 사용 

### §15-h. 환산 — 정정을 모두 반영한 뒤 `[🔖 2026-07-25 §15-d에서 이동]`

§15-e(회귀희석)·§15-f(시각)·§15-g(피복 층화)를 다 읽은 자리에서 환산표를 낸다. 단일 β 한 점이 아니라 **하한(채택 β)~상한(희석보정 β)** 범위로 제시한다.

> ⚠ 이 표는 **"산단이 지금 얼마나 뜨거운가"**이지 AIDC가 얼마나 더 올리는가가 아니다. AIDC 추가분은 케임브리지 감쇠를 쓰는 §11-b·§12-b가 담당한다.

In [ ]:
# ── §15-h. 산단 지표ΔT → 기온ΔT 환산 — **정정을 모두 반영한 범위** [🔖 2026-07-25 이동·확장] ──
# §15-d에 있던 표를 여기로 옮겼다. 그때는 채택 β 하나로만 곱했지만, §15-e가 회귀희석을 기각하지
#   못했으므로 채택값은 **하한**일 수 있다. 그래서 세 값을 나란히 낸다:
#     · 하한 = 채택 β (직접 측정, 눌렸을 수 있음)
#     · 상한 = 희석보정 β (§15-e 역산)
#     · 참고 = 관측망별 β 폭 (ASOS vs AWS, 같은 반경 100m)
_bb,_blo,_bhi,_n15,_ns15=BETA[best]
_BHI15=BETA_L2A_HI                                   # §15-e 희석보정 상한(§9-e 엔진과 같은 값)
print(f'채택 β {_bb:.3f} (95%CI {_blo:.3f}~{_bhi:.3f}) · 희석보정 상한 {_BHI15:.3f}')
print('=== 산단 지표 ΔT → 기온 ΔT 환산 (정정 반영) ===')
# [정정 07-21] 폴리곤 부지=R0만, 점 부지=0-1km (fillna는 R0 결측 장면을 낮은 0-1km로 대체해 오염)
_c15=LST.copy()
_fac15={k:(g['ΔR0내부'].mean() if g['ΔR0내부'].notna().any() else g['Δ0-1km'].mean())
         for k,g in LST.groupby('부지')}
_c15['공장부지ΔLST']=_c15['부지'].map(_fac15)
print(f"{'부지':16}{'지표ΔT':>8}{'기온ΔT 하한':>12}{'기온ΔT 상한':>12}{'하한 95%CI':>18}")
for _k15,_v15 in _c15.groupby('부지')['공장부지ΔLST'].mean().sort_values(ascending=False).items():
    print(f"{_k15:16}{_v15:>7.1f}℃{_v15*_bb:>11.2f}℃{_v15*_BHI15:>11.2f}℃{f'{_v15*_blo:.2f}~{_v15*_bhi:.2f}':>18}")
print()
print('⚠ 이 환산은 **관측소 부지에서 추정한 β를 산단에 외삽**한 것이다. §15-g가 확인했듯 불투수층에는')
print('  관측소가 0개라(초지 설치 규정) 산단 같은 극단 지표로의 외삽은 검증되지 않았다 — 상한 과대 방향.')
print('  반대로 오전 스냅샷(§15-f)은 하한 방향이다. 두 편의가 반대라 순 방향은 미정 → 범위로 읽을 것.')
print('  ★ 이 표는 "산단이 지금 얼마나 뜨거운가"이고, AIDC **추가분**은 별개다(§11-b·§12-b의 케임브리지 감쇠).')


채택 β 0.258 (95%CI 0.220~0.296) · 희석보정 상한 0.459
=== 산단 지표 ΔT → 기온 ΔT 환산 (정정 반영) ===
부지                  지표ΔT     기온ΔT 하한     기온ΔT 상한          하한 95%CI
포항·제철              17.3℃       4.45℃       7.92℃         3.79~5.10
심팩 포항              14.3℃       3.69℃       6.58℃         3.15~4.24
구미·전자              14.0℃       3.61℃       6.43℃         3.08~4.14
당진1철강              12.6℃       3.24℃       5.77℃         2.76~3.72
울산미포               12.1℃       3.12℃       5.55℃         2.66~3.58
동해북평               11.2℃       2.88℃       5.13℃         2.45~3.30
온산                 11.2℃       2.87℃       5.12℃         2.45~3.30
동해 북평2             11.1℃       2.85℃       5.08℃         2.43~3.27
광양·제철              10.2℃       2.63℃       4.69℃         2.24~3.02
여수·석화               8.5℃       2.20℃       3.92℃         1.87~2.52
현대제철 인천             5.9℃       1.52℃       2.71℃         1.30~1.75
동국제강 인천             5.5℃       1.41℃       2.51℃         1.20~1.62

⚠ 이 환산은 **관측소 부지에서 추정한 β를 산단에 외삽**한 것이다. §15-

### §15 결과·해석 규약 `[🔖 2026-07-21 · AWS 459지점 반영]`

#### 측정된 것 `[사실]`
장면 고정효과 회귀에서 **지표 1℃당 기온 β**:

| 추정 | β | 성격 |
|---|---|---|
| ASOS 직접 (반경 100m, n=831·82지점) | **0.181** (95%CI 0.114~0.248) | 하한 |
| AWS 직접 (반경 100m, n=2,953·**347지점**) | **0.258** (95%CI 0.220~0.296) | 하한. 반경 30m~1km에서 0.249~0.258로 불변 |
| 희석 보정 (σ_e=2.24℃ 가정) | **0.459** | 상한 후보. **가정 의존 — 독립 측정 아님** |

> **2026-07-21 갱신**: AWS 일자료를 261→**459지점**으로 넓혀 재추정. AWS β 0.293→**0.258**(ASOS 쪽으로 내려옴), CI도 0.249~0.337→**0.220~0.296**로 좁아짐. β비 0.62→**0.70**(격차 1.6배→1.4배). 결론 방향은 불변, 숫자만 안정.

두 관측망이 1.4배 어긋난 원인은 §15-e에서 **회귀희석을 기각하지 못했다**(설치편의는 기각). σ_e=2.24℃면 정확히 설명되고, 반경 선택만 바꿔도 LST가 SD 2.24℃ 흔들리므로 그 규모의 오차는 실재한다. 즉 **두 직접 추정치 모두 참값보다 눌린 하한**으로 봐야 한다.

| | 지표 ΔT | 기온 ΔT (직접) | (희석 보정) |
|---|---:|---|---|
| 포항·제철 (가동 중) | +17.3℃ | +3.1 ~ +4.5℃ | +7.9℃ |
| 구미·전자 | +14.0℃ | +2.5 ~ +3.6℃ | +6.4℃ |
| 울산미포 | +12.1℃ | +2.2 ~ +3.1℃ | +5.6℃ |
| 광양·제철 | +10.2℃ | +1.8 ~ +2.6℃ | +4.7℃ |
| 현대제철 인천 (도심 폐공장) | +5.9℃ | +1.1 ~ +1.5℃ | +2.7℃ |

#### ⚖ 두 편의가 서로 반대 방향이다 — 이게 현재 불확실성의 핵심
- **회귀희석** → β를 **과소**평가 (직접 추정치가 하한인 이유)
- **극단 지표로의 외삽 포화** → β를 **과대**평가 (관측소는 +17℃ 지표를 겪지 않으므로, 결합이 극단에서 포화하면 상한이 부풀려짐)

둘의 크기를 아직 각각 재지 못했으므로 **어느 쪽이 순효과인지 단정할 수 없다.** 범위로만 말한다.

#### ✅ 이렇게까지만 말한다 `[해석 허용]`
> "우리 관측망 자료에서 지표 1℃당 기온은 최소 **0.18~0.26℃** 오르고, 측정오차를 보정하면 **0.46℃**까지 갈 수 있다. 가동 중 제철소 부지는 주변보다 기온이 **+3~8℃**, 도심 폐공장 부지는 **+1~2.7℃** 높은 것으로 추정된다. §9가 AIDC 입지로 가정한 **+2℃는 도심 폐공장 추정 범위 안에 있어 과장된 가정이 아니며, 오히려 보수적일 수 있다.**"

#### ⛔ 이렇게 말하지 않는다 `[해석 금지]`
1. **"AIDC가 들어오면 +2℃ 오른다"** — §15는 §9 가정이 **터무니없지 않다**는 것만 보인다. AIDC 폐열의 실제 승온은 §9의 물리 계산이고, 제철소 열은 폐열 외에 공정열·복사가 섞여 기전이 다르다.
2. **"산단 때문에 주민 온열질환이 N건 늘었다"** — §6에서 **주민 온열과 산업활동은 무관**했다(작업장 노동으로 판별, 거리 반론 §6-c 기각). β는 §9 가정의 **타당성 검증용**이지 산단을 온열 원인으로 돌리는 근거가 아니다.
3. **희석 보정 0.459를 확정값처럼 사용** — σ_e를 두 관측망이 맞도록 역산한 값이라 독립 증거가 없다. 보정 전 값을 주 추정으로, 보정값을 상한으로만 쓴다.

#### 해소된 한계 — 시각 불일치
§15-f에서 시간자료로 검정한 결과, 시각 차이는 **측정오차가 아니라 결합강도의 실제 일변화**였고, **일최고기온 β ≈ 14~15시 β**임이 확인됐다. 우리 목적(§9-b 곡선이 일최고 기반·온열질환 14~16시 집중)에는 **Tmax가 옳은 지표**다.

#### 해소된 한계 — 설치 편의 재검정 `[🔖 2026-07-21]`
AWS 459지점 확보로 §15-e 가설②(설치 편의)의 ASOS↔AWS 5km 이내 동일일자 짝이 **4→8짝**(4,394일)으로 늘었다. AWS−ASOS 일최고기온 **−0.04℃** — 설치 편의 **더 강하게 기각**.

#### 남은 한계
- **외삽**: 관측소는 개방·통풍 부지에만 있어 +17℃ 산단 적용은 관측 범위 밖. (녹지·지표피복 층화로 완화 — §17 착수)
- **β 선형 가정**: 풍속·습도·지표피복 층화 미실시.
- **회귀희석 잔존**: 독립 검증(반복 측정)이 없어 σ_e를 직접 못 잰다. 직접 β는 하한으로만 사용.

## §16. 에너지바우처 — **별도 문서로 분리** `[🔖 2026-07-25 사용자 요청]`

냉방 지원 제도의 도달 분석은 AIDC 열섬 dose-response 사슬(§7~§14)과 독립적이라 노트북 밖으로 옮겼다.

→ **[`에너지바우처_냉방지원_도달_2026-07-25.md`](에너지바우처_냉방지원_도달_2026-07-25.md)** (코드·출력 포함, 그대로 재실행 가능)

요지만 남긴다: 제도는 확대됐지만 **여름 냉방 바우처는 겨울 난방분에 비해 금액·수급자 모두 작고**, AIDC 후보 부지 시군구의 발급률은 "잘 닿는다"와 "취약한 사람이 많다"를 구분해 읽어야 한다(원문 §16-d 참조).

## §17. 녹지·불투수 분해 (NDVI) — 산단이 **왜** 뜨거운지 + AIDC 냉각완충 판정 `[🔖 2026-07-21]`

§14는 "산단 부지가 주변보다 +10~17℃ 뜨겁다"를 **쟀다**. §17은 그 **물리 기전**을 분해한다 — 식생(NDVI)이 적고
불투수 지표(맨땅·건물·포장)가 많으면 햇빛이 열로 바뀌어 지표가 달궈진다. §14-c의 `GEO` 형상을 그대로 재사용해
같은 부지의 **지표피복**을 잰다. `min_px=30`, 6~8월, 구름<30%.

In [ ]:
# ── §17. 녹지·불투수 분해 (NDVI) — GEO(§14-c) 형상 재사용 ──
# ⚠ NDVI 계산 로직은 §14-d1과 동일 (self-contained 원칙으로 의도적 중복) — 수정 시 양쪽 동기화할 것.
# [🔖 2026-07-21 신규 셀]
"""
Landsat SR 위성 영상에서 구름/수체 등을 제외한 신뢰도 높은 NDVI를 추출하여, 
지정된 각 부지 영역(R0, 1km, 3km 버퍼)별 녹지량 및 불투수면 비율을 
공간적 도넛 형태(도넛 링)로 산출·집계하는 기능
"""
from rasterio.features import geometry_mask

# Landsat 8-9 Surface Reflectance (SR) 스케일링/오프셋 상수 (DN -> 반사율 변환용)
SR_SCALE, SR_OFF = 0.0000275, -0.2               # Surface Reflectance DN → 반사율

def sr_scene_list():
    """SR_B4·B5·B6·QA_PIXEL 다 있고 0바이트 아닌 장면만 (부분 다운로드 배제)"""
    out=[]; z=[]
    # B10(열적외선) 파일 기준으로 검색하여 SR 밴드 세트 구성
    for p in sorted(glob.glob(f'{LSD}/*_ST_B10.TIF')):
        b=p[:-11]; need=['_SR_B4.TIF','_SR_B5.TIF','_SR_B6.TIF','_QA_PIXEL.TIF','_MTL.txt']

        # 필수 파일 존재 여부 검사
        if not all(os.path.exists(b+s) for s in need): continue

        # 다운로드 중단/손상 파일(0바이트) 필터링
        bad0=[b+s for s in need[:4] if os.path.getsize(b+s)==0]        # 0바이트 = 다운로드 실패
        if bad0: z.extend(os.path.basename(f) for f in bad0); continue

        # 메타데이터에서 메타 정보(위성 종류, 관측일자 등) 파싱
        nm=b.replace('\\','/').split('/')[-1]; m=re.search(r'(LC0[89])_L2SP_(\d{3})(\d{3})_(\d{8})',nm)
        out.append(dict(base=b,date=pd.to_datetime(m.group(4))))

    if z: print(f'⚠ 0바이트(재다운로드 필요) {len(z)}개 제외: {z}')
    return pd.DataFrame(out)

def _sr(dn):                                       # DN → 반사율, 유효범위(0~1) 밖은 NaN
    """DN(Digital Number)을 유효 반사율(0~1)로 변환 (범위 밖은 NaN)"""
    r=dn.astype('float64')*SR_SCALE+SR_OFF
    return np.where((dn>0)&(r>=0)&(r<=1), r, np.nan)

def site_veg(base, geom5179, is_poly, cloud_max=30, min_px=30):
    """특정 부지(geom5179) 영역 내 NDVI 및 불투수면 비율 통계 산출"""

    # 1. 메타데이터(_MTL.txt)에서 총 구름 커버율 확인
    m={}
    for ln in open(base+'_MTL.txt',encoding='utf-8',errors='ignore'):
        if 'CLOUD_COVER ' in ln and '=' in ln: m['cc']=float(ln.split('=')[1])
    if m.get('cc',100)>cloud_max: 
        return {'사유':'구름'}  # 허용 구름 비율 초과 시 스킵

    try:
        # 2. B4(Red) 래스터를 열어 관심 영역(부지 3km 버퍼 범위) 설정 및 자르기(Crop)
        with rasterio.open(base+'_SR_B4.TIF') as s:
            g=_to_scene(geom5179,5179,s.crs); gb=g.buffer(3000).bounds
            if not (s.bounds.left<(gb[0]+gb[2])/2<s.bounds.right and s.bounds.bottom<(gb[1]+gb[3])/2<s.bounds.top):
                return {'사유':'장면 밖'}
            w=from_bounds(*gb,s.transform)
            b4=s.read(1,window=w,boundless=True,fill_value=0); tr=s.window_transform(w); shp=b4.shape
        with rasterio.open(base+'_SR_B5.TIF') as s: b5=s.read(1,window=w,boundless=True,fill_value=0)
        with rasterio.open(base+'_SR_B6.TIF') as s: b6=s.read(1,window=w,boundless=True,fill_value=0)
        with rasterio.open(base+'_QA_PIXEL.TIF') as s: qa=s.read(1,window=w,boundless=True,fill_value=1)
    except rasterio.errors.RasterioIOError:
        return {'사유':'읽기 실패(손상)'}
    R4,R5=_sr(b4),_sr(b5)
    ndvi=(R5-R4)/(R5+R4)                            # 식생지수 (높을수록 녹지)
    bad=np.zeros(shp,bool)
    for k in ('fill','dilated','cirrus','cloud','shadow','snow','water'): bad|=((qa>>QA_BITS[k])&1)>0
    ndvi=np.where(bad,np.nan,ndvi)
    inside=lambda gg: geometry_mask([gg],out_shape=shp,transform=tr,invert=True)
    def stat(mask):
        v=ndvi[mask]; v=v[~np.isnan(v)]
        if len(v)<min_px: return (np.nan,np.nan)
        return (float(np.mean(v)), float((v<0.15).mean()))   # 평균 NDVI, 불투수 대리(NDVI<0.15 화소 비율)
    out={'사유':'OK'}
    if is_poly: out['R0_NDVI'],out['R0_불투수%']=stat(inside(g))
    core=inside(g) if is_poly else np.zeros(shp,bool)
    out['1km_NDVI'],_=stat(inside(g.buffer(1000))&~core)
    out['3km_NDVI'],out['3km_불투수%']=stat(inside(g.buffer(3000))&~inside(g.buffer(1000)))
    return out

SRC=sr_scene_list()
print(f"SR 밴드 장면: {len(SRC)} | GEO 부지: {len(GEO)}")
_vrec=[]
for _nm,(_g,_ip) in GEO.items():
    for _,_s in SRC.iterrows():
        if JJA_ONLY and _s.date.month not in (6,7,8): continue
        _r=site_veg(_s.base,_g,_ip)
        if _r.get('사유')!='OK': continue
        _vrec.append(dict(부지=_nm,형상='폴리곤' if _ip else '점',
                          **{k:v for k,v in _r.items() if k!='사유'}))
VEG=pd.DataFrame(_vrec)
_dc=[c for c in VEG.columns if 'NDVI' in c or '불투수' in c]
VEG_AGG=VEG.groupby(['부지','형상']).agg(장면=('3km_NDVI','size'),
              **{c:(c,'mean') for c in _dc}).round(3)
_show=[c for c in ['장면','R0_NDVI','R0_불투수%','1km_NDVI','3km_NDVI','3km_불투수%'] if c in VEG_AGG.columns]
print("\n=== 부지별 평균 NDVI·불투수 (낮을수록 녹지 적음) ===")
print(VEG_AGG[_show].sort_values('R0_NDVI' if 'R0_NDVI' in VEG_AGG else '3km_NDVI').to_string())
print("\n→ 해석은 아래 「§17 결과·해석 규약」 markdown 참조")

SR 밴드 장면: 90 | GEO 부지: 11

=== 부지별 평균 NDVI·불투수 (낮을수록 녹지 적음) ===
             장면  R0_NDVI  R0_불투수%  1km_NDVI  3km_NDVI  3km_불투수%
부지      형상                                                     
당진1철강   폴리곤  20    0.219    0.307     0.355     0.468     0.154
온산      폴리곤  30    0.220    0.492     0.608     0.650     0.051
광양·제철   폴리곤  27    0.235    0.373     0.358     0.534     0.130
울산미포    폴리곤  30    0.255    0.372     0.531     0.580     0.103
동해북평    폴리곤  27    0.319    0.169     0.426     0.559     0.067
구미·전자   폴리곤  25    0.351    0.165     0.561     0.585     0.056
동국제강 인천 점    11      NaN      NaN     0.270     0.234     0.276
심팩 포항   점    16      NaN      NaN     0.305     0.385     0.162
여수·석화   점    27      NaN      NaN     0.236     0.377     0.302
포항·제철   점    16      NaN      NaN     0.205     0.287     0.307
현대제철 인천 점    11      NaN      NaN     0.268     0.232     0.287

→ 해석은 아래 「§17 결과·해석 규약」 markdown 참조


### §17 결과·해석 규약 — 반드시 [사실]과 [해석]을 나눈다 `[🔖 2026-07-21]`

#### 측정된 것 `[사실]`
6개 폴리곤 산단 **모두** 부지 내부(R0)의 NDVI가 주변 3km보다 낮고, 불투수 화소 비율(NDVI<0.15)은 반대로 부지가 훨씬 높다:

| 부지 | R0 NDVI | R0 불투수% | → 3km NDVI | 3km 불투수% |
|---|---:|---:|---:|---:|
| 당진1철강 | 0.219 | 31% | 0.468 | 15% |
| 온산 | 0.220 | **49%** | 0.650 | 5% |
| 광양·제철 | 0.235 | 37% | 0.534 | 13% |
| 울산미포 | 0.255 | 37% | 0.580 | 10% |
| 동해북평 | 0.319 | 17% | 0.559 | 7% |
| 구미·전자 | 0.352 | 16% | 0.587 | 5% |

점 부지(개별 폐공장·YUCH 폴리곤 없는 곳)는 R0가 없어 3km만: 현대제철 인천 **0.232**, 동국제강 인천 0.234, 포항·제철 0.287, 심팩 포항 0.385, 여수·석화 0.377.

#### 이렇게까지 말한다 `[해석 허용]`
1. **산단이 뜨거운 건 우연이 아니라 지표피복 때문이다.** 부지는 주변보다 녹지가 2배 이상 적고(광양 0.235 vs 0.534), 불투수 면적이 3~10배 많다(온산 49% vs 5%). 낮은 녹지 + 높은 불투수 = §14의 부지 고온(+10~17℃)을 설명하는 **물리 기전**. §14가 "얼마나 뜨거운가"를 쟀다면 §17은 "왜 뜨거운가"에 답한다.
2. **AIDC 전환은 이 녹지 결핍을 고정하거나 악화한다.** 데이터센터는 건물·냉각탑·주차장으로 덮여 부지 녹지가 사실상 0이 된다. 이미 낮은 부지 녹지가 더 낮아질 뿐, 냉각 완충이 새로 생기지 않는다.
3. **냉각 완충이 되는 녹지는 부지가 아니라 1~3km 주변에 있다** — 그런데 §4에서 본 데이터센터의 광역 heat footprint(4.5~10km)가 바로 그 주변 녹지대까지 덮는다. 열을 퍼뜨리는 반경과 냉각 완충 녹지가 겹친다.
4. **도심 폐공장(인천)은 완충 녹지 자체가 원래 없다.** 원격 산단은 3km 주변이 녹지(NDVI 0.47~0.65)라 열을 어느 정도 흡수하지만, 인천 폐공장은 부지도 주변 3km도 NDVI 0.23으로 낮다(도심 밀집, 3km 불투수 28%). **AIDC 폐열을 흡수할 녹지 여력이 가장 적은 곳이 사람이 가장 많은 도심 폐공장**이라는 §6·§13 논지를 지표피복이 다시 뒷받침한다.

#### 이렇게 말하지 않는다 `[해석 금지]`
1. **"산단 부지가 RF·MARS의 '취약 극대' 임계를 넘었다"** — 참고 논문(RF·MARS)이 제시한 임계는 **NDVI<0.15·불투수>60%**인데, 부지 **평균** NDVI 최저값은 당진 0.219, 불투수율 최고는 온산 49%로 **폴리곤 평균으로는 그 임계를 넘지 않는다.** 부지 내부 일부 화소는 넘지만 평균은 아래 → "임계에 **근접**"이라고만 말한다.
2. **"녹지를 늘리면 산단이 시원해진다"** — §17은 녹지 결핍과 고온의 **동반**을 보였을 뿐, 녹화의 냉각 효과 크기는 재지 않았다. 그건 별도 인과 설계(전후 비교·대조지) 필요.
3. **NDVI 절대값을 도시 간 순위로 해석** — 촬영 시기·구름·작물 생육에 따라 흔들린다. 여기서는 **같은 장면 안 부지 vs 주변** 대조(같은 시각·대기)만 신뢰한다.

#### 남은 한계
- **불투수 대리**: NDVI<0.15 화소 비율은 진짜 불투수면(콘크리트·아스팔트) 실측이 아니라 대리 지표. 정밀하려면 토지피복도(환경부 세분류) 중첩 필요.
- **완전판 90장면** `[🔖 2026-07-21 갱신]`: 이전에 0바이트로 빠졌던 `LC09_115035_20230609_SR_B6`을 재다운로드해 포함. 89→90 장면, 부지 NDVI·불투수 값 변화 없음(3자리까지 동일) — 결론 재확인.

> `[🔖 2026-07-23 §17-c 검증 결과]` **NDVI<0.15 대리는 실제 불투수를 과소(보수) 추정한다**: 온산 대리 49% vs 실피복 불투수계 81%, 광양 37% vs 87% — NDVI 0.15~0.45의 공업 화소가 대리에서 빠지기 때문. 즉 §17의 "산단 불투수 3~10배" 주장은 절제된 표현. 대리지표는 유지하되 방향(과소)을 명시한다.

> `[🔖 2026-07-24 §17-d]` 피복 유형별 냉각 순위(실측): 활엽수림 −2.5 > 혼효림 −2.4 > 침엽수림 −1.9 ≫ 논 0.0 > 밭·인공초지 +1.6 ≫ 불투수 +5.0(공업 +9.7). **완충숲은 초지·잔디가 아니라 활엽·혼효 수관림**, 논이 밭보다 시원(물 효과). §9 대안⑤ 처방 구체화 근거.

### §17-b. 녹지 용량반응 — NDVI +0.1당 지표온도 몇 ℃ 내려가나 `[🔖 2026-07-23 신규 · 사용자 질문 "녹지 조성의 정량 완화"]`

§17이 "산단이 왜 뜨거운가"(녹지 결핍)를 보였다면, 여기선 방향을 뒤집어 **녹지를 늘리면 얼마나 식는가**를 잰다.
방법: 11개 부지의 기준 도넛(10~20km) 화소에서 (NDVI, LST) 쌍을 표본추출(전량 305만 중 저장 30만), **장면 내 demean**(같은 날·같은 대기 안 비교) 후 구간 곡선과 기울기.

⚠ 정직 단서: ① 단면 연관(공간 비교)이지 식재 전후 실험이 아님 ② 지표온도 기준 — 기온은 β 범위(0.22~0.29)로 축소 환산 ③ 관개 여부 중요(Sailor 관개공원 실측이 공기 수준 증거) ④ NDVI<0.10 구간은 물 경계·고반사면 혼재로 해석 제외.

**§17-b0. 입력 표본 생성 (self-contained)** `[🔖 2026-07-25 사용자 지적]`

§17-b가 읽는 `DERIVED_0723_링화소_NDVI_LST표본.csv`를 만드는 코드가 노트북에 없었다 — 이전 작업에서 별도 스크립트로 돌리고 결과만 들여왔기 때문이다. 처음 보는 사람이 이 노트북만으로 재현할 수 있어야 하므로 생성 코드를 편입한다.

기준 **도넛(10~20km)** 안의 유효 화소에서 (NDVI, LST) 쌍을 장면·부지별로 최대 15,000개씩 뽑고, 장면-부지 평균을 뺀다(`_dm`) — 같은 장면 안에서만 비교하기 위해서다. CSV가 이미 있으면 건너뛴다(전 장면 재스캔은 수십 분).

In [ ]:
# ── §17-b0. 링 화소 (NDVI, LST) 표본 생성 — §17-b의 입력 CSV를 여기서 만든다 [🔖 2026-07-25 편입] ──
# [사용자 지적] §17-b가 읽는 DERIVED_0723_링화소_NDVI_LST표본.csv를 만드는 코드가 노트북에 없었다.
#   (이전 작업에서 별도 스크립트로 돌리고 결과 CSV만 들여왔다 = 처음 보는 사람은 재현 불가.)
#   → 생성 코드를 여기 편입한다. **CSV가 이미 있으면 건너뛴다**(전 장면 재스캔은 수십 분).
#   다시 만들려면 파일을 지우거나 FORCE_17B0=True.
FORCE_17B0=False
_OUT17='DERIVED_0723_링화소_NDVI_LST표본.csv'
if os.path.exists(_OUT17) and not FORCE_17B0:
    print(f'{_OUT17} 이미 존재 — 생성 건너뜀 (다시 만들려면 FORCE_17B0=True)')
else:
    import rasterio as _rio17
    from rasterio.windows import from_bounds as _fb17
    from rasterio.features import geometry_mask as _gm17
    from shapely.ops import transform as _st17
    from pyproj import Transformer as _Tr17
    _REF17=(10000,20000); _RNG17=np.random.default_rng(20260723)     # 기준 도넛 10~20km · 고정 시드
    _SS,_SO=0.00341802,149.0; _SQS=0.01; _RS,_RO=0.0000275,-0.2
    _QB={'fill':0,'dilated':1,'cirrus':2,'cloud':3,'shadow':4,'snow':5,'clear':6,'water':7}

    def _tosc17(geom,dst):
        return _st17(_Tr17.from_crs(5179,dst,always_xy=True).transform,geom)

    _fr17=[]; _sid17=0
    _sc17=[s for _,s in SC.iterrows()
           if s.date.month in (6,7,8) and float(mtl_meta(s.base).get('CLOUD_COVER',100))<=CLOUD_MAX]
    for _j17,(_nm17,_gv17) in enumerate(GEO.items()):
        _g517=_gv17[0] if isinstance(_gv17,tuple) else _gv17      # GEO 값이 (geom, is_poly)인 경우 대응
        _cnt17=0
        for _s17 in _sc17:
            _b17=_s17.base
            if not all(os.path.exists(_b17+_b) and os.path.getsize(_b17+_b)>0 for _b in ('_SR_B4.TIF','_SR_B5.TIF')):
                continue
            try:
                with _rio17.open(_b17+'_ST_B10.TIF') as _s0:
                    _g=_tosc17(_g517,_s0.crs); _gb=_g.bounds
                    if not (_s0.bounds.left<(_gb[0]+_gb[2])/2<_s0.bounds.right
                            and _s0.bounds.bottom<(_gb[1]+_gb[3])/2<_s0.bounds.top):
                        continue
                    _w=_fb17(_gb[0]-_REF17[1]-1000,_gb[1]-_REF17[1]-1000,
                             _gb[2]+_REF17[1]+1000,_gb[3]+_REF17[1]+1000,_s0.transform)
                    _st=_s0.read(1,window=_w,boundless=True,fill_value=0).astype('float64')
                    _tr=_s0.window_transform(_w); _shp=_st.shape
                with _rio17.open(_b17+'_QA_PIXEL.TIF') as _f:
                    _qa=_f.read(1,window=_w,boundless=True,fill_value=1)
                with _rio17.open(_b17+'_ST_QA.TIF') as _f:
                    _sq=_f.read(1,window=_w,boundless=True,fill_value=0).astype('float64')
                with _rio17.open(_b17+'_SR_B4.TIF') as _f:
                    _b4=_f.read(1,window=_w,boundless=True,fill_value=0).astype('float64')*_RS+_RO
                with _rio17.open(_b17+'_SR_B5.TIF') as _f:
                    _b5=_f.read(1,window=_w,boundless=True,fill_value=0).astype('float64')*_RS+_RO
            except Exception:
                continue
            _lst=_st*_SS+_SO-273.15
            _bad=(_st==0)
            for _k in ('fill','dilated','cirrus','cloud','shadow','snow','water'):
                _bad|=((_qa>>_QB[_k])&1)>0
            _bad|=(_sq*_SQS>4.0)
            _ok=(~_bad)&(_b4>0)&(_b4<1)&(_b5>0)&(_b5<1)&((_b4+_b5)>1e-6)
            _ins=lambda gg: _gm17([gg],out_shape=_shp,transform=_tr,invert=True)
            _m=(_ins(_g.buffer(_REF17[1]))&~_ins(_g.buffer(_REF17[0])))&_ok&~np.isnan(_lst)
            if _m.sum()<5000:                                     # §14와 같은 기준선 최소 화소수
                continue
            _nd=((_b5-_b4)/(_b5+_b4))[_m]; _lv=_lst[_m]
            _ix=_RNG17.choice(len(_nd),size=min(15000,len(_nd)),replace=False)
            _fr17.append(pd.DataFrame({'sid':_sid17,'부지':_nm17,'ndvi':_nd[_ix],'lst':_lv[_ix]}))
            _sid17+=1; _cnt17+=1
        print(f'  [{_j17+1}/{len(GEO)}] {_nm17}: {_cnt17}장면',flush=True)
    _D17=pd.concat(_fr17,ignore_index=True)
    _D17['lst_dm']=_D17['lst']-_D17.groupby('sid')['lst'].transform('mean')
    _D17['ndvi_dm']=_D17['ndvi']-_D17.groupby('sid')['ndvi'].transform('mean')
    _D17.sample(min(len(_D17),300000),random_state=1).to_csv(_OUT17,index=False,encoding='utf-8-sig')
    print(f'저장 {_OUT17} · 화소 {len(_D17):,} · 장면-부지 {_sid17}')


DERIVED_0723_링화소_NDVI_LST표본.csv 이미 존재 — 생성 건너뜀 (다시 만들려면 FORCE_17B0=True)


In [ ]:
# ── §17-b. 녹지 용량반응 — 링 화소 NDVI-LST 곡선·기울기 [🔖 2026-07-23 신규] ──
# 입력: 사전 추출 표본 CSV (전량 305만 화소 프로토타입에서 30만 저장 — 재추출 ~4분은 proto_17b 참조)

# 1. 표본 데이터셋 로드 (한글 깨짐 방지를 위해 'utf-8-sig' 인코딩 사용)
_G17=pd.read_csv('DERIVED_0723_링화소_NDVI_LST표본.csv',encoding='utf-8-sig')

# 2. 편차(Demean) 계산: 장면(sid, Scene ID)별 공간 편차를 제거하기 위해 각 장면의 평균값을 뺌
#    - lst_dm: 편차 처리된 지표면 온도 (해당 장면 평균 대비 온도 차이)
#    - ndvi_dm: 편차 처리된 식생지수 (해당 장면 평균 대비 식생지수 차이)
_G17['lst_dm']=_G17['lst']-_G17.groupby('sid')['lst'].transform('mean')
_G17['ndvi_dm']=_G17['ndvi']-_G17.groupby('sid')['ndvi'].transform('mean')

# 3. 분석 대상 작업 구간 필터링 (NDVI가 0.10 이상 0.70 이하인 화소만 추출)
_W17=_G17[(_G17.ndvi>=0.10)&(_G17.ndvi<=0.70)]

# 4. 전체 표본 대상 1차 선형 회귀 기울기 계산
#    - np.polyfit(X, Y, 1)[0]: 1차 다항식 회귀 계수(기울기) 산출
_sl17=np.polyfit(_W17['ndvi_dm'],_W17['lst_dm'],1)[0]

# 5. 전체 구간 기울기 결과 출력 (NDVI +0.1 변화에 따른 온도 변화량으로 변환하여 출력)
print(f'★ 작업구간[0.10,0.70] 기울기: NDVI +0.1당 {_sl17/10:+.2f}℃ (장면 demean · 표본 n={len(_W17):,} · 전량 305만 프로토타입 −1.16과 대조)')

# 6. NDVI를 0.05 단위로 구간화(Binning)하여 새로운 컬럼 생성
_G17['bin']=(_G17['ndvi']//0.05*0.05).round(2)

print()
print('NDVI 구간별 상대 LST (장면 평균 대비 ℃):')
# 7. NDVI 구간(0.0~0.85)별 평균 상대 온도(lst_dm) 및 데이터 개수(count) 집계 후 출력
print(_G17[(_G17.bin>=0.0)&(_G17.bin<=0.85)].groupby('bin')['lst_dm'].agg(['mean','count']).round(2).to_string())

print()
print('부지별 기울기(+0.1 NDVI당 ℃) — 11개 전부 음수인지:')
# 8. 개별 부지별로 그룹화하여 각각의 1차 회귀 기울기 및 데이터 수 출력
#    (식생 증가 시 온도 감소 효과가 모든 부지에서 일관되게 나타나는지 확인)
for _nm,_gd in _W17.groupby('부지'):
    print(f'  {_nm:12}{np.polyfit(_gd.ndvi_dm,_gd.lst_dm,1)[0]/10:+.2f} (n={len(_gd):,})')

print()
# 9. 분석 결과에 대한 도메인 종합 및 분석가 메모/메시지 출력
print('[사실] 같은 링·같은 장면에서 주거혼합 화소는 녹지 화소보다 평균 +7.4℃ (§14-d base_r45−base_grn, 156쌍 — 이진 앵커)')
print('[해석·정책 번역] NDVI 0.2→0.4(가로수·옥상녹화) ≈ 지표 −2.3℃ · 0.2→0.6(완충 공원화) ≈ −4.6℃ · 기온으론 β 범위로 −0.5~−1.3℃')
print('[발견] 최고온은 NDVI 0.15~0.25(어두운 불투수+성긴 식생 혼합 "회색지대") — 완충 설계의 1순위 타깃.')

★ 작업구간[0.10,0.70] 기울기: NDVI +0.1당 -1.16℃ (장면 demean · 표본 n=93,720 · 전량 305만 프로토타입 −1.16과 대조)

NDVI 구간별 상대 LST (장면 평균 대비 ℃):
      mean  count
bin              
0.00  1.66   1691
0.05  2.97   3104
0.10  3.75   4735
0.15  4.91   5166
0.20  4.98   6361
0.25  4.83   7205
0.30  4.51   7490
0.35  4.11   7944
0.40  3.77   8019
0.45  3.40   8384
0.50  2.95   8534
0.55  2.49   9050
0.60  1.97   9636
0.65  1.48  11196
0.70  0.71  14419
0.75 -0.30  22710
0.80 -1.42  51384
0.85 -2.27  93670

부지별 기울기(+0.1 NDVI당 ℃) — 11개 전부 음수인지:
  광양·제철       -0.85 (n=9,981)
  구미·전자       -1.13 (n=9,409)
  당진1철강       -0.80 (n=11,519)
  동국제강 인천     -0.76 (n=10,596)
  동해북평        -0.80 (n=4,193)
  심팩 포항       -1.20 (n=4,880)
  여수·석화       -1.00 (n=8,124)
  온산          -1.26 (n=10,158)
  울산미포        -1.17 (n=9,350)
  포항·제철       -1.05 (n=4,734)
  현대제철 인천     -0.75 (n=10,776)

[사실] 같은 링·같은 장면에서 주거혼합 화소는 녹지 화소보다 평균 +7.4℃ (§14-d base_r45−base_grn, 156쌍 — 이진 앵커)
[해석·정책 번역] NDVI 0.2→0.4(가로수·옥상녹화) ≈ 지표 −2.3℃ · 0.2→0.6(완충 공

### §17-c. 토지피복 세분류(2024) 검증 — 실피복 구성 + NDVI 대리 대조 + 관측소 교차 `[🔖 2026-07-23 신규 · 환경부 EGIS 세분류 1,139도엽]`

⚠ 실행 ~10분 셀 (도엽 SHP 로딩) — 부지별 진행 출력. 입력: `토지피복지도/` 1,139 zip + `DERIVED_0723_토지피복_도엽인덱스.csv`.
기준 도넛은 신청 시군구 범위 한계로 커버 13~57% — 커버율을 함께 출력(정직 표기).

In [ ]:
# ── §17-c. 토지피복 세분류 검증 — 실피복 구성·관측소 교차 [🔖 2026-07-23 신규 · ~10분] ──

import pandas as pd, numpy as np, geopandas as gpd, glob, os, time, warnings; warnings.filterwarnings('ignore')
from shapely.geometry import Point, box
# (노트북 cwd 그대로)
t0=time.time()
IDX=pd.read_csv('DERIVED_0723_토지피복_도엽인덱스.csv',encoding='utf-8-sig').dropna(subset=['w'])
idx_g=gpd.GeoDataFrame(IDX,geometry=[box(w,s,e,n) for w,s,e,n in zip(IDX.w,IDX.s,IDX.e,IDX.n)],crs=4326).to_crs(5179)

PDAN=gpd.read_file('0718_DAM_PDAN/DAM_PDAN.shp',encoding='EUC-KR')
YUCH=gpd.read_file('0718_DAM_YUCH/DAM_YUCH.shp',encoding='EUC-KR')
p4=PDAN.to_crs(4326); PDAN['lon']=p4.geometry.x; PDAN['lat']=p4.geometry.y
def geo_of(nm,ty):
    r=PDAN[(PDAN.DAN_NAME==nm)&(PDAN.DANJI_TYPE==ty)].iloc[0]
    yc=YUCH[YUCH.DAN_ID==r.DAN_ID]
    if len(yc):
        return yc.to_crs(5179).geometry.union_all()
    return gpd.GeoSeries([Point(r.lon,r.lat)],crs=4326).to_crs(5179).iloc[0].buffer(500)
SITES={'울산미포':geo_of('울산·미포','1'),'온산':geo_of('온산','1'),'포항·제철':geo_of('포항','1'),
       '구미·전자':geo_of('구미(2·3단지)','1'),'당진1철강':geo_of('당진1철강','2'),'동해북평':geo_of('북평','1'),
       '광양·제철':geo_of('광양','1'),'여수·석화':geo_of('여수','1'),'동해 북평2':geo_of('북평2','2'),
       '현대제철 인천':gpd.GeoSeries([Point(126.64432,37.48593)],crs=4326).to_crs(5179).iloc[0].buffer(500),
       'KG스틸 인천':gpd.GeoSeries([Point(126.66994,37.48378)],crs=4326).to_crs(5179).iloc[0].buffer(500)}

_cache={}
def load_sheets(target):
    hits=idx_g[idx_g.intersects(target)]
    parts=[]
    for _,r in hits.iterrows():
        if r.도엽 not in _cache:
            try:
                g=gpd.read_file(f"zip://{r.zip}!{r.도엽}.shp")[['L1_CODE','L2_CODE','geometry']].to_crs(5179)
            except Exception:
                g=None
            _cache[r.도엽]=g
        if _cache[r.도엽] is not None:
            parts.append(_cache[r.도엽])
    return (pd.concat(parts,ignore_index=True) if parts else None), len(hits)

L1_NM={'100':'시가화·건조','200':'농업','300':'산림','400':'초지','500':'습지','600':'나지','700':'수역'}
def comp(target,lab):
    lc,nsheet=load_sheets(target)
    A=target.area
    if lc is None or not len(lc):
        print(f'  {lab:16} 커버 0% (도엽 없음)'); return None
    inter=gpd.clip(gpd.GeoDataFrame(lc,geometry='geometry'),target)
    inter['a']=inter.geometry.area
    cov=inter['a'].sum()/A
    g1=inter.groupby(inter['L1_CODE'].astype(str))['a'].sum()/A*100
    top=' · '.join(f"{L1_NM.get(k,k)} {v:.0f}%" for k,v in g1.sort_values(ascending=False).head(4).items())
    imperv=g1.get('100',0)+g1.get('600',0)   # 시가화·건조 + 나지
    print(f'  {lab:16} 커버 {cov*100:3.0f}% | {top} | 불투수계(100+600): {imperv:.0f}%')
    return dict(커버=cov,불투수=imperv,구성=g1.to_dict())

print('① 부지·링 피복 구성 (세분류 2024, L1 기준)')
res={}
for nm,g in SITES.items():
    print(f'[{nm}] ({int(time.time()-t0)}s)', flush=True)
    res[nm]={'부지':comp(g,'부지'),
             '0-1km 링':comp(g.buffer(1000).difference(g),'0-1km 링'),
             '기준도넛(10-20km)':comp(g.buffer(20000).difference(g.buffer(10000)),'기준도넛 10-20km')}

print()
print('② NDVI 대리 대조는 위 §17 규약 md 참조 (온산 49→81% · 광양 37→87% — 대리가 과소·보수)')


print()
print(f'③ AWS 관측소 실제 피복 교차 ({int(time.time()-t0)}s)')
aws=pd.read_csv('OPEN_0720_기상청_AWS지점정보.csv',encoding='cp949')
aws=aws.rename(columns={c:'lat' for c in aws.columns if '위도' in c}|{c:'lon' for c in aws.columns if '경도' in c})
aws=aws.dropna(subset=['lat','lon']).drop_duplicates('지점')
ag=gpd.GeoDataFrame(aws,geometry=gpd.points_from_xy(aws.lon,aws.lat),crs=4326).to_crs(5179)
cover_union=idx_g.geometry.union_all()
inA=ag[ag.geometry.within(cover_union)]
print(f'  커버 영역 내 AWS: {len(inA)}/{len(ag)}')
P=pd.read_csv('DERIVED_0723_관측소패널_LST_NDVI.csv',encoding='utf-8-sig')
snd=P[P.net=='AWS'].groupby('지점')['NDVI_100m'].median()
rows=[]
for _,st in inA.iterrows():
    lc,_=load_sheets(st.geometry.buffer(100))
    if lc is None or not len(lc):
        continue
    hit=lc[lc.contains(st.geometry)]
    if not len(hit):
        continue
    l1=str(hit.iloc[0]['L1_CODE'])
    nd=snd.get(st['지점'],np.nan)
    lay='주거혼합(0.15~0.45)' if nd<0.45 else '녹지(≥0.45)'
    if nd!=nd:
        lay='NDVI없음'
    rows.append(dict(지점=st['지점'],L1=L1_NM.get(l1,l1),NDVI층=lay,NDVI=round(nd,2) if nd==nd else None))
X=pd.DataFrame(rows)
if len(X):
    print(pd.crosstab(X['L1'],X['NDVI층']).to_string())
    print(f'  → 실제 피복(L1) × NDVI 층 교차표 (n={len(X)})')
print(f'\n총 {int(time.time()-t0)}s')


① 부지·링 피복 구성 (세분류 2024, L1 기준)
[울산미포] (1s)
  부지               커버  95% | 시가화·건조 70% · 초지 14% · 나지 10% · 수역 1% | 불투수계(100+600): 79%
  0-1km 링          커버  91% | 수역 32% · 시가화·건조 24% · 산림 16% · 초지 11% | 불투수계(100+600): 29%
  기준도넛 10-20km     커버  31% | 산림 19% · 농업 4% · 초지 3% · 시가화·건조 3% | 불투수계(100+600): 4%
[온산] (134s)
  부지               커버 100% | 시가화·건조 67% · 초지 16% · 나지 14% · 산림 3% | 불투수계(100+600): 81%
  0-1km 링          커버 100% | 수역 36% · 산림 22% · 시가화·건조 20% · 초지 10% | 불투수계(100+600): 25%
  기준도넛 10-20km     커버  28% | 산림 17% · 시가화·건조 3% · 농업 2% · 초지 2% | 불투수계(100+600): 4%
[포항·제철] (170s)
  부지               커버 100% | 시가화·건조 93% · 수역 7% | 불투수계(100+600): 93%
  0-1km 링          커버 100% | 시가화·건조 63% · 수역 28% · 나지 5% · 초지 2% | 불투수계(100+600): 68%
  기준도넛 10-20km     커버  57% | 산림 30% · 수역 10% · 농업 8% · 초지 4% | 불투수계(100+600): 4%
[구미·전자] (226s)
  부지               커버 100% | 시가화·건조 77% · 초지 12% · 산림 9% · 나지 2% | 불투수계(100+600): 79%
  0-1km 링          커버  95% | 시가화·건조 31% · 산림 29% · 초지 13% · 수역 12% | 불투수계(1

### §17-d. 피복 유형별 냉각 효율 — "어떤 녹지를 심어야 하나" `[🔖 2026-07-24 신규 · 사용자 요청 ④]`

§17-b가 NDVI(연속 지표)로 냉각을 봤다면, 여기선 **실제 토지피복 클래스**(환경부 세분류 L2 2024)별로 상대 지표온도를 잰다.
방법: 8개 산단 기준 도넛(10~20km) 화소를 표본추출 → 화소 좌표를 피복 폴리곤과 **point-in-polygon 조인** → 장면 demean 후 클래스별 평균. 입력 CSV `DERIVED_0724_피복별_LST표본.csv`(17만 화소 중 20만 저장 — 재추출 ~11분은 proto_17d 참조).

⚠ 정직 단서: ① 단면 연관(이 자리에 이 피복이 이 온도)이지 식재 전후가 아님 — 수관 성숙에 시간 ② 지표온도 기준(기온은 β 0.22~0.29 환산) ③ **침엽수림·습지는 지형 편향 가능**(침엽수림=산지 고지대, 연안습지=갯벌·해양수 근접) — 절대순위보다 "숲>초지·밭, 논>밭" 방향에 무게 ④ 관개(물) 여부 중요(Sailor 관개공원 실측·논>밭이 방증).

In [ ]:
# ── §17-d. 피복 L2 클래스별 냉각 효율 [🔖 2026-07-24 신규] ──
_D17d=pd.read_csv('DERIVED_0724_피복별_LST표본.csv',encoding='utf-8-sig')
_L2NM={'110':'주거지역','120':'공업지역','130':'상업지역','140':'문화체육','150':'교통시설','160':'공공시설',
       '210':'논','220':'밭','230':'시설재배','240':'과수원','250':'기타재배',
       '310':'활엽수림','320':'침엽수림','330':'혼효림','410':'자연초지','420':'인공초지',
       '510':'내륙습지','520':'연안습지','610':'자연나지','620':'인공나지','710':'내륙수','720':'해양수'}
_D17d['L2']=_D17d['L2_CODE'].astype('Int64').astype(str).map(lambda k:_L2NM.get(k,f'?{k}'))
_D17d=_D17d.dropna(subset=['L2_CODE'])
# lst_dm은 표본 저장 시 이미 장면 demean됨 — 없으면 재계산
if 'lst_dm' not in _D17d.columns:
    _D17d['lst_dm']=_D17d['lst']-_D17d.groupby('sid')['lst'].transform('mean')
_g17=_D17d.groupby('L2').agg(화소=('lst_dm','size'),상대LST=('lst_dm','mean'),NDVI=('ndvi','mean')).round(2)
_g17=_g17[_g17.화소>=500].sort_values('상대LST')
print('피복 L2 클래스별 상대 지표온도 (장면 평균 대비 ℃, 낮을수록 시원) — 화소≥500:')
print(_g17.to_string())
_green=['활엽수림','침엽수림','혼효림','자연초지','인공초지','논','밭','과수원','내륙습지']
_gg=_g17[_g17.index.isin(_green)].sort_values('상대LST')
print()
print('★ 녹지 유형별 냉각 (가장 시원한 순) — 완충 설계 우선순위:')
print(_gg.to_string())
_imp=_D17d[_D17d.L2.isin(['주거지역','공업지역','상업지역','인공나지'])]['lst_dm'].mean()
print()
print(f'대조: 불투수(주거·공업·상업·인공나지) 평균 상대LST {_imp:+.2f}℃')
if len(_gg):
    print(f'→ 최고 냉각 "{_gg.index[0]}"({_gg.iloc[0].상대LST:+.2f}) vs 불투수 {_imp:+.2f} = 격차 {_imp-_gg.iloc[0].상대LST:.1f}℃ (§14-d 이진 앵커 7.4℃와 독립 재현)')
print()
print('[정책 처방] "녹지를 심어라"→"관개된 활엽·혼효 수관림을 심어라": 활엽수림(-2.5)이 인공초지·밭(+1.6)보다 4℃ 더 시원.')
print('  논(+0.02)이 밭(+1.6)보다 시원 = 물의 효과(Sailor 관개공원 실측 방증). 완충 녹지에 관개·수경 요소 권고.')

# ── §17-d2. 프레임 재구성 — 완화 메뉴가 아니라 **상쇄 불가능성 증명** [🔖 2026-07-27 사용자 결정] ──
# [사용자] "§17은 AIDC를 막을 근거가 아니라 합리화할 정보를 제시할 수 있는 것 같다."
#   맞는 우려다. "관개된 활엽수림을 심어라"는 사업자가 **완화 이행 목록**으로 가져다 쓸 수 있다.
#   같은 숫자를 **격차 증명**으로 다시 읽는다 — 이쪽이 자료가 실제로 말하는 것에 더 가깝다.
_ind=float(_g17.loc['공업지역','상대LST']); _best=float(_gg.iloc[0]['상대LST']); _bestnm=_gg.index[0]
_res17=float(_g17.loc['주거지역','상대LST'])
print()
print('§17-d2. 같은 표를 "완화 메뉴"가 아니라 "상쇄 불가능성"으로 읽는다')
print(f'  ① **부지 자체는 무엇을 해도 뜨겁다.** 산단 부지가 공업지역이 되는 순간 상대LST {_ind:+.2f}℃다.')
print(f'     완충 녹지는 부지 **바깥**을 식힐 뿐 부지를 식히지 않는다. AIDC 부지는 건물·냉각탑·주차장으로 덮인다.')
print(f'  ② **최선의 완충으로 상쇄하려면 부지의 몇 배가 필요한가** — 면적가중 평균을 0으로 만드는 비율:')
_ratio=_ind/abs(_best)
print(f'     공업 {_ind:+.2f}℃ · {_bestnm} {_best:+.2f}℃ → 필요 면적비 = {_ind:.2f}/{abs(_best):.2f} = **{_ratio:.1f}배**')
print(f"     {'부지':22}{'부지면적(만평)':>13}{'필요 완충림(만평)':>16}")
for _lb,_a17 in [('북평2 (YUCH 실측)',9.7),('2.4GW 표준 소요 하한',146.0),('2.4GW 표준 소요 상한',235.0)]:
    print(f'     {_lb:22}{_a17:>13.1f}{_a17*_ratio:>16.0f}')
print(f'  ③ 그마저도 **"평균"을 0으로 만드는 것**이지 부지 옆 사람에게는 소용없다 — 열은 평균이 아니라 거리로 온다(§9-e 감쇠).')
print(f'  ④ 주변이 이미 도시면 기준선부터 뜨겁다: 주거지역 {_res17:+.2f}℃ · 교통시설 {float(_g17.loc["교통시설","상대LST"]):+.2f}℃.')
print(f'     시원한 기준선은 **숲에만** 있다. 산단 주변에서 {_bestnm}를 {_ratio:.0f}배 면적으로 확보할 땅이 어디 있나.')
print()
print('  → **재구성된 결론**: §17-d는 "이런 녹지를 심으면 된다"가 아니라')
print(f'    **"최선의 녹지를 부지의 {_ratio:.0f}배로 깔아야 겨우 평균이 0이 되고, 그래도 부지와 그 옆은 그대로 뜨겁다"**는 증명이다.')
print('  → 완충 처방(관개·활엽·수관림)은 **부차적 권고로만** 남긴다 — 상쇄 수단이 아니라 피해 경감 수단으로.')
print('  ⚠ 이 재구성은 숫자를 바꾸지 않는다. **같은 표를 어느 쪽 문장으로 끝내느냐**의 문제다.')
print('    상대가 §17-d를 완화 근거로 인용하면 위 ①~④를 들이대면 된다.')


피복 L2 클래스별 상대 지표온도 (장면 평균 대비 ℃, 낮을수록 시원) — 화소≥500:
         화소  상대LST  NDVI
L2                      
활엽수림  42573  -2.48  0.86
혼효림   12572  -2.36  0.84
침엽수림  40517  -1.91  0.82
내륙수    1659  -0.17  0.55
논     13719   0.02  0.67
자연나지    659   0.59  0.55
과수원    2253   0.81  0.75
해양수     602   1.19  0.24
내륙습지   1958   1.26  0.65
밭     11208   1.62  0.68
인공초지  20050   1.62  0.69
시설재배    742   3.07  0.50
인공나지   6918   3.53  0.49
공공시설    582   3.92  0.44
교통시설  12792   5.22  0.42
주거지역   1507   5.43  0.44
상업지역   1221   6.75  0.34
공업지역   1552   9.74  0.25

★ 녹지 유형별 냉각 (가장 시원한 순) — 완충 설계 우선순위:
         화소  상대LST  NDVI
L2                      
활엽수림  42573  -2.48  0.86
혼효림   12572  -2.36  0.84
침엽수림  40517  -1.91  0.82
논     13719   0.02  0.67
과수원    2253   0.81  0.75
내륙습지   1958   1.26  0.65
밭     11208   1.62  0.68
인공초지  20050   1.62  0.69

대조: 불투수(주거·공업·상업·인공나지) 평균 상대LST +5.00℃
→ 최고 냉각 "활엽수림"(-2.48) vs 불투수 +5.00 = 격차 7.5℃ (§14-d 이진 앵커 7.4℃와 독립 재현)

[정책 처방] "녹지를 심어라"→"관개된 활엽·혼효 수관림을 심어라": 활엽수림(-2.5

## §18. 기사 기반 신규 후보 부지 실측 — 가설→현실 대응 `[사실]` `[🔖 2026-07-22 신규]`

2025-10~2026-07 기사(`AIDC 전환후보 사례.md`)로 **실제 AIDC 계획이 확인된 부지** 중 §13·§14가 아직 안 잰 5곳을 같은 기계(§14-b `site_scene_dT` + §10 집계구 버퍼)로 잰다.

| 부지 | 계획 (기사) | 상태 | 형상 출처 |
|---|---|---|---|
| 포항 광명(일반산단) | 네오AI클라우드 2조, 40MW→260MW | **착공식 2026-07-20** | ILIS YUCH 폴리곤 |
| 동해 북평2(일반산단) | GS 2.4GW 캠퍼스·최대 120조 | MOU (강원도·동해시·GS) | ILIS YUCH 폴리곤 |
| 울산 하이테크밸리 | SKT 1GW 후보 (2공구 9.2만평) | 후보지 검토 | ILIS YUCH 폴리곤 |
| 온산 강양·우봉지구 | SKT 1GW 후보 (10만평, 최유력 평가) | 후보지 검토 | V-World 지오코딩 점 |
| KG스틸 인천공장 | 유휴부지 AIDC 구축 검토 (중장기) | 검토 | V-World 지오코딩 점 |

**형상 주의** — §14와 같은 규약:
- **북평2 ≠ 북평국가산단**: §14의 동해북평은 국가산단. GS MOU가 명시한 부지는 **북평제2일반산업단지**(GS 조성 — 산단공 대장상 2025Q3 기준 **조성중**·분양률 28.4%·입주 0) — 별도 폴리곤으로 잰다.
- **강양·우봉지구**는 지구 경계 자료가 없어 강양리·우봉리의 **중간점 근사**(V-World 검색 2026-07-22) — 참고치.
- **KG스틸 인천공장 = 동부제철(주)인천공장**, 인천 서구 가좌동 백범로 789 (V-World 2026-07-22). 동구 현대제철·동국제강 클러스터에서 약 2.3km — **현재 가동 중**이라 ΔLST에 현 공장 열이 포함된다(폐쇄 부지 아님).
- 광명·하이테크밸리·북평2 ΔR0내부는 **산단 전체 평균**(AIDC는 그 일부 필지).

> **재사용 명시** `[🔖 2026-07-23]`: §18-a = §14-b `site_scene_dT`·§14-c 규약의 **재사용**(같은 자로 새 부지를 잼), §18-b = §10 집계구 면적가중 버퍼·§11-b footprint의 재사용, §18-b2 = §10·§11 온열 공식의 재사용. 새 방법론이 아니다.

> **★ 북평2는 §10~§14 본 파이프라인으로 이관됨** `[🔖 2026-07-24 사용자 지적 반영]`: GS 2.4GW의 실부지가 북평제2일반산단(`북평2`)임이 확인돼, §7 `T`에 정식 대상으로 편입했다. 따라서 **§8-b 형상·§9 열부하·§10-b 폴리곤 인구·§11 연령가중·§11-b footprint·§12-b 형상고령이 모두 북평2를 포함**해 재계산됐고, ΔLST도 §14 표에 통합됐다. 이 §18은 이제 **나머지 4곳(포항 광명·울산 하이테크밸리·온산 강양우봉·KG스틸 인천)의 1차 실측 기록**으로 읽으면 된다(북평2 행은 최초 측정 이력으로 보존).

In [ ]:
# ── §18-a. 신규 부지 형상 + ΔLST — §14 기계 재사용 [🔖 2026-07-22 신규] ──
from shapely.geometry import Point as _P18

# 1. 신규 산업단지 부지 정보 정의
# 폴리곤 부지 목록: (부지 라벨명, 산업단지 DB 내 이름, 단지 유형 코드)
# 폴리곤 부지 (ILIS PDAN/YUCH — DAN_NAME 완전일치)
_POLY18=[('포항 광명','광명','2'),('울산 하이테크밸리','울산 High Tech Valley','2'),('동해 북평2','북평2','2')]

# 점 부지 (V-World 지오코딩 2026-07-22 — 강양·우봉은 두 리 중간점 근사)
_PT18=[('KG스틸 인천',126.66994,37.48378),('온산 강양·우봉지구',129.3510,35.3990)]

# 2. 부지 Spatial Data(GeoSeries) 구축 및 좌표계 변환(WGS84 -> UTM-K 5179)
GEO18={}

# (1) 폴리곤 데이터 처리: ILIS DB(PDAN/YUCH)에서 공간 형상 추출
for _lbl,_nm,_ty in _POLY18:
    _r=PDAN[(PDAN.DAN_NAME==_nm)&(PDAN.DANJI_TYPE==_ty)].iloc[0]
    _yc=YUCH[YUCH.DAN_ID==_r.DAN_ID]

    # 유치업종/구역 형상이 존재하면 폴리곤을 하나로 합쳐서 저장(union_all)
    if len(_yc): 
        GEO18[_lbl]=(_yc.to_crs(5179).geometry.union_all(),True)
    else:        
        GEO18[_lbl]=(gpd.GeoSeries([_P18(_r.lon,_r.lat)],crs=4326).to_crs(5179).iloc[0],False)

# (2) 점 데이터 처리: 위경도를 좌표 변환하여 점 객체로 저장
for _nm,_lo,_la in _PT18:
    GEO18[_nm]=(gpd.GeoSeries([_P18(_lo,_la)],crs=4326).to_crs(5179).iloc[0],False)

print('신규 대상:',len(GEO18),'| 폴리곤:',[k for k,(g,p) in GEO18.items() if p],'| 점:',[k for k,(g,p) in GEO18.items() if not p])

# 3. 신규 부지별 Landsat 위성 장면 대조 및 ΔLST 산출
rec18=[]; why18=[]

for _nm,(_g,_ip) in GEO18.items():
    for _,_s in SC.iterrows():
        # JJA(6, 7, 8월 여름철) 데이터만 처리하는 조건
        if JJA_ONLY and _s.date.month not in (6,7,8):
            continue

        # 메타데이터에서 구름 비율 확인 및 스킵 처리
        _m=mtl_meta(_s.base)
        if float(_m.get('CLOUD_COVER',100))>CLOUD_MAX:
            continue

        # §14에서 정의한 부지별 지표면 온도 편차(ΔLST) 산출 함수 실행
        _r=site_scene_dT(_s.base,_g,_ip)
        why18.append((_nm,_r['사유']))

        if _r['사유']!='OK':
            continue

        # 정상 산출된 결과 레코드에 저장
        rec18.append(dict(부지=_nm,형상='폴리곤' if _ip else '점',날짜=_s.date.date(),위성=_s.sat,
                          구름pct=float(_m.get('CLOUD_COVER',float('nan'))),
                          **{k:v for k,v in _r.items() if k!='사유'}))
        
LST18=pd.DataFrame(rec18)
_W18=pd.DataFrame(why18,columns=['부지','사유'])

# 4. 분석 커버리지(부지별 스킵/성공 사유) 교차표 출력
print('\n커버리지 (부지 × 사유):')
print(_W18.pivot_table(index='부지',columns='사유',aggfunc='size',fill_value=0).to_string())

# 5. 부지별 평균 ΔLST 최종 요약 집계 및 출력
if len(LST18):
    # 집계 대상 컬럼 정의 (R0 내부 및 반경링 버퍼 영역별 ΔLST)
    _dc18=(['ΔR0내부'] if 'ΔR0내부' in LST18.columns else [])+[f'Δ{a//1000}-{b//1000}km' for a,b in RINGS]

    # 부지 및 형상별 평균 ΔLST 계산
    agg18=LST18.groupby(['부지','형상']).agg(장면=('날짜','size'),**{c:(c,'mean') for c in _dc18}).round(2)

    print('\n=== 신규 부지 평균 ΔLST (℃, 기준=동일장면 경계+10~20km) ===')
    print(agg18.sort_values('Δ0-1km',ascending=False).to_string())
    print('\n(§14 기존 측정 대조: 포항·제철 R0 +17.3 / 동해북평 국가산단 R0 +11.2 / 온산 R0 +11.2 / 현대제철 인천 0-1km +5.9)')

신규 대상: 5 | 폴리곤: ['포항 광명', '울산 하이테크밸리', '동해 북평2'] | 점: ['KG스틸 인천', '온산 강양·우봉지구']

커버리지 (부지 × 사유):
사유          OK  기준선 유효화소 부족(구름·바다)  장면 bbox 밖  촬영범위 밖(no-data 모서리)
부지                                                                
KG스틸 인천     10                   1         79                    0
동해 북평2      26                   1         63                    0
온산 강양·우봉지구  26                   4         60                    0
울산 하이테크밸리   16                   0         60                   14
포항 광명       16                   0         74                    0

=== 신규 부지 평균 ΔLST (℃, 기준=동일장면 경계+10~20km) ===
                장면  ΔR0내부  Δ0-1km  Δ1-2km  Δ2-3km  Δ3-5km  Δ5-10km
부지         형상                                                     
동해 북평2     폴리곤  26  11.06    7.61    4.78    6.04    4.59     2.13
KG스틸 인천    점    10    NaN    7.41    4.81    2.77    3.12     0.51
울산 하이테크밸리  폴리곤  16  10.08    5.90    2.70    2.32    1.78     0.46
온산 강양·우봉지구 점    26    NaN    4.07    6.23    4.27   

In [ ]:
# ── §18-b. 신규 부지 노출 인구 — 1·3·4.5·10km + 65+ [🔖 2026-07-22 신규] ──

def _bp18(geom,R,em=None):                       # 폴리곤·점 공용 버퍼 인구 (§11-b(3) _bp3 확장)
    """
    지정된 공간 형상(geom)을 기준으로 반경 R(m) 버퍼 내 총인구 및 고령층 인구 산출
    (면적 비례 가중 기법 적용)
    """
    buf=geom.buffer(R)  # R(m) 버퍼 영역 생성
    b=buf.bounds        # Spatial Index 바운딩 박스 (minx, miny, maxx, maxy)

    # 1. Bounding Box를 이용해 관련 집계구(Shapefile) 데이터 빠르게 읽기 (I/O 최적화)
    oa=pyogrio.read_dataframe(SHP,bbox=(b[0]-100,b[1]-100,b[2]+100,b[3]+100))

    # 2. 집계구 면적 대비 버퍼 교집합 면적 비율(w) 계산 (0~1 사이로 절단)
    w=(oa.geometry.intersection(buf).area/oa.geometry.area).clip(0,1)

    # 3. 집계구 ID(TOT_OA_CD) 매핑으로 총인구 수 집계
    oa['p']=oa['TOT_OA_CD'].map(POP).fillna(0.0)
    out={'tot':float((oa['p']*w).sum())}    # 면적 비례 총인구 합산

    # 4. 고령층(65세 이상) 인구 데이터(em)가 제공된 경우 면적 비례 계산 추가
    if em is not None: 
        out['e']=float((oa['TOT_OA_CD'].map(em).fillna(0.0)*w).sum())

    return out

# 부지별 행정구역 매핑 키 (지자체별 온열질환 발생률 데이터 매핑 목적)
KEY18={'포항 광명':['경상북도 포항시'],'울산 하이테크밸리':['울산광역시 울주군'],
       '동해 북평2':['강원도 동해시','강원특별자치도 동해시'],
       'KG스틸 인천':['인천광역시 서구'],'온산 강양·우봉지구':['울산광역시 울주군']}

rows18=[]

# GEO18에 정의된 각 신규 부지별 노출 인구 산출 반복문
for _nm,(_g,_ip) in GEO18.items():

    # 해당 지자체의 인구 10만 명당 온열질환자 발생률 매핑
    _rt=next((float(_rate[k]) for k in KEY18[_nm] if k in _rate.index),float('nan'))

    # 각 주요 반경별(1km, 3km, 4.5km, 10km) 노출 인구 산출
    _r1=_bp18(_g,1000); _r3=_bp18(_g,3000,_acc); 
    _r45=_bp18(_g,4500); _r10=_bp18(_g,10000,_acc)

    # 결과를 리스트에 저장
    rows18.append(dict(부지=_nm,기준='폴리곤경계' if _ip else '점',온열률10만=round(_rt,1),
        인구_1km=int(_r1['tot']),인구_3km=int(_r3['tot']),노인_3km=int(_r3['e']),
        인구_45km=int(_r45['tot']),인구_10km=int(_r10['tot']),노인_10km=int(_r10['e'])))

# 분석 결과 데이터프레임 변환 및 출력
P18=pd.DataFrame(rows18)
print(P18.to_string(index=False))

# 앞선 지표면 온도(LST18) 계산 결과가 있을 경우 결합하여 CSV 파일 저장
if len(LST18):
    _mg18=P18.merge(agg18.reset_index(),on='부지',how='left')
    _mg18.to_csv('DERIVED_0722_신규부지_LST_인구.csv',index=False,encoding='utf-8-sig')
    print('\n저장: DERIVED_0722_신규부지_LST_인구.csv (LST + 인구 결합)')

# 교차 검증 및 사실 대조(Fact-check)용 요약 문구 출력
print('\n[사실 대조 — 기사 "주거지와 충분히 격리되어 있어 주민 반발 가능성이 낮은 편" 주장]')
for _,_p in P18.iterrows():
    print(f"  {_p.부지:11} 3km {_p.인구_3km:>9,}명(65+ {_p.노인_3km:,}) · 10km {_p.인구_10km:>11,}명(65+ {_p.노인_10km:,})")

print('→ 케임브리지 hyperscaler footprint(10km) 기준으로 "격리"가 성립하는 부지가 있는지 위 숫자로 판단한다.')

        부지    기준  온열률10만  인구_1km  인구_3km  노인_3km  인구_45km  인구_10km  노인_10km
     포항 광명 폴리곤경계     9.4    3435   32805    4647    61302   149380    32173
 울산 하이테크밸리 폴리곤경계     8.7    3774   16436    4738    37195    73068    17023
    동해 북평2 폴리곤경계     6.4     877   25978    4466    69791   123274    28459
   KG스틸 인천     점     5.4    5765  302710   59278   833516  2741108   473449
온산 강양·우봉지구     점     8.7     137    3959     923     5367    64467    10841

저장: DERIVED_0722_신규부지_LST_인구.csv (LST + 인구 결합)

[사실 대조 — 기사 "주거지와 충분히 격리되어 있어 주민 반발 가능성이 낮은 편" 주장]
  포항 광명       3km    32,805명(65+ 4,647) · 10km     149,380명(65+ 32,173)
  울산 하이테크밸리   3km    16,436명(65+ 4,738) · 10km      73,068명(65+ 17,023)
  동해 북평2      3km    25,978명(65+ 4,466) · 10km     123,274명(65+ 28,459)
  KG스틸 인천     3km   302,710명(65+ 59,278) · 10km   2,741,108명(65+ 473,449)
  온산 강양·우봉지구  3km     3,959명(65+ 923) · 10km      64,467명(65+ 10,841)
→ 케임브리지 hyperscaler footprint(10km) 기준으로 "격리"가 성립하는 부지가 있는지 위 숫자로 판단한다.


### §18-b2. 신규 부지 온열질환 추정 — §10·§11 공식 소급 적용 `[🔖 2026-07-23 신규 · 사용자 지적]`

§18-b는 노출 **인구**까지만 구했다. 기존 8개 산단(§10 비가중·§11 연령가중)과 같은 최종 지표를 신규 5곳에도 부여한다 — 이로써 북평2 등이 기존 파이프라인에 완전히 편입된다. 입력은 §18-b가 저장한 CSV(self-contained — §18-a 재실행 불필요).

In [ ]:
import glob as _glob18
# ── §18-b2. 신규 부지 온열질환 추정 — §10 비가중 + §11 연령가중 소급 [🔖 2026-07-23 신규] ──
# (0) §18-b 결과 로드
_nb2=pd.read_csv('DERIVED_0722_신규부지_LST_인구.csv',encoding='utf-8-sig')

# (1) 연령 가중치 재유도 — §11과 동일 로직 (여기서 다시 계산해 §11 값과 대조·검증)
from python_calamine import CalamineWorkbook as _CW18
_rows2=_CW18.from_path('FOIA_0522_질병관리청_NEDIS온열질환_2020-2025.xlsx').get_sheet_by_name('DB(2020-2025)발생지역기준').to_python()
_ne2=pd.DataFrame(_rows2[1:],columns=_rows2[0])
_ne2['dt']=pd.to_datetime(_ne2['발생일자'].astype(str),errors='coerce')
_ne2=_ne2[_ne2['dt'].dt.month.isin([6,7,8])]                  # 여름만 (§3과 동일 규약)
_age2=pd.to_numeric(_ne2['나이'],errors='coerce')
_c65=int((_age2>=65).sum())                                    # 65세 이상 여름 온열 건수
_cyg=int((_age2<65).sum())

# 전국 65세 이상 인구비(NAT65) — 집계구 연령 파일에서 (§11 cell과 동일 원천)
_agef=[f for f in _glob18.glob('0718_2024년 인구총괄/*.csv') if '연령' in f]
_ag2=pd.read_csv(_agef[0],header=None,names=['y','oa','item','val'],dtype={'oa':str},encoding='cp949')
# in_age_001~013 = 0~64세 구간, in_age_014~021 = 65~100+세 구간 (in_age_001은 '0~4세'지 총계가 아님 — 07-23 버그 정정)
_a65=_ag2[_ag2['item'].isin([f'in_age_{i:03d}' for i in range(14,22)])]['val'].sum()
_atot=_ag2[_ag2['item'].isin([f'in_age_{i:03d}' for i in range(1,22)])]['val'].sum()   # 전 연령 합 = 총인구
NAT65_2=_a65/_atot
RR_2=(_c65/_a65)/(_cyg/(_atot-_a65))                           # 65+ 온열률 / 미만 온열률
_avg2=NAT65_2*RR_2+(1-NAT65_2)
E_ELD2=RR_2/_avg2
E_YNG2=1/_avg2
# 검증 게이트: §11의 기존 값(18.8%·1.8)과 크게 다르면 저장 전에 멈춘다 (V9.1 — 잘못된 출력 보존 금지)
assert 0.10<NAT65_2<0.30, f'NAT65 비정상: {NAT65_2}'
assert 1.2<RR_2<3.0, f'RR 비정상: {RR_2}'
print(f'검증 대조 — NAT65 {NAT65_2*100:.1f}% (§11: 18.8%) · RR {RR_2:.2f} (§11: 1.8) · E_ELD {E_ELD2:.2f} · E_YNG {E_YNG2:.2f}')

# (2) +2℃ 시나리오 추가 온열 — §10 공식(비가중)과 §11 공식(연령가중)
# [🔖 2026-07-24] 전국 상수 → 부지별 f_site()(§9-e). 신규 부지는 key 컬럼이 없어 명시 매핑.
_K18={'포항 광명':'경상북도 포항시','울산 하이테크밸리':'울산광역시 울주군','동해 북평2':'강원특별자치도 동해시',
      'KG스틸 인천':'인천광역시 서구','온산 강양·우봉지구':'울산광역시 울주군'}
assert set(_nb2['부지'])<=set(_K18), f'매핑 누락: {set(_nb2["부지"])-set(_K18)}'
_nb2['f_부지']=[round(f_site(_K18[b]),3) for b in _nb2['부지']]
_f2=HEAT_MULT_PER_C**2-1                                       # 전국 참조값(대조 전용)

# [🔖 2026-07-25] 여기가 마지막 남은 '3km까지 전부 +2℃' 계산이었다 → §9-e 엔진의 링별 감쇠로 통일.
#   누적 인구(1·3·4.5·10km)를 링 증분으로 바꾸고, 링마다 다른 ΔT(dT_at)를 적용해 누적한다.
#   고령 비율은 3km 이내는 노인_3km/인구_3km, 그 바깥은 (노인_10km−노인_3km)/(인구_10km−인구_3km).
_RG18=[('인구_1km','노인_3km',0.5),('인구_3km','노인_3km',2.0),
       ('인구_45km',None,3.75),('인구_10km',None,7.25)]
_res18={'추가온열_3km':[],'추가온열_3km_상':[],'추가온열_10km':[],'추가온열_10km_상':[],'그중65세이상_3km':[]}
for _,_r18 in _nb2.iterrows():
    _key18=_K18[_r18['부지']]; _rt18=_r18['온열률10만']
    _sh_in=_r18['노인_3km']/max(_r18['인구_3km'],1)                       # 3km 이내 고령비
    _dn=max(_r18['인구_10km']-_r18['인구_3km'],1)
    _sh_out=max(_r18['노인_10km']-_r18['노인_3km'],0)/_dn                 # 3km 바깥 고령비
    _pv=0.0; _lo=_hi=_e65=0.0; _lo3=_hi3=_e3=0.0
    for _col,_,_mid in _RG18:
        _cum=_r18[_col]; _ring=max(_cum-_pv,0.0); _pv=_cum
        _sh=_sh_in if _mid<=2.0 else _sh_out
        _eld=_ring*_sh
        _lo+=excess_ring(_key18,_ring,_rt18,_mid,eld=_eld,E_ELD=E_ELD2,E_YNG=E_YNG2)
        _hi+=excess_ring(_key18,_ring,_rt18,_mid,eld=_eld,E_ELD=E_ELD2,E_YNG=E_YNG2,beta=BETA_L2A_HI)
        _e65+=_eld*_rt18*E_ELD2/1e5*f_site(_key18,dT_at(_mid))
        if _mid<=2.0:
            _lo3,_hi3,_e3=_lo,_hi,_e65
    _res18['추가온열_3km'].append(round(_lo3,2)); _res18['추가온열_3km_상'].append(round(_hi3,2))
    _res18['추가온열_10km'].append(round(_lo,2)); _res18['추가온열_10km_상'].append(round(_hi,2))
    _res18['그중65세이상_3km'].append(round(_e3,2))
for _k,_v in _res18.items():
    _nb2[_k]=_v
_nb2['연령가중_년']=_nb2['추가온열_3km']        # 이름 하위호환(3km 누적 하한)
_nb2['추가온열_년']=_nb2['추가온열_3km']
print()
print(_nb2[['부지','기준','인구_3km','노인_3km','온열률10만','추가온열_3km','추가온열_3km_상',
            '추가온열_10km','추가온열_10km_상','그중65세이상_3km']].to_string(index=False))
print()
print('→ 이 표로 신규 5곳이 기존 8산단(§10·§11)과 동일한 최종 지표를 갖는다. 북평2가 §9~§12 파이프라인에 완전 편입(사용자 지적 반영).')
_kg18=_nb2[_nb2['부지']=='KG스틸 인천'].iloc[0]
print(f"  KG스틸 인천 3km 연 +{_kg18['추가온열_3km']:.1f}~{_kg18['추가온열_3km_상']:.1f}건 · "
      f"10km 연 +{_kg18['추가온열_10km']:.1f}~{_kg18['추가온열_10km_상']:.1f}건 = 수도권 최대.")
print("  [🔖 07-25] 여기가 마지막 남은 '3km 전부 +2℃' 계산이었다 → 링별 감쇠(§9-e 엔진)로 통일했다.")
print(f"  하한/상한 = β {BETA_L2A}(채택) ~ {BETA_L2A_HI}(§15-e 희석보정). ⚠ 시나리오 추정 — 확정 예측 아님.")

검증 대조 — NAT65 19.0% (§11: 18.8%) · RR 1.80 (§11: 1.8) · E_ELD 1.56 · E_YNG 0.87

        부지    기준  인구_3km  노인_3km  온열률10만  추가온열_3km  추가온열_3km_상  추가온열_10km  추가온열_10km_상  그중65세이상_3km
     포항 광명 폴리곤경계   32805    4647     9.4      0.39        0.74       1.10         2.02         0.09
 울산 하이테크밸리 폴리곤경계   16436    4738     8.7      0.31        0.59       0.80         1.50         0.13
    동해 북평2 폴리곤경계   25978    4466     6.4      0.31        0.60       0.98         1.84         0.09
   KG스틸 인천     점  302710   59278     5.4      3.09        5.90      14.57        27.09         0.94
온산 강양·우봉지구     점    3959     923     8.7      0.07        0.13       0.46         0.85         0.02

→ 이 표로 신규 5곳이 기존 8산단(§10·§11)과 동일한 최종 지표를 갖는다. 북평2가 §9~§12 파이프라인에 완전 편입(사용자 지적 반영).
  KG스틸 인천 3km 연 +3.1~5.9건 · 10km 연 +14.6~27.1건 = 수도권 최대.
  [🔖 07-25] 여기가 마지막 남은 '3km 전부 +2℃' 계산이었다 → 링별 감쇠(§9-e 엔진)로 통일했다.
  하한/상한 = β 0.258(채택) ~ 0.459(§15-e 희석보정). ⚠ 시나리오 추정 — 확정 예측 아님.


### §18 결과·해석 규약 `[🔖 2026-07-22]`

**[사실]** 위 ΔLST(위성 실측)와 버퍼 인구(집계구 실측). **[해석]** AIDC 전환 후 온도장은 미래 시나리오 — §11-b(3)의 지수 감쇠 × dose 범위 논리를 그대로 적용하되, 부지별 상태 구분:

- **착공**(포항 광명): 계획 확정 — 노출 인구는 [사실], 열 영향은 케임브리지 footprint 적용 시나리오.
- **MOU**(북평2): 규모 확정(2.4GW) — 동일.
- **검토**(하이테크밸리·강양우봉·KG스틸): 후보 단계 — "만약 여기면" 조건부.
- KG스틸·광명·하이테크밸리·북평2는 **가동/조성 중 부지**라 §11-b(2) 열원교체 논리 적용(순수 더하기 아님). 폐쇄 확정 부지(현대제철 인천 2026-01)만 순수 더하기.
- 강양·우봉지구 수치는 점 근사 참고치 — 지구 경계 확보 시 재계산.